# Swinging research notebook 02

Run the cells from top to bottom on Google Colab. The notebook writes its
Python package to `/content/JackSparrow` and its datasets to
`/content/JackSparrow/data`.

Data is downloaded from the public Delta Exchange candle API. Do not mount
Google Drive. After a runtime reset, run this notebook again. It rebuilds the
package from the cells and can rebuild the parquet inputs from Delta or from
parquet already stored under `/content/JackSparrow/data`.

These artifacts are research only. They do not enable live orders.


## Install

In [ ]:
import subprocess
import sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "pandas", "pyarrow", "numpy", "scipy", "scikit-learn",
    "xgboost", "lightgbm", "optuna", "joblib", "requests",
])
print("swinging dependencies ready")


swinging dependencies ready


## Local package

In [ ]:
import json
import sys
from pathlib import Path
ROOT = Path('/content/JackSparrow')
FILES = json.loads("{\"research/__init__.py\": \"\\\"\\\"\\\"Research packages used by offline pipelines and forecast inference.\\\"\\\"\\\"\\n\", \"research/cpr_mtf/__init__.py\": \"\\\"\\\"\\\"BTCUSD multi-timeframe chart-candidate research.\\n\\nThis package is offline. It must not register a live model or send orders.\\n\\\"\\\"\\\"\\n\\nfrom research.cpr_mtf.contract import SPEC_VERSION\\n\\n__all__ = [\\\"SPEC_VERSION\\\"]\\n\", \"research/cpr_mtf/contract.py\": \"\\\"\\\"\\\"Frozen constants for the CPR-MTF research track.\\n\\nNothing here is an order. Proposal tolerances are candidate-generation\\nparameters, not ground-truth labels.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nSPEC_VERSION = \\\"cpr_mtf_v1\\\"\\nINSTRUMENT = \\\"BTCUSD\\\"\\nSOURCE_MINUTES = 5\\nTIMEFRAMES = (\\\"15m\\\", \\\"1h\\\", \\\"4h\\\")\\nBAR_MINUTES = {\\\"15m\\\": 15, \\\"1h\\\": 60, \\\"4h\\\": 240}\\nSOURCE_BARS_PER_BUCKET = {\\\"15m\\\": 3, \\\"1h\\\": 12, \\\"4h\\\": 48}\\nTIMEFRAME_ID = {\\\"15m\\\": 0, \\\"1h\\\": 1, \\\"4h\\\": 2}\\n\\nWINDOW_BARS = 192\\nSUFFIX_BARS = (48, 96, 192)\\nATR_PERIOD = 14\\nVOLUME_BASELINE = 48\\nFIRST_FINITE_BAR = VOLUME_BASELINE - 1\\nZZ_TAU = 1.5\\n\\nPURGE_BARS = 384\\nTRAIN_FRACTION = 0.70\\nVAL_FRACTION = 0.15\\nTRAIN_STRIDE = 4\\nEVAL_EVENT_STRIDE = 384\\n\\nMAX_PER_FAMILY = 2\\nMAX_CANDIDATES_PER_TIMEFRAME = 12\\nMAX_CANDIDATES = 36\\nPAIR_LIMIT = 32\\n\\nSIMILAR_PRICE_ATR = 0.75\\nMIN_SEPARATION_BARS = 3\\nMIN_PATTERN_DEPTH_ATR = 1.0\\nHEAD_PROMINENCE_ATR = 0.75\\nIMPULSE_ATR_MIN = 3.0\\nFLAG_RETRACE_MAX = 0.55\\nSLOPE_FLAT_ATR = 0.015\\nPARALLEL_SLOPE_ATR = 0.01\\nCONVERGENCE_SLOPE_ATR = 0.01\\nTOUCH_TOL_ATR = 0.50\\nMIN_BOUNDARY_POINTS = 2\\nCUP_DEPTH_ATR = 2.0\\nMIN_CUP_SIDE_BARS = 5\\nHANDLE_RETRACE_MIN = 0.05\\nHANDLE_RETRACE_MAX = 0.50\\n\\nBREAKOUT_BUFFER_ATR = 0.20\\nBREAKOUT_STRONG_ATR = 0.40\\nBREAKOUT_HORIZONS = (12, 24)\\nBREAKOUT_STATUSES = (\\n    \\\"no_breakout\\\",\\n    \\\"upside_breakout\\\",\\n    \\\"downside_breakout\\\",\\n    \\\"invalidated\\\",\\n    \\\"ambiguous\\\",\\n    \\\"masked\\\",\\n)\\n\\nPATTERN_CLASSES = (\\n    \\\"double\\\",\\n    \\\"triple\\\",\\n    \\\"head_and_shoulders\\\",\\n    \\\"flag\\\",\\n    \\\"pennant\\\",\\n    \\\"ascending_triangle\\\",\\n    \\\"descending_triangle\\\",\\n    \\\"symmetrical_triangle\\\",\\n    \\\"rising_wedge\\\",\\n    \\\"falling_wedge\\\",\\n    \\\"ascending_channel\\\",\\n    \\\"descending_channel\\\",\\n    \\\"rectangle\\\",\\n    \\\"broadening\\\",\\n    \\\"cup_and_handle\\\",\\n)\\nDIRECTIONS = (\\\"bullish\\\", \\\"bearish\\\", \\\"neutral\\\")\\nMATURITY_STATES = (\\\"developing\\\", \\\"mature\\\", \\\"confirmed\\\", \\\"invalidated\\\")\\nGEOMETRY_KINDS = (\\\"swing\\\", \\\"rounded_proxy\\\")\\n\\nSEQUENCE_CHANNELS = (\\n    \\\"open_atr\\\",\\n    \\\"high_atr\\\",\\n    \\\"low_atr\\\",\\n    \\\"close_atr\\\",\\n    \\\"body_size_atr\\\",\\n    \\\"upper_wick_atr\\\",\\n    \\\"lower_wick_atr\\\",\\n    \\\"range_atr\\\",\\n    \\\"body_to_range\\\",\\n    \\\"close_location\\\",\\n    \\\"volume_ratio\\\",\\n    \\\"swing_high_flag\\\",\\n    \\\"swing_low_flag\\\",\\n    \\\"swing_high_prominence_atr\\\",\\n    \\\"swing_low_prominence_atr\\\",\\n    \\\"bars_since_confirmed_swing\\\",\\n    \\\"swing_high_slope\\\",\\n    \\\"swing_low_slope\\\",\\n    \\\"swing_range_atr\\\",\\n    \\\"price_position_in_swing_range\\\",\\n)\\nMODEL_CANDIDATE_FEATURES = (\\n    \\\"timeframe_id\\\",\\n    \\\"formation_width_bars\\\",\\n    \\\"formation_height_atr\\\",\\n    \\\"upper_boundary_slope\\\",\\n    \\\"lower_boundary_slope\\\",\\n    \\\"upper_boundary_distance_atr\\\",\\n    \\\"lower_boundary_distance_atr\\\",\\n    \\\"boundary_convergence_ratio\\\",\\n    \\\"boundary_divergence_ratio\\\",\\n    \\\"swing_high_count\\\",\\n    \\\"swing_low_count\\\",\\n    \\\"upper_boundary_touches\\\",\\n    \\\"lower_boundary_touches\\\",\\n    \\\"upper_boundary_error_atr\\\",\\n    \\\"lower_boundary_error_atr\\\",\\n    \\\"price_position_in_formation\\\",\\n    \\\"volume_contraction_ratio\\\",\\n    \\\"volume_expansion_ratio\\\",\\n    \\\"bars_since_last_interaction\\\",\\n    \\\"impulse_move_atr\\\",\\n    \\\"retracement_ratio\\\",\\n    \\\"formation_maturity_ratio\\\",\\n)\\nPROPOSAL_ONLY_FIELDS = (\\n    \\\"candidate_type_proposal\\\",\\n    \\\"candidate_quality_rule_score\\\",\\n    \\\"proposed_direction\\\",\\n)\\n\\nANNOTATION_FIELDS = (\\n    \\\"candidate_id\\\",\\n    \\\"identity\\\",\\n    \\\"direction\\\",\\n    \\\"maturity\\\",\\n    \\\"upper_boundary_atr\\\",\\n    \\\"lower_boundary_atr\\\",\\n    \\\"start_frac\\\",\\n    \\\"end_frac\\\",\\n    \\\"annotator\\\",\\n    \\\"notes\\\",\\n)\\n\\nFORBIDDEN_PREFIXES = (\\\"cdl_\\\", \\\"chp_\\\", \\\"sr_\\\", \\\"tl_\\\", \\\"bo_\\\")\\nFORECAST_LABEL_COLUMNS = (\\n    \\\"y_direction\\\",\\n    \\\"forward_return_24\\\",\\n    \\\"forward_log_return_24\\\",\\n    \\\"long_path_code\\\",\\n    \\\"short_path_code\\\",\\n    \\\"long_tp_first_24\\\",\\n    \\\"short_tp_first_24\\\",\\n    \\\"tp_pct\\\",\\n    \\\"sl_pct\\\",\\n    \\\"neutral_band\\\",\\n    \\\"buy\\\",\\n    \\\"sell\\\",\\n    \\\"hold\\\",\\n    \\\"expectancy\\\",\\n    \\\"sharpe\\\",\\n    \\\"win_rate\\\",\\n    \\\"breakout_status\\\",\\n)\\nFORBIDDEN_EXACT = frozenset(\\n    FORECAST_LABEL_COLUMNS\\n    + PROPOSAL_ONLY_FIELDS\\n    + PATTERN_CLASSES\\n    + DIRECTIONS\\n    + MATURITY_STATES\\n    + BREAKOUT_STATUSES\\n    + (\\\"reject\\\", \\\"identity\\\", \\\"maturity\\\")\\n)\\n\\nSOURCE_RELATIVE = \\\"export/fusion_diagnostics/btcusd_5m_raw.parquet\\\"\\nPREVIEW_RELATIVE = \\\"export/cpr_mtf/preview\\\"\\nAUDIT_RELATIVE = \\\"export/cpr_mtf/audit.json\\\"\\n\\n__all__ = [\\n    \\\"ANNOTATION_FIELDS\\\",\\n    \\\"ATR_PERIOD\\\",\\n    \\\"AUDIT_RELATIVE\\\",\\n    \\\"BAR_MINUTES\\\",\\n    \\\"BREAKOUT_BUFFER_ATR\\\",\\n    \\\"BREAKOUT_HORIZONS\\\",\\n    \\\"BREAKOUT_STATUSES\\\",\\n    \\\"BREAKOUT_STRONG_ATR\\\",\\n    \\\"CONVERGENCE_SLOPE_ATR\\\",\\n    \\\"CUP_DEPTH_ATR\\\",\\n    \\\"DIRECTIONS\\\",\\n    \\\"EVAL_EVENT_STRIDE\\\",\\n    \\\"FIRST_FINITE_BAR\\\",\\n    \\\"FLAG_RETRACE_MAX\\\",\\n    \\\"FORBIDDEN_EXACT\\\",\\n    \\\"FORBIDDEN_PREFIXES\\\",\\n    \\\"FORECAST_LABEL_COLUMNS\\\",\\n    \\\"GEOMETRY_KINDS\\\",\\n    \\\"HANDLE_RETRACE_MAX\\\",\\n    \\\"HANDLE_RETRACE_MIN\\\",\\n    \\\"HEAD_PROMINENCE_ATR\\\",\\n    \\\"IMPULSE_ATR_MIN\\\",\\n    \\\"INSTRUMENT\\\",\\n    \\\"MATURITY_STATES\\\",\\n    \\\"MAX_CANDIDATES\\\",\\n    \\\"MAX_CANDIDATES_PER_TIMEFRAME\\\",\\n    \\\"MAX_PER_FAMILY\\\",\\n    \\\"MIN_BOUNDARY_POINTS\\\",\\n    \\\"MIN_CUP_SIDE_BARS\\\",\\n    \\\"MIN_PATTERN_DEPTH_ATR\\\",\\n    \\\"MIN_SEPARATION_BARS\\\",\\n    \\\"MODEL_CANDIDATE_FEATURES\\\",\\n    \\\"PAIR_LIMIT\\\",\\n    \\\"PARALLEL_SLOPE_ATR\\\",\\n    \\\"PATTERN_CLASSES\\\",\\n    \\\"PREVIEW_RELATIVE\\\",\\n    \\\"PROPOSAL_ONLY_FIELDS\\\",\\n    \\\"PURGE_BARS\\\",\\n    \\\"SEQUENCE_CHANNELS\\\",\\n    \\\"SIMILAR_PRICE_ATR\\\",\\n    \\\"SLOPE_FLAT_ATR\\\",\\n    \\\"SOURCE_BARS_PER_BUCKET\\\",\\n    \\\"SOURCE_MINUTES\\\",\\n    \\\"SOURCE_RELATIVE\\\",\\n    \\\"SPEC_VERSION\\\",\\n    \\\"SUFFIX_BARS\\\",\\n    \\\"TIMEFRAMES\\\",\\n    \\\"TIMEFRAME_ID\\\",\\n    \\\"TOUCH_TOL_ATR\\\",\\n    \\\"TRAIN_FRACTION\\\",\\n    \\\"TRAIN_STRIDE\\\",\\n    \\\"VAL_FRACTION\\\",\\n    \\\"VOLUME_BASELINE\\\",\\n    \\\"WINDOW_BARS\\\",\\n    \\\"ZZ_TAU\\\",\\n]\\n\", \"research/cpr_mtf/swings.py\": \"\\\"\\\"\\\"Causal ATR swings with separate pivot and confirmation times.\\n\\nA swing enters the record on the bar where price has reversed by 1.5 ATR.\\nLater bars are not read, and the first extreme on each side is kept.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nfrom dataclasses import dataclass\\nfrom typing import Sequence\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom research.cpr_mtf.contract import WINDOW_BARS, ZZ_TAU\\n\\n_EPS = 1e-9\\n\\n\\n@dataclass(frozen=True)\\nclass Swing:\\n    \\\"\\\"\\\"One confirmed swing extreme.\\n\\n    Attributes:\\n        swing_type: ``high`` or ``low``.\\n        pivot_index: Bar of the extreme.\\n        confirmation_index: Bar where the reversal became known.\\n        pivot_price: Extreme price.\\n        confirmation_price: Close at the confirmation bar.\\n        atr_at_pivot: Causal ATR at the pivot, with a price fallback.\\n        prominence_atr: Confirming reversal distance divided by ATR.\\n        timeframe: ``15m``, ``1h``, or ``4h``.\\n        pivot_time: Timestamp of the pivot bar.\\n        confirmation_time: Timestamp of the confirmation bar.\\n    \\\"\\\"\\\"\\n\\n    swing_type: str\\n    pivot_index: int\\n    confirmation_index: int\\n    pivot_price: float\\n    confirmation_price: float\\n    atr_at_pivot: float\\n    prominence_atr: float\\n    timeframe: str\\n    pivot_time: pd.Timestamp\\n    confirmation_time: pd.Timestamp\\n\\n\\ndef positive_atr(atr_value: float, price: float) -> float:\\n    \\\"\\\"\\\"ATR divisor. A non-positive reading falls back to one percent of price.\\n\\n    Args:\\n        atr_value: Causal ATR reading.\\n        price: Price used for the fallback scale.\\n\\n    Returns:\\n        A positive ATR value.\\n    \\\"\\\"\\\"\\n    if np.isfinite(atr_value) and float(atr_value) > 0.0:\\n        return float(atr_value)\\n    return max(abs(float(price)) * 0.01, _EPS)\\n\\n\\ndef detect_swings(\\n    frame: pd.DataFrame,\\n    timeframe: str,\\n    atr: np.ndarray,\\n) -> list[Swing]:\\n    \\\"\\\"\\\"Record every extreme confirmed by a 1.5 ATR reversal.\\n\\n    Args:\\n        frame: OHLCV sorted so that row position is the bar index.\\n        timeframe: Timeframe stored on each swing.\\n        atr: Causal ATR aligned to ``frame``.\\n\\n    Returns:\\n        Swings in confirmation order. The pivot index is the extreme bar,\\n        which is always at or before the confirmation index.\\n    \\\"\\\"\\\"\\n    high = frame[\\\"high\\\"].to_numpy(dtype=np.float64)\\n    low = frame[\\\"low\\\"].to_numpy(dtype=np.float64)\\n    close = frame[\\\"close\\\"].to_numpy(dtype=np.float64)\\n    times = pd.to_datetime(frame[\\\"time\\\"], utc=True)\\n    n = len(close)\\n    if n == 0:\\n        return []\\n    direction = 1\\n    extreme_index = 0\\n    extreme_price = float(high[0])\\n    swings: list[Swing] = []\\n    for index in range(n):\\n        atr_value = positive_atr(float(atr[index]), float(close[index]))\\n        threshold = ZZ_TAU * atr_value\\n        if direction == 1:\\n            if float(high[index]) >= extreme_price:\\n                extreme_price = float(high[index])\\n                extreme_index = index\\n            elif (extreme_price - float(low[index])) >= threshold:\\n                distance = extreme_price - float(low[index])\\n                swings.append(\\n                    _record(\\n                        \\\"high\\\",\\n                        extreme_index,\\n                        index,\\n                        extreme_price,\\n                        float(close[index]),\\n                        distance / atr_value,\\n                        high,\\n                        close,\\n                        atr,\\n                        times,\\n                        timeframe,\\n                    )\\n                )\\n                direction = -1\\n                extreme_price = float(low[index])\\n                extreme_index = index\\n        elif float(low[index]) <= extreme_price:\\n            extreme_price = float(low[index])\\n            extreme_index = index\\n        elif (float(high[index]) - extreme_price) >= threshold:\\n            distance = float(high[index]) - extreme_price\\n            swings.append(\\n                _record(\\n                    \\\"low\\\",\\n                    extreme_index,\\n                    index,\\n                    extreme_price,\\n                    float(close[index]),\\n                    distance / atr_value,\\n                    low,\\n                    close,\\n                    atr,\\n                    times,\\n                    timeframe,\\n                )\\n            )\\n            direction = 1\\n            extreme_price = float(high[index])\\n            extreme_index = index\\n    return swings\\n\\n\\ndef visible_swings(swings: Sequence[Swing], asof_index: int) -> list[Swing]:\\n    \\\"\\\"\\\"Return swings whose confirmation bar is at or before ``asof_index``.\\n\\n    Args:\\n        swings: Swings in any order.\\n        asof_index: Last bar the caller is allowed to know.\\n\\n    Returns:\\n        Confirmed swings, stable in input order.\\n    \\\"\\\"\\\"\\n    return [swing for swing in swings if swing.confirmation_index <= asof_index]\\n\\n\\ndef swings_on_chart(\\n    swings: Sequence[Swing],\\n    asof_index: int,\\n    window_start: int,\\n) -> list[Swing]:\\n    \\\"\\\"\\\"Return confirmed swings whose pivot lies inside the drawn window.\\n\\n    Args:\\n        swings: Full swing list.\\n        asof_index: Last visible bar.\\n        window_start: First bar index drawn on the chart.\\n\\n    Returns:\\n        Swings that a preview is allowed to mark.\\n    \\\"\\\"\\\"\\n    return [\\n        swing\\n        for swing in swings\\n        if swing.confirmation_index <= asof_index\\n        and window_start <= swing.pivot_index <= asof_index\\n    ]\\n\\n\\ndef _record(\\n    swing_type: str,\\n    pivot_index: int,\\n    confirmation_index: int,\\n    pivot_price: float,\\n    confirmation_price: float,\\n    prominence_atr: float,\\n    prices: np.ndarray,\\n    close: np.ndarray,\\n    atr: np.ndarray,\\n    times: pd.Series,\\n    timeframe: str,\\n) -> Swing:\\n    \\\"\\\"\\\"Build one swing using the ATR that was known at the pivot bar.\\\"\\\"\\\"\\n    return Swing(\\n        swing_type=swing_type,\\n        pivot_index=int(pivot_index),\\n        confirmation_index=int(confirmation_index),\\n        pivot_price=float(pivot_price),\\n        confirmation_price=float(confirmation_price),\\n        atr_at_pivot=positive_atr(float(atr[pivot_index]), float(prices[pivot_index])),\\n        prominence_atr=float(prominence_atr),\\n        timeframe=timeframe,\\n        pivot_time=pd.Timestamp(times.iloc[pivot_index]),\\n        confirmation_time=pd.Timestamp(times.iloc[confirmation_index]),\\n    )\\n\\n\\ndef bars_since_scale() -> float:\\n    \\\"\\\"\\\"Denominator for the recency channel, fixed at the 192-bar window.\\\"\\\"\\\"\\n    return float(np.log1p(WINDOW_BARS))\\n\", \"research/cpr_mtf/features.py\": \"\\\"\\\"\\\"Causal candle, volume, and swing channels for one timeframe.\\n\\nEvery value at bar ``t`` uses bars at or before ``t``. Pattern-rule columns\\nare not computed here.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom research.cpr_mtf.contract import (\\n    ATR_PERIOD,\\n    SEQUENCE_CHANNELS,\\n    VOLUME_BASELINE,\\n)\\nfrom research.cpr_mtf.swings import Swing, bars_since_scale, detect_swings, positive_atr\\n\\n_EPS = 1e-9\\n_OHLCV = (\\\"open\\\", \\\"high\\\", \\\"low\\\", \\\"close\\\", \\\"volume\\\")\\n\\n\\ndef causal_atr(high: np.ndarray, low: np.ndarray, close: np.ndarray, period: int) -> np.ndarray:\\n    \\\"\\\"\\\"Simple moving average of true range, using the previous close.\\n\\n    Args:\\n        high: High prices.\\n        low: Low prices.\\n        close: Close prices.\\n        period: Averaging window. Earlier bars stay NaN.\\n\\n    Returns:\\n        ATR aligned to ``close``. Bar ``t`` uses true ranges through ``t``.\\n    \\\"\\\"\\\"\\n    n = len(close)\\n    prev = np.empty(n, dtype=np.float64)\\n    prev[0] = close[0]\\n    prev[1:] = close[:-1]\\n    true_range = np.maximum(\\n        high - low,\\n        np.maximum(np.abs(high - prev), np.abs(low - prev)),\\n    )\\n    atr = np.full(n, np.nan, dtype=np.float64)\\n    if n < period:\\n        return atr\\n    window = np.ones(period, dtype=np.float64) / float(period)\\n    valid = np.convolve(true_range, window, mode=\\\"valid\\\")\\n    atr[period - 1 :] = valid\\n    return atr\\n\\n\\ndef volume_baseline(volume: np.ndarray, window: int) -> np.ndarray:\\n    \\\"\\\"\\\"Trailing mean of volume ending at the current bar.\\n\\n    Args:\\n        volume: Volume per bar.\\n        window: Lookback length, including the current bar.\\n\\n    Returns:\\n        Baseline aligned to ``volume``. Earlier bars stay NaN.\\n    \\\"\\\"\\\"\\n    n = len(volume)\\n    baseline = np.full(n, np.nan, dtype=np.float64)\\n    if n < window:\\n        return baseline\\n    kernel = np.ones(window, dtype=np.float64) / float(window)\\n    valid = np.convolve(volume.astype(np.float64), kernel, mode=\\\"valid\\\")\\n    baseline[window - 1 :] = valid\\n    return baseline\\n\\n\\ndef add_sequence_features(\\n    frame: pd.DataFrame,\\n    timeframe: str,\\n) -> tuple[pd.DataFrame, list[Swing]]:\\n    \\\"\\\"\\\"Append the 20 causal channels and the swing list for one timeframe.\\n\\n    Args:\\n        frame: Completed OHLCV sorted by time.\\n        timeframe: ``15m``, ``1h``, or ``4h``.\\n\\n    Returns:\\n        Feature frame aligned to a reset index, and swings in confirmation\\n        order. Candle channels stay NaN until ATR is finite. Volume ratio\\n        stays NaN until the trailing baseline is finite.\\n\\n    Raises:\\n        ValueError: If a required column is missing or a channel was not built.\\n    \\\"\\\"\\\"\\n    required = {\\\"time\\\", *_OHLCV}\\n    missing = required - set(frame.columns)\\n    if missing:\\n        raise ValueError(f\\\"feature frame is missing columns: {sorted(missing)}\\\")\\n    out = frame.loc[:, [\\\"time\\\", *_OHLCV]].copy()\\n    out[\\\"time\\\"] = pd.to_datetime(out[\\\"time\\\"], utc=True)\\n    out = out.sort_values(\\\"time\\\", kind=\\\"mergesort\\\").reset_index(drop=True)\\n    high = out[\\\"high\\\"].to_numpy(dtype=np.float64)\\n    low = out[\\\"low\\\"].to_numpy(dtype=np.float64)\\n    close = out[\\\"close\\\"].to_numpy(dtype=np.float64)\\n    open_ = out[\\\"open\\\"].to_numpy(dtype=np.float64)\\n    volume = out[\\\"volume\\\"].to_numpy(dtype=np.float64)\\n    atr = causal_atr(high, low, close, ATR_PERIOD)\\n    baseline = volume_baseline(volume, VOLUME_BASELINE)\\n    out[\\\"atr\\\"] = atr\\n    _write_candle_channels(out, open_, high, low, close, volume, atr, baseline)\\n    swings = detect_swings(out, timeframe, atr)\\n    for name, values in swing_channels(len(out), swings, close, atr).items():\\n        out[name] = values\\n    for name in SEQUENCE_CHANNELS:\\n        if name not in out.columns:\\n            raise ValueError(f\\\"sequence channel was not computed: {name}\\\")\\n    return out, swings\\n\\n\\ndef swing_channels(\\n    n_bars: int,\\n    swings: list[Swing],\\n    close: np.ndarray,\\n    atr: np.ndarray,\\n) -> dict[str, np.ndarray]:\\n    \\\"\\\"\\\"Fill swing event and geometry channels in one forward pass.\\n\\n    Args:\\n        n_bars: Length of the series.\\n        swings: Swings in confirmation order.\\n        close: Close prices.\\n        atr: Causal ATR aligned to ``close``.\\n\\n    Returns:\\n        The nine swing and geometry channels. A bar only sees swings whose\\n        confirmation index is at or before that bar.\\n    \\\"\\\"\\\"\\n    high_flag = np.zeros(n_bars, dtype=np.float64)\\n    low_flag = np.zeros(n_bars, dtype=np.float64)\\n    high_prom = np.zeros(n_bars, dtype=np.float64)\\n    low_prom = np.zeros(n_bars, dtype=np.float64)\\n    bars_since = np.ones(n_bars, dtype=np.float64)\\n    high_slope = np.zeros(n_bars, dtype=np.float64)\\n    low_slope = np.zeros(n_bars, dtype=np.float64)\\n    swing_range = np.zeros(n_bars, dtype=np.float64)\\n    position = np.full(n_bars, 0.5, dtype=np.float64)\\n    by_bar: dict[int, list[Swing]] = {}\\n    for swing in swings:\\n        by_bar.setdefault(swing.confirmation_index, []).append(swing)\\n    scale = bars_since_scale()\\n    last_confirmation: int | None = None\\n    recent_high: Swing | None = None\\n    previous_high: Swing | None = None\\n    recent_low: Swing | None = None\\n    previous_low: Swing | None = None\\n    for index in range(n_bars):\\n        for swing in by_bar.get(index, []):\\n            if swing.swing_type == \\\"high\\\":\\n                high_flag[index] = 1.0\\n                high_prom[index] = swing.prominence_atr\\n                previous_high = recent_high\\n                recent_high = swing\\n            else:\\n                low_flag[index] = 1.0\\n                low_prom[index] = swing.prominence_atr\\n                previous_low = recent_low\\n                recent_low = swing\\n            last_confirmation = index\\n        if last_confirmation is not None:\\n            bars_since[index] = float(np.log1p(index - last_confirmation) / scale)\\n        atr_value = positive_atr(\\n            float(atr[index]) if index < len(atr) else np.nan,\\n            float(close[index]),\\n        )\\n        high_slope[index] = _pair_slope(previous_high, recent_high, atr_value)\\n        low_slope[index] = _pair_slope(previous_low, recent_low, atr_value)\\n        if recent_high is not None and recent_low is not None:\\n            upper = max(recent_high.pivot_price, recent_low.pivot_price)\\n            lower = min(recent_high.pivot_price, recent_low.pivot_price)\\n            span = upper - lower\\n            swing_range[index] = span / atr_value\\n            if span > _EPS:\\n                position[index] = (float(close[index]) - lower) / span\\n            else:\\n                position[index] = 0.5\\n    return {\\n        \\\"swing_high_flag\\\": high_flag,\\n        \\\"swing_low_flag\\\": low_flag,\\n        \\\"swing_high_prominence_atr\\\": high_prom,\\n        \\\"swing_low_prominence_atr\\\": low_prom,\\n        \\\"bars_since_confirmed_swing\\\": bars_since,\\n        \\\"swing_high_slope\\\": high_slope,\\n        \\\"swing_low_slope\\\": low_slope,\\n        \\\"swing_range_atr\\\": swing_range,\\n        \\\"price_position_in_swing_range\\\": position,\\n    }\\n\\n\\ndef _write_candle_channels(\\n    frame: pd.DataFrame,\\n    open_: np.ndarray,\\n    high: np.ndarray,\\n    low: np.ndarray,\\n    close: np.ndarray,\\n    volume: np.ndarray,\\n    atr: np.ndarray,\\n    baseline: np.ndarray,\\n) -> None:\\n    \\\"\\\"\\\"Write the 11 candle and volume channels onto ``frame``.\\\"\\\"\\\"\\n    n = len(close)\\n    prev = np.empty(n, dtype=np.float64)\\n    prev[0] = close[0]\\n    prev[1:] = close[:-1]\\n    ready = np.isfinite(atr) & (atr > 0.0)\\n    body = np.abs(close - open_)\\n    upper = high - np.maximum(open_, close)\\n    lower = np.minimum(open_, close) - low\\n    span = high - low\\n    frame[\\\"open_atr\\\"] = _divide(open_ - prev, atr, ready)\\n    frame[\\\"high_atr\\\"] = _divide(high - prev, atr, ready)\\n    frame[\\\"low_atr\\\"] = _divide(low - prev, atr, ready)\\n    frame[\\\"close_atr\\\"] = _divide(close - prev, atr, ready)\\n    frame[\\\"body_size_atr\\\"] = _divide(body, atr, ready)\\n    frame[\\\"upper_wick_atr\\\"] = _divide(upper, atr, ready)\\n    frame[\\\"lower_wick_atr\\\"] = _divide(lower, atr, ready)\\n    frame[\\\"range_atr\\\"] = _divide(span, atr, ready)\\n    frame[\\\"body_to_range\\\"] = _ratio(body, span, ready)\\n    frame[\\\"close_location\\\"] = _ratio(close - low, span, ready)\\n    volume_ready = ready & np.isfinite(baseline) & (baseline > 0.0)\\n    frame[\\\"volume_ratio\\\"] = _divide(volume, baseline, volume_ready)\\n\\n\\ndef _divide(numer: np.ndarray, denom: np.ndarray, ready: np.ndarray) -> np.ndarray:\\n    \\\"\\\"\\\"Divide where ``ready`` is true and leave the other rows NaN.\\\"\\\"\\\"\\n    out = np.full(len(numer), np.nan, dtype=np.float64)\\n    out[ready] = numer[ready] / denom[ready]\\n    return out\\n\\n\\ndef _ratio(numer: np.ndarray, denom: np.ndarray, ready: np.ndarray) -> np.ndarray:\\n    \\\"\\\"\\\"Divide by a bar range, using 0 when the range is flat.\\\"\\\"\\\"\\n    out = np.full(len(numer), np.nan, dtype=np.float64)\\n    safe = ready & (np.abs(denom) > _EPS)\\n    flat = ready & ~safe\\n    out[safe] = numer[safe] / denom[safe]\\n    out[flat] = 0.0\\n    return out\\n\\n\\ndef _pair_slope(older: Swing | None, newer: Swing | None, atr_value: float) -> float:\\n    \\\"\\\"\\\"ATR-per-bar slope between two confirmed pivots.\\\"\\\"\\\"\\n    if older is None or newer is None:\\n        return 0.0\\n    span = newer.pivot_index - older.pivot_index\\n    if span <= 0 or atr_value <= 0.0:\\n        return 0.0\\n    return (newer.pivot_price - older.pivot_price) / atr_value / float(span)\\n\", \"research/cpr_mtf/bars.py\": \"\\\"\\\"\\\"Resample completed 5-minute BTCUSD candles into 15m, 1h, and 4h bars.\\n\\nA bucket is emitted only when it contains every clock-aligned 5-minute bar.\\nThe next bucket is never consulted.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nfrom pathlib import Path\\nfrom typing import Iterable\\n\\nimport pandas as pd\\n\\nfrom research.cpr_mtf.contract import (\\n    BAR_MINUTES,\\n    SOURCE_BARS_PER_BUCKET,\\n    SOURCE_MINUTES,\\n    TIMEFRAMES,\\n)\\n\\n_OHLCV = (\\\"open\\\", \\\"high\\\", \\\"low\\\", \\\"close\\\", \\\"volume\\\")\\n_STEP = pd.Timedelta(minutes=SOURCE_MINUTES)\\n\\n\\ndef _as_utc(values: pd.Series) -> pd.Series:\\n    \\\"\\\"\\\"Parse timestamps as UTC.\\n\\n    Args:\\n        values: Datetime-like values. Naive values are treated as UTC.\\n\\n    Returns:\\n        Timezone-aware UTC timestamps.\\n    \\\"\\\"\\\"\\n    parsed = pd.to_datetime(values, utc=True)\\n    return pd.Series(parsed, index=values.index)\\n\\n\\ndef dedupe_timestamps(frame: pd.DataFrame) -> pd.DataFrame:\\n    \\\"\\\"\\\"Keep the last row for each timestamp.\\n\\n    Args:\\n        frame: Candle rows with a ``time`` or ``timestamp`` column.\\n\\n    Returns:\\n        Frame sorted by time, with one row per timestamp.\\n\\n    Raises:\\n        ValueError: If neither time column is present.\\n    \\\"\\\"\\\"\\n    work = _require_ohlcv(frame)\\n    work[\\\"time\\\"] = _as_utc(work[\\\"time\\\"])\\n    work = work.sort_values(\\\"time\\\", kind=\\\"mergesort\\\")\\n    return work.drop_duplicates(\\\"time\\\", keep=\\\"last\\\").reset_index(drop=True)\\n\\n\\ndef load_btcusd_5m(path: Path) -> pd.DataFrame:\\n    \\\"\\\"\\\"Load and dedupe a 5-minute BTCUSD parquet file.\\n\\n    Args:\\n        path: Parquet path with ``time`` or ``timestamp`` plus OHLCV.\\n\\n    Returns:\\n        Deduped frame sorted by UTC time.\\n    \\\"\\\"\\\"\\n    frame = pd.read_parquet(path)\\n    return dedupe_timestamps(frame)\\n\\n\\ndef resample_ohlcv(frame: pd.DataFrame, timeframe: str) -> pd.DataFrame:\\n    \\\"\\\"\\\"Aggregate contiguous 5-minute bars into completed higher-timeframe bars.\\n\\n    Args:\\n        frame: Columns ``time``, ``open``, ``high``, ``low``, ``close``,\\n            and ``volume``.\\n        timeframe: One of ``15m``, ``1h``, or ``4h``.\\n\\n    Returns:\\n        One row per complete bucket, sorted by bucket open time.\\n\\n    Raises:\\n        ValueError: If the timeframe or a required column is unknown.\\n    \\\"\\\"\\\"\\n    if timeframe not in TIMEFRAMES:\\n        raise ValueError(f\\\"unknown timeframe: {timeframe}\\\")\\n    work = dedupe_timestamps(frame)\\n    bucket_delta = pd.Timedelta(minutes=BAR_MINUTES[timeframe])\\n    expected_count = SOURCE_BARS_PER_BUCKET[timeframe]\\n    bucket = work[\\\"time\\\"].dt.floor(bucket_delta)\\n    grouped = work.groupby(bucket, sort=True)\\n    rows: list[dict[str, object]] = []\\n    for start, group in grouped:\\n        if not _bucket_is_complete(group[\\\"time\\\"].tolist(), start, expected_count):\\n            continue\\n        ordered = group.sort_values(\\\"time\\\", kind=\\\"mergesort\\\")\\n        rows.append(\\n            {\\n                \\\"time\\\": pd.Timestamp(start),\\n                \\\"open\\\": float(ordered[\\\"open\\\"].iloc[0]),\\n                \\\"high\\\": float(ordered[\\\"high\\\"].max()),\\n                \\\"low\\\": float(ordered[\\\"low\\\"].min()),\\n                \\\"close\\\": float(ordered[\\\"close\\\"].iloc[-1]),\\n                \\\"volume\\\": float(ordered[\\\"volume\\\"].sum()),\\n            }\\n        )\\n    if not rows:\\n        return pd.DataFrame(columns=[\\\"time\\\", *_OHLCV])\\n    out = pd.DataFrame(rows)\\n    out[\\\"time\\\"] = pd.to_datetime(out[\\\"time\\\"], utc=True)\\n    return out.reset_index(drop=True)\\n\\n\\ndef _require_ohlcv(frame: pd.DataFrame) -> pd.DataFrame:\\n    \\\"\\\"\\\"Return the time and OHLCV columns, accepting ``timestamp`` as time.\\\"\\\"\\\"\\n    work = frame.copy()\\n    if \\\"time\\\" not in work.columns:\\n        if \\\"timestamp\\\" not in work.columns:\\n            raise ValueError(\\\"candle frame needs a time or timestamp column\\\")\\n        work[\\\"time\\\"] = work[\\\"timestamp\\\"]\\n    missing = {\\\"time\\\", *_OHLCV} - set(work.columns)\\n    if missing:\\n        raise ValueError(f\\\"candle frame is missing columns: {sorted(missing)}\\\")\\n    return work.loc[:, [\\\"time\\\", *_OHLCV]].copy()\\n\\n\\ndef _bucket_is_complete(\\n    stamps: Iterable[pd.Timestamp],\\n    start: pd.Timestamp,\\n    expected_count: int,\\n) -> bool:\\n    \\\"\\\"\\\"Return True when the bucket holds exactly the expected source stamps.\\\"\\\"\\\"\\n    origin = pd.Timestamp(start)\\n    expected = {origin + index * _STEP for index in range(expected_count)}\\n    seen = [pd.Timestamp(stamp) for stamp in stamps]\\n    if len(seen) != expected_count or len(set(seen)) != expected_count:\\n        return False\\n    return set(seen) == expected\\n\", \"research/cpr_mtf/candidates.py\": \"\\\"\\\"\\\"Chart-candidate proposals from confirmed swings.\\n\\nProposal type and the geometric rule score stay on the proposal record.\\nThey are not columns of the sequence tensor or the geometry vector.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport math\\nfrom dataclasses import dataclass\\nfrom itertools import combinations\\nfrom typing import Sequence\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom research.cpr_mtf.contract import (\\n    CONVERGENCE_SLOPE_ATR,\\n    CUP_DEPTH_ATR,\\n    FLAG_RETRACE_MAX,\\n    HANDLE_RETRACE_MAX,\\n    HANDLE_RETRACE_MIN,\\n    HEAD_PROMINENCE_ATR,\\n    IMPULSE_ATR_MIN,\\n    MAX_CANDIDATES,\\n    MAX_CANDIDATES_PER_TIMEFRAME,\\n    MAX_PER_FAMILY,\\n    MIN_CUP_SIDE_BARS,\\n    MIN_PATTERN_DEPTH_ATR,\\n    MIN_SEPARATION_BARS,\\n    MODEL_CANDIDATE_FEATURES,\\n    PAIR_LIMIT,\\n    PARALLEL_SLOPE_ATR,\\n    SIMILAR_PRICE_ATR,\\n    SLOPE_FLAT_ATR,\\n    TIMEFRAMES,\\n    TIMEFRAME_ID,\\n    TOUCH_TOL_ATR,\\n    WINDOW_BARS,\\n)\\nfrom research.cpr_mtf.swings import Swing\\n\\n_EPS = 1e-9\\n_BOUNDARY_DIRECTION = {\\n    \\\"ascending_triangle\\\": \\\"bullish\\\",\\n    \\\"descending_triangle\\\": \\\"bearish\\\",\\n    \\\"symmetrical_triangle\\\": \\\"neutral\\\",\\n    \\\"rising_wedge\\\": \\\"bullish\\\",\\n    \\\"falling_wedge\\\": \\\"bearish\\\",\\n    \\\"ascending_channel\\\": \\\"bullish\\\",\\n    \\\"descending_channel\\\": \\\"bearish\\\",\\n    \\\"rectangle\\\": \\\"neutral\\\",\\n    \\\"broadening\\\": \\\"neutral\\\",\\n}\\n\\n\\n@dataclass\\nclass Candidate:\\n    \\\"\\\"\\\"One geometric proposal at a single as-of bar.\\n\\n    Attributes:\\n        candidate_id: Stable id from timeframe, class, and confirmation times.\\n        timeframe: Native timeframe.\\n        pattern: Proposed shape class.\\n        direction: Structural proposal direction, not an annotation.\\n        rule_score: Preliminary geometric score in ``[0, 1]``.\\n        geometry_kind: ``swing`` or ``rounded_proxy``.\\n        swing_confirmation_indices: Constituent confirmation bars.\\n        first_confirmation_index: Earliest constituent confirmation.\\n        last_confirmation_index: Latest constituent confirmation.\\n        first_pivot_index: Earliest constituent pivot.\\n        last_pivot_index: Latest constituent pivot.\\n        upper_price_at_asof: Upper boundary price at the as-of bar.\\n        lower_price_at_asof: Lower boundary price at the as-of bar.\\n        upper_slope: Upper boundary in price per bar.\\n        lower_slope: Lower boundary in price per bar.\\n        atr: ATR at the as-of bar.\\n        origin_close: Close at the as-of bar, the price origin.\\n        start_time: Timestamp of the earliest constituent pivot.\\n        asof_time: Timestamp of the as-of bar.\\n        features: Geometry vector in ``MODEL_CANDIDATE_FEATURES`` order.\\n    \\\"\\\"\\\"\\n\\n    candidate_id: str\\n    timeframe: str\\n    pattern: str\\n    direction: str\\n    rule_score: float\\n    geometry_kind: str\\n    swing_confirmation_indices: tuple[int, ...]\\n    first_confirmation_index: int\\n    last_confirmation_index: int\\n    first_pivot_index: int\\n    last_pivot_index: int\\n    upper_price_at_asof: float\\n    lower_price_at_asof: float\\n    upper_slope: float\\n    lower_slope: float\\n    atr: float\\n    origin_close: float\\n    start_time: pd.Timestamp\\n    asof_time: pd.Timestamp\\n    features: np.ndarray\\n\\n\\ndef generate_candidates(\\n    frame: pd.DataFrame,\\n    swings: Sequence[Swing],\\n    timeframe: str,\\n    asof_index: int,\\n) -> list[Candidate]:\\n    \\\"\\\"\\\"Propose chart formations visible at ``asof_index``.\\n\\n    Args:\\n        frame: Feature frame for ``timeframe``, position-aligned to ``swings``.\\n        swings: Swings detected on ``frame``.\\n        timeframe: Timeframe of the series.\\n        asof_index: Last bar the generator may use.\\n\\n    Returns:\\n        Deduplicated proposals after the per-family and per-timeframe caps.\\n\\n    Raises:\\n        ValueError: If ``asof_index`` is outside ``frame``.\\n    \\\"\\\"\\\"\\n    if asof_index < 0 or asof_index >= len(frame):\\n        raise ValueError(f\\\"as-of index {asof_index} is outside the series\\\")\\n    atr = _asof_atr(frame, asof_index)\\n    visible = _visible_in_window(swings, asof_index)\\n    found: list[Candidate] = []\\n    found.extend(_doubles(visible, frame, timeframe, asof_index, atr))\\n    found.extend(_triples(visible, frame, timeframe, asof_index, atr))\\n    found.extend(_head_and_shoulders(visible, frame, timeframe, asof_index, atr))\\n    found.extend(_impulse_patterns(visible, frame, timeframe, asof_index, atr))\\n    found.extend(_boundary_pattern(visible, frame, timeframe, asof_index, atr))\\n    found.extend(_cups(visible, frame, timeframe, asof_index, atr))\\n    return apply_quota(deduplicate_candidates(found))\\n\\n\\ndef deduplicate_candidates(candidates: Sequence[Candidate]) -> list[Candidate]:\\n    \\\"\\\"\\\"Keep the higher rule score for each candidate id.\\n\\n    Args:\\n        candidates: Proposals that may repeat the same confirmation span.\\n\\n    Returns:\\n        One proposal per candidate id.\\n    \\\"\\\"\\\"\\n    best: dict[str, Candidate] = {}\\n    for candidate in candidates:\\n        current = best.get(candidate.candidate_id)\\n        if current is None or candidate.rule_score > current.rule_score:\\n            best[candidate.candidate_id] = candidate\\n    return list(best.values())\\n\\n\\ndef apply_quota(candidates: Sequence[Candidate]) -> list[Candidate]:\\n    \\\"\\\"\\\"Keep at most two proposals per family and twelve per call.\\n\\n    Args:\\n        candidates: Proposals for a single timeframe.\\n\\n    Returns:\\n        Proposals ordered by descending rule score.\\n    \\\"\\\"\\\"\\n    ordered = sorted(candidates, key=lambda item: (-item.rule_score, item.candidate_id))\\n    kept: list[Candidate] = []\\n    counts: dict[str, int] = {}\\n    for candidate in ordered:\\n        if counts.get(candidate.pattern, 0) >= MAX_PER_FAMILY:\\n            continue\\n        if len(kept) >= MAX_CANDIDATES_PER_TIMEFRAME:\\n            break\\n        counts[candidate.pattern] = counts.get(candidate.pattern, 0) + 1\\n        kept.append(candidate)\\n    return kept\\n\\n\\ndef stack_candidates(\\n    by_timeframe: dict[str, Sequence[Candidate]],\\n) -> tuple[np.ndarray, np.ndarray, list[Candidate]]:\\n    \\\"\\\"\\\"Pad proposals to 36 rows in timeframe order.\\n\\n    Args:\\n        by_timeframe: Proposals keyed by ``15m``, ``1h``, and ``4h``.\\n\\n    Returns:\\n        Geometry array ``(36, 22)``, validity mask ``(36,)``, and the\\n        unpadded proposals in the same order as the mask's positive rows.\\n    \\\"\\\"\\\"\\n    ordered: list[Candidate] = []\\n    for timeframe in TIMEFRAMES:\\n        ordered.extend(by_timeframe.get(timeframe, []))\\n    ordered = ordered[:MAX_CANDIDATES]\\n    width = len(MODEL_CANDIDATE_FEATURES)\\n    array = np.zeros((MAX_CANDIDATES, width), dtype=np.float64)\\n    mask = np.zeros(MAX_CANDIDATES, dtype=np.float64)\\n    for index, candidate in enumerate(ordered):\\n        array[index] = candidate.features\\n        mask[index] = 1.0\\n    return array, mask, ordered\\n\\n\\ndef proposal_record(candidate: Candidate) -> dict[str, object]:\\n    \\\"\\\"\\\"Return proposal metadata that is excluded from the geometry vector.\\n\\n    Args:\\n        candidate: One proposal.\\n\\n    Returns:\\n        JSON-ready metadata, including the rule score and proposed type.\\n    \\\"\\\"\\\"\\n    atr = max(candidate.atr, _EPS)\\n    return {\\n        \\\"candidate_id\\\": candidate.candidate_id,\\n        \\\"timeframe\\\": candidate.timeframe,\\n        \\\"candidate_type_proposal\\\": candidate.pattern,\\n        \\\"proposed_direction\\\": candidate.direction,\\n        \\\"candidate_quality_rule_score\\\": candidate.rule_score,\\n        \\\"geometry_kind\\\": candidate.geometry_kind,\\n        \\\"swing_confirmation_indices\\\": list(candidate.swing_confirmation_indices),\\n        \\\"upper_price\\\": candidate.upper_price_at_asof,\\n        \\\"lower_price\\\": candidate.lower_price_at_asof,\\n        \\\"upper_slope_price_per_bar\\\": candidate.upper_slope,\\n        \\\"lower_slope_price_per_bar\\\": candidate.lower_slope,\\n        \\\"origin_close\\\": candidate.origin_close,\\n        \\\"upper_boundary_atr\\\": (candidate.upper_price_at_asof - candidate.origin_close) / atr,\\n        \\\"lower_boundary_atr\\\": (candidate.lower_price_at_asof - candidate.origin_close) / atr,\\n    }\\n\\n\\ndef propose_pairs(candidates: Sequence[Candidate]) -> list[dict[str, object]]:\\n    \\\"\\\"\\\"List cross-timeframe pairs, overlaps first, capped at 32.\\n\\n    Args:\\n        candidates: Proposals from one or more timeframes.\\n\\n    Returns:\\n        Pair rows with an empty ``relationship`` field. Same-timeframe\\n        pairs are omitted.\\n    \\\"\\\"\\\"\\n    overlaps: list[dict[str, object]] = []\\n    negatives: list[dict[str, object]] = []\\n    for left, right in combinations(candidates, 2):\\n        if left.timeframe == right.timeframe:\\n            continue\\n        lower, higher = _timeframe_order(left, right)\\n        price_overlap = _ranges_overlap(_price_span(lower), _price_span(higher))\\n        time_overlap = _ranges_overlap(\\n            (lower.start_time, lower.asof_time),\\n            (higher.start_time, higher.asof_time),\\n        )\\n        row = {\\n            \\\"lower_timeframe\\\": lower.timeframe,\\n            \\\"higher_timeframe\\\": higher.timeframe,\\n            \\\"left_id\\\": lower.candidate_id,\\n            \\\"right_id\\\": higher.candidate_id,\\n            \\\"price_overlap\\\": price_overlap,\\n            \\\"time_overlap\\\": time_overlap,\\n            \\\"relationship\\\": \\\"\\\",\\n        }\\n        if price_overlap and time_overlap:\\n            overlaps.append(row)\\n        else:\\n            negatives.append(row)\\n    return (overlaps + negatives)[:PAIR_LIMIT]\\n\\n\\ndef _doubles(\\n    visible: Sequence[Swing],\\n    frame: pd.DataFrame,\\n    timeframe: str,\\n    asof_index: int,\\n    atr: float,\\n) -> list[Candidate]:\\n    highs = [swing for swing in visible if swing.swing_type == \\\"high\\\"]\\n    lows = [swing for swing in visible if swing.swing_type == \\\"low\\\"]\\n    found = _paired_extremes(highs, lows, \\\"bearish\\\", frame, timeframe, asof_index, atr, 2)\\n    found.extend(\\n        _paired_extremes(lows, highs, \\\"bullish\\\", frame, timeframe, asof_index, atr, 2)\\n    )\\n    return found\\n\\n\\ndef _triples(\\n    visible: Sequence[Swing],\\n    frame: pd.DataFrame,\\n    timeframe: str,\\n    asof_index: int,\\n    atr: float,\\n) -> list[Candidate]:\\n    highs = [swing for swing in visible if swing.swing_type == \\\"high\\\"]\\n    lows = [swing for swing in visible if swing.swing_type == \\\"low\\\"]\\n    found = _paired_extremes(highs, lows, \\\"bearish\\\", frame, timeframe, asof_index, atr, 3)\\n    found.extend(\\n        _paired_extremes(lows, highs, \\\"bullish\\\", frame, timeframe, asof_index, atr, 3)\\n    )\\n    return found\\n\\n\\ndef _paired_extremes(\\n    extremes: Sequence[Swing],\\n    opposites: Sequence[Swing],\\n    direction: str,\\n    frame: pd.DataFrame,\\n    timeframe: str,\\n    asof_index: int,\\n    atr: float,\\n    count: int,\\n) -> list[Candidate]:\\n    \\\"\\\"\\\"Build double or triple proposals from consecutive same-side swings.\\\"\\\"\\\"\\n    found: list[Candidate] = []\\n    if len(extremes) < count:\\n        return found\\n    pattern = \\\"double\\\" if count == 2 else \\\"triple\\\"\\n    for start in range(len(extremes) - count + 1):\\n        group = list(extremes[start : start + count])\\n        prices = [swing.pivot_price for swing in group]\\n        gap = max(prices) - min(prices)\\n        if gap > SIMILAR_PRICE_ATR * atr:\\n            continue\\n        if group[-1].pivot_index - group[0].pivot_index < MIN_SEPARATION_BARS:\\n            continue\\n        between: list[Swing] = []\\n        valid = True\\n        for left, right in zip(group, group[1:]):\\n            mids = [\\n                swing\\n                for swing in opposites\\n                if left.pivot_index < swing.pivot_index < right.pivot_index\\n            ]\\n            if not mids:\\n                valid = False\\n                break\\n            if direction == \\\"bearish\\\":\\n                between.append(min(mids, key=lambda swing: swing.pivot_price))\\n            else:\\n                between.append(max(mids, key=lambda swing: swing.pivot_price))\\n        if not valid:\\n            continue\\n        if direction == \\\"bearish\\\":\\n            depth = min(prices) - max(swing.pivot_price for swing in between)\\n        else:\\n            depth = min(swing.pivot_price for swing in between) - max(prices)\\n        if depth < MIN_PATTERN_DEPTH_ATR * atr:\\n            continue\\n        upper_slope, upper_at, lower_slope, lower_at = _extreme_lines(\\n            group, between, direction, asof_index\\n        )\\n        used = tuple(group + between)\\n        impulse = abs(upper_at - lower_at) / max(atr, _EPS)\\n        found.append(\\n            _assemble(\\n                frame,\\n                timeframe,\\n                pattern,\\n                direction,\\n                used,\\n                asof_index,\\n                atr,\\n                upper_slope,\\n                lower_slope,\\n                upper_at,\\n                lower_at,\\n                impulse,\\n                depth / max(abs(upper_at - lower_at), _EPS),\\n                \\\"swing\\\",\\n                gap,\\n            )\\n        )\\n    return found\\n\\n\\ndef _extreme_lines(\\n    extremes: Sequence[Swing],\\n    between: Sequence[Swing],\\n    direction: str,\\n    asof_index: int,\\n) -> tuple[float, float, float, float]:\\n    \\\"\\\"\\\"Return upper slope, upper price, lower slope, and lower price.\\\"\\\"\\\"\\n    extreme_slope, extreme_intercept = _fit_line(\\n        [(swing.pivot_index, swing.pivot_price) for swing in extremes]\\n    )\\n    other_slope, other_intercept = _fit_line(\\n        [(swing.pivot_index, swing.pivot_price) for swing in between]\\n    )\\n    extreme_at = extreme_intercept + extreme_slope * asof_index\\n    other_at = other_intercept + other_slope * asof_index\\n    if direction == \\\"bearish\\\":\\n        return extreme_slope, extreme_at, other_slope, other_at\\n    return other_slope, other_at, extreme_slope, extreme_at\\n\\n\\ndef _head_and_shoulders(\\n    visible: Sequence[Swing],\\n    frame: pd.DataFrame,\\n    timeframe: str,\\n    asof_index: int,\\n    atr: float,\\n) -> list[Candidate]:\\n    found: list[Candidate] = []\\n    for index in range(len(visible) - 4):\\n        group = list(visible[index : index + 5])\\n        kinds = [swing.swing_type for swing in group]\\n        if kinds == [\\\"high\\\", \\\"low\\\", \\\"high\\\", \\\"low\\\", \\\"high\\\"]:\\n            candidate = _shoulder_candidate(\\n                group, \\\"bearish\\\", frame, timeframe, asof_index, atr\\n            )\\n        elif kinds == [\\\"low\\\", \\\"high\\\", \\\"low\\\", \\\"high\\\", \\\"low\\\"]:\\n            candidate = _shoulder_candidate(\\n                group, \\\"bullish\\\", frame, timeframe, asof_index, atr\\n            )\\n        else:\\n            candidate = None\\n        if candidate is not None:\\n            found.append(candidate)\\n    return found\\n\\n\\ndef _shoulder_candidate(\\n    group: list[Swing],\\n    direction: str,\\n    frame: pd.DataFrame,\\n    timeframe: str,\\n    asof_index: int,\\n    atr: float,\\n) -> Candidate | None:\\n    left, first_mid, head, second_mid, right = group\\n    if direction == \\\"bearish\\\":\\n        prominence = head.pivot_price - max(left.pivot_price, right.pivot_price)\\n        above = min(left.pivot_price, right.pivot_price) - max(\\n            first_mid.pivot_price, second_mid.pivot_price\\n        )\\n    else:\\n        prominence = min(left.pivot_price, right.pivot_price) - head.pivot_price\\n        above = min(first_mid.pivot_price, second_mid.pivot_price) - max(\\n            left.pivot_price, right.pivot_price\\n        )\\n    gap = abs(left.pivot_price - right.pivot_price)\\n    if prominence < HEAD_PROMINENCE_ATR * atr or above <= 0.0 or gap > SIMILAR_PRICE_ATR * atr:\\n        return None\\n    if direction == \\\"bearish\\\":\\n        upper_points = [\\n            (left.pivot_index, left.pivot_price),\\n            (right.pivot_index, right.pivot_price),\\n        ]\\n        lower_points = [\\n            (first_mid.pivot_index, first_mid.pivot_price),\\n            (second_mid.pivot_index, second_mid.pivot_price),\\n        ]\\n    else:\\n        lower_points = [\\n            (left.pivot_index, left.pivot_price),\\n            (right.pivot_index, right.pivot_price),\\n        ]\\n        upper_points = [\\n            (first_mid.pivot_index, first_mid.pivot_price),\\n            (second_mid.pivot_index, second_mid.pivot_price),\\n        ]\\n    upper_slope, upper_intercept = _fit_line(upper_points)\\n    lower_slope, lower_intercept = _fit_line(lower_points)\\n    return _assemble(\\n        frame,\\n        timeframe,\\n        \\\"head_and_shoulders\\\",\\n        direction,\\n        tuple(group),\\n        asof_index,\\n        atr,\\n        upper_slope,\\n        lower_slope,\\n        upper_intercept + upper_slope * asof_index,\\n        lower_intercept + lower_slope * asof_index,\\n        prominence / max(atr, _EPS),\\n        0.0,\\n        \\\"swing\\\",\\n        gap,\\n    )\\n\\n\\ndef _impulse_patterns(\\n    visible: Sequence[Swing],\\n    frame: pd.DataFrame,\\n    timeframe: str,\\n    asof_index: int,\\n    atr: float,\\n) -> list[Candidate]:\\n    found: list[Candidate] = []\\n    for index in range(len(visible) - 1):\\n        start = visible[index]\\n        end = visible[index + 1]\\n        if start.swing_type == end.swing_type:\\n            continue\\n        impulse = abs(end.pivot_price - start.pivot_price)\\n        if impulse < IMPULSE_ATR_MIN * atr:\\n            continue\\n        direction = \\\"bullish\\\" if end.pivot_price > start.pivot_price else \\\"bearish\\\"\\n        packed = _consolidation_points(end, visible[index + 2 : index + 8])\\n        if packed is None:\\n            continue\\n        upper_points, lower_points, rest = packed\\n        prices = [price for _, price in upper_points + lower_points]\\n        span = max(prices) - min(prices)\\n        if span > FLAG_RETRACE_MAX * impulse:\\n            continue\\n        drift = rest[-1].pivot_price - end.pivot_price\\n        if direction == \\\"bullish\\\" and drift >= 0.0:\\n            continue\\n        if direction == \\\"bearish\\\" and drift <= 0.0:\\n            continue\\n        upper_slope, upper_intercept = _fit_line(upper_points)\\n        lower_slope, lower_intercept = _fit_line(lower_points)\\n        upper_at = upper_intercept + upper_slope * asof_index\\n        lower_at = lower_intercept + lower_slope * asof_index\\n        used = (start, end, *rest)\\n        retrace = span / max(impulse, _EPS)\\n        impulse_atr = impulse / max(atr, _EPS)\\n        found.append(\\n            _assemble(\\n                frame,\\n                timeframe,\\n                \\\"flag\\\",\\n                direction,\\n                used,\\n                asof_index,\\n                atr,\\n                upper_slope,\\n                lower_slope,\\n                upper_at,\\n                lower_at,\\n                impulse_atr,\\n                retrace,\\n                \\\"swing\\\",\\n                None,\\n            )\\n        )\\n        slope_gap = (lower_slope - upper_slope) / max(atr, _EPS)\\n        if slope_gap > CONVERGENCE_SLOPE_ATR:\\n            found.append(\\n                _assemble(\\n                    frame,\\n                    timeframe,\\n                    \\\"pennant\\\",\\n                    direction,\\n                    used,\\n                    asof_index,\\n                    atr,\\n                    upper_slope,\\n                    lower_slope,\\n                    upper_at,\\n                    lower_at,\\n                    impulse_atr,\\n                    retrace,\\n                    \\\"swing\\\",\\n                    None,\\n                )\\n            )\\n    return found\\n\\n\\ndef _consolidation_points(\\n    impulse_end: Swing,\\n    rest: Sequence[Swing],\\n) -> tuple[list[tuple[int, float]], list[tuple[int, float]], list[Swing]] | None:\\n    \\\"\\\"\\\"Collect two boundary points on each side, including the impulse end.\\\"\\\"\\\"\\n    upper: list[tuple[int, float]] = []\\n    lower: list[tuple[int, float]] = []\\n    if impulse_end.swing_type == \\\"high\\\":\\n        upper.append((impulse_end.pivot_index, impulse_end.pivot_price))\\n    else:\\n        lower.append((impulse_end.pivot_index, impulse_end.pivot_price))\\n    used: list[Swing] = []\\n    for swing in rest:\\n        used.append(swing)\\n        point = (swing.pivot_index, swing.pivot_price)\\n        if swing.swing_type == \\\"high\\\":\\n            upper.append(point)\\n        else:\\n            lower.append(point)\\n        if len(upper) >= 2 and len(lower) >= 2:\\n            return upper, lower, used\\n    return None\\n\\n\\ndef _boundary_pattern(\\n    visible: Sequence[Swing],\\n    frame: pd.DataFrame,\\n    timeframe: str,\\n    asof_index: int,\\n    atr: float,\\n) -> list[Candidate]:\\n    highs = [swing for swing in visible if swing.swing_type == \\\"high\\\"][-4:]\\n    lows = [swing for swing in visible if swing.swing_type == \\\"low\\\"][-4:]\\n    if len(highs) < 2 or len(lows) < 2:\\n        return []\\n    upper_points = [(swing.pivot_index, swing.pivot_price) for swing in highs]\\n    lower_points = [(swing.pivot_index, swing.pivot_price) for swing in lows]\\n    upper_slope, upper_intercept = _fit_line(upper_points)\\n    lower_slope, lower_intercept = _fit_line(lower_points)\\n    family = _family_from_slopes(upper_slope / max(atr, _EPS), lower_slope / max(atr, _EPS))\\n    if family is None:\\n        return []\\n    used = tuple(highs + lows)\\n    upper_at = upper_intercept + upper_slope * asof_index\\n    lower_at = lower_intercept + lower_slope * asof_index\\n    return [\\n        _assemble(\\n            frame,\\n            timeframe,\\n            family,\\n            _BOUNDARY_DIRECTION[family],\\n            used,\\n            asof_index,\\n            atr,\\n            upper_slope,\\n            lower_slope,\\n            upper_at,\\n            lower_at,\\n            abs(upper_at - lower_at) / max(atr, _EPS),\\n            0.0,\\n            \\\"swing\\\",\\n            None,\\n        )\\n    ]\\n\\n\\ndef _family_from_slopes(upper_slope: float, lower_slope: float) -> str | None:\\n    \\\"\\\"\\\"Classify two ATR-per-bar slopes into one boundary family.\\\"\\\"\\\"\\n    flat = SLOPE_FLAT_ATR\\n    if abs(upper_slope) <= flat and abs(lower_slope) <= flat:\\n        return \\\"rectangle\\\"\\n    if abs(upper_slope) <= flat and lower_slope > flat:\\n        return \\\"ascending_triangle\\\"\\n    if abs(lower_slope) <= flat and upper_slope < -flat:\\n        return \\\"descending_triangle\\\"\\n    if upper_slope < -flat and lower_slope > flat:\\n        return \\\"symmetrical_triangle\\\"\\n    if upper_slope > flat and lower_slope < -flat:\\n        return \\\"broadening\\\"\\n    if upper_slope > flat and lower_slope > flat:\\n        if abs(upper_slope - lower_slope) <= PARALLEL_SLOPE_ATR:\\n            return \\\"ascending_channel\\\"\\n        if lower_slope > upper_slope + CONVERGENCE_SLOPE_ATR:\\n            return \\\"rising_wedge\\\"\\n        return None\\n    if upper_slope < -flat and lower_slope < -flat:\\n        if abs(upper_slope - lower_slope) <= PARALLEL_SLOPE_ATR:\\n            return \\\"descending_channel\\\"\\n        if upper_slope < lower_slope - CONVERGENCE_SLOPE_ATR:\\n            return \\\"falling_wedge\\\"\\n        return None\\n    return None\\n\\n\\ndef _cups(\\n    visible: Sequence[Swing],\\n    frame: pd.DataFrame,\\n    timeframe: str,\\n    asof_index: int,\\n    atr: float,\\n) -> list[Candidate]:\\n    highs = [swing for swing in visible if swing.swing_type == \\\"high\\\"]\\n    lows = [swing for swing in visible if swing.swing_type == \\\"low\\\"]\\n    found = _cup_side(lows, highs, \\\"bullish\\\", frame, timeframe, asof_index, atr)\\n    found.extend(_cup_side(highs, lows, \\\"bearish\\\", frame, timeframe, asof_index, atr))\\n    return found\\n\\n\\ndef _cup_side(\\n    bottoms: Sequence[Swing],\\n    lips: Sequence[Swing],\\n    direction: str,\\n    frame: pd.DataFrame,\\n    timeframe: str,\\n    asof_index: int,\\n    atr: float,\\n) -> list[Candidate]:\\n    \\\"\\\"\\\"Rounded recovery. ``bottoms`` are the cup extremes and ``lips`` the rims.\\\"\\\"\\\"\\n    found: list[Candidate] = []\\n    for bottom in bottoms:\\n        prior = [swing for swing in lips if swing.pivot_index < bottom.pivot_index]\\n        later = [swing for swing in lips if swing.pivot_index > bottom.pivot_index]\\n        if not prior or not later:\\n            continue\\n        left = prior[-1]\\n        right = later[0]\\n        gap = abs(left.pivot_price - right.pivot_price)\\n        if gap > SIMILAR_PRICE_ATR * atr:\\n            continue\\n        if bottom.pivot_index - left.pivot_index < MIN_CUP_SIDE_BARS:\\n            continue\\n        if right.pivot_index - bottom.pivot_index < MIN_CUP_SIDE_BARS:\\n            continue\\n        lip = (left.pivot_price + right.pivot_price) / 2.0\\n        depth = (lip - bottom.pivot_price) if direction == \\\"bullish\\\" else (bottom.pivot_price - lip)\\n        if depth < CUP_DEPTH_ATR * atr:\\n            continue\\n        handles = [swing for swing in bottoms if swing.pivot_index > right.pivot_index]\\n        if not handles:\\n            continue\\n        handle = handles[0]\\n        retrace = (\\n            right.pivot_price - handle.pivot_price\\n            if direction == \\\"bullish\\\"\\n            else handle.pivot_price - right.pivot_price\\n        )\\n        if retrace < HANDLE_RETRACE_MIN * depth or retrace > HANDLE_RETRACE_MAX * depth:\\n            continue\\n        if direction == \\\"bullish\\\" and handle.pivot_price <= bottom.pivot_price:\\n            continue\\n        if direction == \\\"bearish\\\" and handle.pivot_price >= bottom.pivot_price:\\n            continue\\n        upper_points, lower_points = _cup_lines(left, right, bottom, handle, direction)\\n        upper_slope, upper_intercept = _fit_line(upper_points)\\n        lower_slope, lower_intercept = _fit_line(lower_points)\\n        found.append(\\n            _assemble(\\n                frame,\\n                timeframe,\\n                \\\"cup_and_handle\\\",\\n                direction,\\n                (left, bottom, right, handle),\\n                asof_index,\\n                atr,\\n                upper_slope,\\n                lower_slope,\\n                upper_intercept + upper_slope * asof_index,\\n                lower_intercept + lower_slope * asof_index,\\n                depth / max(atr, _EPS),\\n                retrace / max(depth, _EPS),\\n                \\\"rounded_proxy\\\",\\n                gap,\\n            )\\n        )\\n    return found\\n\\n\\ndef _cup_lines(\\n    left: Swing,\\n    right: Swing,\\n    bottom: Swing,\\n    handle: Swing,\\n    direction: str,\\n) -> tuple[list[tuple[int, float]], list[tuple[int, float]]]:\\n    lips = [\\n        (left.pivot_index, left.pivot_price),\\n        (right.pivot_index, right.pivot_price),\\n    ]\\n    bowl = [\\n        (bottom.pivot_index, bottom.pivot_price),\\n        (handle.pivot_index, handle.pivot_price),\\n    ]\\n    if direction == \\\"bullish\\\":\\n        return lips, bowl\\n    return bowl, lips\\n\\n\\ndef _assemble(\\n    frame: pd.DataFrame,\\n    timeframe: str,\\n    pattern: str,\\n    direction: str,\\n    used: Sequence[Swing],\\n    asof_index: int,\\n    atr: float,\\n    upper_slope: float,\\n    lower_slope: float,\\n    upper_at: float,\\n    lower_at: float,\\n    impulse_atr: float,\\n    retrace: float,\\n    geometry_kind: str,\\n    similar_gap: float | None,\\n) -> Candidate:\\n    \\\"\\\"\\\"Build one candidate from boundary lines known at ``asof_index``.\\\"\\\"\\\"\\n    swings = tuple(used)\\n    first_conf = min(swings, key=lambda swing: swing.confirmation_index)\\n    last_conf = max(swings, key=lambda swing: swing.confirmation_index)\\n    first_pivot = min(swing.pivot_index for swing in swings)\\n    last_pivot = max(swing.pivot_index for swing in swings)\\n    width = max(last_pivot - first_pivot, 1)\\n    height_start = _height(upper_at, upper_slope, lower_at, lower_slope, asof_index, first_pivot)\\n    height_end = _height(upper_at, upper_slope, lower_at, lower_slope, asof_index, last_pivot)\\n    raw_change = (height_start - height_end) / max(abs(height_start), _EPS)\\n    high_points = [\\n        (swing.pivot_index, swing.pivot_price)\\n        for swing in swings\\n        if swing.swing_type == \\\"high\\\"\\n    ]\\n    low_points = [\\n        (swing.pivot_index, swing.pivot_price)\\n        for swing in swings\\n        if swing.swing_type == \\\"low\\\"\\n    ]\\n    upper_error, upper_touches = _line_fit(high_points, upper_slope, upper_at, asof_index, atr)\\n    lower_error, lower_touches = _line_fit(low_points, lower_slope, lower_at, asof_index, atr)\\n    close = float(frame[\\\"close\\\"].iloc[asof_index])\\n    span = upper_at - lower_at\\n    if abs(span) > _EPS:\\n        position = (close - lower_at) / span\\n    else:\\n        position = 0.5\\n    contraction, expansion = _volume_ratios(\\n        frame[\\\"volume\\\"].to_numpy(dtype=np.float64),\\n        first_pivot,\\n        last_pivot,\\n    )\\n    last_touch = _last_touch(\\n        high_points,\\n        low_points,\\n        upper_slope,\\n        upper_at,\\n        lower_slope,\\n        lower_at,\\n        asof_index,\\n        atr,\\n        last_pivot,\\n    )\\n    recency = float(np.log1p(max(asof_index - last_touch, 0)) / np.log1p(WINDOW_BARS))\\n    maturity = float(np.clip((asof_index - first_pivot) / width, 0.0, 3.0))\\n    values = {\\n        \\\"timeframe_id\\\": float(TIMEFRAME_ID[timeframe]),\\n        \\\"formation_width_bars\\\": float(width),\\n        \\\"formation_height_atr\\\": abs(span) / max(atr, _EPS),\\n        \\\"upper_boundary_slope\\\": upper_slope / max(atr, _EPS),\\n        \\\"lower_boundary_slope\\\": lower_slope / max(atr, _EPS),\\n        \\\"upper_boundary_distance_atr\\\": (upper_at - close) / max(atr, _EPS),\\n        \\\"lower_boundary_distance_atr\\\": (close - lower_at) / max(atr, _EPS),\\n        \\\"boundary_convergence_ratio\\\": max(0.0, raw_change),\\n        \\\"boundary_divergence_ratio\\\": max(0.0, -raw_change),\\n        \\\"swing_high_count\\\": float(len(high_points)),\\n        \\\"swing_low_count\\\": float(len(low_points)),\\n        \\\"upper_boundary_touches\\\": float(upper_touches),\\n        \\\"lower_boundary_touches\\\": float(lower_touches),\\n        \\\"upper_boundary_error_atr\\\": upper_error,\\n        \\\"lower_boundary_error_atr\\\": lower_error,\\n        \\\"price_position_in_formation\\\": position,\\n        \\\"volume_contraction_ratio\\\": contraction,\\n        \\\"volume_expansion_ratio\\\": expansion,\\n        \\\"bars_since_last_interaction\\\": recency,\\n        \\\"impulse_move_atr\\\": float(impulse_atr),\\n        \\\"retracement_ratio\\\": float(retrace),\\n        \\\"formation_maturity_ratio\\\": maturity,\\n    }\\n    features = np.array(\\n        [values[name] for name in MODEL_CANDIDATE_FEATURES],\\n        dtype=np.float64,\\n    )\\n    features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)\\n    score = _rule_score(upper_touches, lower_touches, upper_error, lower_error, similar_gap, atr)\\n    start_swing = min(swings, key=lambda swing: swing.pivot_index)\\n    return Candidate(\\n        candidate_id=(\\n            f\\\"{timeframe}|{pattern}|{_iso(first_conf.confirmation_time)}|\\\"\\n            f\\\"{_iso(last_conf.confirmation_time)}\\\"\\n        ),\\n        timeframe=timeframe,\\n        pattern=pattern,\\n        direction=direction,\\n        rule_score=score,\\n        geometry_kind=geometry_kind,\\n        swing_confirmation_indices=tuple(\\n            sorted(swing.confirmation_index for swing in swings)\\n        ),\\n        first_confirmation_index=first_conf.confirmation_index,\\n        last_confirmation_index=last_conf.confirmation_index,\\n        first_pivot_index=first_pivot,\\n        last_pivot_index=last_pivot,\\n        upper_price_at_asof=float(upper_at),\\n        lower_price_at_asof=float(lower_at),\\n        upper_slope=float(upper_slope),\\n        lower_slope=float(lower_slope),\\n        atr=float(atr),\\n        origin_close=close,\\n        start_time=start_swing.pivot_time,\\n        asof_time=pd.Timestamp(frame[\\\"time\\\"].iloc[asof_index]),\\n        features=features,\\n    )\\n\\n\\ndef _rule_score(\\n    upper_touches: int,\\n    lower_touches: int,\\n    upper_error: float,\\n    lower_error: float,\\n    similar_gap: float | None,\\n    atr: float,\\n) -> float:\\n    \\\"\\\"\\\"Preliminary score from touches, fit error, and price similarity.\\\"\\\"\\\"\\n    touch_score = min(upper_touches + lower_touches, 6) / 6.0\\n    error_score = math.exp(-0.5 * (upper_error + lower_error))\\n    if similar_gap is None:\\n        gap_score = 1.0\\n    else:\\n        gap_score = 1.0 - min(1.0, similar_gap / max(SIMILAR_PRICE_ATR * atr, _EPS))\\n    return float(np.clip((touch_score + error_score + gap_score) / 3.0, 0.0, 1.0))\\n\\n\\ndef _fit_line(points: Sequence[tuple[int, float]]) -> tuple[float, float]:\\n    \\\"\\\"\\\"Return price-per-bar slope and intercept of ``price = intercept + slope * index``.\\\"\\\"\\\"\\n    if not points:\\n        return 0.0, 0.0\\n    xs = np.array([point[0] for point in points], dtype=np.float64)\\n    ys = np.array([point[1] for point in points], dtype=np.float64)\\n    if len(points) == 1 or np.unique(xs).size < 2:\\n        return 0.0, float(np.mean(ys))\\n    slope, intercept = np.polyfit(xs, ys, 1)\\n    return float(slope), float(intercept)\\n\\n\\ndef _line_fit(\\n    points: Sequence[tuple[int, float]],\\n    slope: float,\\n    price_at_asof: float,\\n    asof_index: int,\\n    atr: float,\\n) -> tuple[float, int]:\\n    \\\"\\\"\\\"Mean absolute residual in ATR, and the number of points within tolerance.\\\"\\\"\\\"\\n    if not points:\\n        return 0.0, 0\\n    errors: list[float] = []\\n    touches = 0\\n    for index, price in points:\\n        predicted = price_at_asof + slope * (index - asof_index)\\n        error = abs(price - predicted)\\n        errors.append(error)\\n        if error <= TOUCH_TOL_ATR * atr:\\n            touches += 1\\n    return float(np.mean(errors) / max(atr, _EPS)), touches\\n\\n\\ndef _height(\\n    upper_at: float,\\n    upper_slope: float,\\n    lower_at: float,\\n    lower_slope: float,\\n    asof_index: int,\\n    index: int,\\n) -> float:\\n    \\\"\\\"\\\"Signed upper-minus-lower distance at ``index``.\\\"\\\"\\\"\\n    upper = upper_at + upper_slope * (index - asof_index)\\n    lower = lower_at + lower_slope * (index - asof_index)\\n    return upper - lower\\n\\n\\ndef _volume_ratios(volume: np.ndarray, first: int, last: int) -> tuple[float, float]:\\n    \\\"\\\"\\\"Return contraction and expansion of the second half versus the first.\\\"\\\"\\\"\\n    if last <= first:\\n        return 1.0, 1.0\\n    mid = first + max((last - first) // 2, 1)\\n    mid = min(mid, last)\\n    first_mean = float(np.mean(volume[first:mid])) if mid > first else float(volume[first])\\n    second_mean = float(np.mean(volume[mid : last + 1]))\\n    expansion = second_mean / first_mean if first_mean > _EPS else 1.0\\n    contraction = first_mean / second_mean if second_mean > _EPS else 1.0\\n    return float(contraction), float(expansion)\\n\\n\\ndef _last_touch(\\n    highs: Sequence[tuple[int, float]],\\n    lows: Sequence[tuple[int, float]],\\n    upper_slope: float,\\n    upper_at: float,\\n    lower_slope: float,\\n    lower_at: float,\\n    asof_index: int,\\n    atr: float,\\n    fallback: int,\\n) -> int:\\n    \\\"\\\"\\\"Latest pivot that sits within the touch tolerance of either boundary.\\\"\\\"\\\"\\n    latest = fallback\\n    groups = (\\n        (highs, upper_slope, upper_at),\\n        (lows, lower_slope, lower_at),\\n    )\\n    for points, slope, price_at in groups:\\n        for index, price in points:\\n            predicted = price_at + slope * (index - asof_index)\\n            if abs(price - predicted) <= TOUCH_TOL_ATR * atr:\\n                latest = max(latest, index)\\n    return latest\\n\\n\\ndef _visible_in_window(swings: Sequence[Swing], asof_index: int) -> list[Swing]:\\n    \\\"\\\"\\\"Confirmed swings whose pivot lies in the 192 bars ending at ``asof_index``.\\\"\\\"\\\"\\n    start = max(0, asof_index - WINDOW_BARS + 1)\\n    chosen = [\\n        swing\\n        for swing in swings\\n        if swing.confirmation_index <= asof_index and swing.pivot_index >= start\\n    ]\\n    chosen.sort(key=lambda swing: (swing.pivot_index, swing.confirmation_index))\\n    return chosen\\n\\n\\ndef _asof_atr(frame: pd.DataFrame, asof_index: int) -> float:\\n    \\\"\\\"\\\"Positive ATR at the as-of bar.\\\"\\\"\\\"\\n    atr = float(frame[\\\"atr\\\"].iloc[asof_index]) if \\\"atr\\\" in frame.columns else np.nan\\n    if np.isfinite(atr) and atr > 0.0:\\n        return atr\\n    return max(abs(float(frame[\\\"close\\\"].iloc[asof_index])) * 0.01, _EPS)\\n\\n\\ndef _price_span(candidate: Candidate) -> tuple[float, float]:\\n    lower = min(candidate.lower_price_at_asof, candidate.upper_price_at_asof)\\n    upper = max(candidate.lower_price_at_asof, candidate.upper_price_at_asof)\\n    return lower, upper\\n\\n\\ndef _ranges_overlap(\\n    left: tuple[float, float] | tuple[pd.Timestamp, pd.Timestamp],\\n    right: tuple[float, float] | tuple[pd.Timestamp, pd.Timestamp],\\n) -> bool:\\n    \\\"\\\"\\\"True when two closed intervals share a point.\\\"\\\"\\\"\\n    return left[0] <= right[1] and right[0] <= left[1]\\n\\n\\ndef _timeframe_order(left: Candidate, right: Candidate) -> tuple[Candidate, Candidate]:\\n    if TIMEFRAME_ID[left.timeframe] <= TIMEFRAME_ID[right.timeframe]:\\n        return left, right\\n    return right, left\\n\\n\\ndef _iso(stamp: pd.Timestamp) -> str:\\n    value = pd.Timestamp(stamp)\\n    if value.tzinfo is None:\\n        value = value.tz_localize(\\\"UTC\\\")\\n    else:\\n        value = value.tz_convert(\\\"UTC\\\")\\n    return value.strftime(\\\"%Y-%m-%dT%H:%M:%SZ\\\")\\n\", \"feature_store/__init__.py\": \"\\\"\\\"\\\"\\nFeature store: canonical feature list, batch pipeline, and live cache.\\n\\nTraining and inference use the same feature logic via:\\n- feature_registry: canonical feature names and schema\\n- feature_pipeline: compute features from OHLCV DataFrame (batch)\\n- feature_cache: rolling candle window for live incremental features\\n\\\"\\\"\\\"\\n\\nfrom feature_store.feature_registry import (\\n    FEATURE_LIST,\\n    EXPECTED_FEATURE_COUNT,\\n    get_feature_list,\\n    get_feature_schema,\\n    validate_feature_count,\\n    validate_feature_order,\\n)\\nfrom feature_store.feature_pipeline import FeaturePipeline, compute_features\\nfrom feature_store.feature_cache import FeatureCache\\nfrom feature_store.drift import (\\n    load_training_stats,\\n    check_drift,\\n    drift_checker_from_schema,\\n)\\n\\n__all__ = [\\n    \\\"FEATURE_LIST\\\",\\n    \\\"EXPECTED_FEATURE_COUNT\\\",\\n    \\\"get_feature_list\\\",\\n    \\\"get_feature_schema\\\",\\n    \\\"validate_feature_count\\\",\\n    \\\"validate_feature_order\\\",\\n    \\\"FeaturePipeline\\\",\\n    \\\"compute_features\\\",\\n    \\\"FeatureCache\\\",\\n    \\\"load_training_stats\\\",\\n    \\\"check_drift\\\",\\n    \\\"drift_checker_from_schema\\\",\\n]\\n\", \"feature_store/feature_registry.py\": \"\\\"\\\"\\\"\\nCanonical feature registry: single source of truth for feature names and order.\\n\\nTraining and inference must use this list and order. Schema is used for\\nfeature_schema.json and drift validation.\\n\\\"\\\"\\\"\\n\\nfrom typing import Any, Dict, List\\n\\n# 50 features: 16 price + 10 momentum + 8 trend + 8 volatility + 6 volume + 2 returns\\n# Canonical source used by both runtime and training paths.\\nFEATURE_LIST: List[str] = [\\n    # Price-based (16)\\n    \\\"sma_10\\\", \\\"sma_20\\\", \\\"sma_50\\\", \\\"sma_100\\\", \\\"sma_200\\\",\\n    \\\"ema_12\\\", \\\"ema_26\\\", \\\"ema_50\\\",\\n    \\\"close_sma_20_ratio\\\", \\\"close_sma_50_ratio\\\", \\\"close_sma_200_ratio\\\",\\n    \\\"high_low_spread\\\", \\\"close_open_ratio\\\", \\\"body_size\\\", \\\"upper_shadow\\\", \\\"lower_shadow\\\",\\n    # Momentum (10)\\n    \\\"rsi_14\\\", \\\"rsi_7\\\", \\\"stochastic_k_14\\\", \\\"stochastic_d_14\\\",\\n    \\\"williams_r_14\\\", \\\"cci_20\\\", \\\"roc_10\\\", \\\"roc_20\\\",\\n    \\\"momentum_10\\\", \\\"momentum_20\\\",\\n    # Trend (8)\\n    \\\"macd\\\", \\\"macd_signal\\\", \\\"macd_histogram\\\",\\n    \\\"adx_14\\\", \\\"aroon_up\\\", \\\"aroon_down\\\", \\\"aroon_oscillator\\\",\\n    \\\"trend_strength\\\",\\n    # Volatility (8)\\n    \\\"bb_upper\\\", \\\"bb_lower\\\", \\\"bb_width\\\", \\\"bb_position\\\",\\n    \\\"atr_14\\\", \\\"atr_20\\\",\\n    \\\"volatility_10\\\", \\\"volatility_20\\\",\\n    # Volume (6)\\n    \\\"volume_sma_20\\\", \\\"volume_ratio\\\", \\\"obv\\\",\\n    \\\"volume_price_trend\\\", \\\"accumulation_distribution\\\", \\\"chaikin_oscillator\\\",\\n    # Returns (2)\\n    \\\"returns_1h\\\", \\\"returns_24h\\\",\\n]\\n\\nEXPECTED_FEATURE_COUNT: int = len(FEATURE_LIST)\\n\\n# Candlestick pattern features (~40)\\nCANDLESTICK_FEATURES: List[str] = [\\n    \\\"cdl_doji\\\", \\\"cdl_long_legged_doji\\\", \\\"cdl_dragonfly_doji\\\", \\\"cdl_gravestone_doji\\\",\\n    \\\"cdl_hammer\\\", \\\"cdl_inv_hammer\\\", \\\"cdl_hanging_man\\\", \\\"cdl_shooting_star\\\",\\n    \\\"cdl_bull_marubozu\\\", \\\"cdl_bear_marubozu\\\", \\\"cdl_spinning_top\\\",\\n    \\\"cdl_bull_engulfing\\\", \\\"cdl_bear_engulfing\\\", \\\"cdl_bull_harami\\\", \\\"cdl_bear_harami\\\",\\n    \\\"cdl_piercing\\\", \\\"cdl_dark_cloud\\\", \\\"cdl_tweezer_top\\\", \\\"cdl_tweezer_bottom\\\",\\n    \\\"cdl_bull_kicker\\\", \\\"cdl_bear_kicker\\\",\\n    \\\"cdl_morning_star\\\", \\\"cdl_evening_star\\\", \\\"cdl_three_white_soldiers\\\",\\n    \\\"cdl_three_black_crows\\\", \\\"cdl_three_inside_up\\\", \\\"cdl_three_inside_down\\\",\\n    \\\"cdl_abandoned_baby_bull\\\", \\\"cdl_abandoned_baby_bear\\\",\\n    \\\"cdl_bull_score\\\", \\\"cdl_bear_score\\\", \\\"cdl_net_score\\\", \\\"cdl_reversal_signal\\\",\\n    \\\"cdl_indecision_score\\\", \\\"cdl_body_ratio\\\", \\\"cdl_upper_wick_ratio\\\",\\n    \\\"cdl_lower_wick_ratio\\\", \\\"cdl_consecutive_bull\\\", \\\"cdl_consecutive_bear\\\",\\n]\\n\\n# Chart pattern features (~32)\\nCHART_PATTERN_FEATURES: List[str] = [\\n    \\\"sr_support_dist_pct\\\", \\\"sr_resistance_dist_pct\\\",\\n    \\\"sr_at_support\\\", \\\"sr_at_resistance\\\",\\n    \\\"sr_support_strength\\\", \\\"sr_resistance_strength\\\", \\\"sr_range_position\\\",\\n    \\\"tl_uptrend_detected\\\", \\\"tl_downtrend_detected\\\", \\\"tl_trend_slope\\\",\\n    \\\"tl_dist_to_trendline\\\", \\\"tl_near_trendline\\\", \\\"tl_breakout_up\\\", \\\"tl_breakout_down\\\",\\n    \\\"chp_bull_flag\\\", \\\"chp_bear_flag\\\", \\\"chp_bull_flag_strength\\\", \\\"chp_bear_flag_strength\\\",\\n    \\\"chp_asc_triangle\\\", \\\"chp_desc_triangle\\\", \\\"chp_sym_triangle\\\", \\\"chp_triangle_apex_dist\\\",\\n    \\\"chp_double_top\\\", \\\"chp_double_bottom\\\", \\\"chp_double_top_dist\\\", \\\"chp_double_bottom_dist\\\",\\n    \\\"chp_hs_detected\\\", \\\"chp_ihs_detected\\\",\\n    \\\"bo_at_high\\\", \\\"bo_at_low\\\", \\\"bo_volume_confirmation\\\", \\\"bo_breakout_score\\\",\\n]\\n\\n# Multi-timeframe context (derived from primary OHLCV via resample; 5m primary in production).\\n# Appended after patterns for backward-compatible ordering of base blocks.\\nMTF_CONTEXT_FEATURES: List[str] = [\\n    \\\"mtf_15m_rsi_14\\\",\\n    \\\"mtf_15m_ema_12\\\",\\n    \\\"mtf_3m_rsi_14\\\",\\n    \\\"mtf_3m_ema_12\\\",\\n    \\\"mtf_1m_vol_ratio\\\",\\n]\\n\\n# Regime-awareness features produced by notebook/live feature layer.\\nREGIME_FEATURES: List[str] = [\\n    \\\"regime_state\\\",\\n    \\\"regime_is_ranging\\\",\\n    \\\"regime_is_trending\\\",\\n    \\\"regime_is_volatile\\\",\\n]\\n\\n# Perpetual futures-specific features (non-canonical but included for v6 futures models)\\nPERPETUAL_FEATURES: List[str] = [\\n    \\\"basis\\\", \\\"basis_pct\\\", \\\"basis_zscore\\\", \\\"basis_direction\\\",\\n    \\\"funding_rate\\\", \\\"funding_cumsum3\\\", \\\"funding_zscore\\\",\\n    \\\"funding_spike\\\", \\\"funding_sign\\\",\\n    \\\"oi\\\", \\\"oi_change_pct\\\", \\\"oi_change_5\\\", \\\"oi_price_confirm\\\", \\\"oi_zscore\\\",\\n    \\\"long_squeeze_risk\\\", \\\"short_squeeze_risk\\\",\\n    \\\"ob_imbalance\\\",\\n]\\n\\n# Expanded: canonical + candlestick + chart patterns + MTF context + regime flags + perpetual features.\\nEXPANDED_FEATURE_LIST: List[str] = (\\n    FEATURE_LIST\\n    + CANDLESTICK_FEATURES\\n    + CHART_PATTERN_FEATURES\\n    + MTF_CONTEXT_FEATURES\\n    + REGIME_FEATURES\\n    + PERPETUAL_FEATURES\\n)\\n\\nFEATURE_VERSION = \\\"1.2\\\"\\n\\n\\ndef get_feature_list() -> List[str]:\\n    \\\"\\\"\\\"Return the canonical feature list (copy).\\\"\\\"\\\"\\n    return list(FEATURE_LIST)\\n\\n\\ndef validate_feature_count(features: List[str]) -> bool:\\n    \\\"\\\"\\\"Return True if feature count matches canonical.\\\"\\\"\\\"\\n    return len(features) == EXPECTED_FEATURE_COUNT\\n\\n\\ndef validate_feature_order(feature_names: List[str]) -> bool:\\n    \\\"\\\"\\\"Return True if names match canonical order.\\\"\\\"\\\"\\n    if len(feature_names) != EXPECTED_FEATURE_COUNT:\\n        return False\\n    return all(a == b for a, b in zip(FEATURE_LIST, feature_names))\\n\\n\\ndef get_feature_schema(\\n    include_stats: bool = False,\\n    training_stats: Dict[str, Dict[str, float]] | None = None,\\n) -> Dict[str, Any]:\\n    \\\"\\\"\\\"\\n    Return schema dict for feature_schema.json.\\n    If include_stats and training_stats are provided, add mean/std for drift baseline.\\n    \\\"\\\"\\\"\\n    schema: Dict[str, Any] = {\\n        \\\"version\\\": FEATURE_VERSION,\\n        \\\"feature_count\\\": EXPECTED_FEATURE_COUNT,\\n        \\\"features\\\": FEATURE_LIST,\\n    }\\n    if include_stats and training_stats:\\n        schema[\\\"training_stats\\\"] = training_stats\\n    return schema\\n\", \"feature_store/feature_pipeline.py\": \"\\\"\\\"\\\"\\nBatch feature pipeline: identical logic for training and live inference.\\n\\nDelegates to UnifiedFeatureEngine for single source of truth.\\n\\\"\\\"\\\"\\n\\nfrom typing import Optional\\n\\nimport pandas as pd\\n\\nfrom feature_store.feature_registry import FEATURE_LIST, EXPECTED_FEATURE_COUNT\\nfrom feature_store.unified_feature_engine import UnifiedFeatureEngine\\n\\n\\ndef compute_features(\\n    df: pd.DataFrame,\\n    resolution_minutes: int = 15,\\n    fill_invalid: bool = True,\\n) -> pd.DataFrame:\\n    \\\"\\\"\\\"\\n    Compute canonical 50 features from OHLCV DataFrame.\\n    Columns required: open, high, low, close, volume.\\n    Optional: timestamp (ignored).\\n    \\\"\\\"\\\"\\n    engine = UnifiedFeatureEngine()\\n    return engine.compute_batch(\\n        df,\\n        resolution_minutes=resolution_minutes,\\n        fill_invalid=fill_invalid,\\n    )\\n\\n\\nclass FeaturePipeline:\\n    \\\"\\\"\\\"\\n    Pipeline that computes canonical features from OHLCV.\\n    Optionally validates output count/order.\\n    \\\"\\\"\\\"\\n\\n    def __init__(\\n        self,\\n        resolution_minutes: int = 15,\\n        fill_invalid: bool = True,\\n        validate: bool = True,\\n    ):\\n        self.resolution_minutes = resolution_minutes\\n        self.fill_invalid = fill_invalid\\n        self.validate = validate\\n        self._engine = UnifiedFeatureEngine()\\n\\n    def transform(self, df: pd.DataFrame) -> pd.DataFrame:\\n        feat = self._engine.compute_batch(\\n            df,\\n            resolution_minutes=self.resolution_minutes,\\n            fill_invalid=self.fill_invalid,\\n        )\\n        if self.validate:\\n            if list(feat.columns) != FEATURE_LIST:\\n                raise ValueError(\\\"Feature order mismatch\\\")\\n            if len(feat.columns) != EXPECTED_FEATURE_COUNT:\\n                raise ValueError(\\n                    f\\\"Feature count: expected {EXPECTED_FEATURE_COUNT}, \\\"\\n                    f\\\"got {len(feat.columns)}\\\"\\n                )\\n        return feat\\n\\n    def fit_transform(self, df: pd.DataFrame) -> pd.DataFrame:\\n        return self.transform(df)\\n\", \"feature_store/feature_cache.py\": \"\\\"\\\"\\\"\\nLive feature cache: rolling window of candles for incremental feature computation.\\n\\nThe agent maintains this cache and passes the window to FeaturePipeline\\nso live inference uses the same feature logic as training.\\n\\\"\\\"\\\"\\n\\nfrom collections import deque\\nfrom typing import Any, Dict, List, Optional\\n\\nimport pandas as pd\\n\\nfrom feature_store.feature_pipeline import FeaturePipeline, compute_features\\nfrom feature_store.feature_registry import FEATURE_LIST, EXPECTED_FEATURE_COUNT\\n\\n\\nclass FeatureCache:\\n    \\\"\\\"\\\"\\n    Rolling window of OHLCV candles. Call update(candle) on each new candle,\\n    then dataframe() to get the window and compute_features() for the last row.\\n    \\\"\\\"\\\"\\n\\n    def __init__(\\n        self,\\n        maxlen: int = 500,\\n        resolution_minutes: int = 15,\\n    ):\\n        self._maxlen = maxlen\\n        self._resolution_minutes = resolution_minutes\\n        self._deque: deque = deque(maxlen=maxlen)\\n        self._pipeline = FeaturePipeline(\\n            resolution_minutes=resolution_minutes,\\n            fill_invalid=True,\\n            validate=True,\\n        )\\n\\n    def update(self, candle: Dict[str, Any]) -> None:\\n        \\\"\\\"\\\"Append one candle. Expect keys: open, high, low, close, volume; optional timestamp.\\\"\\\"\\\"\\n        self._deque.append(dict(candle))\\n\\n    def extend(self, candles: List[Dict[str, Any]]) -> None:\\n        \\\"\\\"\\\"Append multiple candles.\\\"\\\"\\\"\\n        for c in candles:\\n            self.update(c)\\n\\n    def dataframe(self) -> pd.DataFrame:\\n        \\\"\\\"\\\"Return current window as DataFrame with columns open, high, low, close, volume.\\\"\\\"\\\"\\n        if not self._deque:\\n            return pd.DataFrame(columns=[\\\"open\\\", \\\"high\\\", \\\"low\\\", \\\"close\\\", \\\"volume\\\"])\\n        return pd.DataFrame(list(self._deque))\\n\\n    def get_latest_features(self) -> Optional[pd.DataFrame]:\\n        \\\"\\\"\\\"\\n        Compute features for the full window and return the last row as (1, 50) DataFrame.\\n        Returns None if window has too few rows for feature computation.\\n        \\\"\\\"\\\"\\n        df = self.dataframe()\\n        if df.empty or len(df) < 30:\\n            return None\\n        feat = self._pipeline.transform(df)\\n        return feat.iloc[[-1]]\\n\\n    def get_latest_feature_vector(self) -> Optional[List[float]]:\\n        \\\"\\\"\\\"Return the last row of features as a list of 50 floats, or None.\\\"\\\"\\\"\\n        latest = self.get_latest_features()\\n        if latest is None or latest.empty:\\n            return None\\n        return latest.iloc[0].tolist()\\n\\n    @property\\n    def expected_feature_count(self) -> int:\\n        return EXPECTED_FEATURE_COUNT\\n\\n    @property\\n    def feature_names(self) -> List[str]:\\n        return list(FEATURE_LIST)\\n\\n    def __len__(self) -> int:\\n        return len(self._deque)\\n\", \"feature_store/drift.py\": \"\\\"\\\"\\\"\\nLive feature drift monitoring against training baseline.\\n\\nLoad training_stats from feature_schema.json (or training_metadata.json)\\nand check live feature vectors for drift beyond a sigma threshold.\\n\\\"\\\"\\\"\\n\\nimport json\\nfrom pathlib import Path\\nfrom typing import Dict, List, Optional, Union\\n\\n\\ndef load_training_stats(path: Union[str, Path]) -> Dict[str, Dict[str, float]]:\\n    \\\"\\\"\\\"\\n    Load training feature statistics (mean/std) from feature_schema.json\\n    or training_metadata.json. Returns dict mapping feature name -> {mean, std}.\\n    \\\"\\\"\\\"\\n    path = Path(path)\\n    if not path.exists():\\n        return {}\\n    try:\\n        data = json.loads(path.read_text(encoding=\\\"utf-8\\\"))\\n    except Exception:\\n        return {}\\n    stats = data.get(\\\"training_stats\\\") or data.get(\\\"feature_drift_stats\\\")\\n    if isinstance(stats, dict) and stats:\\n        return stats\\n    # feature_drift_{tag}.json has \\\"features\\\" key\\n    if \\\"features\\\" in data and isinstance(data[\\\"features\\\"], dict):\\n        return data[\\\"features\\\"]\\n    return {}\\n\\n\\ndef check_drift(\\n    feature_vector: List[float],\\n    feature_names: List[str],\\n    training_stats: Dict[str, Dict[str, float]],\\n    threshold_sigma: float = 4.0,\\n) -> List[str]:\\n    \\\"\\\"\\\"\\n    Return names of features that exceed the drift threshold (live value vs\\n    training mean in units of training std).\\n    \\\"\\\"\\\"\\n    drifted: List[str] = []\\n    for name, val in zip(feature_names, feature_vector):\\n        stat = training_stats.get(name)\\n        if stat is None:\\n            continue\\n        mean = stat.get(\\\"mean\\\", 0.0)\\n        std = stat.get(\\\"std\\\", 0.0)\\n        if std < 1e-12:\\n            continue\\n        z = abs((float(val) - mean) / std)\\n        if z > threshold_sigma:\\n            drifted.append(name)\\n    return drifted\\n\\n\\ndef drift_checker_from_schema(\\n    schema_path: Union[str, Path],\\n    feature_names: Optional[List[str]] = None,\\n    threshold_sigma: float = 4.0,\\n):\\n    \\\"\\\"\\\"\\n    Return a callable (feature_vector) -> List[str] that checks drift using\\n    stats loaded from schema_path. If feature_names is None, use the\\n    feature list from the schema.\\n    \\\"\\\"\\\"\\n    stats = load_training_stats(schema_path)\\n    if not stats:\\n        return lambda _: []\\n    names = feature_names or list(stats.keys())\\n    return lambda vec: check_drift(vec, names, stats, threshold_sigma)\\n\\n\\n# Pattern feature prefixes for activation rate monitoring\\nPATTERN_FEATURE_PREFIXES = (\\\"cdl_\\\", \\\"chp_\\\", \\\"sr_\\\", \\\"tl_\\\", \\\"bo_\\\")\\n\\n\\ndef get_pattern_feature_activation_rates(\\n    feature_vectors: List[List[float]],\\n    feature_names: List[str],\\n) -> Dict[str, float]:\\n    \\\"\\\"\\\"\\n    Compute activation rate (fraction of 1s) for binary pattern features\\n    across a batch of feature vectors. Used to track live distribution\\n    vs training baseline.\\n    \\\"\\\"\\\"\\n    if not feature_vectors:\\n        return {}\\n    n = len(feature_vectors)\\n    sums: Dict[str, float] = {}\\n    for vec in feature_vectors:\\n        for name, val in zip(feature_names, vec):\\n            if any(name.startswith(p) for p in PATTERN_FEATURE_PREFIXES):\\n                sums[name] = sums.get(name, 0) + (1.0 if float(val) > 0.5 else 0.0)\\n    return {k: v / n for k, v in sums.items()}\\n\\n\\ndef check_pattern_activation_drift(\\n    live_rates: Dict[str, float],\\n    training_rates: Dict[str, float],\\n    threshold: float = 0.15,\\n) -> List[str]:\\n    \\\"\\\"\\\"\\n    Return pattern features whose live activation rate diverges from training\\n    by more than threshold (absolute difference).\\n    \\\"\\\"\\\"\\n    drifted = []\\n    for name, live_val in live_rates.items():\\n        train_val = training_rates.get(name)\\n        if train_val is None:\\n            continue\\n        if abs(live_val - train_val) > threshold:\\n            drifted.append(name)\\n    return drifted\\n\", \"feature_store/perpetual_features.py\": \"\\\"\\\"\\\"\\nperpetual_features.py\\nComputes futures-specific derived features from the merged feature matrix.\\nAll features are designed to be train/serve consistent.\\n\\\"\\\"\\\"\\nimport numpy as np\\nimport pandas as pd\\n\\n\\ndef compute_perpetual_features(df: pd.DataFrame) -> pd.DataFrame:\\n    out = pd.DataFrame(index=df.index)\\n\\n    # \\u2500\\u2500 Basis Features \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    out[\\\"basis\\\"] = df[\\\"mark_price\\\"] - df[\\\"close\\\"]\\n    out[\\\"basis_pct\\\"] = out[\\\"basis\\\"] / df[\\\"close\\\"].replace(0, np.nan)\\n    out[\\\"basis_zscore\\\"] = (\\n        (out[\\\"basis_pct\\\"] - out[\\\"basis_pct\\\"].rolling(48).mean())\\n        / out[\\\"basis_pct\\\"].rolling(48).std().replace(0, np.nan)\\n    )\\n    out[\\\"basis_direction\\\"] = np.sign(out[\\\"basis\\\"])\\n\\n    # \\u2500\\u2500 Funding Rate Features \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    out[\\\"funding_rate\\\"] = df[\\\"funding_rate\\\"].fillna(0.0)\\n    out[\\\"funding_cumsum_3\\\"] = out[\\\"funding_rate\\\"].rolling(3).sum()\\n    funding_mean = out[\\\"funding_rate\\\"].rolling(168).mean()\\n    funding_std = out[\\\"funding_rate\\\"].rolling(168).std().replace(0, np.nan)\\n    out[\\\"funding_zscore\\\"] = (out[\\\"funding_rate\\\"] - funding_mean) / funding_std\\n    out[\\\"funding_spike\\\"] = (out[\\\"funding_zscore\\\"].abs() > 2.0).astype(float)\\n    out[\\\"funding_sign\\\"] = np.sign(out[\\\"funding_rate\\\"])\\n\\n    # \\u2500\\u2500 Open Interest Features \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    oi = df[\\\"open_interest\\\"].ffill().fillna(0.0)\\n    out[\\\"oi\\\"] = oi\\n    out[\\\"oi_change_pct\\\"] = oi.pct_change().fillna(0.0)\\n    out[\\\"oi_change_5\\\"] = oi.pct_change(5).fillna(0.0)\\n    price_dir = np.sign(df[\\\"close\\\"].diff().fillna(0.0))\\n    oi_dir = np.sign(oi.diff().fillna(0.0))\\n    out[\\\"oi_price_confirm\\\"] = (price_dir == oi_dir).astype(float) * 2 - 1\\n    oi_mean = oi.rolling(336).mean()\\n    oi_std = oi.rolling(336).std().replace(0, np.nan)\\n    out[\\\"oi_zscore\\\"] = (oi - oi_mean) / oi_std\\n\\n    # \\u2500\\u2500 Squeeze Risk \\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\u2500\\n    out[\\\"long_squeeze_risk\\\"] = (\\n        (out[\\\"funding_sign\\\"] > 0) &\\n        (out[\\\"funding_spike\\\"] > 0) &\\n        (out[\\\"oi_change_pct\\\"] < -0.01)\\n    ).astype(float)\\n\\n    out[\\\"short_squeeze_risk\\\"] = (\\n        (out[\\\"funding_sign\\\"] < 0) &\\n        (out[\\\"funding_spike\\\"] > 0) &\\n        (out[\\\"oi_change_pct\\\"] < -0.01)\\n    ).astype(float)\\n\\n    return out.replace([np.inf, -np.inf], 0.0).fillna(0.0)\\n\\n\\nPERPETUAL_FEATURE_NAMES = [\\n    \\\"basis\\\", \\\"basis_pct\\\", \\\"basis_zscore\\\", \\\"basis_direction\\\",\\n    \\\"funding_rate\\\", \\\"funding_cumsum_3\\\", \\\"funding_zscore\\\",\\n    \\\"funding_spike\\\", \\\"funding_sign\\\",\\n    \\\"oi\\\", \\\"oi_change_pct\\\", \\\"oi_change_5\\\", \\\"oi_price_confirm\\\", \\\"oi_zscore\\\",\\n    \\\"long_squeeze_risk\\\", \\\"short_squeeze_risk\\\",\\n    \\\"ob_imbalance\\\",\\n]\\n\", \"feature_store/unified_feature_engine.py\": \"\\\"\\\"\\\"\\nUnified feature computation engine: single source of truth for all features.\\n\\nBoth training (batch) and live (single) paths MUST use this class to ensure\\ntrain-serve parity. No separate implementations elsewhere.\\n\\nTrain-serve parity for the canonical feature registry. Transformer models use\\n``feature_store/transformer_btcusd/`` instead.\\n\\\"\\\"\\\"\\n\\nfrom typing import List, Optional\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom feature_store.feature_registry import (\\n    CANDLESTICK_FEATURES,\\n    CHART_PATTERN_FEATURES,\\n    EXPECTED_FEATURE_COUNT,\\n    FEATURE_LIST,\\n    REGIME_FEATURES,\\n    MTF_CONTEXT_FEATURES,\\n)\\nfrom feature_store.perpetual_features import (\\n    compute_perpetual_features,\\n    PERPETUAL_FEATURE_NAMES,\\n)\\nfrom feature_store.pattern_features.candlestick_patterns import CandlestickPatternEngine\\nfrom feature_store.pattern_features.chart_patterns import ChartPatternEngine\\n\\n# V4 model extras (not in canonical 50 but used by jacksparrow_v4)\\nV4_EXTRA_FEATURES = [\\n    \\\"ema_9\\\",\\n    \\\"ema_21\\\",\\n    \\\"macd_hist\\\",  # alias for macd_histogram\\n    \\\"bb_pct\\\",  # alias for bb_position\\n    \\\"vol_zscore\\\",\\n    \\\"vol_ratio\\\",\\n    \\\"ema_cross\\\",\\n    \\\"returns_1\\\",\\n    \\\"atr_pct\\\",\\n]\\nFEATURE_ALIASES = {\\n    \\\"macd_hist\\\": \\\"macd_histogram\\\",\\n    \\\"bb_pct\\\": \\\"bb_position\\\",\\n    \\\"bb_pos\\\": \\\"bb_position\\\",\\n    \\\"vol_ratio\\\": \\\"volume_ratio\\\",\\n    \\\"volatility\\\": \\\"volatility_20\\\",\\n}\\n\\n\\ndef _ema_series(s: pd.Series, period: int) -> pd.Series:\\n    \\\"\\\"\\\"Exponential moving average.\\\"\\\"\\\"\\n    return s.ewm(span=period, adjust=False).mean()\\n\\n\\ndef _rsi_14_from_close(c: pd.Series) -> pd.Series:\\n    \\\"\\\"\\\"RSI(14) on close; matches rolling-mean variant used in compute_batch.\\\"\\\"\\\"\\n    delta = c.diff()\\n    gain = delta.where(delta > 0, 0.0)\\n    loss = (-delta).where(delta < 0, 0.0)\\n    avg_gain = gain.rolling(14, min_periods=1).mean()\\n    avg_loss = loss.rolling(14, min_periods=1).mean()\\n    rs = avg_gain / avg_loss.replace(0, np.nan)\\n    return 100 - (100 / (1 + rs))\\n\\n\\ndef _ensure_timestamp_series(df: pd.DataFrame) -> pd.Series:\\n    \\\"\\\"\\\"Return UTC datetime series aligned to df rows (from timestamp or time).\\\"\\\"\\\"\\n    if \\\"timestamp\\\" in df.columns:\\n        ts = pd.to_datetime(df[\\\"timestamp\\\"], utc=True)\\n    elif \\\"time\\\" in df.columns:\\n        ts = pd.to_datetime(df[\\\"time\\\"], unit=\\\"s\\\", utc=True)\\n    else:\\n        raise ValueError(\\\"DataFrame must have 'timestamp' or 'time' for MTF context\\\")\\n    return ts\\n\\n\\ndef _compute_regime_series(\\n    df: pd.DataFrame,\\n    atr_window: int = 14,\\n    adx_window: int = 14,\\n) -> pd.Series:\\n    \\\"\\\"\\\"Compute regime label: 0=ranging, 1=trending, 2=volatile.\\\"\\\"\\\"\\n    tr = pd.concat(\\n        [\\n            df[\\\"high\\\"] - df[\\\"low\\\"],\\n            (df[\\\"high\\\"] - df[\\\"close\\\"].shift()).abs(),\\n            (df[\\\"low\\\"].shift() - df[\\\"low\\\"]).abs(),\\n        ],\\n        axis=1,\\n    ).max(axis=1)\\n    atr = tr.rolling(atr_window).mean()\\n\\n    dm_pos = (df[\\\"high\\\"] - df[\\\"high\\\"].shift()).clip(lower=0)\\n    dm_neg = (df[\\\"low\\\"].shift() - df[\\\"low\\\"]).clip(lower=0)\\n\\n    di_pos = 100 * (dm_pos.rolling(adx_window).mean() / atr)\\n    di_neg = 100 * (dm_neg.rolling(adx_window).mean() / atr)\\n    dx = 100 * (di_pos - di_neg).abs() / (di_pos + di_neg)\\n    adx = dx.rolling(adx_window).mean()\\n\\n    atr_pct = atr / df[\\\"close\\\"]\\n    atr_rank = atr_pct.rolling(200).rank(pct=True)\\n\\n    regime = pd.Series(1, index=df.index)\\n    regime[adx < 25] = 0\\n    regime[atr_rank > 0.85] = 2\\n    return regime.fillna(1).astype(int)\\n\\n\\nclass UnifiedFeatureEngine:\\n    \\\"\\\"\\\"\\n    Single source of truth for all feature computation.\\n    Both live and training MUST use this class.\\n    \\\"\\\"\\\"\\n\\n    SUPPORTED_FEATURES = (\\n        list(FEATURE_LIST)\\n        + list({f for f in V4_EXTRA_FEATURES if f not in FEATURE_LIST})\\n        + list(CANDLESTICK_FEATURES)\\n        + list(CHART_PATTERN_FEATURES)\\n        + list(MTF_CONTEXT_FEATURES)\\n        + list(REGIME_FEATURES)\\n        + list(PERPETUAL_FEATURE_NAMES)\\n    )\\n\\n    def __init__(self):\\n        self._cdl_engine = CandlestickPatternEngine()\\n        self._chp_engine = ChartPatternEngine()\\n\\n    def _mtf_context_from_primary(\\n        self, df: pd.DataFrame, resolution_minutes: int\\n    ) -> pd.DataFrame:\\n        \\\"\\\"\\\"\\n        Multi-timeframe context on the primary bar index: resample OHLCV to 3m/15m,\\n        compute RSI/EMA on resampled closes, forward-fill to primary timestamps.\\n        mtf_1m_vol_ratio is a short-horizon volume spike proxy on the primary grid.\\n        When resolution is not 5m, returns zeros (caller should skip MTF in training).\\n        \\\"\\\"\\\"\\n        n = len(df)\\n        zero = pd.DataFrame(\\n            {name: np.zeros(n, dtype=float) for name in MTF_CONTEXT_FEATURES}\\n        )\\n        if resolution_minutes != 5:\\n            return zero\\n        try:\\n            ts = _ensure_timestamp_series(df)\\n        except ValueError:\\n            return zero\\n\\n        work = pd.DataFrame(\\n            {\\n                \\\"timestamp\\\": ts,\\n                \\\"open\\\": df[\\\"open\\\"].astype(float),\\n                \\\"high\\\": df[\\\"high\\\"].astype(float),\\n                \\\"low\\\": df[\\\"low\\\"].astype(float),\\n                \\\"close\\\": df[\\\"close\\\"].astype(float),\\n                \\\"volume\\\": df[\\\"volume\\\"].astype(float),\\n            }\\n        )\\n        work = work.set_index(\\\"timestamp\\\").sort_index()\\n        idx = work.index\\n        out = pd.DataFrame(index=idx)\\n        for rule, prefix in [(\\\"3min\\\", \\\"mtf_3m\\\"), (\\\"15min\\\", \\\"mtf_15m\\\")]:\\n            r = (\\n                work.resample(rule, label=\\\"right\\\", closed=\\\"right\\\")\\n                .agg(\\n                    {\\n                        \\\"open\\\": \\\"first\\\",\\n                        \\\"high\\\": \\\"max\\\",\\n                        \\\"low\\\": \\\"min\\\",\\n                        \\\"close\\\": \\\"last\\\",\\n                        \\\"volume\\\": \\\"sum\\\",\\n                    }\\n                )\\n                .dropna(how=\\\"any\\\")\\n            )\\n            c = r[\\\"close\\\"].astype(float)\\n            rsi = _rsi_14_from_close(c).reindex(idx, method=\\\"ffill\\\")\\n            ema12 = _ema_series(c, 12).reindex(idx, method=\\\"ffill\\\")\\n            out[f\\\"{prefix}_rsi_14\\\"] = rsi\\n            out[f\\\"{prefix}_ema_12\\\"] = ema12\\n\\n        v = work[\\\"volume\\\"].astype(float)\\n        vm = v.rolling(3, min_periods=1).mean()\\n        out[\\\"mtf_1m_vol_ratio\\\"] = v / vm.replace(0, np.nan)\\n\\n        out = out.reindex(columns=list(MTF_CONTEXT_FEATURES))\\n        out = out.fillna(0).replace([np.inf, -np.inf], 0)\\n        return out.reset_index(drop=True)\\n\\n    def compute_batch(\\n        self,\\n        df: pd.DataFrame,\\n        resolution_minutes: int = 15,\\n        fill_invalid: bool = True,\\n        include_pattern_features: bool = False,\\n        include_mtf_context: bool = False,\\n    ) -> pd.DataFrame:\\n        \\\"\\\"\\\"\\n        Compute canonical 50 features from OHLCV DataFrame.\\n        Used during training. Returns full feature matrix.\\n        \\\"\\\"\\\"\\n        o = df[\\\"open\\\"].astype(float)\\n        h = df[\\\"high\\\"].astype(float)\\n        l = df[\\\"low\\\"].astype(float)\\n        c = df[\\\"close\\\"].astype(float)\\n        v = df[\\\"volume\\\"].astype(float)\\n        out: dict[str, pd.Series] = {}\\n\\n        # ---- Price ----\\n        for period in [10, 20, 50, 100, 200]:\\n            key = f\\\"sma_{period}\\\"\\n            out[key] = c.rolling(period, min_periods=1).mean()\\n        for period in [12, 26, 50]:\\n            key = f\\\"ema_{period}\\\"\\n            out[key] = _ema_series(c, period)\\n        for period in [20, 50, 200]:\\n            sma = c.rolling(period, min_periods=1).mean()\\n            out[f\\\"close_sma_{period}_ratio\\\"] = c / sma.replace(0, np.nan).ffill().bfill()\\n        out[\\\"high_low_spread\\\"] = (h - l) / l.replace(0, np.nan)\\n        out[\\\"close_open_ratio\\\"] = c / o.replace(0, np.nan)\\n        out[\\\"body_size\\\"] = (c - o).abs() / o.replace(0, np.nan)\\n        body_top = np.maximum(c, o)\\n        body_bot = np.minimum(c, o)\\n        out[\\\"upper_shadow\\\"] = (h - body_top) / body_top.replace(0, np.nan)\\n        out[\\\"lower_shadow\\\"] = (body_bot - l) / body_bot.replace(0, np.nan)\\n\\n        # ---- Momentum ----\\n        delta = c.diff()\\n        gain = delta.where(delta > 0, 0.0)\\n        loss = (-delta).where(delta < 0, 0.0)\\n        for period in [14, 7]:\\n            avg_gain = gain.rolling(period, min_periods=1).mean()\\n            avg_loss = loss.rolling(period, min_periods=1).mean()\\n            rs = avg_gain / avg_loss.replace(0, np.nan)\\n            out[f\\\"rsi_{period}\\\"] = 100 - (100 / (1 + rs))\\n        hh = h.rolling(14, min_periods=1).max()\\n        ll = l.rolling(14, min_periods=1).min()\\n        out[\\\"stochastic_k_14\\\"] = 100 * (c - ll) / (hh - ll).replace(0, np.nan)\\n        out[\\\"stochastic_d_14\\\"] = out[\\\"stochastic_k_14\\\"].rolling(3, min_periods=1).mean()\\n        out[\\\"williams_r_14\\\"] = -100 * (hh - c) / (hh - ll).replace(0, np.nan)\\n        tp = (h + l + c) / 3\\n        sma_tp = tp.rolling(20, min_periods=1).mean()\\n        mad = tp.rolling(20, min_periods=1).apply(\\n            lambda x: np.abs(x - x.mean()).mean(), raw=True\\n        )\\n        out[\\\"cci_20\\\"] = (tp - sma_tp) / (0.015 * mad.replace(0, np.nan))\\n        out[\\\"roc_10\\\"] = (c - c.shift(10)) / c.shift(10).replace(0, np.nan) * 100\\n        out[\\\"roc_20\\\"] = (c - c.shift(20)) / c.shift(20).replace(0, np.nan) * 100\\n        out[\\\"momentum_10\\\"] = c - c.shift(10)\\n        out[\\\"momentum_20\\\"] = c - c.shift(20)\\n\\n        # ---- Trend ----\\n        ema12 = _ema_series(c, 12)\\n        ema26 = _ema_series(c, 26)\\n        macd_line = ema12 - ema26\\n        out[\\\"macd\\\"] = macd_line\\n        out[\\\"macd_signal\\\"] = _ema_series(macd_line, 9)\\n        out[\\\"macd_histogram\\\"] = macd_line - out[\\\"macd_signal\\\"]\\n        tr = np.maximum(\\n            h - l,\\n            np.maximum((h - c.shift(1)).abs(), (l - c.shift(1)).abs()),\\n        )\\n        plus_dm = (h - h.shift(1)).where(\\n            (h - h.shift(1)) > (l.shift(1) - l), 0\\n        ).clip(lower=0)\\n        minus_dm = (l.shift(1) - l).where(\\n            (l.shift(1) - l) > (h - h.shift(1)), 0\\n        ).clip(lower=0)\\n        atr_14 = tr.rolling(14, min_periods=1).mean()\\n        plus_di = 100 * plus_dm.rolling(14, min_periods=1).mean() / atr_14.replace(\\n            0, np.nan\\n        )\\n        minus_di = 100 * minus_dm.rolling(14, min_periods=1).mean() / atr_14.replace(\\n            0, np.nan\\n        )\\n        dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)\\n        out[\\\"adx_14\\\"] = dx.rolling(14, min_periods=1).mean()\\n        aroon_period = 14\\n        out[\\\"aroon_up\\\"] = (\\n            (aroon_period - h.rolling(aroon_period).apply(np.argmax, raw=True))\\n            / aroon_period\\n            * 100\\n        )\\n        out[\\\"aroon_down\\\"] = (\\n            (aroon_period - l.rolling(aroon_period).apply(np.argmin, raw=True))\\n            / aroon_period\\n            * 100\\n        )\\n        out[\\\"aroon_oscillator\\\"] = out[\\\"aroon_up\\\"] - out[\\\"aroon_down\\\"]\\n        sma5 = c.rolling(5, min_periods=1).mean()\\n        sma20 = c.rolling(20, min_periods=1).mean()\\n        out[\\\"trend_strength\\\"] = (sma5 - sma20) / sma20.replace(0, np.nan) * 100\\n\\n        # ---- Volatility ----\\n        sma20 = c.rolling(20, min_periods=1).mean()\\n        std20 = c.rolling(20, min_periods=1).std()\\n        out[\\\"bb_upper\\\"] = sma20 + 2 * std20\\n        out[\\\"bb_lower\\\"] = sma20 - 2 * std20\\n        out[\\\"bb_width\\\"] = (out[\\\"bb_upper\\\"] - out[\\\"bb_lower\\\"]) / c.replace(0, np.nan) * 100\\n        out[\\\"bb_position\\\"] = (c - out[\\\"bb_lower\\\"]) / (\\n            (out[\\\"bb_upper\\\"] - out[\\\"bb_lower\\\"]).replace(0, np.nan)\\n        )\\n        out[\\\"atr_14\\\"] = tr.rolling(14, min_periods=1).mean()\\n        out[\\\"atr_20\\\"] = tr.rolling(20, min_periods=1).mean()\\n        ret = c.pct_change()\\n        out[\\\"volatility_10\\\"] = ret.rolling(10, min_periods=1).std().fillna(0) * 100\\n        out[\\\"volatility_20\\\"] = ret.rolling(20, min_periods=1).std().fillna(0) * 100\\n\\n        # ---- Volume ----\\n        out[\\\"volume_sma_20\\\"] = v.rolling(20, min_periods=1).mean()\\n        out[\\\"volume_ratio\\\"] = v / out[\\\"volume_sma_20\\\"].replace(0, np.nan)\\n        obv = (np.sign(c.diff()) * v).fillna(0).cumsum()\\n        out[\\\"obv\\\"] = obv\\n        vpt = (v * ret).fillna(0).cumsum()\\n        out[\\\"volume_price_trend\\\"] = vpt\\n        mfm = ((c - l) - (h - c)) / (h - l).replace(0, np.nan)\\n        ad = (mfm * v).fillna(0).cumsum()\\n        out[\\\"accumulation_distribution\\\"] = ad\\n        ad_ema3 = ad.ewm(span=3, adjust=False).mean()\\n        ad_ema10 = ad.ewm(span=10, adjust=False).mean()\\n        out[\\\"chaikin_oscillator\\\"] = ad_ema3 - ad_ema10\\n\\n        # ---- Returns (resolution-dependent) ----\\n        bars_1h = max(1, 60 // resolution_minutes)\\n        bars_24h = max(1, 24 * 60 // resolution_minutes)\\n        out[\\\"returns_1h\\\"] = (c / c.shift(bars_1h).replace(0, np.nan) - 1) * 100\\n        out[\\\"returns_24h\\\"] = (c / c.shift(bars_24h).replace(0, np.nan) - 1) * 100\\n\\n        result = pd.DataFrame({k: out[k] for k in FEATURE_LIST})\\n        if include_pattern_features:\\n            cdl = self._cdl_engine.compute_all(df)\\n            chp = self._chp_engine.compute_all(df)\\n            result = pd.concat([result, cdl, chp], axis=1)\\n\\n        # Perpetual futures custom features\\n        if \\\"mark_price\\\" in df.columns and \\\"funding_rate\\\" in df.columns and \\\"open_interest\\\" in df.columns:\\n            perp = compute_perpetual_features(df)\\n            result = pd.concat([result, perp], axis=1)\\n\\n        if include_mtf_context:\\n            mtf = self._mtf_context_from_primary(df, resolution_minutes)\\n            result = pd.concat([result, mtf], axis=1)\\n        if fill_invalid:\\n            result = result.fillna(0).replace([np.inf, -np.inf], 0)\\n        return result\\n\\n    def compute_single(\\n        self,\\n        feature_name: str,\\n        candles: List[dict],\\n        resolution_minutes: int = 15,\\n    ) -> float:\\n        \\\"\\\"\\\"\\n        Compute a single feature value for live inference.\\n        Returns the latest scalar for the given feature.\\n        Supports canonical 50, aliases, v4 extras, and candlestick pattern features.\\n        \\\"\\\"\\\"\\n        # Candlestick pattern features\\n        if feature_name in CANDLESTICK_FEATURES:\\n            df = pd.DataFrame(candles)\\n            cdl = self._cdl_engine.compute_all(df)\\n            return float(cdl[feature_name].iloc[-1])\\n\\n        # Chart pattern features\\n        if feature_name in CHART_PATTERN_FEATURES:\\n            df = pd.DataFrame(candles)\\n            chp = self._chp_engine.compute_all(df)\\n            return float(chp[feature_name].iloc[-1])\\n\\n        if feature_name in MTF_CONTEXT_FEATURES:\\n            df = pd.DataFrame(candles)\\n            mtf = self._mtf_context_from_primary(df, resolution_minutes)\\n            return float(mtf[feature_name].iloc[-1])\\n\\n        if feature_name in PERPETUAL_FEATURE_NAMES:\\n            df = pd.DataFrame(candles)\\n            perp = compute_perpetual_features(df)\\n            return float(perp[feature_name].iloc[-1])\\n\\n        if feature_name in REGIME_FEATURES:\\n            df = pd.DataFrame(candles)\\n            # Ensure numeric types; compute_regime uses OHLC only.\\n            for col in (\\\"open\\\", \\\"high\\\", \\\"low\\\", \\\"close\\\", \\\"volume\\\"):\\n                if col in df.columns:\\n                    df[col] = pd.to_numeric(df[col], errors=\\\"coerce\\\")\\n            regime = _compute_regime_series(df)\\n            last = int(regime.iloc[-1]) if not regime.empty else 1\\n            if feature_name == \\\"regime_state\\\":\\n                return float(last)\\n            if feature_name == \\\"regime_is_ranging\\\":\\n                return float(1 if last == 0 else 0)\\n            if feature_name == \\\"regime_is_trending\\\":\\n                return float(1 if last == 1 else 0)\\n            if feature_name == \\\"regime_is_volatile\\\":\\n                return float(1 if last == 2 else 0)\\n\\n        # Backward-compat aliases for legacy/model-exported feature names.\\n        if feature_name.endswith(\\\"_15m\\\"):\\n            base_name = feature_name[:-4]\\n            resolved_base = FEATURE_ALIASES.get(base_name, base_name)\\n            if (\\n                resolved_base in FEATURE_LIST\\n                or base_name in V4_EXTRA_FEATURES\\n                or base_name in CANDLESTICK_FEATURES\\n                or base_name in CHART_PATTERN_FEATURES\\n                or base_name in MTF_CONTEXT_FEATURES\\n                or base_name in REGIME_FEATURES\\n                or base_name in PERPETUAL_FEATURE_NAMES\\n                or base_name in (\\\"di_diff\\\", \\\"ema_cross_50_200\\\")\\n            ):\\n                feature_name = base_name\\n\\n        resolved = FEATURE_ALIASES.get(feature_name, feature_name)\\n        if resolved in FEATURE_LIST:\\n            df = pd.DataFrame(candles)\\n            batch = self.compute_batch(\\n                df,\\n                resolution_minutes=resolution_minutes,\\n                fill_invalid=True,\\n            )\\n            return float(batch[resolved].iloc[-1])\\n\\n        # V4 extras: compute on demand\\n        df = pd.DataFrame(candles)\\n        batch = self.compute_batch(\\n            df,\\n            resolution_minutes=resolution_minutes,\\n            fill_invalid=True,\\n        )\\n        c = df[\\\"close\\\"].astype(float)\\n        v = df[\\\"volume\\\"].astype(float)\\n        o = df[\\\"open\\\"].astype(float)\\n        h = df[\\\"high\\\"].astype(float)\\n        l = df[\\\"low\\\"].astype(float)\\n\\n        if feature_name == \\\"ema_9\\\":\\n            return float(_ema_series(c, 9).iloc[-1])\\n        if feature_name == \\\"ema_21\\\":\\n            return float(_ema_series(c, 21).iloc[-1])\\n        if feature_name == \\\"ema_cross\\\":\\n            ema9 = _ema_series(c, 9).iloc[-1]\\n            ema21 = _ema_series(c, 21).iloc[-1]\\n            return float(ema9 - ema21)\\n        if feature_name == \\\"ema_cross_9_21\\\":\\n            ema9 = _ema_series(c, 9).iloc[-1]\\n            ema21 = _ema_series(c, 21).iloc[-1]\\n            return float(ema9 - ema21)\\n        if feature_name == \\\"ema_cross_21_50\\\":\\n            ema21 = _ema_series(c, 21).iloc[-1]\\n            ema50 = _ema_series(c, 50).iloc[-1]\\n            return float(ema21 - ema50)\\n        if feature_name == \\\"ema_cross_50_200\\\":\\n            ema50 = _ema_series(c, 50).iloc[-1]\\n            ema200 = _ema_series(c, 200).iloc[-1]\\n            return float(ema50 - ema200)\\n        if feature_name == \\\"returns_1\\\":\\n            if len(c) < 2:\\n                return 0.0\\n            return float((c.iloc[-1] / c.iloc[-2] - 1) * 100) if c.iloc[-2] > 0 else 0.0\\n        if feature_name == \\\"atr_pct\\\":\\n            atr = batch[\\\"atr_14\\\"].iloc[-1]\\n            close = c.iloc[-1]\\n            return float(atr / close * 100) if close > 0 else 0.0\\n        if feature_name == \\\"vol_zscore\\\":\\n            if len(v) < 21:\\n                return 0.0\\n            recent = v.iloc[-20:]\\n            mean_v = recent.mean()\\n            std_v = recent.std()\\n            return float((v.iloc[-1] - mean_v) / std_v) if std_v > 0 else 0.0\\n        if feature_name == \\\"hl_range\\\":\\n            return float((h.iloc[-1] - l.iloc[-1]))\\n        if feature_name in (\\\"plus_di\\\", \\\"minus_di\\\", \\\"di_diff\\\"):\\n            tr = np.maximum(\\n                h - l,\\n                np.maximum((h - c.shift(1)).abs(), (l - c.shift(1)).abs()),\\n            )\\n            plus_dm = (h - h.shift(1)).where(\\n                (h - h.shift(1)) > (l.shift(1) - l), 0\\n            ).clip(lower=0)\\n            minus_dm = (l.shift(1) - l).where(\\n                (l.shift(1) - l) > (h - h.shift(1)), 0\\n            ).clip(lower=0)\\n            atr_14 = tr.rolling(14, min_periods=1).mean()\\n            plus_di = 100 * plus_dm.rolling(14, min_periods=1).mean() / atr_14.replace(\\n                0, np.nan\\n            )\\n            minus_di = 100 * minus_dm.rolling(14, min_periods=1).mean() / atr_14.replace(\\n                0, np.nan\\n            )\\n            plus_di_last = float(plus_di.fillna(0).iloc[-1])\\n            minus_di_last = float(minus_di.fillna(0).iloc[-1])\\n            if feature_name == \\\"plus_di\\\":\\n                return plus_di_last\\n            if feature_name == \\\"minus_di\\\":\\n                return minus_di_last\\n            return plus_di_last - minus_di_last\\n        if feature_name == \\\"price_vs_ema200\\\":\\n            ema200 = _ema_series(c, 200).iloc[-1]\\n            if ema200 == 0:\\n                return 0.0\\n            return float((c.iloc[-1] - ema200) / ema200)\\n        if feature_name in (\\\"sr_near_high\\\", \\\"sr_near_low\\\"):\\n            window = 50\\n            recent_high = h.rolling(window, min_periods=1).max().iloc[-1]\\n            recent_low = l.rolling(window, min_periods=1).min().iloc[-1]\\n            close = c.iloc[-1]\\n            if feature_name == \\\"sr_near_high\\\":\\n                if recent_high <= 0:\\n                    return 0.0\\n                return float(1.0 if ((recent_high - close) / recent_high) <= 0.002 else 0.0)\\n            if recent_low <= 0:\\n                return 0.0\\n            return float(1.0 if ((close - recent_low) / recent_low) <= 0.002 else 0.0)\\n        if feature_name in (\\\"sr_breakout_up\\\", \\\"sr_breakout_dn\\\"):\\n            window = 50\\n            recent_high = h.rolling(window, min_periods=1).max().iloc[-1]\\n            recent_low = l.rolling(window, min_periods=1).min().iloc[-1]\\n            close = c.iloc[-1]\\n            if feature_name == \\\"sr_breakout_up\\\":\\n                return float(1.0 if close > recent_high else 0.0)\\n            return float(1.0 if close < recent_low else 0.0)\\n\\n        raise ValueError(f\\\"Unknown feature: {feature_name}\\\")\\n\\n\\n# Module-level singleton for convenience (optional; callers may instantiate)\\n_default_engine: Optional[UnifiedFeatureEngine] = None\\n\\n\\ndef get_unified_engine() -> UnifiedFeatureEngine:\\n    \\\"\\\"\\\"Return the default unified feature engine instance.\\\"\\\"\\\"\\n    global _default_engine\\n    if _default_engine is None:\\n        _default_engine = UnifiedFeatureEngine()\\n    return _default_engine\\n\", \"feature_store/pattern_features/__init__.py\": \"\\\"\\\"\\\"Pattern feature engines for candlestick and chart patterns.\\\"\\\"\\\"\\n\\nfrom feature_store.pattern_features.candlestick_patterns import (\\n    CANDLESTICK_FEATURES,\\n    CandlestickPatternEngine,\\n)\\nfrom feature_store.pattern_features.chart_patterns import (\\n    CHART_PATTERN_FEATURES,\\n    ChartPatternEngine,\\n)\\n\\n__all__ = [\\n    \\\"CandlestickPatternEngine\\\",\\n    \\\"ChartPatternEngine\\\",\\n    \\\"CANDLESTICK_FEATURES\\\",\\n    \\\"CHART_PATTERN_FEATURES\\\",\\n]\\n\", \"feature_store/pattern_features/pattern_utils.py\": \"\\\"\\\"\\\"Shared utilities for pattern feature computation.\\\"\\\"\\\"\\n\\nfrom dataclasses import dataclass\\n\\nimport pandas as pd\\n\\n\\n@dataclass\\nclass CandleGeometry:\\n    \\\"\\\"\\\"Pre-computed geometry for a single candle \\u2014 avoids redundant calculations.\\\"\\\"\\\"\\n\\n    body: float\\n    upper_wick: float\\n    lower_wick: float\\n    total_range: float\\n    body_ratio: float\\n    upper_ratio: float\\n    lower_ratio: float\\n    is_bullish: bool\\n\\n    @classmethod\\n    def from_row(cls, row: pd.Series) -> \\\"CandleGeometry\\\":\\n        body = abs(row[\\\"close\\\"] - row[\\\"open\\\"])\\n        upper_wick = row[\\\"high\\\"] - max(row[\\\"open\\\"], row[\\\"close\\\"])\\n        lower_wick = min(row[\\\"open\\\"], row[\\\"close\\\"]) - row[\\\"low\\\"]\\n        total_range = row[\\\"high\\\"] - row[\\\"low\\\"]\\n        safe_range = total_range if total_range > 1e-10 else 1e-10\\n        return cls(\\n            body=body,\\n            upper_wick=upper_wick,\\n            lower_wick=lower_wick,\\n            total_range=total_range,\\n            body_ratio=body / safe_range,\\n            upper_ratio=upper_wick / safe_range,\\n            lower_ratio=lower_wick / safe_range,\\n            is_bullish=row[\\\"close\\\"] >= row[\\\"open\\\"],\\n        )\\n\\n\\ndef compute_atr(df: pd.DataFrame, period: int = 14) -> pd.Series:\\n    \\\"\\\"\\\"Compute Average True Range.\\\"\\\"\\\"\\n    high_low = df[\\\"high\\\"] - df[\\\"low\\\"]\\n    high_close = (df[\\\"high\\\"] - df[\\\"close\\\"].shift()).abs()\\n    low_close = (df[\\\"low\\\"] - df[\\\"close\\\"].shift()).abs()\\n    tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)\\n    return tr.rolling(period).mean()\\n\", \"feature_store/pattern_features/candlestick_patterns.py\": \"\\\"\\\"\\\"\\nCandlestick pattern detection for ML features.\\n\\nDetects single-candle, two-candle, and three-candle patterns.\\nProduces ML-ready feature columns aligned with input DataFrame index.\\n\\\"\\\"\\\"\\n\\nimport pandas as pd\\n\\nfrom feature_store.pattern_features.pattern_utils import CandleGeometry\\n\\n# Feature names for registration\\nCANDLESTICK_FEATURES = [\\n    \\\"cdl_doji\\\",\\n    \\\"cdl_long_legged_doji\\\",\\n    \\\"cdl_dragonfly_doji\\\",\\n    \\\"cdl_gravestone_doji\\\",\\n    \\\"cdl_hammer\\\",\\n    \\\"cdl_inv_hammer\\\",\\n    \\\"cdl_hanging_man\\\",\\n    \\\"cdl_shooting_star\\\",\\n    \\\"cdl_bull_marubozu\\\",\\n    \\\"cdl_bear_marubozu\\\",\\n    \\\"cdl_spinning_top\\\",\\n    \\\"cdl_bull_engulfing\\\",\\n    \\\"cdl_bear_engulfing\\\",\\n    \\\"cdl_bull_harami\\\",\\n    \\\"cdl_bear_harami\\\",\\n    \\\"cdl_piercing\\\",\\n    \\\"cdl_dark_cloud\\\",\\n    \\\"cdl_tweezer_top\\\",\\n    \\\"cdl_tweezer_bottom\\\",\\n    \\\"cdl_bull_kicker\\\",\\n    \\\"cdl_bear_kicker\\\",\\n    \\\"cdl_morning_star\\\",\\n    \\\"cdl_evening_star\\\",\\n    \\\"cdl_three_white_soldiers\\\",\\n    \\\"cdl_three_black_crows\\\",\\n    \\\"cdl_three_inside_up\\\",\\n    \\\"cdl_three_inside_down\\\",\\n    \\\"cdl_abandoned_baby_bull\\\",\\n    \\\"cdl_abandoned_baby_bear\\\",\\n    \\\"cdl_bull_score\\\",\\n    \\\"cdl_bear_score\\\",\\n    \\\"cdl_net_score\\\",\\n    \\\"cdl_reversal_signal\\\",\\n    \\\"cdl_indecision_score\\\",\\n    \\\"cdl_body_ratio\\\",\\n    \\\"cdl_upper_wick_ratio\\\",\\n    \\\"cdl_lower_wick_ratio\\\",\\n    \\\"cdl_consecutive_bull\\\",\\n    \\\"cdl_consecutive_bear\\\",\\n]\\n\\n\\nclass CandlestickPatternEngine:\\n    \\\"\\\"\\\"\\n    Detects candlestick patterns and produces ML-ready feature columns.\\n    All methods operate on a full OHLCV DataFrame and return a feature DataFrame\\n    aligned by index \\u2014 safe for both training (batch) and live (last-row) use.\\n\\n    Hammer vs hanging man (and inverted hammer vs shooting star) share geometry;\\n    they are disambiguated only by prior trend context, never by candle color alone.\\n    \\\"\\\"\\\"\\n\\n    # Bars of prior closes (ending at t-1) used for trend context.\\n    TREND_LOOKBACK = 5\\n\\n    BULL_PATTERN_WEIGHTS = {\\n        \\\"cdl_hammer\\\": 0.8,\\n        \\\"cdl_dragonfly_doji\\\": 0.6,\\n        \\\"cdl_bull_engulfing\\\": 0.95,\\n        \\\"cdl_bull_harami\\\": 0.6,\\n        \\\"cdl_piercing\\\": 0.75,\\n        \\\"cdl_morning_star\\\": 0.9,\\n        \\\"cdl_three_white_soldiers\\\": 0.85,\\n        \\\"cdl_bull_marubozu\\\": 0.7,\\n        \\\"cdl_tweezer_bottom\\\": 0.65,\\n        \\\"cdl_three_inside_up\\\": 0.7,\\n        \\\"cdl_abandoned_baby_bull\\\": 0.95,\\n        \\\"cdl_bull_kicker\\\": 0.9,\\n        \\\"cdl_inv_hammer\\\": 0.5,\\n    }\\n\\n    BEAR_PATTERN_WEIGHTS = {\\n        \\\"cdl_shooting_star\\\": 0.8,\\n        \\\"cdl_gravestone_doji\\\": 0.6,\\n        \\\"cdl_bear_engulfing\\\": 0.95,\\n        \\\"cdl_bear_harami\\\": 0.6,\\n        \\\"cdl_dark_cloud\\\": 0.75,\\n        \\\"cdl_evening_star\\\": 0.9,\\n        \\\"cdl_three_black_crows\\\": 0.85,\\n        \\\"cdl_bear_marubozu\\\": 0.7,\\n        \\\"cdl_tweezer_top\\\": 0.65,\\n        \\\"cdl_three_inside_down\\\": 0.7,\\n        \\\"cdl_abandoned_baby_bear\\\": 0.95,\\n        \\\"cdl_bear_kicker\\\": 0.9,\\n        \\\"cdl_hanging_man\\\": 0.7,\\n    }\\n\\n    def compute_all(self, df: pd.DataFrame) -> pd.DataFrame:\\n        \\\"\\\"\\\"\\n        Main entry point. Returns DataFrame of all candlestick pattern features.\\n        Aligned with input df index.\\n        \\\"\\\"\\\"\\n        required = [\\\"open\\\", \\\"high\\\", \\\"low\\\", \\\"close\\\"]\\n        for col in required:\\n            if col not in df.columns:\\n                raise ValueError(f\\\"Missing required column: {col}\\\")\\n\\n        geo = list(df.apply(CandleGeometry.from_row, axis=1))\\n        out = pd.DataFrame(index=df.index)\\n\\n        # Single-candle patterns\\n        out[\\\"cdl_doji\\\"] = self._doji(df, geo)\\n        out[\\\"cdl_long_legged_doji\\\"] = self._long_legged_doji(df, geo)\\n        out[\\\"cdl_dragonfly_doji\\\"] = self._dragonfly_doji(df, geo)\\n        out[\\\"cdl_gravestone_doji\\\"] = self._gravestone_doji(df, geo)\\n        out[\\\"cdl_hammer\\\"] = self._hammer(df, geo)\\n        out[\\\"cdl_inv_hammer\\\"] = self._inverted_hammer(df, geo)\\n        out[\\\"cdl_hanging_man\\\"] = self._hanging_man(df, geo)\\n        out[\\\"cdl_shooting_star\\\"] = self._shooting_star(df, geo)\\n        out[\\\"cdl_bull_marubozu\\\"] = self._bull_marubozu(df, geo)\\n        out[\\\"cdl_bear_marubozu\\\"] = self._bear_marubozu(df, geo)\\n        out[\\\"cdl_spinning_top\\\"] = self._spinning_top(df, geo)\\n\\n        # Two-candle patterns\\n        out[\\\"cdl_bull_engulfing\\\"] = self._bull_engulfing(df, geo)\\n        out[\\\"cdl_bear_engulfing\\\"] = self._bear_engulfing(df, geo)\\n        out[\\\"cdl_bull_harami\\\"] = self._bull_harami(df, geo)\\n        out[\\\"cdl_bear_harami\\\"] = self._bear_harami(df, geo)\\n        out[\\\"cdl_piercing\\\"] = self._piercing(df, geo)\\n        out[\\\"cdl_dark_cloud\\\"] = self._dark_cloud(df, geo)\\n        out[\\\"cdl_tweezer_top\\\"] = self._tweezer_top(df, geo)\\n        out[\\\"cdl_tweezer_bottom\\\"] = self._tweezer_bottom(df, geo)\\n        out[\\\"cdl_bull_kicker\\\"] = self._bull_kicker(df, geo)\\n        out[\\\"cdl_bear_kicker\\\"] = self._bear_kicker(df, geo)\\n\\n        # Three-candle patterns\\n        out[\\\"cdl_morning_star\\\"] = self._morning_star(df, geo)\\n        out[\\\"cdl_evening_star\\\"] = self._evening_star(df, geo)\\n        out[\\\"cdl_three_white_soldiers\\\"] = self._three_white_soldiers(df, geo)\\n        out[\\\"cdl_three_black_crows\\\"] = self._three_black_crows(df, geo)\\n        out[\\\"cdl_three_inside_up\\\"] = self._three_inside_up(df, geo)\\n        out[\\\"cdl_three_inside_down\\\"] = self._three_inside_down(df, geo)\\n        out[\\\"cdl_abandoned_baby_bull\\\"] = self._abandoned_baby(df, geo, direction=\\\"bull\\\")\\n        out[\\\"cdl_abandoned_baby_bear\\\"] = self._abandoned_baby(df, geo, direction=\\\"bear\\\")\\n\\n        # Composite scores\\n        out[\\\"cdl_bull_score\\\"] = self._bull_composite(out)\\n        out[\\\"cdl_bear_score\\\"] = self._bear_composite(out)\\n        out[\\\"cdl_net_score\\\"] = out[\\\"cdl_bull_score\\\"] - out[\\\"cdl_bear_score\\\"]\\n        out[\\\"cdl_reversal_signal\\\"] = out[\\\"cdl_net_score\\\"].clip(-1, 1)\\n        out[\\\"cdl_indecision_score\\\"] = (\\n            out[\\\"cdl_doji\\\"] * 0.6\\n            + out[\\\"cdl_long_legged_doji\\\"] * 0.8\\n            + out[\\\"cdl_spinning_top\\\"] * 0.4\\n        ).clip(0, 1)\\n\\n        # Raw geometry features\\n        out[\\\"cdl_body_ratio\\\"] = [g.body_ratio for g in geo]\\n        out[\\\"cdl_upper_wick_ratio\\\"] = [g.upper_ratio for g in geo]\\n        out[\\\"cdl_lower_wick_ratio\\\"] = [g.lower_ratio for g in geo]\\n        out[\\\"cdl_consecutive_bull\\\"] = self._consecutive_direction(df, direction=\\\"bull\\\")\\n        out[\\\"cdl_consecutive_bear\\\"] = self._consecutive_direction(df, direction=\\\"bear\\\")\\n\\n        return out.fillna(0)\\n\\n    def _doji(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        return pd.Series(\\n            [1 if g.body_ratio < 0.10 and g.total_range > 0 else 0 for g in geo],\\n            index=df.index,\\n        )\\n\\n    def _long_legged_doji(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        return pd.Series(\\n            [\\n                1\\n                if g.body_ratio < 0.10 and g.upper_ratio > 0.30 and g.lower_ratio > 0.30\\n                else 0\\n                for g in geo\\n            ],\\n            index=df.index,\\n        )\\n\\n    def _dragonfly_doji(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        return pd.Series(\\n            [\\n                1\\n                if g.body_ratio < 0.10 and g.upper_ratio < 0.05 and g.lower_ratio > 0.60\\n                else 0\\n                for g in geo\\n            ],\\n            index=df.index,\\n        )\\n\\n    def _gravestone_doji(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        return pd.Series(\\n            [\\n                1\\n                if g.body_ratio < 0.10 and g.lower_ratio < 0.05 and g.upper_ratio > 0.60\\n                else 0\\n                for g in geo\\n            ],\\n            index=df.index,\\n        )\\n\\n    def _prior_trend(self, df: pd.DataFrame) -> tuple[pd.Series, pd.Series]:\\n        \\\"\\\"\\\"Causal prior up/down trend from closes ending at the previous bar.\\n\\n        Uses prior close vs its trailing SMA over ``TREND_LOOKBACK`` bars.\\n        Flat or insufficient history yields neither up nor down.\\n        \\\"\\\"\\\"\\n        prior_close = df[\\\"close\\\"].astype(float).shift(1)\\n        sma = prior_close.rolling(\\n            self.TREND_LOOKBACK, min_periods=self.TREND_LOOKBACK\\n        ).mean()\\n        prior_up = (prior_close > sma).fillna(False)\\n        prior_down = (prior_close < sma).fillna(False)\\n        return prior_up, prior_down\\n\\n    @staticmethod\\n    def _atr_proxy(df: pd.DataFrame) -> pd.Series:\\n        atr = df[\\\"close\\\"].diff().abs().rolling(14).mean()\\n        fallback = df[\\\"close\\\"].astype(float) * 0.01\\n        return atr.where(atr.notna() & (atr > 0), fallback)\\n\\n    @staticmethod\\n    def _is_hammer_shape(g: CandleGeometry, atr_val: float) -> bool:\\n        \\\"\\\"\\\"Long lower wick, short upper wick, meaningful body and range.\\\"\\\"\\\"\\n        return (\\n            g.lower_wick >= 2.0 * g.body\\n            and g.upper_ratio < 0.20\\n            and g.body_ratio > 0.05\\n            and g.total_range > float(atr_val) * 0.5\\n        )\\n\\n    @staticmethod\\n    def _is_inverted_hammer_shape(g: CandleGeometry) -> bool:\\n        \\\"\\\"\\\"Long upper wick, short lower wick, meaningful body.\\\"\\\"\\\"\\n        return (\\n            g.upper_wick >= 2.0 * g.body\\n            and g.lower_ratio < 0.20\\n            and g.body_ratio > 0.05\\n        )\\n\\n    def _hammer(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        \\\"\\\"\\\"Bullish hammer: hammer shape after a prior downtrend.\\\"\\\"\\\"\\n        atr = self._atr_proxy(df)\\n        _, prior_down = self._prior_trend(df)\\n        result = [\\n            1\\n            if self._is_hammer_shape(g, float(atr.iloc[i])) and bool(prior_down.iloc[i])\\n            else 0\\n            for i, g in enumerate(geo)\\n        ]\\n        return pd.Series(result, index=df.index)\\n\\n    def _hanging_man(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        \\\"\\\"\\\"Bearish hanging man: same shape as hammer after a prior uptrend.\\\"\\\"\\\"\\n        atr = self._atr_proxy(df)\\n        prior_up, _ = self._prior_trend(df)\\n        result = [\\n            1\\n            if self._is_hammer_shape(g, float(atr.iloc[i])) and bool(prior_up.iloc[i])\\n            else 0\\n            for i, g in enumerate(geo)\\n        ]\\n        return pd.Series(result, index=df.index)\\n\\n    def _inverted_hammer(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        \\\"\\\"\\\"Bullish inverted hammer: inverted shape after a prior downtrend.\\\"\\\"\\\"\\n        _, prior_down = self._prior_trend(df)\\n        result = [\\n            1\\n            if self._is_inverted_hammer_shape(g) and bool(prior_down.iloc[i])\\n            else 0\\n            for i, g in enumerate(geo)\\n        ]\\n        return pd.Series(result, index=df.index)\\n\\n    def _shooting_star(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        \\\"\\\"\\\"Bearish shooting star: same shape as inverted hammer after uptrend.\\\"\\\"\\\"\\n        prior_up, _ = self._prior_trend(df)\\n        result = [\\n            1\\n            if self._is_inverted_hammer_shape(g) and bool(prior_up.iloc[i])\\n            else 0\\n            for i, g in enumerate(geo)\\n        ]\\n        return pd.Series(result, index=df.index)\\n\\n    def _bull_marubozu(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        return pd.Series(\\n            [1 if g.is_bullish and g.body_ratio > 0.95 else 0 for g in geo],\\n            index=df.index,\\n        )\\n\\n    def _bear_marubozu(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        return pd.Series(\\n            [1 if not g.is_bullish and g.body_ratio > 0.95 else 0 for g in geo],\\n            index=df.index,\\n        )\\n\\n    def _spinning_top(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        return pd.Series(\\n            [\\n                1\\n                if 0.10 < g.body_ratio < 0.40\\n                and g.upper_ratio > 0.20\\n                and g.lower_ratio > 0.20\\n                else 0\\n                for g in geo\\n            ],\\n            index=df.index,\\n        )\\n\\n    def _bull_engulfing(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        result = [0] * len(df)\\n        for i in range(1, len(df)):\\n            prev, curr = geo[i - 1], geo[i]\\n            if (\\n                not prev.is_bullish\\n                and curr.is_bullish\\n                and df[\\\"open\\\"].iloc[i] <= df[\\\"close\\\"].iloc[i - 1]\\n                and df[\\\"close\\\"].iloc[i] >= df[\\\"open\\\"].iloc[i - 1]\\n            ):\\n                result[i] = 1\\n        return pd.Series(result, index=df.index)\\n\\n    def _bear_engulfing(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        result = [0] * len(df)\\n        for i in range(1, len(df)):\\n            prev, curr = geo[i - 1], geo[i]\\n            if (\\n                prev.is_bullish\\n                and not curr.is_bullish\\n                and df[\\\"open\\\"].iloc[i] >= df[\\\"close\\\"].iloc[i - 1]\\n                and df[\\\"close\\\"].iloc[i] <= df[\\\"open\\\"].iloc[i - 1]\\n            ):\\n                result[i] = 1\\n        return pd.Series(result, index=df.index)\\n\\n    def _bull_harami(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        result = [0] * len(df)\\n        for i in range(1, len(df)):\\n            prev, curr = geo[i - 1], geo[i]\\n            if (\\n                not prev.is_bullish\\n                and curr.is_bullish\\n                and curr.body < prev.body * 0.5\\n                and df[\\\"open\\\"].iloc[i] > df[\\\"close\\\"].iloc[i - 1]\\n                and df[\\\"close\\\"].iloc[i] < df[\\\"open\\\"].iloc[i - 1]\\n            ):\\n                result[i] = 1\\n        return pd.Series(result, index=df.index)\\n\\n    def _bear_harami(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        result = [0] * len(df)\\n        for i in range(1, len(df)):\\n            prev, curr = geo[i - 1], geo[i]\\n            if (\\n                prev.is_bullish\\n                and not curr.is_bullish\\n                and curr.body < prev.body * 0.5\\n                and df[\\\"open\\\"].iloc[i] < df[\\\"close\\\"].iloc[i - 1]\\n                and df[\\\"close\\\"].iloc[i] > df[\\\"open\\\"].iloc[i - 1]\\n            ):\\n                result[i] = 1\\n        return pd.Series(result, index=df.index)\\n\\n    def _piercing(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        result = [0] * len(df)\\n        for i in range(1, len(df)):\\n            prev, curr = geo[i - 1], geo[i]\\n            midpoint = (df[\\\"open\\\"].iloc[i - 1] + df[\\\"close\\\"].iloc[i - 1]) / 2\\n            if (\\n                not prev.is_bullish\\n                and curr.is_bullish\\n                and df[\\\"open\\\"].iloc[i] < df[\\\"close\\\"].iloc[i - 1]\\n                and df[\\\"close\\\"].iloc[i] > midpoint\\n            ):\\n                result[i] = 1\\n        return pd.Series(result, index=df.index)\\n\\n    def _dark_cloud(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        result = [0] * len(df)\\n        for i in range(1, len(df)):\\n            prev, curr = geo[i - 1], geo[i]\\n            midpoint = (df[\\\"open\\\"].iloc[i - 1] + df[\\\"close\\\"].iloc[i - 1]) / 2\\n            if (\\n                prev.is_bullish\\n                and not curr.is_bullish\\n                and df[\\\"open\\\"].iloc[i] > df[\\\"close\\\"].iloc[i - 1]\\n                and df[\\\"close\\\"].iloc[i] < midpoint\\n            ):\\n                result[i] = 1\\n        return pd.Series(result, index=df.index)\\n\\n    def _tweezer_top(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        result = [0] * len(df)\\n        for i in range(1, len(df)):\\n            high_diff = abs(df[\\\"high\\\"].iloc[i] - df[\\\"high\\\"].iloc[i - 1])\\n            atr = df[\\\"close\\\"].iloc[max(0, i - 14) : i].diff().abs().mean()\\n            atr_val = atr if not pd.isna(atr) and atr > 0 else df[\\\"close\\\"].iloc[i] * 0.01\\n            if not geo[i].is_bullish and high_diff < atr_val * 0.1:\\n                result[i] = 1\\n        return pd.Series(result, index=df.index)\\n\\n    def _tweezer_bottom(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        result = [0] * len(df)\\n        for i in range(1, len(df)):\\n            low_diff = abs(df[\\\"low\\\"].iloc[i] - df[\\\"low\\\"].iloc[i - 1])\\n            atr = df[\\\"close\\\"].iloc[max(0, i - 14) : i].diff().abs().mean()\\n            atr_val = atr if not pd.isna(atr) and atr > 0 else df[\\\"close\\\"].iloc[i] * 0.01\\n            if geo[i].is_bullish and low_diff < atr_val * 0.1:\\n                result[i] = 1\\n        return pd.Series(result, index=df.index)\\n\\n    def _bull_kicker(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        result = [0] * len(df)\\n        for i in range(1, len(df)):\\n            if (\\n                not geo[i - 1].is_bullish\\n                and geo[i].is_bullish\\n                and df[\\\"open\\\"].iloc[i] > df[\\\"open\\\"].iloc[i - 1]\\n                and geo[i].body_ratio > 0.5\\n            ):\\n                result[i] = 1\\n        return pd.Series(result, index=df.index)\\n\\n    def _bear_kicker(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        result = [0] * len(df)\\n        for i in range(1, len(df)):\\n            if (\\n                geo[i - 1].is_bullish\\n                and not geo[i].is_bullish\\n                and df[\\\"open\\\"].iloc[i] < df[\\\"open\\\"].iloc[i - 1]\\n                and geo[i].body_ratio > 0.5\\n            ):\\n                result[i] = 1\\n        return pd.Series(result, index=df.index)\\n\\n    def _morning_star(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        result = [0] * len(df)\\n        for i in range(2, len(df)):\\n            a, b, c = geo[i - 2], geo[i - 1], geo[i]\\n            mid_a = (df[\\\"open\\\"].iloc[i - 2] + df[\\\"close\\\"].iloc[i - 2]) / 2\\n            if (\\n                not a.is_bullish\\n                and a.body_ratio > 0.4\\n                and b.body_ratio < 0.3\\n                and c.is_bullish\\n                and df[\\\"close\\\"].iloc[i] > mid_a\\n            ):\\n                result[i] = 1\\n        return pd.Series(result, index=df.index)\\n\\n    def _evening_star(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        result = [0] * len(df)\\n        for i in range(2, len(df)):\\n            a, b, c = geo[i - 2], geo[i - 1], geo[i]\\n            mid_a = (df[\\\"open\\\"].iloc[i - 2] + df[\\\"close\\\"].iloc[i - 2]) / 2\\n            if (\\n                a.is_bullish\\n                and a.body_ratio > 0.4\\n                and b.body_ratio < 0.3\\n                and not c.is_bullish\\n                and df[\\\"close\\\"].iloc[i] < mid_a\\n            ):\\n                result[i] = 1\\n        return pd.Series(result, index=df.index)\\n\\n    def _three_white_soldiers(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        result = [0] * len(df)\\n        for i in range(2, len(df)):\\n            a, b, c = geo[i - 2], geo[i - 1], geo[i]\\n            if (\\n                a.is_bullish\\n                and b.is_bullish\\n                and c.is_bullish\\n                and df[\\\"close\\\"].iloc[i] > df[\\\"close\\\"].iloc[i - 1] > df[\\\"close\\\"].iloc[i - 2]\\n                and a.body_ratio > 0.5\\n                and b.body_ratio > 0.5\\n                and c.body_ratio > 0.5\\n            ):\\n                result[i] = 1\\n        return pd.Series(result, index=df.index)\\n\\n    def _three_black_crows(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        result = [0] * len(df)\\n        for i in range(2, len(df)):\\n            a, b, c = geo[i - 2], geo[i - 1], geo[i]\\n            if (\\n                not a.is_bullish\\n                and not b.is_bullish\\n                and not c.is_bullish\\n                and df[\\\"close\\\"].iloc[i] < df[\\\"close\\\"].iloc[i - 1] < df[\\\"close\\\"].iloc[i - 2]\\n                and a.body_ratio > 0.5\\n                and b.body_ratio > 0.5\\n                and c.body_ratio > 0.5\\n            ):\\n                result[i] = 1\\n        return pd.Series(result, index=df.index)\\n\\n    def _three_inside_up(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        result = [0] * len(df)\\n        for i in range(2, len(df)):\\n            a, b, c = geo[i - 2], geo[i - 1], geo[i]\\n            if (\\n                not a.is_bullish\\n                and b.is_bullish\\n                and c.is_bullish\\n                and b.body < a.body\\n                and df[\\\"close\\\"].iloc[i] > df[\\\"close\\\"].iloc[i - 1]\\n            ):\\n                result[i] = 1\\n        return pd.Series(result, index=df.index)\\n\\n    def _three_inside_down(self, df: pd.DataFrame, geo: list) -> pd.Series:\\n        result = [0] * len(df)\\n        for i in range(2, len(df)):\\n            a, b, c = geo[i - 2], geo[i - 1], geo[i]\\n            if (\\n                a.is_bullish\\n                and not b.is_bullish\\n                and not c.is_bullish\\n                and b.body < a.body\\n                and df[\\\"close\\\"].iloc[i] < df[\\\"close\\\"].iloc[i - 1]\\n            ):\\n                result[i] = 1\\n        return pd.Series(result, index=df.index)\\n\\n    def _abandoned_baby(\\n        self, df: pd.DataFrame, geo: list, direction: str\\n    ) -> pd.Series:\\n        result = [0] * len(df)\\n        for i in range(2, len(df)):\\n            a, b, c = geo[i - 2], geo[i - 1], geo[i]\\n            is_doji_b = b.body_ratio < 0.10\\n            if direction == \\\"bull\\\":\\n                gap1 = df[\\\"low\\\"].iloc[i - 1] < df[\\\"low\\\"].iloc[i - 2]\\n                gap2 = df[\\\"low\\\"].iloc[i] > df[\\\"high\\\"].iloc[i - 1]\\n                if not a.is_bullish and is_doji_b and c.is_bullish and gap1 and gap2:\\n                    result[i] = 1\\n            else:\\n                gap1 = df[\\\"high\\\"].iloc[i - 1] > df[\\\"high\\\"].iloc[i - 2]\\n                gap2 = df[\\\"high\\\"].iloc[i] < df[\\\"low\\\"].iloc[i - 1]\\n                if a.is_bullish and is_doji_b and not c.is_bullish and gap1 and gap2:\\n                    result[i] = 1\\n        return pd.Series(result, index=df.index)\\n\\n    def _bull_composite(self, out: pd.DataFrame) -> pd.Series:\\n        score = pd.Series(0.0, index=out.index)\\n        for col, weight in self.BULL_PATTERN_WEIGHTS.items():\\n            if col in out.columns:\\n                score += out[col] * weight\\n        total_weight = sum(self.BULL_PATTERN_WEIGHTS.values())\\n        return (score / total_weight).clip(0, 1)\\n\\n    def _bear_composite(self, out: pd.DataFrame) -> pd.Series:\\n        score = pd.Series(0.0, index=out.index)\\n        for col, weight in self.BEAR_PATTERN_WEIGHTS.items():\\n            if col in out.columns:\\n                score += out[col] * weight\\n        total_weight = sum(self.BEAR_PATTERN_WEIGHTS.values())\\n        return (score / total_weight).clip(0, 1)\\n\\n    def _consecutive_direction(\\n        self, df: pd.DataFrame, direction: str\\n    ) -> pd.Series:\\n        is_bull = (df[\\\"close\\\"] >= df[\\\"open\\\"]).astype(int)\\n        values = is_bull if direction == \\\"bull\\\" else (1 - is_bull)\\n        result = []\\n        count = 0\\n        for v in values:\\n            count = count + 1 if v == 1 else 0\\n            result.append(count)\\n        return pd.Series(result, index=df.index)\\n\", \"feature_store/pattern_features/chart_patterns.py\": \"\\\"\\\"\\\"\\nChart pattern detection for ML features.\\n\\nDetects support/resistance, trendlines, flags, triangles, reversals, and breakouts.\\nProduces ML-ready feature columns aligned with input DataFrame index.\\n\\\"\\\"\\\"\\n\\nimport numpy as np\\nimport pandas as pd\\nfrom scipy.signal import argrelextrema\\nfrom scipy.stats import linregress\\n\\n# Feature names for registration\\nCHART_PATTERN_FEATURES = [\\n    \\\"sr_support_dist_pct\\\",\\n    \\\"sr_resistance_dist_pct\\\",\\n    \\\"sr_at_support\\\",\\n    \\\"sr_at_resistance\\\",\\n    \\\"sr_support_strength\\\",\\n    \\\"sr_resistance_strength\\\",\\n    \\\"sr_range_position\\\",\\n    \\\"tl_uptrend_detected\\\",\\n    \\\"tl_downtrend_detected\\\",\\n    \\\"tl_trend_slope\\\",\\n    \\\"tl_dist_to_trendline\\\",\\n    \\\"tl_near_trendline\\\",\\n    \\\"tl_breakout_up\\\",\\n    \\\"tl_breakout_down\\\",\\n    \\\"chp_bull_flag\\\",\\n    \\\"chp_bear_flag\\\",\\n    \\\"chp_bull_flag_strength\\\",\\n    \\\"chp_bear_flag_strength\\\",\\n    \\\"chp_asc_triangle\\\",\\n    \\\"chp_desc_triangle\\\",\\n    \\\"chp_sym_triangle\\\",\\n    \\\"chp_triangle_apex_dist\\\",\\n    \\\"chp_double_top\\\",\\n    \\\"chp_double_bottom\\\",\\n    \\\"chp_double_top_dist\\\",\\n    \\\"chp_double_bottom_dist\\\",\\n    \\\"chp_hs_detected\\\",\\n    \\\"chp_ihs_detected\\\",\\n    \\\"bo_at_high\\\",\\n    \\\"bo_at_low\\\",\\n    \\\"bo_volume_confirmation\\\",\\n    \\\"bo_breakout_score\\\",\\n]\\n\\n\\n\\ndef _chart_safe_atr(atr_value: object, price: float) -> float:\\n    \\\"\\\"\\\"Positive ATR for ratios. Zero ATR appears on flat / warmup bars.\\\"\\\"\\\"\\n    try:\\n        a = float(atr_value)\\n    except (TypeError, ValueError):\\n        a = float(\\\"nan\\\")\\n    if not np.isfinite(a) or a <= 0.0:\\n        return max(abs(float(price)) * 0.01, 1e-9)\\n    return a\\n\\n\\nclass ChartPatternEngine:\\n    \\\"\\\"\\\"\\n    Detects chart patterns and computes structural market features.\\n    All methods return full-length DataFrames aligned with input index.\\n    \\\"\\\"\\\"\\n\\n    def compute_all(self, df: pd.DataFrame, atr_period: int = 14) -> pd.DataFrame:\\n        \\\"\\\"\\\"Main entry point \\u2014 computes all chart pattern features.\\\"\\\"\\\"\\n        atr = self._compute_atr(df, atr_period)\\n        out = pd.DataFrame(index=df.index)\\n\\n        sr = self._compute_support_resistance(df, atr)\\n        out = pd.concat([out, sr], axis=1)\\n\\n        tl = self._compute_trendlines(df, atr)\\n        out = pd.concat([out, tl], axis=1)\\n\\n        flags = self._compute_flags(df, atr)\\n        out = pd.concat([out, flags], axis=1)\\n\\n        tri = self._compute_triangles(df, atr)\\n        out = pd.concat([out, tri], axis=1)\\n\\n        rev = self._compute_reversal_patterns(df, atr)\\n        out = pd.concat([out, rev], axis=1)\\n\\n        bo = self._compute_breakouts(df, atr)\\n        out = pd.concat([out, bo], axis=1)\\n\\n        return out.replace([np.inf, -np.inf], 0).fillna(0)\\n\\n    def _compute_atr(self, df: pd.DataFrame, period: int = 14) -> pd.Series:\\n        high_low = df[\\\"high\\\"] - df[\\\"low\\\"]\\n        high_close = (df[\\\"high\\\"] - df[\\\"close\\\"].shift()).abs()\\n        low_close = (df[\\\"low\\\"] - df[\\\"close\\\"].shift()).abs()\\n        tr = pd.concat([high_low, high_close, low_close], axis=1).max(axis=1)\\n        return tr.rolling(period).mean()\\n\\n    def _compute_support_resistance(\\n        self,\\n        df: pd.DataFrame,\\n        atr: pd.Series,\\n        window: int = 5,\\n        lookback: int = 60,\\n    ) -> pd.DataFrame:\\n        out = pd.DataFrame(index=df.index)\\n        highs = df[\\\"high\\\"].values\\n        lows = df[\\\"low\\\"].values\\n        closes = df[\\\"close\\\"].values\\n\\n        high_idx = argrelextrema(highs, np.greater, order=window)[0]\\n        low_idx = argrelextrema(lows, np.less, order=window)[0]\\n\\n        sr_support_dist = np.full(len(df), np.nan)\\n        sr_resistance_dist = np.full(len(df), np.nan)\\n        sr_at_support = np.zeros(len(df))\\n        sr_at_resistance = np.zeros(len(df))\\n        sr_support_strength = np.zeros(len(df))\\n        sr_resistance_strength = np.zeros(len(df))\\n        sr_range_position = np.full(len(df), 0.5)\\n\\n        for i in range(lookback, len(df)):\\n            current_price = closes[i]\\n            current_atr = _chart_safe_atr(atr.iloc[i], float(current_price))\\n\\n            recent_highs_idx = high_idx[(high_idx >= i - lookback) & (high_idx < i)]\\n            recent_lows_idx = low_idx[(low_idx >= i - lookback) & (low_idx < i)]\\n\\n            if len(recent_highs_idx) > 0:\\n                resistance_levels = highs[recent_highs_idx]\\n                above = resistance_levels[resistance_levels > current_price]\\n                if len(above) > 0:\\n                    nearest_res = above.min()\\n                    sr_resistance_dist[i] = (\\n                        (nearest_res - current_price) / current_price * 100\\n                    )\\n                    sr_at_resistance[i] = (\\n                        1 if abs(nearest_res - current_price) < current_atr else 0\\n                    )\\n                    sr_resistance_strength[i] = np.sum(\\n                        np.abs(resistance_levels - nearest_res) < current_atr * 0.5\\n                    )\\n\\n            if len(recent_lows_idx) > 0:\\n                support_levels = lows[recent_lows_idx]\\n                below = support_levels[support_levels < current_price]\\n                if len(below) > 0:\\n                    nearest_sup = below.max()\\n                    sr_support_dist[i] = (\\n                        (current_price - nearest_sup) / current_price * 100\\n                    )\\n                    sr_at_support[i] = (\\n                        1 if abs(current_price - nearest_sup) < current_atr else 0\\n                    )\\n                    sr_support_strength[i] = np.sum(\\n                        np.abs(support_levels - nearest_sup) < current_atr * 0.5\\n                    )\\n\\n            sup_d = sr_support_dist[i] if not np.isnan(sr_support_dist[i]) else 0\\n            res_d = sr_resistance_dist[i] if not np.isnan(sr_resistance_dist[i]) else 0\\n            total = sup_d + res_d\\n            if total > 0:\\n                sr_range_position[i] = res_d / total\\n\\n        out[\\\"sr_support_dist_pct\\\"] = sr_support_dist\\n        out[\\\"sr_resistance_dist_pct\\\"] = sr_resistance_dist\\n        out[\\\"sr_at_support\\\"] = sr_at_support\\n        out[\\\"sr_at_resistance\\\"] = sr_at_resistance\\n        out[\\\"sr_support_strength\\\"] = sr_support_strength\\n        out[\\\"sr_resistance_strength\\\"] = sr_resistance_strength\\n        out[\\\"sr_range_position\\\"] = sr_range_position\\n\\n        return out\\n\\n    def _compute_trendlines(\\n        self,\\n        df: pd.DataFrame,\\n        atr: pd.Series,\\n        min_touches: int = 3,\\n        lookback: int = 50,\\n    ) -> pd.DataFrame:\\n        out = pd.DataFrame(index=df.index)\\n        highs = df[\\\"high\\\"].values\\n        lows = df[\\\"low\\\"].values\\n        closes = df[\\\"close\\\"].values\\n\\n        high_idx = argrelextrema(highs, np.greater, order=5)[0]\\n        low_idx = argrelextrema(lows, np.less, order=5)[0]\\n\\n        tl_uptrend = np.zeros(len(df))\\n        tl_downtrend = np.zeros(len(df))\\n        tl_slope = np.zeros(len(df))\\n        tl_dist = np.zeros(len(df))\\n        tl_near = np.zeros(len(df))\\n        tl_break_up = np.zeros(len(df))\\n        tl_break_down = np.zeros(len(df))\\n\\n        for i in range(lookback, len(df)):\\n            current_atr = _chart_safe_atr(atr.iloc[i], float(closes[i]))\\n\\n            ul_idx = low_idx[(low_idx >= i - lookback) & (low_idx < i)]\\n            if len(ul_idx) >= min_touches:\\n                x = ul_idx.astype(float)\\n                y = lows[ul_idx]\\n                slope, intercept, r, _, _ = linregress(x, y)\\n                if slope > 0 and r > 0.7:\\n                    tl_uptrend[i] = 1\\n                    trendline_val = slope * i + intercept\\n                    dist = (closes[i] - trendline_val) / current_atr\\n                    tl_slope[i] = slope / (closes[i] * 0.0001)\\n                    tl_dist[i] = max(0, dist)\\n                    tl_near[i] = 1 if abs(dist) < 1.0 else 0\\n                    if closes[i] < trendline_val and closes[i - 1] >= (\\n                        slope * (i - 1) + intercept\\n                    ):\\n                        tl_break_down[i] = 1\\n\\n            dl_idx = high_idx[(high_idx >= i - lookback) & (high_idx < i)]\\n            if len(dl_idx) >= min_touches:\\n                x = dl_idx.astype(float)\\n                y = highs[dl_idx]\\n                slope, intercept, r, _, _ = linregress(x, y)\\n                if slope < 0 and r > 0.7:\\n                    tl_downtrend[i] = 1\\n                    trendline_val = slope * i + intercept\\n                    if closes[i] > trendline_val and closes[i - 1] <= (\\n                        slope * (i - 1) + intercept\\n                    ):\\n                        tl_break_up[i] = 1\\n\\n        out[\\\"tl_uptrend_detected\\\"] = tl_uptrend\\n        out[\\\"tl_downtrend_detected\\\"] = tl_downtrend\\n        out[\\\"tl_trend_slope\\\"] = tl_slope\\n        out[\\\"tl_dist_to_trendline\\\"] = tl_dist\\n        out[\\\"tl_near_trendline\\\"] = tl_near\\n        out[\\\"tl_breakout_up\\\"] = tl_break_up\\n        out[\\\"tl_breakout_down\\\"] = tl_break_down\\n\\n        return out\\n\\n    def _compute_flags(\\n        self,\\n        df: pd.DataFrame,\\n        atr: pd.Series,\\n        pole_bars: int = 10,\\n        flag_bars: int = 20,\\n    ) -> pd.DataFrame:\\n        out = pd.DataFrame(index=df.index)\\n        closes = df[\\\"close\\\"].values\\n        n = len(df)\\n\\n        bull_flag = np.zeros(n)\\n        bear_flag = np.zeros(n)\\n        bull_flag_strength = np.zeros(n)\\n        bear_flag_strength = np.zeros(n)\\n\\n        for i in range(pole_bars + flag_bars, n):\\n            current_atr = _chart_safe_atr(atr.iloc[i], float(closes[i]))\\n\\n            pole_start = i - pole_bars - flag_bars\\n            pole_end = i - flag_bars\\n            pole_move = closes[pole_end] - closes[pole_start]\\n            pole_magnitude = abs(pole_move) / (current_atr * pole_bars)\\n\\n            if pole_magnitude > 0.5:\\n                flag_closes = closes[pole_end:i]\\n                flag_range = np.max(flag_closes) - np.min(flag_closes)\\n                flag_drift = closes[i - 1] - closes[pole_end]\\n\\n                if pole_move > 0:\\n                    if flag_range < abs(pole_move) * 0.5 and flag_drift < 0:\\n                        bull_flag[i] = 1\\n                        bull_flag_strength[i] = pole_magnitude\\n                else:\\n                    if flag_range < abs(pole_move) * 0.5 and flag_drift > 0:\\n                        bear_flag[i] = 1\\n                        bear_flag_strength[i] = pole_magnitude\\n\\n        out[\\\"chp_bull_flag\\\"] = bull_flag\\n        out[\\\"chp_bear_flag\\\"] = bear_flag\\n        out[\\\"chp_bull_flag_strength\\\"] = bull_flag_strength\\n        out[\\\"chp_bear_flag_strength\\\"] = bear_flag_strength\\n\\n        return out\\n\\n    def _compute_triangles(\\n        self,\\n        df: pd.DataFrame,\\n        atr: pd.Series,\\n        lookback: int = 40,\\n    ) -> pd.DataFrame:\\n        out = pd.DataFrame(index=df.index)\\n        highs = df[\\\"high\\\"].values\\n        lows = df[\\\"low\\\"].values\\n        n = len(df)\\n\\n        asc_tri = np.zeros(n)\\n        desc_tri = np.zeros(n)\\n        sym_tri = np.zeros(n)\\n        apex_dist = np.zeros(n)\\n\\n        for i in range(lookback, n):\\n            window_highs = highs[i - lookback : i]\\n            window_lows = lows[i - lookback : i]\\n            x = np.arange(lookback, dtype=float)\\n\\n            slope_h, _, _, _, _ = linregress(x, window_highs)\\n            slope_l, _, _, _, _ = linregress(x, window_lows)\\n\\n            atr_val = _chart_safe_atr(atr.iloc[i], float(highs[i]))\\n\\n            if abs(slope_h) < 0.1 * atr_val and slope_l > 0:\\n                asc_tri[i] = 1\\n            elif slope_h < 0 and abs(slope_l) < 0.1 * atr_val:\\n                desc_tri[i] = 1\\n            elif slope_h < 0 and slope_l > 0:\\n                sym_tri[i] = 1\\n                price_range = window_highs[-1] - window_lows[-1]\\n                if abs(slope_h - slope_l) > 1e-10:\\n                    bars_to_apex = price_range / abs(slope_h - slope_l)\\n                    apex_dist[i] = bars_to_apex / lookback\\n\\n        out[\\\"chp_asc_triangle\\\"] = asc_tri\\n        out[\\\"chp_desc_triangle\\\"] = desc_tri\\n        out[\\\"chp_sym_triangle\\\"] = sym_tri\\n        out[\\\"chp_triangle_apex_dist\\\"] = apex_dist\\n\\n        return out\\n\\n    def _compute_reversal_patterns(\\n        self,\\n        df: pd.DataFrame,\\n        atr: pd.Series,\\n        lookback: int = 60,\\n    ) -> pd.DataFrame:\\n        out = pd.DataFrame(index=df.index)\\n        highs = df[\\\"high\\\"].values\\n        lows = df[\\\"low\\\"].values\\n        closes = df[\\\"close\\\"].values\\n        n = len(df)\\n\\n        double_top = np.zeros(n)\\n        double_bottom = np.zeros(n)\\n        double_top_dist = np.zeros(n)\\n        double_bottom_dist = np.zeros(n)\\n        hs_detected = np.zeros(n)\\n        ihs_detected = np.zeros(n)\\n\\n        high_idx = argrelextrema(highs, np.greater, order=5)[0]\\n        low_idx = argrelextrema(lows, np.less, order=5)[0]\\n\\n        for i in range(lookback, n):\\n            current_atr = _chart_safe_atr(atr.iloc[i], float(closes[i]))\\n\\n            recent_hi = high_idx[(high_idx >= i - lookback) & (high_idx < i)]\\n            if len(recent_hi) >= 2:\\n                h1, h2 = highs[recent_hi[-2]], highs[recent_hi[-1]]\\n                separation = recent_hi[-1] - recent_hi[-2]\\n                if abs(h1 - h2) < current_atr * 0.5 and 5 <= separation <= lookback // 2:\\n                    double_top[i] = 1\\n                    double_top_dist[i] = (closes[i] - min(h1, h2)) / current_atr\\n\\n            recent_lo = low_idx[(low_idx >= i - lookback) & (low_idx < i)]\\n            if len(recent_lo) >= 2:\\n                l1, l2 = lows[recent_lo[-2]], lows[recent_lo[-1]]\\n                separation = recent_lo[-1] - recent_lo[-2]\\n                if abs(l1 - l2) < current_atr * 0.5 and 5 <= separation <= lookback // 2:\\n                    double_bottom[i] = 1\\n                    double_bottom_dist[i] = (max(l1, l2) - closes[i]) / current_atr\\n\\n            if len(recent_hi) >= 3:\\n                p1, p2, p3 = (\\n                    highs[recent_hi[-3]],\\n                    highs[recent_hi[-2]],\\n                    highs[recent_hi[-1]],\\n                )\\n                if p2 > p1 and p2 > p3 and abs(p1 - p3) < current_atr:\\n                    hs_detected[i] = 1\\n\\n            if len(recent_lo) >= 3:\\n                t1, t2, t3 = (\\n                    lows[recent_lo[-3]],\\n                    lows[recent_lo[-2]],\\n                    lows[recent_lo[-1]],\\n                )\\n                if t2 < t1 and t2 < t3 and abs(t1 - t3) < current_atr:\\n                    ihs_detected[i] = 1\\n\\n        out[\\\"chp_double_top\\\"] = double_top\\n        out[\\\"chp_double_bottom\\\"] = double_bottom\\n        out[\\\"chp_double_top_dist\\\"] = double_top_dist\\n        out[\\\"chp_double_bottom_dist\\\"] = double_bottom_dist\\n        out[\\\"chp_hs_detected\\\"] = hs_detected\\n        out[\\\"chp_ihs_detected\\\"] = ihs_detected\\n\\n        return out\\n\\n    def _compute_breakouts(\\n        self,\\n        df: pd.DataFrame,\\n        atr: pd.Series,\\n        lookback: int = 20,\\n    ) -> pd.DataFrame:\\n        out = pd.DataFrame(index=df.index)\\n        closes = df[\\\"close\\\"].values\\n        n = len(df)\\n\\n        bo_at_high = np.zeros(n)\\n        bo_at_low = np.zeros(n)\\n        bo_volume_conf = np.zeros(n)\\n        bo_score = np.zeros(n)\\n\\n        for i in range(lookback, n):\\n            window_closes = closes[i - lookback : i]\\n            range_high = np.max(window_closes)\\n            range_low = np.min(window_closes)\\n            current = closes[i]\\n            current_atr = _chart_safe_atr(atr.iloc[i], float(current))\\n\\n            if current >= range_high - current_atr * 0.3:\\n                bo_at_high[i] = 1\\n\\n            if current <= range_low + current_atr * 0.3:\\n                bo_at_low[i] = 1\\n\\n            if \\\"volume\\\" in df.columns:\\n                vol = df[\\\"volume\\\"].values\\n                avg_vol = np.mean(vol[i - lookback : i])\\n                if vol[i] > avg_vol * 1.5:\\n                    bo_volume_conf[i] = 1\\n\\n            bo_score[i] = float(bo_at_high[i]) * 0.5 + float(bo_volume_conf[i]) * 0.5\\n\\n        out[\\\"bo_at_high\\\"] = bo_at_high\\n        out[\\\"bo_at_low\\\"] = bo_at_low\\n        out[\\\"bo_volume_confirmation\\\"] = bo_volume_conf\\n        out[\\\"bo_breakout_score\\\"] = bo_score\\n\\n        return out\\n\", \"feature_store/transformer_btcusd/__init__.py\": \"\\\"\\\"\\\"Per-TF BTCUSD transformer feature pipeline (train/serve parity with Colab).\\\"\\\"\\\"\\n\\nfrom feature_store.transformer_btcusd.contract import (\\n    CANDLE_CLASS_NAMES,\\n    CONTINUOUS_LABEL_COLS,\\n    FEATURE_COLS,\\n    FEATURE_CONTRACT_VERSION,\\n    FUSION_INPUT_RESOLUTIONS,\\n    FUTURE_CANDLE_COL,\\n    HTF_STRUCTURE_COLS,\\n    NATIVE_STRUCTURE_COLS,\\n    ONNX_OUTPUT_NAMES,\\n    PATH_LABEL_HORIZON_BARS,\\n    REGIME_NAMES,\\n    RESOLUTION_MINUTES,\\n    STRUCTURE_OUTCOME_COL,\\n    STRUCTURE_OUTCOME_NAMES,\\n    SUPPORTED_RESOLUTIONS,\\n    TF_KEYS,\\n    candle_family_from_class,\\n    compute_long_edge,\\n    compute_path_edge,\\n    compute_short_edge,\\n    default_training_config,\\n    feature_cols_for_resolution,\\n    path_favorable_adverse,\\n)\\nfrom feature_store.transformer_btcusd.features import (\\n    add_features,\\n    assemble_raw_frame,\\n    classify_candle_shape,\\n    prepare_raw_frame,\\n    add_candle_structure_features,\\n)\\nfrom feature_store.transformer_btcusd.inference import (\\n    build_candle_class_window,\\n    build_continuous_window,\\n    build_inference_window,\\n    load_feature_config,\\n    unstandardize_continuous,\\n)\\nfrom feature_store.transformer_btcusd.structure import (\\n    add_htf_structure_features,\\n    add_market_structure_features,\\n)\\n\\n__all__ = [\\n    \\\"CANDLE_CLASS_NAMES\\\",\\n    \\\"CONTINUOUS_LABEL_COLS\\\",\\n    \\\"FEATURE_COLS\\\",\\n    \\\"FEATURE_CONTRACT_VERSION\\\",\\n    \\\"FUTURE_CANDLE_COL\\\",\\n    \\\"HTF_STRUCTURE_COLS\\\",\\n    \\\"NATIVE_STRUCTURE_COLS\\\",\\n    \\\"ONNX_OUTPUT_NAMES\\\",\\n    \\\"PATH_LABEL_HORIZON_BARS\\\",\\n    \\\"REGIME_NAMES\\\",\\n    \\\"RESOLUTION_MINUTES\\\",\\n    \\\"STRUCTURE_OUTCOME_COL\\\",\\n    \\\"STRUCTURE_OUTCOME_NAMES\\\",\\n    \\\"SUPPORTED_RESOLUTIONS\\\",\\n    \\\"FUSION_INPUT_RESOLUTIONS\\\",\\n    \\\"TF_KEYS\\\",\\n    \\\"add_candle_structure_features\\\",\\n    \\\"add_features\\\",\\n    \\\"add_htf_structure_features\\\",\\n    \\\"add_market_structure_features\\\",\\n    \\\"assemble_raw_frame\\\",\\n    \\\"build_candle_class_window\\\",\\n    \\\"build_continuous_window\\\",\\n    \\\"build_inference_window\\\",\\n    \\\"candle_family_from_class\\\",\\n    \\\"classify_candle_shape\\\",\\n    \\\"compute_long_edge\\\",\\n    \\\"compute_path_edge\\\",\\n    \\\"compute_short_edge\\\",\\n    \\\"default_training_config\\\",\\n    \\\"feature_cols_for_resolution\\\",\\n    \\\"load_feature_config\\\",\\n    \\\"path_favorable_adverse\\\",\\n    \\\"prepare_raw_frame\\\",\\n    \\\"unstandardize_continuous\\\",\\n]\\n\", \"feature_store/transformer_btcusd/contract.py\": \"\\\"\\\"\\\"Constants shared between per-TF Colab training and agent inference.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any, Dict, Mapping, Optional, Tuple\\n\\n# Bump when FEATURE_COLS, label heads, or ONNX outputs change (requires retrain).\\nFEATURE_CONTRACT_VERSION_V6 = \\\"transformer_btcusd_per_tf_features_v6\\\"\\nFEATURE_CONTRACT_VERSION_V7 = \\\"transformer_btcusd_per_tf_features_v7\\\"\\nFEATURE_CONTRACT_VERSION_V8 = \\\"transformer_btcusd_per_tf_features_v8\\\"\\nFEATURE_CONTRACT_VERSION_V9 = \\\"transformer_btcusd_per_tf_features_v9\\\"\\nFEATURE_CONTRACT_VERSION_V10 = \\\"transformer_btcusd_mtf_fusion_v10\\\"\\nFEATURE_CONTRACT_VERSION_V11 = \\\"transformer_btcusd_mtf_fusion_v11\\\"\\nFEATURE_CONTRACT_VERSION_V12 = \\\"transformer_btcusd_mtf_fusion_v12\\\"\\nFEATURE_CONTRACT_VERSION_V13 = \\\"transformer_btcusd_mtf_fusion_v13\\\"\\nFEATURE_CONTRACT_VERSION_V14 = \\\"transformer_btcusd_mtf_fusion_v14\\\"\\nFEATURE_CONTRACT_VERSION_V15 = \\\"transformer_btcusd_mtf_fusion_v15\\\"\\n# Live per-TF bundles remain v9. The fused live path uses V11 (2-class, no h10m head).\\n# V12/V13/V14/V15 are Label V2 research training only and must not load in FusionModelNode.\\nFEATURE_CONTRACT_VERSION = FEATURE_CONTRACT_VERSION_V9\\n\\nSUPPORTED_RESOLUTIONS: Tuple[str, ...] = (\\\"5m\\\", \\\"15m\\\", \\\"30m\\\", \\\"1h\\\", \\\"2h\\\")\\nFUSION_INPUT_RESOLUTIONS: Tuple[str, ...] = (\\\"5m\\\", \\\"10m\\\", \\\"30m\\\", \\\"1h\\\", \\\"2h\\\")\\nFUSION_BUNDLE_DIR_NAME = \\\"JackSparrow_Transformer_BTCUSD_mtf_fusion\\\"\\nFUSION_V12_BUNDLE_DIR_NAME = \\\"JackSparrow_Transformer_BTCUSD_mtf_fusion_v12\\\"\\nFUSION_V13_BUNDLE_DIR_NAME = \\\"JackSparrow_Transformer_BTCUSD_mtf_fusion_v13\\\"\\nFUSION_V14_BUNDLE_DIR_NAME = \\\"JackSparrow_Transformer_BTCUSD_mtf_fusion_v14\\\"\\nFUSION_V15_BUNDLE_DIR_NAME = \\\"JackSparrow_Transformer_BTCUSD_mtf_fusion_v15\\\"\\nFUSION_MODEL_FAMILY = \\\"jacksparrow_transformer_btcusd_mtf_fusion\\\"\\nFUSION_ONNX_FILENAME = \\\"btcusd_mtf_fusion.onnx\\\"\\nFUSION_TARGET_WINDOW_MINUTES: int = 10 * 60\\nFUSION_EMBARGO_BARS: int = 24\\n\\nRESOLUTION_MINUTES: Dict[str, int] = {\\n    \\\"5m\\\": 5,\\n    \\\"10m\\\": 10,\\n    \\\"15m\\\": 15,\\n    \\\"30m\\\": 30,\\n    \\\"1h\\\": 60,\\n    \\\"2h\\\": 120,\\n}\\n\\nTF_KEYS: Tuple[str, ...] = tuple(f\\\"tf_{r}\\\" for r in SUPPORTED_RESOLUTIONS)\\n\\nHTF_SOURCE_TFS: Tuple[str, ...] = (\\\"15m\\\", \\\"30m\\\", \\\"1h\\\", \\\"2h\\\")\\nHTF_FEATURE_FIELDS: Tuple[str, ...] = (\\n    \\\"structure_bias\\\",\\n    \\\"trend_efficiency\\\",\\n    \\\"ema21_slope_atr\\\",\\n    \\\"dist_support_atr\\\",\\n    \\\"dist_resistance_atr\\\",\\n    \\\"range_width_atr\\\",\\n)\\n\\nNATIVE_STRUCTURE_COLS: Tuple[str, ...] = (\\n    \\\"hh_count\\\",\\n    \\\"hl_count\\\",\\n    \\\"lh_count\\\",\\n    \\\"ll_count\\\",\\n    \\\"structure_bias\\\",\\n    \\\"last_swing_dir\\\",\\n    \\\"bars_since_swing\\\",\\n    \\\"swing_amp_atr\\\",\\n    \\\"unconfirmed_ext_atr\\\",\\n    \\\"trend_efficiency\\\",\\n    \\\"displacement_atr\\\",\\n    \\\"pct_with_trend\\\",\\n    \\\"price_vs_ema9_atr\\\",\\n    \\\"price_vs_ema21_atr\\\",\\n    \\\"price_vs_ema50_atr\\\",\\n    \\\"price_vs_ema200_atr\\\",\\n    \\\"ema9_vs_21_atr\\\",\\n    \\\"ema21_vs_50_atr\\\",\\n    \\\"ema50_vs_200_atr\\\",\\n    \\\"ema21_slope_atr\\\",\\n    \\\"ema50_slope_atr\\\",\\n    \\\"dist_to_support_atr\\\",\\n    \\\"dist_to_resistance_atr\\\",\\n    \\\"support_touch_count\\\",\\n    \\\"resistance_touch_count\\\",\\n    \\\"range_width_atr\\\",\\n    \\\"range_width_pctile\\\",\\n    \\\"atr_contraction\\\",\\n    \\\"breakout_size_atr\\\",\\n    \\\"breakout_vol_ratio\\\",\\n    \\\"pre_breakout_comp\\\",\\n    \\\"bars_since_breakout\\\",\\n    \\\"retest_dist_atr\\\",\\n    \\\"failed_break\\\",\\n    \\\"peak_diff_atr\\\",\\n    \\\"trough_diff_atr\\\",\\n    \\\"peak_sep_bars\\\",\\n    \\\"trough_sep_bars\\\",\\n    \\\"dist_neck_atr\\\",\\n    \\\"high_slope_atr\\\",\\n    \\\"low_slope_atr\\\",\\n    \\\"convergence\\\",\\n    \\\"width_now_atr\\\",\\n    \\\"pole_disp_atr\\\",\\n    \\\"flag_width_atr\\\",\\n    \\\"flag_slope_atr\\\",\\n)\\n\\nHTF_STRUCTURE_COLS: Tuple[str, ...] = tuple(\\n    f\\\"htf_{tf}_{field}\\\" for tf in HTF_SOURCE_TFS for field in HTF_FEATURE_FIELDS\\n)\\n\\n_V4_FEATURE_COLS: Tuple[str, ...] = (\\n    \\\"ret_1\\\",\\n    \\\"rv_16\\\",\\n    \\\"rv_96\\\",\\n    \\\"ema50_dist_pct\\\",\\n    \\\"macd_hist\\\",\\n    \\\"rsi_14\\\",\\n    \\\"adx_14\\\",\\n    \\\"obv_z\\\",\\n    \\\"vol_z\\\",\\n    \\\"body_ratio\\\",\\n    \\\"upper_wick_ratio\\\",\\n    \\\"lower_wick_ratio\\\",\\n    \\\"close_loc\\\",\\n    \\\"range_atr\\\",\\n    \\\"body_atr\\\",\\n    \\\"gap_atr\\\",\\n    \\\"inside_bar\\\",\\n    \\\"outside_bar\\\",\\n    \\\"engulf_score\\\",\\n    \\\"hour_sin\\\",\\n    \\\"hour_cos\\\",\\n    \\\"dow_sin\\\",\\n    \\\"dow_cos\\\",\\n    \\\"dist_to_resistance_pct\\\",\\n    \\\"dist_to_support_pct\\\",\\n    \\\"funding_rate\\\",\\n    \\\"oi_z\\\",\\n    \\\"funding_zscore\\\",\\n    \\\"funding_mom\\\",\\n    \\\"funding_rate_roc\\\",\\n    \\\"oi_change_2\\\",\\n    \\\"oi_delta_z\\\",\\n    \\\"oi_price_divergence\\\",\\n    \\\"oi_acceleration\\\",\\n    \\\"funding_x_oi\\\",\\n)\\n\\n# Native all-TF continuous columns (v4 plus causal structure/geometry).\\nFEATURE_COLS: Tuple[str, ...] = _V4_FEATURE_COLS + NATIVE_STRUCTURE_COLS\\n\\n# v14 research continuous vector: drop duplicate scale encodings and pattern flags.\\nV14_DROPPED_COLS: Tuple[str, ...] = (\\n    \\\"ema50_dist_pct\\\",\\n    \\\"dist_to_support_pct\\\",\\n    \\\"dist_to_resistance_pct\\\",\\n    \\\"funding_rate\\\",\\n)\\n\\nV14_FEATURE_GROUP_ORDER: Tuple[str, ...] = (\\n    \\\"price\\\",\\n    \\\"volatility\\\",\\n    \\\"trend\\\",\\n    \\\"momentum\\\",\\n    \\\"structure\\\",\\n    \\\"candle\\\",\\n    \\\"chart\\\",\\n    \\\"flow\\\",\\n    \\\"context\\\",\\n)\\n\\nV14_FEATURE_GROUPS: Dict[str, Tuple[str, ...]] = {\\n    \\\"price\\\": (\\\"ret_1\\\",),\\n    \\\"volatility\\\": (\\\"rv_16\\\", \\\"rv_96\\\", \\\"atr_contraction\\\"),\\n    \\\"trend\\\": (\\n        \\\"adx_14\\\",\\n        \\\"trend_efficiency\\\",\\n        \\\"displacement_atr\\\",\\n        \\\"pct_with_trend\\\",\\n        \\\"price_vs_ema9_atr\\\",\\n        \\\"price_vs_ema21_atr\\\",\\n        \\\"price_vs_ema50_atr\\\",\\n        \\\"price_vs_ema200_atr\\\",\\n        \\\"ema9_vs_21_atr\\\",\\n        \\\"ema21_vs_50_atr\\\",\\n        \\\"ema50_vs_200_atr\\\",\\n        \\\"ema21_slope_atr\\\",\\n        \\\"ema50_slope_atr\\\",\\n    ),\\n    \\\"momentum\\\": (\\\"macd_hist\\\", \\\"rsi_14\\\"),\\n    \\\"structure\\\": (\\n        \\\"hh_count\\\",\\n        \\\"hl_count\\\",\\n        \\\"lh_count\\\",\\n        \\\"ll_count\\\",\\n        \\\"structure_bias\\\",\\n        \\\"last_swing_dir\\\",\\n        \\\"bars_since_swing\\\",\\n        \\\"swing_amp_atr\\\",\\n        \\\"unconfirmed_ext_atr\\\",\\n        \\\"dist_to_support_atr\\\",\\n        \\\"dist_to_resistance_atr\\\",\\n        \\\"support_touch_count\\\",\\n        \\\"resistance_touch_count\\\",\\n        \\\"range_width_atr\\\",\\n        \\\"range_width_pctile\\\",\\n    ),\\n    \\\"candle\\\": (\\n        \\\"body_ratio\\\",\\n        \\\"upper_wick_ratio\\\",\\n        \\\"lower_wick_ratio\\\",\\n        \\\"close_loc\\\",\\n        \\\"range_atr\\\",\\n        \\\"body_atr\\\",\\n        \\\"gap_atr\\\",\\n        \\\"inside_bar\\\",\\n        \\\"outside_bar\\\",\\n        \\\"engulf_score\\\",\\n    ),\\n    \\\"chart\\\": (\\n        \\\"breakout_size_atr\\\",\\n        \\\"breakout_vol_ratio\\\",\\n        \\\"pre_breakout_comp\\\",\\n        \\\"bars_since_breakout\\\",\\n        \\\"retest_dist_atr\\\",\\n        \\\"failed_break\\\",\\n        \\\"peak_diff_atr\\\",\\n        \\\"trough_diff_atr\\\",\\n        \\\"peak_sep_bars\\\",\\n        \\\"trough_sep_bars\\\",\\n        \\\"dist_neck_atr\\\",\\n        \\\"high_slope_atr\\\",\\n        \\\"low_slope_atr\\\",\\n        \\\"convergence\\\",\\n        \\\"width_now_atr\\\",\\n        \\\"pole_disp_atr\\\",\\n        \\\"flag_width_atr\\\",\\n        \\\"flag_slope_atr\\\",\\n    ),\\n    \\\"flow\\\": (\\n        \\\"obv_z\\\",\\n        \\\"vol_z\\\",\\n        \\\"oi_z\\\",\\n        \\\"funding_zscore\\\",\\n        \\\"funding_mom\\\",\\n        \\\"funding_rate_roc\\\",\\n        \\\"oi_change_2\\\",\\n        \\\"oi_delta_z\\\",\\n        \\\"oi_price_divergence\\\",\\n        \\\"oi_acceleration\\\",\\n        \\\"funding_x_oi\\\",\\n    ),\\n    \\\"context\\\": (\\\"hour_sin\\\", \\\"hour_cos\\\", \\\"dow_sin\\\", \\\"dow_cos\\\"),\\n}\\n\\nV14_COUNT_COLS: Tuple[str, ...] = (\\n    \\\"hh_count\\\",\\n    \\\"hl_count\\\",\\n    \\\"lh_count\\\",\\n    \\\"ll_count\\\",\\n    \\\"bars_since_swing\\\",\\n    \\\"support_touch_count\\\",\\n    \\\"resistance_touch_count\\\",\\n    \\\"bars_since_breakout\\\",\\n    \\\"peak_sep_bars\\\",\\n    \\\"trough_sep_bars\\\",\\n)\\n\\nV14_WINDOW_ZSCORE_COLS: Tuple[str, ...] = (\\\"ret_1\\\",)\\nV14_STATE_GROUP_ORDER: Tuple[str, ...] = (\\\"candle_state\\\", \\\"chart_state\\\")\\nV14_STRUCTURE_LOOKBACK_BARS: int = 96\\n\\nPATH_LABEL_COLS: Tuple[str, ...] = (\\n    \\\"mfe\\\",\\n    \\\"mae\\\",\\n    \\\"future_volatility\\\",\\n    \\\"trend_strength\\\",\\n    \\\"drawdown_before_mfe\\\",\\n    \\\"future_oi_change_pct\\\",\\n    \\\"future_volume_change_pct\\\",\\n    \\\"candle_follow_through_atr\\\",\\n    \\\"structure_delta\\\",\\n)\\n\\nCONTINUOUS_LABEL_COLS: Tuple[str, ...] = PATH_LABEL_COLS\\n\\nPATH_LABEL_HORIZON_BARS: int = 8\\n\\n# Legacy 8h reference (btcusd_15m_transformer used 32 bars on 15m).\\nREFERENCE_LABEL_HORIZON_MINUTES: int = 480\\n\\nDEFAULT_PATH_LABEL_HORIZON_MINUTES: Dict[str, int] = {\\n    \\\"5m\\\": 240,\\n    \\\"15m\\\": 240,\\n    \\\"30m\\\": 480,\\n    \\\"1h\\\": 480,\\n    \\\"2h\\\": 480,\\n}\\n\\n# Volume change excluded from loss (dominates shared encoder).\\nDEFAULT_CONTINUOUS_LOSS_WEIGHTS: Dict[str, float] = {\\n    \\\"mfe\\\": 0.5,\\n    \\\"mae\\\": 0.5,\\n    \\\"future_volatility\\\": 1.0,\\n    \\\"trend_strength\\\": 0.5,\\n    \\\"drawdown_before_mfe\\\": 0.5,\\n    \\\"future_oi_change_pct\\\": 0.25,\\n    \\\"future_volume_change_pct\\\": 0.0,\\n    \\\"candle_follow_through_atr\\\": 0.5,\\n    \\\"structure_delta\\\": 0.5,\\n}\\n\\nREGIME_NAMES: Dict[int, str] = {\\n    0: \\\"LOW\\\",\\n    1: \\\"NORMAL\\\",\\n    2: \\\"HIGH\\\",\\n    3: \\\"EXTREME\\\",\\n}\\n\\nCANDLE_CLASS_COL = \\\"candle_class_id\\\"\\nCANDLE_CLASS_CARDINALITY = 13\\nCANDLE_EMBED_DIM = 8\\nCHART_EMBED_DIM = 8\\n\\nCANDLE_CLASS_NAMES: Dict[int, str] = {\\n    0: \\\"FLAT_ZERO_RANGE\\\",\\n    1: \\\"DOJI_DRAGONFLY\\\",\\n    2: \\\"DOJI_GRAVESTONE\\\",\\n    3: \\\"DOJI_STANDARD\\\",\\n    4: \\\"MARUBOZU_BULL\\\",\\n    5: \\\"MARUBOZU_BEAR\\\",\\n    6: \\\"HAMMER_SHAPE\\\",\\n    7: \\\"INV_HAMMER_SHAPE\\\",\\n    8: \\\"SPINNING_TOP\\\",\\n    9: \\\"BELT_HOLD_BULL\\\",\\n    10: \\\"BELT_HOLD_BEAR\\\",\\n    11: \\\"STANDARD_BULL\\\",\\n    12: \\\"STANDARD_BEAR\\\",\\n}\\n\\n# Discrete training/inference columns (not continuous ONNX heads).\\nSTRUCTURE_OUTCOME_COL = \\\"future_structure_outcome\\\"\\nFUTURE_CANDLE_COL = \\\"future_candle_class\\\"\\nSTRUCTURE_OUTCOME_CARDINALITY = 6\\nN_AUX_CLASS_HEADS = 3  # vol regime + structure outcome + next-bar candle\\n\\n# v7 next-candle structure (kept for legacy 5m ONNX decode).\\nNEXT_DIRECTION_COL = \\\"next_direction\\\"\\nNEXT_BODY_COL = \\\"next_body\\\"\\nNEXT_WICK_COL = \\\"next_wick\\\"\\nNEXT_RANGE_COL = \\\"next_range\\\"\\nCHART_PATTERN_COL = \\\"chart_pattern_id\\\"\\nVOLUME_STATE_COL = \\\"volume_state\\\"\\nVOLUME_CONFIRMS_COL = \\\"volume_confirms\\\"\\nPATTERN_ACTIVE_COL = \\\"pattern_active\\\"\\nSAMPLE_WEIGHT_COL = \\\"sample_weight\\\"\\n\\n# v8 wall-clock behavior packets on the 5m grid (bars).\\nHORIZON_SPECS: Tuple[Tuple[str, int], ...] = (\\n    (\\\"h5m\\\", 1),\\n    (\\\"h10m\\\", 2),\\n    (\\\"h15m\\\", 3),\\n    (\\\"h30m\\\", 6),\\n    (\\\"h1h\\\", 12),\\n    (\\\"h2h\\\", 24),\\n)\\nHORIZON_KEYS: Tuple[str, ...] = tuple(key for key, _ in HORIZON_SPECS)\\nHORIZON_BARS_5M: Tuple[int, ...] = tuple(bars for _, bars in HORIZON_SPECS)\\nN_HORIZONS: int = len(HORIZON_SPECS)\\nMAX_V8_HORIZON_BARS: int = max(HORIZON_BARS_5M)\\nHORIZON_CONTINUOUS_FIELDS: Tuple[str, ...] = (\\\"mfe\\\", \\\"mae\\\", \\\"vol\\\", \\\"trend_strength\\\")\\nHORIZON_DIR_COLS: Tuple[str, ...] = tuple(f\\\"{key}_dir\\\" for key in HORIZON_KEYS)\\nHORIZON_STRUCTURE_COLS: Tuple[str, ...] = tuple(\\n    f\\\"{key}_structure\\\" for key in HORIZON_KEYS\\n)\\nV8_CONTINUOUS_LABEL_COLS: Tuple[str, ...] = tuple(\\n    f\\\"{key}_{field}\\\"\\n    for key in HORIZON_KEYS\\n    for field in HORIZON_CONTINUOUS_FIELDS\\n)\\nHORIZON_DIR_COLS_V7: Tuple[str, ...] = (\\n    \\\"horizon_t1_dir\\\",\\n    \\\"horizon_t3_dir\\\",\\n    \\\"horizon_t6_dir\\\",\\n    \\\"horizon_t12_dir\\\",\\n    \\\"horizon_t24_dir\\\",\\n)\\n\\nNEXT_DIRECTION_CARDINALITY_V8 = 3\\nNEXT_DIRECTION_CARDINALITY = 5\\nNEXT_BODY_CARDINALITY = 3\\nNEXT_WICK_CARDINALITY = 4\\nNEXT_RANGE_CARDINALITY = 3\\nVOLUME_STATE_CARDINALITY = 3\\nCHART_PATTERN_CARDINALITY = 9\\n\\nNEXT_DIRECTION_NAMES_V8: Dict[int, str] = {0: \\\"BEARISH\\\", 1: \\\"NEUTRAL\\\", 2: \\\"BULLISH\\\"}\\nNEXT_DIRECTION_NAMES: Dict[int, str] = {\\n    0: \\\"STRONG_DOWN\\\",\\n    1: \\\"DOWN\\\",\\n    2: \\\"NEUTRAL\\\",\\n    3: \\\"UP\\\",\\n    4: \\\"STRONG_UP\\\",\\n}\\nBULLISH_DIRECTION_NAMES = frozenset({\\\"UP\\\", \\\"STRONG_UP\\\", \\\"BULLISH\\\"})\\nBEARISH_DIRECTION_NAMES = frozenset({\\\"DOWN\\\", \\\"STRONG_DOWN\\\", \\\"BEARISH\\\"})\\nNEUTRAL_DIRECTION_NAMES = frozenset({\\\"NEUTRAL\\\"})\\nNEXT_BODY_NAMES: Dict[int, str] = {0: \\\"SMALL\\\", 1: \\\"MEDIUM\\\", 2: \\\"LARGE\\\"}\\nNEXT_WICK_NAMES: Dict[int, str] = {\\n    0: \\\"BALANCED\\\",\\n    1: \\\"UPPER_REJECTION\\\",\\n    2: \\\"LOWER_REJECTION\\\",\\n    3: \\\"BOTH_REJECTION\\\",\\n}\\nNEXT_RANGE_NAMES: Dict[int, str] = {0: \\\"COMPRESSED\\\", 1: \\\"NORMAL\\\", 2: \\\"EXPANDED\\\"}\\nVOLUME_STATE_NAMES: Dict[int, str] = {0: \\\"DRY\\\", 1: \\\"NORMAL\\\", 2: \\\"EXPANSION\\\"}\\nCHART_PATTERN_NAMES: Dict[int, str] = {\\n    0: \\\"NONE\\\",\\n    1: \\\"FLAG_BULL\\\",\\n    2: \\\"FLAG_BEAR\\\",\\n    3: \\\"TRIANGLE\\\",\\n    4: \\\"DOUBLE_TOP\\\",\\n    5: \\\"DOUBLE_BOTTOM\\\",\\n    6: \\\"CHANNEL\\\",\\n    7: \\\"BREAKOUT\\\",\\n    8: \\\"FAILED_BREAK\\\",\\n}\\n\\nVOL_Z_EXPANSION = 0.5\\nVOL_Z_DRY = -0.5\\nBREAKOUT_VOL_CONFIRM = 1.2\\nHORIZON_DIR_ATR_WEAK = 0.5\\nHORIZON_DIR_ATR_STRONG = 2.0\\nHORIZON_DIR_ATR_DEADZONE = HORIZON_DIR_ATR_WEAK\\n# Fusion heads: h2h uses a wider ignore band so chop does not train the trunk.\\nFUSION_HORIZON_ATR_WEAK: Dict[str, float] = {\\n    \\\"h30m\\\": 0.5,\\n    \\\"h1h\\\": 0.5,\\n    \\\"h2h\\\": 0.75,\\n}\\n\\nV7_STRUCTURE_LOSS_WEIGHTS: Dict[str, float] = {\\n    \\\"direction\\\": 1.0,\\n    \\\"body\\\": 0.5,\\n    \\\"wick\\\": 0.5,\\n    \\\"range\\\": 0.5,\\n    \\\"pattern\\\": 0.5,\\n    \\\"path\\\": 1.0,\\n    \\\"volume_state\\\": 0.25,\\n    \\\"pattern_validates\\\": 0.25,\\n    \\\"horizon\\\": 0.35,\\n}\\n\\nV8_STRUCTURE_LOSS_WEIGHTS: Dict[str, float] = {\\n    \\\"path\\\": 1.0,\\n    \\\"direction\\\": 1.0,\\n    \\\"structure\\\": 0.5,\\n    \\\"volume_state\\\": 0.25,\\n}\\nV9_STRUCTURE_LOSS_WEIGHTS: Dict[str, float] = {\\n    \\\"path\\\": 1.0,\\n    \\\"direction\\\": 1.0,\\n    \\\"structure\\\": 0.5,\\n    \\\"volume_state\\\": 0.25,\\n    \\\"pattern\\\": 0.25,\\n}\\n\\nV7_CONTINUOUS_LOSS_WEIGHTS: Dict[str, float] = {\\n    \\\"mfe\\\": 0.5,\\n    \\\"mae\\\": 0.5,\\n    \\\"future_volatility\\\": 1.0,\\n    \\\"trend_strength\\\": 0.5,\\n    \\\"drawdown_before_mfe\\\": 0.5,\\n    \\\"future_oi_change_pct\\\": 0.25,\\n    \\\"future_volume_change_pct\\\": 0.25,\\n    \\\"candle_follow_through_atr\\\": 0.5,\\n    \\\"structure_delta\\\": 0.5,\\n}\\n\\nSTRUCTURE_OUTCOME_NAMES: Dict[int, str] = {\\n    0: \\\"RANGE\\\",\\n    1: \\\"CONTINUATION_LONG\\\",\\n    2: \\\"CONTINUATION_SHORT\\\",\\n    3: \\\"BREAKOUT\\\",\\n    4: \\\"FAILED_BREAK\\\",\\n    5: \\\"REVERSAL\\\",\\n}\\n\\nONNX_OUTPUT_NAMES_V6: Tuple[str, ...] = (\\n    \\\"continuous_pred\\\",\\n    \\\"regime_logits\\\",\\n    \\\"structure_outcome_logits\\\",\\n    \\\"future_candle_logits\\\",\\n)\\n# Live 15m\\u20132h bundles still export v6 heads. 5m research is v8.\\nONNX_OUTPUT_NAMES: Tuple[str, ...] = ONNX_OUTPUT_NAMES_V6\\n\\nONNX_OUTPUT_NAMES_V7_BASE: Tuple[str, ...] = (\\n    \\\"next_direction_logits\\\",\\n    \\\"next_body_logits\\\",\\n    \\\"next_wick_logits\\\",\\n    \\\"next_range_logits\\\",\\n    \\\"next_pattern_logits\\\",\\n    \\\"continuous_pred\\\",\\n    \\\"volume_state_logits\\\",\\n    \\\"pattern_validates_logit\\\",\\n)\\nONNX_OUTPUT_NAMES_V7_HORIZONS: Tuple[str, ...] = (\\n    \\\"horizon_t3_dir_logits\\\",\\n    \\\"horizon_t6_dir_logits\\\",\\n    \\\"horizon_t12_dir_logits\\\",\\n    \\\"horizon_t24_dir_logits\\\",\\n)\\nONNX_OUTPUT_NAMES_V7: Tuple[str, ...] = (\\n    ONNX_OUTPUT_NAMES_V7_BASE + ONNX_OUTPUT_NAMES_V7_HORIZONS\\n)\\nONNX_OUTPUT_NAMES_V8_DIR: Tuple[str, ...] = tuple(\\n    f\\\"{key}_dir_logits\\\" for key in HORIZON_KEYS\\n)\\nONNX_OUTPUT_NAMES_V8_STRUCTURE: Tuple[str, ...] = tuple(\\n    f\\\"{key}_structure_logits\\\" for key in HORIZON_KEYS\\n)\\nONNX_OUTPUT_NAMES_V8: Tuple[str, ...] = (\\n    ONNX_OUTPUT_NAMES_V8_DIR\\n    + ONNX_OUTPUT_NAMES_V8_STRUCTURE\\n    + (\\\"continuous_pred\\\", \\\"volume_state_logits\\\")\\n)\\nONNX_OUTPUT_NAMES_V9: Tuple[str, ...] = ONNX_OUTPUT_NAMES_V8 + (\\\"chart_pattern_logits\\\",)\\n\\n# v11 fused multi-TF model: 2-class BEAR/BULL heads, no MFE/MAE.\\n# 5m/10m stay encoder inputs. h10m is not a trading head. HOLD is not a class.\\nFUSION_HORIZON_SPECS: Tuple[Tuple[str, int], ...] = (\\n    (\\\"h30m\\\", 6),\\n    (\\\"h1h\\\", 12),\\n    (\\\"h2h\\\", 24),\\n)\\nFUSION_HORIZON_KEYS: Tuple[str, ...] = tuple(key for key, _ in FUSION_HORIZON_SPECS)\\nFUSION_HORIZON_BARS_5M: Tuple[int, ...] = tuple(bars for _, bars in FUSION_HORIZON_SPECS)\\nN_FUSION_HORIZONS: int = len(FUSION_HORIZON_SPECS)\\nMAX_FUSION_HORIZON_BARS: int = max(FUSION_HORIZON_BARS_5M)\\nFUSION_DIR_COLS: Tuple[str, ...] = tuple(f\\\"{key}_dir\\\" for key in FUSION_HORIZON_KEYS)\\nFUSION_RETIRED_DIR_COLS: Tuple[str, ...] = (\\\"h10m_dir\\\",)\\nFUSION_IGNORE_INDEX = -1\\nFUSION_DIRECTION_CARDINALITY = 2\\nFUSION_DIRECTION_NAMES: Dict[int, str] = {0: \\\"BEAR\\\", 1: \\\"BULL\\\"}\\nFUSION_POSITION_LONG = \\\"LONG\\\"\\nFUSION_POSITION_SHORT = \\\"SHORT\\\"\\nFUSION_POSITION_HOLD = \\\"HOLD\\\"\\nFUSION_GRADE_HIGH = \\\"HIGH\\\"\\nFUSION_GRADE_MEDIUM = \\\"MEDIUM\\\"\\nFUSION_GRADE_LOW = \\\"LOW\\\"\\n# 2-class chance is 0.50; floors sit above noise (old 0.40/0.45 would auto-pass).\\nFUSION_HIGH_BALANCED_ACC = 0.58\\nFUSION_HIGH_MAX_ECE = 0.08\\nFUSION_MEDIUM_BALANCED_ACC = 0.55\\nFUSION_MIN_PROBABILITY = 0.60\\n# Frozen v10 4-head / 3-class names \\u2014 live fusion must not load this layout.\\nONNX_OUTPUT_NAMES_V10: Tuple[str, ...] = (\\n    \\\"h10m_dir_logits\\\",\\n    \\\"h30m_dir_logits\\\",\\n    \\\"h1h_dir_logits\\\",\\n    \\\"h2h_dir_logits\\\",\\n    \\\"tf_fusion_logits\\\",\\n)\\nONNX_OUTPUT_NAMES_V11: Tuple[str, ...] = tuple(\\n    f\\\"{key}_dir_logits\\\" for key in FUSION_HORIZON_KEYS\\n) + (\\\"tf_fusion_logits\\\",)\\nFUSION_DURATION_ATR_MULT: Dict[str, Tuple[float, float]] = {\\n    \\\"h30m\\\": (1.0, 1.5),\\n    \\\"h1h\\\": (1.5, 2.25),\\n    \\\"h2h\\\": (2.0, 3.0),\\n}\\nFUSION_HORIZON_MINUTES: Dict[str, int] = {\\n    \\\"h30m\\\": 30,\\n    \\\"h1h\\\": 60,\\n    \\\"h2h\\\": 120,\\n}\\n\\n# Label V2 research heads. Live fusion stays h30m/h1h/h2h. Encoder still sees 2h.\\nLABEL_V2_HORIZON_SPECS: Tuple[Tuple[str, int], ...] = (\\n    (\\\"h10m\\\", 2),\\n    (\\\"h15m\\\", 3),\\n    (\\\"h30m\\\", 6),\\n    (\\\"h1h\\\", 12),\\n)\\nLABEL_V2_HORIZON_KEYS: Tuple[str, ...] = tuple(key for key, _ in LABEL_V2_HORIZON_SPECS)\\nLABEL_V2_HORIZON_BARS_5M: Tuple[int, ...] = tuple(\\n    bars for _, bars in LABEL_V2_HORIZON_SPECS\\n)\\nN_LABEL_V2_HORIZONS: int = len(LABEL_V2_HORIZON_SPECS)\\nMAX_LABEL_V2_HORIZON_BARS: int = max(LABEL_V2_HORIZON_BARS_5M)\\nLABEL_V2_DURATION_ATR_MULT: Dict[str, Tuple[float, float]] = {\\n    \\\"h10m\\\": (0.5, 0.75),\\n    \\\"h15m\\\": (0.75, 1.25),\\n    \\\"h30m\\\": (1.0, 1.5),\\n    \\\"h1h\\\": (1.5, 2.25),\\n}\\nLABEL_V2_HORIZON_MINUTES: Dict[str, int] = {\\n    \\\"h10m\\\": 10,\\n    \\\"h15m\\\": 15,\\n    \\\"h30m\\\": 30,\\n    \\\"h1h\\\": 60,\\n}\\n\\n# Label V2 research targets. Live v11 stays 2-class with ignore_index NEUTRAL.\\nLABEL_V2_DIRECTION_CARDINALITY = 3\\nLABEL_V2_DIRECTION_NAMES: Dict[int, str] = {0: \\\"BEAR\\\", 1: \\\"NEUTRAL\\\", 2: \\\"BULL\\\"}\\nLABEL_V2_THETA_GRID: Tuple[float, ...] = (0.25, 0.40, 0.50, 0.60, 0.75, 1.00)\\nLABEL_V2_TP_SL_GRID: Tuple[Tuple[float, float], ...] = (\\n    (0.5, 0.75),\\n    (0.75, 1.25),\\n    (1.0, 1.5),\\n    (1.5, 2.25),\\n    (2.0, 3.0),\\n)\\nLABEL_V2_TP_SL_TP_FIRST = 0\\nLABEL_V2_TP_SL_SL_FIRST = 1\\nLABEL_V2_TP_SL_NEITHER = 2\\nLABEL_V2_TP_SL_AMBIGUOUS = 3\\nLABEL_V2_TP_SL_NAMES: Dict[int, str] = {\\n    LABEL_V2_TP_SL_TP_FIRST: \\\"TP_FIRST\\\",\\n    LABEL_V2_TP_SL_SL_FIRST: \\\"SL_FIRST\\\",\\n    LABEL_V2_TP_SL_NEITHER: \\\"NEITHER\\\",\\n    LABEL_V2_TP_SL_AMBIGUOUS: \\\"AMBIGUOUS\\\",\\n}\\nLABEL_V2_PATH_FIELDS: Tuple[str, ...] = (\\n    \\\"ret\\\",\\n    \\\"long_mfe\\\",\\n    \\\"long_mae\\\",\\n    \\\"short_mfe\\\",\\n    \\\"short_mae\\\",\\n    \\\"persist\\\",\\n    \\\"t_long_mfe\\\",\\n    \\\"t_short_mfe\\\",\\n)\\nLABEL_V2_COLS: Tuple[str, ...] = tuple(\\n    f\\\"{key}_{field}\\\"\\n    for key in LABEL_V2_HORIZON_KEYS\\n    for field in LABEL_V2_PATH_FIELDS\\n)\\nLABEL_V2_MINORITY_RATE = 0.15\\nLABEL_V2_DISAGREE_MIN = 0.02\\nLABEL_V2_PERSIST_RANGE_MIN = 0.10\\n# Frozen from the four-horizon mix report (259,188 labeled 5m bars).\\nLABEL_V2_THETA_FROZEN: Dict[str, float] = {\\n    \\\"h10m\\\": 0.50,\\n    \\\"h15m\\\": 0.50,\\n    \\\"h30m\\\": 0.50,\\n    \\\"h1h\\\": 0.50,\\n}\\nLABEL_V2_REG_FIELDS: Tuple[str, ...] = (\\n    \\\"ret\\\",\\n    \\\"long_mfe\\\",\\n    \\\"long_mae\\\",\\n    \\\"short_mfe\\\",\\n    \\\"short_mae\\\",\\n)\\nLABEL_V2_PATH_LOSS_WEIGHTS: Dict[str, float] = {\\n    \\\"dir\\\": 1.0,\\n    \\\"ret\\\": 1.0,\\n    \\\"long_mfe\\\": 0.35,\\n    \\\"long_mae\\\": 0.35,\\n    \\\"short_mfe\\\": 0.35,\\n    \\\"short_mae\\\": 0.35,\\n}\\n# later_direction_only research pass: train 3-class CE, keep path heads untrained.\\nLABEL_V2_DIRECTION_ONLY_PATH_LOSS_WEIGHTS: Dict[str, float] = {\\n    \\\"dir\\\": 1.0,\\n    \\\"ret\\\": 0.0,\\n    \\\"long_mfe\\\": 0.0,\\n    \\\"long_mae\\\": 0.0,\\n    \\\"short_mfe\\\": 0.0,\\n    \\\"short_mae\\\": 0.0,\\n}\\n# Frozen v12 long-centric path outputs (ret/mfe/mae). Do not reuse for v13.\\nLABEL_V2_V12_REG_FIELDS: Tuple[str, ...] = (\\\"ret\\\", \\\"mfe\\\", \\\"mae\\\")\\n# 3-class chance is ~0.33; research grades only (not a live promote gate).\\nLABEL_V2_HIGH_BALANCED_ACC = 0.45\\nLABEL_V2_MEDIUM_BALANCED_ACC = 0.40\\nONNX_OUTPUT_NAMES_V12: Tuple[str, ...] = (\\n    tuple(f\\\"{key}_dir_logits\\\" for key in FUSION_HORIZON_KEYS)\\n    + tuple(\\n        f\\\"{key}_{field}\\\"\\n        for key in FUSION_HORIZON_KEYS\\n        for field in LABEL_V2_V12_REG_FIELDS\\n    )\\n    + (\\\"tf_fusion_logits\\\",)\\n)\\nONNX_OUTPUT_NAMES_V13: Tuple[str, ...] = (\\n    tuple(f\\\"{key}_dir_logits\\\" for key in FUSION_HORIZON_KEYS)\\n    + tuple(\\n        f\\\"{key}_{field}\\\"\\n        for key in FUSION_HORIZON_KEYS\\n        for field in LABEL_V2_REG_FIELDS\\n    )\\n    + (\\\"tf_fusion_logits\\\",)\\n)\\nONNX_OUTPUT_NAMES_V15: Tuple[str, ...] = (\\n    tuple(f\\\"{key}_dir_logits\\\" for key in LABEL_V2_HORIZON_KEYS)\\n    + tuple(\\n        f\\\"{key}_{field}\\\"\\n        for key in LABEL_V2_HORIZON_KEYS\\n        for field in LABEL_V2_REG_FIELDS\\n    )\\n    + (\\\"tf_fusion_logits\\\",)\\n)\\n\\n# Next-bar candle families for 5m timing (not model classes).\\nCANDLE_FAMILY_DOJI = frozenset({0, 1, 2, 3, 8})\\nCANDLE_FAMILY_BULL = frozenset({4, 6, 9, 11})\\nCANDLE_FAMILY_BEAR = frozenset({5, 7, 10, 12})\\n\\n# Minimum test-set correlation for future_volatility before ONNX export.\\nMIN_EXPORT_VOL_CORR: Dict[str, float] = {\\n    \\\"5m\\\": 0.10,\\n    \\\"15m\\\": 0.10,\\n    \\\"30m\\\": 0.10,\\n    \\\"1h\\\": 0.08,\\n    \\\"2h\\\": 0.08,\\n}\\n\\nPROMOTION_VOL_CORR: float = 0.15\\nPROMOTION_REGIME_ACCURACY: float = 0.35\\n\\nEXPORT_QUALITY_DISCLAIMER = (\\n    \\\"Sanity gates detect broken exports, not trading edge. \\\"\\n    \\\"Promotion tier targets are informational.\\\"\\n)\\n\\nTRANSFORMER_METADATA_FILENAME = \\\"metadata_transformer.json\\\"\\nTRANSFORMER_FEATURE_CONFIG_FILENAME = \\\"feature_config.json\\\"\\n\\n\\ndef candle_family_from_class(class_id: int) -> str:\\n    \\\"\\\"\\\"Map a candle class id to bull / bear / doji for timing modifiers.\\\"\\\"\\\"\\n    cid = int(class_id)\\n    if cid in CANDLE_FAMILY_BULL:\\n        return \\\"bull\\\"\\n    if cid in CANDLE_FAMILY_BEAR:\\n        return \\\"bear\\\"\\n    return \\\"doji\\\"\\n\\n\\ndef compute_path_edge(mfe: float, mae: float) -> float:\\n    \\\"\\\"\\\"Directional edge from predicted path asymmetry (MFE minus MAE).\\n\\n    Alias of :func:`compute_long_edge` kept for train/serve telemetry compatibility.\\n    \\\"\\\"\\\"\\n    return compute_long_edge(mfe, mae)\\n\\n\\ndef compute_long_edge(mfe: float, mae: float) -> float:\\n    \\\"\\\"\\\"Long-side path edge: upside (MFE) minus downside (MAE).\\\"\\\"\\\"\\n    return float(mfe) - float(mae)\\n\\n\\ndef compute_short_edge(mfe: float, mae: float) -> float:\\n    \\\"\\\"\\\"Short-side path edge: downside (MAE) minus upside (MFE).\\\"\\\"\\\"\\n    return float(mae) - float(mfe)\\n\\n\\ndef path_favorable_adverse(\\n    mfe: float,\\n    mae: float,\\n    *,\\n    side: str,\\n) -> Tuple[float, float]:\\n    \\\"\\\"\\\"Return (favorable_pct, adverse_pct) for bracket sizing.\\n\\n    Labels are long-centric (MFE = upside, MAE = downside). For shorts, favorable\\n    excursion is downside (MAE) and adverse is upside (MFE).\\n    \\\"\\\"\\\"\\n    s = str(side or \\\"BUY\\\").strip().upper()\\n    if s in (\\\"SELL\\\", \\\"SHORT\\\", \\\"STRONG_SELL\\\"):\\n        return float(mae), float(mfe)\\n    return float(mfe), float(mae)\\n\\n\\ndef model_family_for_resolution(resolution: str) -> str:\\n    \\\"\\\"\\\"Canonical model_family string for a TF bundle.\\\"\\\"\\\"\\n    res = resolution.strip().lower()\\n    if res == \\\"mtf_fusion\\\":\\n        return FUSION_MODEL_FAMILY\\n    if res not in RESOLUTION_MINUTES:\\n        raise ValueError(f\\\"Unsupported resolution: {resolution!r}\\\")\\n    return f\\\"jacksparrow_transformer_btcusd_{res}\\\"\\n\\n\\ndef onnx_filename_for_resolution(resolution: str) -> str:\\n    res = resolution.strip().lower()\\n    if res == \\\"mtf_fusion\\\":\\n        return FUSION_ONNX_FILENAME\\n    if res not in RESOLUTION_MINUTES:\\n        raise ValueError(f\\\"Unsupported resolution: {resolution!r}\\\")\\n    return f\\\"btcusd_{res}_transformer.onnx\\\"\\n\\n\\ndef bundle_dir_name(resolution: str) -> str:\\n    res = resolution.strip().lower()\\n    if res == \\\"mtf_fusion\\\":\\n        return FUSION_BUNDLE_DIR_NAME\\n    return f\\\"JackSparrow_Transformer_BTCUSD_{res}\\\"\\n\\n\\ndef horizon_bars_for_wall_minutes(wall_minutes: int, resolution_minutes: int) -> int:\\n    \\\"\\\"\\\"Convert wall-clock minutes to native-TF bar count.\\\"\\\"\\\"\\n    return max(1, int(round(int(wall_minutes) / resolution_minutes)))\\n\\n\\ndef fusion_window_len(\\n    resolution: str,\\n    *,\\n    target_minutes: int = FUSION_TARGET_WINDOW_MINUTES,\\n) -> int:\\n    \\\"\\\"\\\"Bar count so one TF window covers ``target_minutes`` of wall-clock time.\\\"\\\"\\\"\\n    res = resolution.strip().lower()\\n    if res not in RESOLUTION_MINUTES:\\n        raise ValueError(f\\\"Unsupported resolution: {resolution!r}\\\")\\n    return horizon_bars_for_wall_minutes(int(target_minutes), RESOLUTION_MINUTES[res])\\n\\n\\ndef fusion_window_lens(\\n    resolutions: Tuple[str, ...] = FUSION_INPUT_RESOLUTIONS,\\n    *,\\n    target_minutes: int = FUSION_TARGET_WINDOW_MINUTES,\\n) -> Dict[str, int]:\\n    \\\"\\\"\\\"Per-TF window lengths derived from the shared wall-clock target.\\\"\\\"\\\"\\n    return {\\n        res: fusion_window_len(res, target_minutes=target_minutes) for res in resolutions\\n    }\\n\\n\\ndef resolve_fusion_window_lens(\\n    window_lens: Optional[Mapping[str, int]] = None,\\n    window_len: Optional[int] = None,\\n    *,\\n    target_minutes: Optional[int] = None,\\n) -> Dict[str, int]:\\n    \\\"\\\"\\\"Prefer an explicit per-TF map; else a uniform int; else the 10h target.\\\"\\\"\\\"\\n    if window_lens:\\n        missing = [res for res in FUSION_INPUT_RESOLUTIONS if res not in window_lens]\\n        if missing:\\n            raise KeyError(f\\\"window_lens missing resolutions: {missing}\\\")\\n        return {res: int(window_lens[res]) for res in FUSION_INPUT_RESOLUTIONS}\\n    if window_len is not None:\\n        width = int(window_len)\\n        return {res: width for res in FUSION_INPUT_RESOLUTIONS}\\n    minutes = (\\n        int(target_minutes)\\n        if target_minutes is not None\\n        else FUSION_TARGET_WINDOW_MINUTES\\n    )\\n    return fusion_window_lens(target_minutes=minutes)\\n\\n\\n# Longest TF window (5m) for PE max_len and 5m decision-clock warm-up.\\nFUSION_WINDOW_LEN: int = fusion_window_len(\\\"5m\\\")\\n\\n\\ndef label_horizon_bars_for_resolution(resolution_minutes: int) -> int:\\n    \\\"\\\"\\\"Legacy helper: 8h wall-clock in bars for a TF grid.\\\"\\\"\\\"\\n    return horizon_bars_for_wall_minutes(REFERENCE_LABEL_HORIZON_MINUTES, resolution_minutes)\\n\\n\\ndef path_label_horizon_bars_for_resolution(resolution: str) -> int:\\n    \\\"\\\"\\\"Path-label forward window in bars for a TF.\\\"\\\"\\\"\\n    res = resolution.strip().lower()\\n    if res not in RESOLUTION_MINUTES:\\n        raise ValueError(f\\\"Unsupported resolution: {resolution!r}\\\")\\n    minutes = RESOLUTION_MINUTES[res]\\n    wall = DEFAULT_PATH_LABEL_HORIZON_MINUTES.get(res, REFERENCE_LABEL_HORIZON_MINUTES)\\n    return horizon_bars_for_wall_minutes(wall, minutes)\\n\\n\\ndef continuous_loss_weights_for_resolution(resolution: str) -> Tuple[float, ...]:\\n    \\\"\\\"\\\"Per-head loss weights aligned with CONTINUOUS_LABEL_COLS (0 = no gradient).\\\"\\\"\\\"\\n    weights = dict(DEFAULT_CONTINUOUS_LOSS_WEIGHTS)\\n    return tuple(float(weights.get(col, 1.0)) for col in CONTINUOUS_LABEL_COLS)\\n\\n\\ndef default_training_config(resolution: str) -> Dict[str, Any]:\\n    \\\"\\\"\\\"Default Colab training config for a single TF model.\\\"\\\"\\\"\\n    res = resolution.strip().lower()\\n    if res not in RESOLUTION_MINUTES:\\n        raise ValueError(f\\\"Unsupported resolution: {resolution!r}\\\")\\n    minutes = RESOLUTION_MINUTES[res]\\n    path_horizon = path_label_horizon_bars_for_resolution(res)\\n    return {\\n        \\\"symbol\\\": \\\"BTCUSD\\\",\\n        \\\"resolution\\\": res,\\n        \\\"resolution_minutes\\\": minutes,\\n        \\\"history_days\\\": 900,\\n        \\\"base_url\\\": \\\"https://api.india.delta.exchange\\\",\\n        \\\"atr_period\\\": 14,\\n        \\\"path_label_horizon_bars\\\": path_horizon,\\n        \\\"label_horizon_minutes_path\\\": DEFAULT_PATH_LABEL_HORIZON_MINUTES.get(\\n            res, REFERENCE_LABEL_HORIZON_MINUTES\\n        ),\\n        \\\"continuous_loss_weights\\\": list(continuous_loss_weights_for_resolution(res)),\\n        \\\"mae_floor_atr_mult\\\": 0.25,\\n        \\\"vol_regime_quantiles\\\": [0.25, 0.5, 0.75],\\n        \\\"window_len\\\": 128,\\n        \\\"stride\\\": 8,\\n        \\\"train_frac\\\": 0.65,\\n        \\\"val_frac\\\": 0.15,\\n        \\\"embargo_bars\\\": path_horizon,\\n        \\\"batch_size\\\": 128,\\n        \\\"epochs\\\": 120,\\n        \\\"lr\\\": 1e-4,\\n        \\\"d_model\\\": 64,\\n        \\\"nhead\\\": 4,\\n        \\\"num_layers\\\": 2,\\n        \\\"dropout\\\": 0.25,\\n        \\\"weight_decay\\\": 1e-2,\\n        \\\"early_stop_patience\\\": 12,\\n        \\\"early_stopping_enabled\\\": True,\\n        \\\"min_derivatives_coverage\\\": 0.5,\\n        \\\"derivatives_coverage_warn\\\": 0.9,\\n        \\\"min_export_vol_corr\\\": MIN_EXPORT_VOL_CORR.get(res, 0.08),\\n        \\\"default_threshold\\\": 0.005,\\n        \\\"seed\\\": 42,\\n    }\\n\\n\\ndef max_label_horizon_bars(\\n    path_label_horizon_bars: int = PATH_LABEL_HORIZON_BARS,\\n) -> int:\\n    \\\"\\\"\\\"Maximum forward bars across all training labels.\\\"\\\"\\\"\\n    return int(path_label_horizon_bars)\\n\\n\\ndef scale_period(period: int, resolution_minutes: int, *, base_minutes: int = 5) -> int:\\n    \\\"\\\"\\\"Scale indicator lookback to preserve wall-clock semantics across TFs.\\\"\\\"\\\"\\n    return max(1, int(round(period * resolution_minutes / base_minutes)))\\n\\n\\ndef feature_cols_for_resolution(resolution: str) -> Tuple[str, ...]:\\n    \\\"\\\"\\\"Continuous feature columns for a TF bundle (5m appends closed HTF context).\\\"\\\"\\\"\\n    res = resolution.strip().lower()\\n    if res not in RESOLUTION_MINUTES:\\n        raise ValueError(f\\\"Unsupported resolution: {resolution!r}\\\")\\n    if res == \\\"5m\\\":\\n        return FEATURE_COLS + HTF_STRUCTURE_COLS\\n    return FEATURE_COLS\\n\\n\\ndef fusion_native_feature_cols() -> Tuple[str, ...]:\\n    \\\"\\\"\\\"Native-TF columns for the fused model (no resampled HTF context).\\\"\\\"\\\"\\n    return FEATURE_COLS\\n\\n\\ndef fusion_feature_cols_v14() -> Tuple[str, ...]:\\n    \\\"\\\"\\\"Disjoint v14 continuous columns (no pattern flags, no duplicate scales).\\\"\\\"\\\"\\n    cols: Tuple[str, ...] = tuple(\\n        name\\n        for group in V14_FEATURE_GROUP_ORDER\\n        for name in V14_FEATURE_GROUPS[group]\\n    )\\n    return cols\\n\\n\\ndef fusion_feature_groups_v14() -> Dict[str, Tuple[str, ...]]:\\n    \\\"\\\"\\\"Copy of the disjoint v14 group map.\\\"\\\"\\\"\\n    return {str(key): tuple(val) for key, val in V14_FEATURE_GROUPS.items()}\\n\\n\\ndef v7_feature_cols_for_resolution(resolution: str) -> Tuple[str, ...]:\\n    \\\"\\\"\\\"Input columns: native features plus causal chart_pattern_id.\\\"\\\"\\\"\\n    return feature_cols_for_resolution(resolution) + (CHART_PATTERN_COL,)\\n\\n\\ndef v8_feature_cols_for_resolution(resolution: str) -> Tuple[str, ...]:\\n    \\\"\\\"\\\"v8 5m input columns (native features plus causal chart_pattern_id).\\\"\\\"\\\"\\n    return v7_feature_cols_for_resolution(resolution)\\n\\n\\ndef v9_feature_cols_for_resolution(resolution: str) -> Tuple[str, ...]:\\n    \\\"\\\"\\\"v9 5m input columns: geometry only; chart_pattern_id is a target head.\\\"\\\"\\\"\\n    return feature_cols_for_resolution(resolution)\\n\\n\\ndef onnx_output_names_for_contract(\\n    contract_version: str,\\n    *,\\n    resolution: str = \\\"5m\\\",\\n) -> Tuple[str, ...]:\\n    \\\"\\\"\\\"ONNX head names for a bundle contract.\\\"\\\"\\\"\\n    ver = str(contract_version or \\\"\\\").strip()\\n    res = resolution.strip().lower()\\n    if ver == FEATURE_CONTRACT_VERSION_V15:\\n        return ONNX_OUTPUT_NAMES_V15\\n    if ver in (FEATURE_CONTRACT_VERSION_V13, FEATURE_CONTRACT_VERSION_V14):\\n        return ONNX_OUTPUT_NAMES_V13\\n    if ver == FEATURE_CONTRACT_VERSION_V12:\\n        return ONNX_OUTPUT_NAMES_V12\\n    if ver == FEATURE_CONTRACT_VERSION_V11 or res == \\\"mtf_fusion\\\":\\n        return ONNX_OUTPUT_NAMES_V11\\n    if ver == FEATURE_CONTRACT_VERSION_V10:\\n        return ONNX_OUTPUT_NAMES_V10\\n    if ver in (FEATURE_CONTRACT_VERSION, FEATURE_CONTRACT_VERSION_V9):\\n        if res == \\\"5m\\\":\\n            return ONNX_OUTPUT_NAMES_V9\\n        return ONNX_OUTPUT_NAMES_V6\\n    if ver == FEATURE_CONTRACT_VERSION_V8:\\n        if res == \\\"5m\\\":\\n            return ONNX_OUTPUT_NAMES_V8\\n        return ONNX_OUTPUT_NAMES_V6\\n    if ver == FEATURE_CONTRACT_VERSION_V7:\\n        if res == \\\"5m\\\":\\n            return ONNX_OUTPUT_NAMES_V7\\n        return ONNX_OUTPUT_NAMES_V7_BASE\\n    return ONNX_OUTPUT_NAMES_V6\\n\\n\\ndef v8_future_leak_cols() -> frozenset:\\n    \\\"\\\"\\\"Label/target columns that must never appear in 5m v8 model inputs.\\\"\\\"\\\"\\n    leaked = set(V8_CONTINUOUS_LABEL_COLS)\\n    leaked.update(HORIZON_DIR_COLS)\\n    leaked.update(HORIZON_STRUCTURE_COLS)\\n    leaked.update(\\n        {\\n            VOLUME_STATE_COL,\\n            NEXT_DIRECTION_COL,\\n            NEXT_BODY_COL,\\n            NEXT_WICK_COL,\\n            NEXT_RANGE_COL,\\n            FUTURE_CANDLE_COL,\\n            \\\"mfe\\\",\\n            \\\"mae\\\",\\n            \\\"future_volatility\\\",\\n            \\\"pattern_validates\\\",\\n            *HORIZON_DIR_COLS_V7,\\n        }\\n    )\\n    return frozenset(leaked)\\n\\n\\ndef expected_direction_from_chart_pattern(pattern_id: int) -> int:\\n    \\\"\\\"\\\"Map chart pattern to expected next-candle direction (0/1/2). Neutral if none.\\\"\\\"\\\"\\n    pid = int(pattern_id)\\n    if pid in (1, 5, 7):  # FLAG_BULL, DOUBLE_BOTTOM, BREAKOUT (unsigned handled elsewhere)\\n        return 2\\n    if pid in (2, 4, 8):  # FLAG_BEAR, DOUBLE_TOP, FAILED_BREAK\\n        return 0\\n    return 1\\n\\n\\ndef default_research_config() -> Dict[str, Any]:\\n    \\\"\\\"\\\"Colab research-pipeline defaults (5m multi-horizon path).\\\"\\\"\\\"\\n    return {\\n        \\\"symbol\\\": \\\"BTCUSD\\\",\\n        \\\"resolution\\\": \\\"5m\\\",\\n        \\\"resolution_minutes\\\": 5,\\n        \\\"base_timeframe\\\": \\\"5m\\\",\\n        \\\"context_timeframes\\\": [\\\"15m\\\", \\\"30m\\\", \\\"1h\\\", \\\"2h\\\"],\\n        \\\"sequence_length\\\": 64,\\n        \\\"prediction_horizon\\\": 1,\\n        \\\"horizon_bars\\\": list(HORIZON_BARS_5M),\\n        \\\"train_ratio\\\": 0.70,\\n        \\\"validation_ratio\\\": 0.15,\\n        \\\"test_ratio\\\": 0.15,\\n        \\\"batch_size\\\": 256,\\n        \\\"epochs\\\": 50,\\n        \\\"learning_rate\\\": 1e-4,\\n        \\\"weight_decay\\\": 1e-4,\\n        \\\"dropout\\\": 0.15,\\n        \\\"early_stopping_patience\\\": 8,\\n        \\\"seed\\\": 42,\\n        \\\"d_model\\\": 64,\\n        \\\"nhead\\\": 4,\\n        \\\"num_layers\\\": 2,\\n        \\\"stride\\\": 4,\\n        \\\"scaler_mode\\\": \\\"train_fit\\\",\\n        \\\"gate_chart_volume\\\": False,\\n        \\\"path_label_horizon_bars\\\": MAX_V8_HORIZON_BARS,\\n        \\\"embargo_bars\\\": MAX_V8_HORIZON_BARS,\\n        \\\"mae_floor_atr_mult\\\": 0.25,\\n        \\\"atr_period\\\": 14,\\n        \\\"history_days\\\": 900,\\n        \\\"base_url\\\": \\\"https://api.india.delta.exchange\\\",\\n        \\\"loss_weights\\\": dict(V9_STRUCTURE_LOSS_WEIGHTS),\\n        \\\"run_optuna\\\": False,\\n        \\\"optuna_trials\\\": 0,\\n        \\\"run_shap\\\": False,\\n        \\\"run_ablations\\\": False,\\n        \\\"run_walk_forward\\\": False,\\n        \\\"walk_forward_folds\\\": 3,\\n    }\\n\\n\\ndef default_fusion_training_config() -> Dict[str, Any]:\\n    \\\"\\\"\\\"Defaults for the single multi-TF fusion trainer.\\\"\\\"\\\"\\n    return {\\n        \\\"symbol\\\": \\\"BTCUSD\\\",\\n        \\\"resolutions\\\": list(FUSION_INPUT_RESOLUTIONS),\\n        \\\"horizon_keys\\\": list(LABEL_V2_HORIZON_KEYS),\\n        \\\"horizon_bars\\\": list(LABEL_V2_HORIZON_BARS_5M),\\n        \\\"window_len\\\": FUSION_WINDOW_LEN,\\n        \\\"window_lens\\\": fusion_window_lens(),\\n        \\\"target_window_minutes\\\": FUSION_TARGET_WINDOW_MINUTES,\\n        \\\"stride\\\": 4,\\n        \\\"train_frac\\\": 0.70,\\n        \\\"val_frac\\\": 0.15,\\n        \\\"embargo_bars\\\": FUSION_EMBARGO_BARS,\\n        \\\"batch_size\\\": 128,\\n        \\\"epochs\\\": 40,\\n        \\\"lr\\\": 1e-4,\\n        \\\"weight_decay\\\": 1e-3,\\n        \\\"dropout\\\": 0.30,\\n        \\\"d_model\\\": 64,\\n        \\\"nhead\\\": 4,\\n        \\\"num_layers\\\": 2,\\n        \\\"early_stop_patience\\\": 8,\\n        \\\"label_smoothing\\\": 0.05,\\n        \\\"horizon_loss_weights\\\": [1.0, 1.0, 1.0, 1.0],\\n        \\\"lr_schedule\\\": \\\"plateau\\\",\\n        \\\"seed\\\": 42,\\n        \\\"history_days\\\": 900,\\n        \\\"base_url\\\": \\\"https://api.india.delta.exchange\\\",\\n        \\\"atr_period\\\": 14,\\n        \\\"run_optuna\\\": False,\\n        \\\"optuna_trials\\\": 3,\\n        \\\"optuna_trial_epochs\\\": 4,\\n        \\\"optuna_refresh\\\": False,\\n        \\\"windows_in_ram\\\": True,\\n        \\\"pin_memory\\\": True,\\n        \\\"dataloader_workers\\\": 2,\\n        \\\"prefetch_factor\\\": 4,\\n        \\\"amp\\\": True,\\n        \\\"run_walk_forward\\\": True,\\n        \\\"walk_forward_folds\\\": 3,\\n        \\\"walk_forward_embargo\\\": FUSION_EMBARGO_BARS,\\n        \\\"min_probability\\\": FUSION_MIN_PROBABILITY,\\n        \\\"high_balanced_acc\\\": LABEL_V2_HIGH_BALANCED_ACC,\\n        \\\"high_max_ece\\\": FUSION_HIGH_MAX_ECE,\\n        \\\"medium_balanced_acc\\\": LABEL_V2_MEDIUM_BALANCED_ACC,\\n        \\\"n_classes\\\": LABEL_V2_DIRECTION_CARDINALITY,\\n        \\\"label_scheme\\\": \\\"label_v2\\\",\\n        \\\"experiment\\\": \\\"later_direction_only\\\",\\n        \\\"use_class_weights\\\": False,\\n        \\\"path_loss_weights\\\": dict(LABEL_V2_DIRECTION_ONLY_PATH_LOSS_WEIGHTS),\\n        \\\"label_v2_theta\\\": dict(LABEL_V2_THETA_FROZEN),\\n        \\\"run_shap\\\": False,\\n        \\\"shap_background\\\": 32,\\n        \\\"shap_explain_n\\\": 64,\\n        \\\"use_state_embeddings\\\": True,\\n        \\\"run_leave_one_group_out\\\": False,\\n    }\\n\\n\\ndef ablation_feature_groups(resolution: str = \\\"5m\\\") -> Dict[str, Tuple[str, ...]]:\\n    \\\"\\\"\\\"Nested feature sets A-F for out-of-sample ablation.\\\"\\\"\\\"\\n    all_cols = v9_feature_cols_for_resolution(resolution)\\n    ohlcv = (\\\"ret_1\\\", \\\"rv_16\\\", \\\"rv_96\\\", \\\"hour_sin\\\", \\\"hour_cos\\\", \\\"dow_sin\\\", \\\"dow_cos\\\")\\n    geometry = ohlcv + (\\n        \\\"body_ratio\\\",\\n        \\\"upper_wick_ratio\\\",\\n        \\\"lower_wick_ratio\\\",\\n        \\\"close_loc\\\",\\n        \\\"range_atr\\\",\\n        \\\"body_atr\\\",\\n        \\\"gap_atr\\\",\\n        \\\"inside_bar\\\",\\n        \\\"outside_bar\\\",\\n        \\\"engulf_score\\\",\\n    )\\n    trend = geometry + (\\n        \\\"ema50_dist_pct\\\",\\n        \\\"macd_hist\\\",\\n        \\\"rsi_14\\\",\\n        \\\"adx_14\\\",\\n        \\\"price_vs_ema9_atr\\\",\\n        \\\"price_vs_ema21_atr\\\",\\n        \\\"price_vs_ema50_atr\\\",\\n        \\\"price_vs_ema200_atr\\\",\\n        \\\"ema9_vs_21_atr\\\",\\n        \\\"ema21_vs_50_atr\\\",\\n        \\\"ema50_vs_200_atr\\\",\\n        \\\"ema21_slope_atr\\\",\\n        \\\"ema50_slope_atr\\\",\\n        \\\"trend_efficiency\\\",\\n        \\\"displacement_atr\\\",\\n        \\\"pct_with_trend\\\",\\n    )\\n    structure = trend + (\\n        \\\"hh_count\\\",\\n        \\\"hl_count\\\",\\n        \\\"lh_count\\\",\\n        \\\"ll_count\\\",\\n        \\\"structure_bias\\\",\\n        \\\"last_swing_dir\\\",\\n        \\\"bars_since_swing\\\",\\n        \\\"swing_amp_atr\\\",\\n        \\\"dist_to_support_atr\\\",\\n        \\\"dist_to_resistance_atr\\\",\\n        \\\"support_touch_count\\\",\\n        \\\"resistance_touch_count\\\",\\n        \\\"range_width_atr\\\",\\n        \\\"dist_to_resistance_pct\\\",\\n        \\\"dist_to_support_pct\\\",\\n    )\\n    chart = structure + (\\n        \\\"peak_diff_atr\\\",\\n        \\\"trough_diff_atr\\\",\\n        \\\"peak_sep_bars\\\",\\n        \\\"trough_sep_bars\\\",\\n        \\\"dist_neck_atr\\\",\\n        \\\"high_slope_atr\\\",\\n        \\\"low_slope_atr\\\",\\n        \\\"convergence\\\",\\n        \\\"width_now_atr\\\",\\n        \\\"pole_disp_atr\\\",\\n        \\\"flag_width_atr\\\",\\n        \\\"flag_slope_atr\\\",\\n        \\\"breakout_size_atr\\\",\\n        \\\"breakout_vol_ratio\\\",\\n        \\\"pre_breakout_comp\\\",\\n        \\\"bars_since_breakout\\\",\\n        \\\"retest_dist_atr\\\",\\n        \\\"failed_break\\\",\\n    )\\n    return {\\n        \\\"A\\\": tuple(c for c in ohlcv if c in all_cols),\\n        \\\"B\\\": tuple(c for c in geometry if c in all_cols),\\n        \\\"C\\\": tuple(c for c in trend if c in all_cols),\\n        \\\"D\\\": tuple(c for c in structure if c in all_cols),\\n        \\\"E\\\": tuple(c for c in chart if c in all_cols),\\n        \\\"F\\\": all_cols,\\n    }\\n\", \"feature_store/transformer_btcusd/derivatives.py\": \"\\\"\\\"\\\"Funding and OI derivative features scaled per resolution.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom feature_store.transformer_btcusd.contract import scale_period\\n\\n_EPS = 1e-9\\n\\n\\ndef compute_funding_derivatives(\\n    fund_rate: pd.Series,\\n    ret_2: pd.Series,\\n    *,\\n    resolution_minutes: int,\\n) -> pd.DataFrame:\\n    \\\"\\\"\\\"Funding z-score, momentum, and rate-of-change on native TF bars.\\\"\\\"\\\"\\n    z_window = scale_period(16, resolution_minutes)\\n    roc_diff = max(1, scale_period(1, resolution_minutes))\\n    fz_mean = fund_rate.rolling(z_window, min_periods=5).mean()\\n    fz_std = fund_rate.rolling(z_window, min_periods=5).std().replace(0, _EPS)\\n    funding_zscore = ((fund_rate - fz_mean) / fz_std).fillna(0.0).clip(-4.0, 4.0)\\n    funding_mom = funding_zscore * ret_2\\n    diff = fund_rate.diff(roc_diff)\\n    roc_mu = diff.rolling(z_window, min_periods=5).mean()\\n    roc_std = diff.rolling(z_window, min_periods=5).std().replace(0, _EPS)\\n    funding_rate_roc = ((diff - roc_mu) / roc_std).fillna(0.0).clip(-4.0, 4.0)\\n    return pd.DataFrame(\\n        {\\n            \\\"funding_zscore\\\": funding_zscore,\\n            \\\"funding_mom\\\": funding_mom.fillna(0.0),\\n            \\\"funding_rate_roc\\\": funding_rate_roc,\\n        }\\n    )\\n\\n\\ndef compute_oi_derivatives(\\n    primary: pd.DataFrame,\\n    *,\\n    resolution_minutes: int,\\n) -> pd.DataFrame:\\n    \\\"\\\"\\\"OI-derived features on native TF bars.\\\"\\\"\\\"\\n    n = len(primary)\\n    zero = pd.DataFrame(\\n        {\\n            \\\"oi_change_2\\\": np.zeros(n),\\n            \\\"oi_delta_z\\\": np.zeros(n),\\n            \\\"oi_price_divergence\\\": np.zeros(n),\\n            \\\"oi_acceleration\\\": np.zeros(n),\\n            \\\"oi_zscore\\\": np.zeros(n),\\n        },\\n        index=primary.index,\\n    )\\n    if \\\"open_interest\\\" not in primary.columns:\\n        return zero\\n\\n    z_window = scale_period(16, resolution_minutes)\\n    change_window = max(1, scale_period(2, resolution_minutes))\\n\\n    oi_s = primary[\\\"open_interest\\\"].astype(float)\\n    if oi_s.notna().sum() == 0 or float(oi_s.max()) < _EPS:\\n        return zero\\n\\n    oi_mu = oi_s.rolling(z_window, min_periods=max(2, z_window // 4)).mean()\\n    oi_std = oi_s.rolling(z_window, min_periods=max(2, z_window // 4)).std().clip(\\n        lower=_EPS\\n    )\\n    oi_zscore = ((oi_s - oi_mu) / oi_std).fillna(0.0).clip(-4.0, 4.0)\\n\\n    oi_lagged = oi_s.shift(change_window)\\n    oi_change_2 = (\\n        ((oi_s - oi_lagged) / (oi_lagged.abs() + _EPS)).fillna(0.0).clip(-0.05, 0.05)\\n    )\\n\\n    close_s = primary[\\\"close\\\"].astype(float)\\n    close_lagged = close_s.shift(change_window)\\n    ret_2 = ((close_s - close_lagged) / (close_lagged.abs() + _EPS)).fillna(0.0)\\n    oi_price_divergence = (\\n        np.sign(oi_change_2.values) * -np.sign(ret_2.values)\\n    ).astype(np.float32)\\n    oi_acceleration = oi_change_2.diff().fillna(0.0).clip(-0.02, 0.02)\\n\\n    oi_delta_1 = oi_s.diff(1).fillna(0.0)\\n    oi_delta_mu = oi_delta_1.rolling(z_window, min_periods=max(2, z_window // 4)).mean()\\n    oi_delta_std = oi_delta_1.rolling(z_window, min_periods=max(2, z_window // 4)).std().clip(\\n        lower=_EPS\\n    )\\n    oi_delta_z = ((oi_delta_1 - oi_delta_mu) / oi_delta_std).fillna(0.0).clip(-4.0, 4.0)\\n\\n    return pd.DataFrame(\\n        {\\n            \\\"oi_zscore\\\": oi_zscore,\\n            \\\"oi_change_2\\\": oi_change_2,\\n            \\\"oi_price_divergence\\\": pd.Series(oi_price_divergence).fillna(0.0),\\n            \\\"oi_acceleration\\\": oi_acceleration,\\n            \\\"oi_delta_z\\\": oi_delta_z,\\n        },\\n        index=primary.index,\\n    ).replace([np.inf, -np.inf], 0.0).fillna(0.0)\\n\", \"feature_store/transformer_btcusd/structure.py\": \"\\\"\\\"\\\"Causal OHLCV market-structure and chart-geometry features.\\n\\nAll values at bar t use information available at or before t. Swings are ATR\\nZigZag confirmations (not future-looking fractals). Donchian/breakout ranges\\nexclude bar t. Higher-TF context is merged from fully closed resampled bars.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nfrom collections import deque\\nfrom typing import Deque, List, Tuple\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom feature_store.transformer_btcusd.contract import (\\n    CHART_PATTERN_COL,\\n    HTF_FEATURE_FIELDS,\\n    HTF_SOURCE_TFS,\\n    HTF_STRUCTURE_COLS,\\n    NATIVE_STRUCTURE_COLS,\\n    RESOLUTION_MINUTES,\\n)\\n\\n_EPS = 1e-9\\n_ATR_CLIP = 8.0\\n_SLOPE_CLIP = 2.0\\n_ZZ_TAU = 1.5\\n_ZZ_EVENT_M = 6\\n_ZZ_EPS_ATR = 0.15\\n_SWING_K = 20\\n_ER_N = 32\\n_SLOPE_LAG = 8\\n_DONCHIAN_N = 32\\n_RANGE_PCTILE_W = 256\\n_TOUCH_W = 64\\n_SR_DELTA_ATR = 0.5\\n_POLE_BARS = 12\\n_FLAG_BARS = 16\\n_BREAKOUT_THRESH = 0.25\\n_FAILED_BREAK_BARS = 8\\n_CHANNEL_K = 3\\n_ATR_SHORT = 14\\n_ATR_LONG = 56\\n_HTF_NATIVE_MINUTES = 5\\n\\n_HTF_NATIVE_MAP = {\\n    \\\"structure_bias\\\": \\\"structure_bias\\\",\\n    \\\"trend_efficiency\\\": \\\"trend_efficiency\\\",\\n    \\\"ema21_slope_atr\\\": \\\"ema21_slope_atr\\\",\\n    \\\"dist_support_atr\\\": \\\"dist_to_support_atr\\\",\\n    \\\"dist_resistance_atr\\\": \\\"dist_to_resistance_atr\\\",\\n    \\\"range_width_atr\\\": \\\"range_width_atr\\\",\\n}\\n\\n\\ndef _structure_safe_atr(atr: np.ndarray, close: np.ndarray, i: int) -> float:\\n    val = float(atr[i]) if np.isfinite(atr[i]) else float(\\\"nan\\\")\\n    if not np.isfinite(val) or val <= 0:\\n        c = float(close[i]) if np.isfinite(close[i]) else 0.0\\n        return max(abs(c) * 0.01, _EPS)\\n    return val\\n\\n\\ndef _clip_atr(val: float) -> float:\\n    if not np.isfinite(val):\\n        return 0.0\\n    return float(np.clip(val, -_ATR_CLIP, _ATR_CLIP))\\n\\n\\ndef _ols_slope(x: np.ndarray, y: np.ndarray) -> float:\\n    n = float(len(x))\\n    if n < 2:\\n        return 0.0\\n    sx = float(x.sum())\\n    sy = float(y.sum())\\n    sxy = float((x * y).sum())\\n    sx2 = float((x * x).sum())\\n    den = n * sx2 - sx * sx\\n    if abs(den) < 1e-12:\\n        return 0.0\\n    return float((n * sxy - sx * sy) / den)\\n\\n\\ndef _ensure_atr(df: pd.DataFrame, period: int = 14) -> pd.DataFrame:\\n    out = df.copy()\\n    if \\\"atr\\\" in out.columns and out[\\\"atr\\\"].notna().any():\\n        return out\\n    prev_close = out[\\\"close\\\"].shift(1)\\n    tr = pd.concat(\\n        [\\n            out[\\\"high\\\"] - out[\\\"low\\\"],\\n            (out[\\\"high\\\"] - prev_close).abs(),\\n            (out[\\\"low\\\"] - prev_close).abs(),\\n        ],\\n        axis=1,\\n    ).max(axis=1)\\n    out[\\\"atr\\\"] = tr.rolling(period, min_periods=1).mean()\\n    return out\\n\\n\\ndef _vectorized_trend_and_ma(out: pd.DataFrame) -> pd.DataFrame:\\n    \\\"\\\"\\\"ER, displacement, MA distances/slopes, Donchian compression, flag geometry.\\\"\\\"\\\"\\n    close = out[\\\"close\\\"].astype(float)\\n    high = out[\\\"high\\\"].astype(float)\\n    low = out[\\\"low\\\"].astype(float)\\n    volume = out[\\\"volume\\\"].astype(float) if \\\"volume\\\" in out.columns else pd.Series(\\n        0.0, index=out.index\\n    )\\n    atr = out[\\\"atr\\\"].astype(float) if \\\"atr\\\" in out.columns else pd.Series(\\n        close.abs() * 0.01, index=out.index\\n    )\\n    atr_safe = atr.replace(0, np.nan).fillna(close.abs() * 0.01 + _EPS)\\n\\n    abs_move = (close - close.shift(_ER_N)).abs()\\n    path = close.diff().abs().rolling(_ER_N, min_periods=2).sum()\\n    out[\\\"trend_efficiency\\\"] = (abs_move / (path + _EPS)).clip(0.0, 1.0).fillna(0.0)\\n    out[\\\"displacement_atr\\\"] = (abs_move / (atr_safe + _EPS)).clip(\\n        0.0, _ATR_CLIP\\n    ).fillna(0.0)\\n\\n    window_dir = np.sign(close - close.shift(_ER_N))\\n    up_frac = (close.diff() > 0).astype(float).rolling(_ER_N, min_periods=1).mean()\\n    pct = np.where(window_dir >= 0, up_frac, 1.0 - up_frac)\\n    out[\\\"pct_with_trend\\\"] = pd.Series(pct, index=out.index).fillna(0.0)\\n\\n    ema9 = close.ewm(span=9, adjust=False).mean()\\n    ema21 = close.ewm(span=21, adjust=False).mean()\\n    ema50 = close.ewm(span=50, adjust=False).mean()\\n    ema200 = close.ewm(span=200, adjust=False).mean()\\n    out[\\\"price_vs_ema9_atr\\\"] = ((close - ema9) / (atr_safe + _EPS)).clip(\\n        -_ATR_CLIP, _ATR_CLIP\\n    )\\n    out[\\\"price_vs_ema21_atr\\\"] = ((close - ema21) / (atr_safe + _EPS)).clip(\\n        -_ATR_CLIP, _ATR_CLIP\\n    )\\n    out[\\\"price_vs_ema50_atr\\\"] = ((close - ema50) / (atr_safe + _EPS)).clip(\\n        -_ATR_CLIP, _ATR_CLIP\\n    )\\n    out[\\\"price_vs_ema200_atr\\\"] = ((close - ema200) / (atr_safe + _EPS)).clip(\\n        -_ATR_CLIP, _ATR_CLIP\\n    )\\n    out[\\\"ema9_vs_21_atr\\\"] = ((ema9 - ema21) / (atr_safe + _EPS)).clip(\\n        -_ATR_CLIP, _ATR_CLIP\\n    )\\n    out[\\\"ema21_vs_50_atr\\\"] = ((ema21 - ema50) / (atr_safe + _EPS)).clip(\\n        -_ATR_CLIP, _ATR_CLIP\\n    )\\n    out[\\\"ema50_vs_200_atr\\\"] = ((ema50 - ema200) / (atr_safe + _EPS)).clip(\\n        -_ATR_CLIP, _ATR_CLIP\\n    )\\n    lag = float(_SLOPE_LAG)\\n    out[\\\"ema21_slope_atr\\\"] = (\\n        (ema21 - ema21.shift(_SLOPE_LAG)) / (atr_safe * lag + _EPS)\\n    ).clip(-_SLOPE_CLIP, _SLOPE_CLIP).fillna(0.0)\\n    out[\\\"ema50_slope_atr\\\"] = (\\n        (ema50 - ema50.shift(_SLOPE_LAG)) / (atr_safe * lag + _EPS)\\n    ).clip(-_SLOPE_CLIP, _SLOPE_CLIP).fillna(0.0)\\n\\n    prior_high = high.shift(1)\\n    prior_low = low.shift(1)\\n    donch_high = prior_high.rolling(_DONCHIAN_N, min_periods=1).max()\\n    donch_low = prior_low.rolling(_DONCHIAN_N, min_periods=1).min()\\n    range_width_atr = ((donch_high - donch_low) / (atr_safe + _EPS)).clip(\\n        0.0, _ATR_CLIP\\n    )\\n    out[\\\"range_width_atr\\\"] = range_width_atr.fillna(0.0)\\n    out[\\\"range_width_pctile\\\"] = range_width_atr.rolling(\\n        _RANGE_PCTILE_W, min_periods=8\\n    ).rank(pct=True).fillna(0.0)\\n    atr_short = atr_safe.rolling(_ATR_SHORT, min_periods=1).mean()\\n    atr_long = atr_safe.rolling(_ATR_LONG, min_periods=1).mean()\\n    out[\\\"atr_contraction\\\"] = (atr_short / (atr_long + _EPS)).clip(0.0, _ATR_CLIP).fillna(\\n        1.0\\n    )\\n\\n    out[\\\"breakout_size_atr\\\"] = ((close - donch_high) / (atr_safe + _EPS)).clip(\\n        -_ATR_CLIP, _ATR_CLIP\\n    ).fillna(0.0)\\n    vol_mean = volume.shift(1).rolling(_DONCHIAN_N, min_periods=1).mean()\\n    out[\\\"breakout_vol_ratio\\\"] = (volume / (vol_mean + _EPS)).clip(0.0, _ATR_CLIP).fillna(\\n        0.0\\n    )\\n    out[\\\"pre_breakout_comp\\\"] = out[\\\"range_width_pctile\\\"].shift(1).fillna(0.0)\\n\\n    pole_end = close.shift(_FLAG_BARS)\\n    pole_start = close.shift(_POLE_BARS + _FLAG_BARS)\\n    out[\\\"pole_disp_atr\\\"] = ((pole_end - pole_start) / (atr_safe + _EPS)).clip(\\n        -_ATR_CLIP, _ATR_CLIP\\n    ).fillna(0.0)\\n    flag_high = high.rolling(_FLAG_BARS, min_periods=1).max()\\n    flag_low = low.rolling(_FLAG_BARS, min_periods=1).min()\\n    out[\\\"flag_width_atr\\\"] = ((flag_high - flag_low) / (atr_safe + _EPS)).clip(\\n        0.0, _ATR_CLIP\\n    ).fillna(0.0)\\n    flag_lag = float(_FLAG_BARS)\\n    out[\\\"flag_slope_atr\\\"] = (\\n        (close - close.shift(_FLAG_BARS)) / (atr_safe * flag_lag + _EPS)\\n    ).clip(-_SLOPE_CLIP, _SLOPE_CLIP).fillna(0.0)\\n    return out\\n\\n\\ndef _zigzag_structure_loop(out: pd.DataFrame) -> pd.DataFrame:\\n    \\\"\\\"\\\"Single-pass ATR ZigZag: HH/HL counts, S/R, breakout events, pattern geometry.\\\"\\\"\\\"\\n    n = len(out)\\n    high = out[\\\"high\\\"].to_numpy(dtype=np.float64)\\n    low = out[\\\"low\\\"].to_numpy(dtype=np.float64)\\n    close = out[\\\"close\\\"].to_numpy(dtype=np.float64)\\n    atr = out[\\\"atr\\\"].to_numpy(dtype=np.float64) if \\\"atr\\\" in out.columns else np.full(\\n        n, np.nan\\n    )\\n    prior_high = np.roll(high, 1)\\n    prior_high[0] = high[0]\\n    prior_low = np.roll(low, 1)\\n    prior_low[0] = low[0]\\n    donch_h = np.empty(n, dtype=np.float64)\\n    donch_l = np.empty(n, dtype=np.float64)\\n    max_q: Deque[int] = deque()\\n    min_q: Deque[int] = deque()\\n    for t in range(n):\\n        left = t - _DONCHIAN_N\\n        while max_q and max_q[0] <= left:\\n            max_q.popleft()\\n        while min_q and min_q[0] <= left:\\n            min_q.popleft()\\n        src_h = prior_high[t]\\n        src_l = prior_low[t]\\n        while max_q and prior_high[max_q[-1]] <= src_h:\\n            max_q.pop()\\n        while min_q and prior_low[min_q[-1]] >= src_l:\\n            min_q.pop()\\n        max_q.append(t)\\n        min_q.append(t)\\n        donch_h[t] = prior_high[max_q[0]]\\n        donch_l[t] = prior_low[min_q[0]]\\n\\n    hh_count = np.zeros(n, dtype=np.float64)\\n    hl_count = np.zeros(n, dtype=np.float64)\\n    lh_count = np.zeros(n, dtype=np.float64)\\n    ll_count = np.zeros(n, dtype=np.float64)\\n    structure_bias = np.zeros(n, dtype=np.float64)\\n    last_swing_dir = np.zeros(n, dtype=np.float64)\\n    bars_since_swing = np.zeros(n, dtype=np.float64)\\n    swing_amp_atr = np.zeros(n, dtype=np.float64)\\n    unconfirmed_ext_atr = np.zeros(n, dtype=np.float64)\\n    dist_to_support_atr = np.zeros(n, dtype=np.float64)\\n    dist_to_resistance_atr = np.zeros(n, dtype=np.float64)\\n    support_touch = np.zeros(n, dtype=np.float64)\\n    resistance_touch = np.zeros(n, dtype=np.float64)\\n    bars_since_breakout = np.zeros(n, dtype=np.float64)\\n    retest_dist_atr = np.zeros(n, dtype=np.float64)\\n    failed_break = np.zeros(n, dtype=np.float64)\\n    peak_diff_atr = np.zeros(n, dtype=np.float64)\\n    trough_diff_atr = np.zeros(n, dtype=np.float64)\\n    peak_sep_bars = np.zeros(n, dtype=np.float64)\\n    trough_sep_bars = np.zeros(n, dtype=np.float64)\\n    dist_neck_atr = np.zeros(n, dtype=np.float64)\\n    high_slope_atr = np.zeros(n, dtype=np.float64)\\n    low_slope_atr = np.zeros(n, dtype=np.float64)\\n    convergence = np.zeros(n, dtype=np.float64)\\n    width_now_atr = np.zeros(n, dtype=np.float64)\\n\\n    direction = 1\\n    e_idx = 0\\n    e_price = high[0] if n else 0.0\\n    last_high_price = float(\\\"nan\\\")\\n    last_low_price = float(\\\"nan\\\")\\n    last_high_idx = -1\\n    last_low_idx = -1\\n    last_swing_idx = -1\\n    last_dir = 0.0\\n    highs: List[Tuple[int, float]] = []\\n    lows: List[Tuple[int, float]] = []\\n    events: Deque[str] = deque(maxlen=_ZZ_EVENT_M)\\n    last_bo_idx = -1\\n    last_bo_level = float(\\\"nan\\\")\\n    last_bo_side = 0\\n\\n    for t in range(n):\\n        atr_t = _structure_safe_atr(atr, close, t)\\n        thresh = _ZZ_TAU * atr_t\\n        eps = _ZZ_EPS_ATR * atr_t\\n\\n        if direction == 1:\\n            if high[t] >= e_price:\\n                e_price = high[t]\\n                e_idx = t\\n            elif (e_price - low[t]) >= thresh:\\n                highs.append((e_idx, e_price))\\n                if len(highs) > _SWING_K:\\n                    highs = highs[-_SWING_K:]\\n                if np.isfinite(last_high_price):\\n                    if e_price > last_high_price + eps:\\n                        events.append(\\\"HH\\\")\\n                    elif e_price < last_high_price - eps:\\n                        events.append(\\\"LH\\\")\\n                last_high_price = e_price\\n                last_high_idx = e_idx\\n                last_swing_idx = e_idx\\n                last_dir = 1.0\\n                direction = -1\\n                e_price = low[t]\\n                e_idx = t\\n        else:\\n            if low[t] <= e_price:\\n                e_price = low[t]\\n                e_idx = t\\n            elif (high[t] - e_price) >= thresh:\\n                lows.append((e_idx, e_price))\\n                if len(lows) > _SWING_K:\\n                    lows = lows[-_SWING_K:]\\n                if np.isfinite(last_low_price):\\n                    if e_price > last_low_price + eps:\\n                        events.append(\\\"HL\\\")\\n                    elif e_price < last_low_price - eps:\\n                        events.append(\\\"LL\\\")\\n                last_low_price = e_price\\n                last_low_idx = e_idx\\n                last_swing_idx = e_idx\\n                last_dir = -1.0\\n                direction = 1\\n                e_price = high[t]\\n                e_idx = t\\n\\n        n_hh = sum(1 for e in events if e == \\\"HH\\\")\\n        n_hl = sum(1 for e in events if e == \\\"HL\\\")\\n        n_lh = sum(1 for e in events if e == \\\"LH\\\")\\n        n_ll = sum(1 for e in events if e == \\\"LL\\\")\\n        n_ev = max(len(events), 1)\\n        hh_count[t] = float(n_hh)\\n        hl_count[t] = float(n_hl)\\n        lh_count[t] = float(n_lh)\\n        ll_count[t] = float(n_ll)\\n        structure_bias[t] = (n_hh + n_hl - n_lh - n_ll) / float(n_ev)\\n        last_swing_dir[t] = last_dir\\n        bars_since_swing[t] = float(t - last_swing_idx) if last_swing_idx >= 0 else float(t)\\n        if np.isfinite(last_high_price) and np.isfinite(last_low_price):\\n            swing_amp_atr[t] = _clip_atr(abs(last_high_price - last_low_price) / atr_t)\\n        unconfirmed_ext_atr[t] = _clip_atr((e_price - close[t]) / atr_t)\\n\\n        swing_lows = [p for _, p in lows if p <= close[t]]\\n        swing_highs = [p for _, p in highs if p >= close[t]]\\n        support = max(swing_lows) if swing_lows else float(donch_l[t])\\n        resistance = min(swing_highs) if swing_highs else float(donch_h[t])\\n        dist_to_support_atr[t] = _clip_atr((close[t] - support) / atr_t)\\n        dist_to_resistance_atr[t] = _clip_atr((resistance - close[t]) / atr_t)\\n        delta = _SR_DELTA_ATR * atr_t\\n        if low[t] <= support + delta and close[t] >= support - delta:\\n            support_touch[t] = 1.0\\n        if high[t] >= resistance - delta and close[t] <= resistance + delta:\\n            resistance_touch[t] = 1.0\\n\\n        up_ext = (close[t] - donch_h[t]) / atr_t\\n        dn_ext = (donch_l[t] - close[t]) / atr_t\\n        if up_ext >= _BREAKOUT_THRESH:\\n            last_bo_idx = t\\n            last_bo_level = float(donch_h[t])\\n            last_bo_side = 1\\n        elif dn_ext >= _BREAKOUT_THRESH:\\n            last_bo_idx = t\\n            last_bo_level = float(donch_l[t])\\n            last_bo_side = -1\\n        if last_bo_idx >= 0:\\n            bars_since_breakout[t] = float(t - last_bo_idx)\\n            retest_dist_atr[t] = _clip_atr((close[t] - last_bo_level) / atr_t)\\n            if last_bo_side > 0 and close[t] < last_bo_level:\\n                failed_break[t] = 1.0\\n            elif last_bo_side < 0 and close[t] > last_bo_level:\\n                failed_break[t] = 1.0\\n        else:\\n            bars_since_breakout[t] = float(t)\\n\\n        if len(highs) >= 2:\\n            (i1, p1), (i2, p2) = highs[-2], highs[-1]\\n            peak_diff_atr[t] = _clip_atr((p2 - p1) / atr_t)\\n            peak_sep_bars[t] = float(i2 - i1)\\n            between = [lp for li, lp in lows if i1 < li < i2]\\n            if between:\\n                neck = min(between)\\n                dist_neck_atr[t] = _clip_atr((close[t] - neck) / atr_t)\\n        if len(lows) >= 2:\\n            (j1, q1), (j2, q2) = lows[-2], lows[-1]\\n            trough_diff_atr[t] = _clip_atr((q2 - q1) / atr_t)\\n            trough_sep_bars[t] = float(j2 - j1)\\n\\n        if len(highs) >= _CHANNEL_K:\\n            hx = np.array([idx for idx, _ in highs[-_CHANNEL_K:]], dtype=np.float64)\\n            hy = np.array([px for _, px in highs[-_CHANNEL_K:]], dtype=np.float64)\\n            hs = _ols_slope(hx, hy)\\n            high_slope_atr[t] = float(np.clip(hs / atr_t, -_SLOPE_CLIP, _SLOPE_CLIP))\\n        if len(lows) >= _CHANNEL_K:\\n            lx = np.array([idx for idx, _ in lows[-_CHANNEL_K:]], dtype=np.float64)\\n            ly = np.array([px for _, px in lows[-_CHANNEL_K:]], dtype=np.float64)\\n            ls = _ols_slope(lx, ly)\\n            low_slope_atr[t] = float(np.clip(ls / atr_t, -_SLOPE_CLIP, _SLOPE_CLIP))\\n        convergence[t] = float(\\n            np.clip(high_slope_atr[t] - low_slope_atr[t], -_SLOPE_CLIP, _SLOPE_CLIP)\\n        )\\n        if len(highs) >= 1 and len(lows) >= 1:\\n            h_line = highs[-1][1] + high_slope_atr[t] * atr_t * (t - highs[-1][0])\\n            l_line = lows[-1][1] + low_slope_atr[t] * atr_t * (t - lows[-1][0])\\n            width_now_atr[t] = _clip_atr((h_line - l_line) / atr_t)\\n\\n    touch_sup = pd.Series(support_touch).rolling(_TOUCH_W, min_periods=1).sum()\\n    touch_res = pd.Series(resistance_touch).rolling(_TOUCH_W, min_periods=1).sum()\\n\\n    out[\\\"hh_count\\\"] = hh_count\\n    out[\\\"hl_count\\\"] = hl_count\\n    out[\\\"lh_count\\\"] = lh_count\\n    out[\\\"ll_count\\\"] = ll_count\\n    out[\\\"structure_bias\\\"] = structure_bias\\n    out[\\\"last_swing_dir\\\"] = last_swing_dir\\n    out[\\\"bars_since_swing\\\"] = bars_since_swing\\n    out[\\\"swing_amp_atr\\\"] = swing_amp_atr\\n    out[\\\"unconfirmed_ext_atr\\\"] = unconfirmed_ext_atr\\n    out[\\\"dist_to_support_atr\\\"] = dist_to_support_atr\\n    out[\\\"dist_to_resistance_atr\\\"] = dist_to_resistance_atr\\n    out[\\\"support_touch_count\\\"] = touch_sup.to_numpy(dtype=np.float64)\\n    out[\\\"resistance_touch_count\\\"] = touch_res.to_numpy(dtype=np.float64)\\n    out[\\\"bars_since_breakout\\\"] = bars_since_breakout\\n    out[\\\"retest_dist_atr\\\"] = retest_dist_atr\\n    out[\\\"failed_break\\\"] = failed_break\\n    out[\\\"peak_diff_atr\\\"] = peak_diff_atr\\n    out[\\\"trough_diff_atr\\\"] = trough_diff_atr\\n    out[\\\"peak_sep_bars\\\"] = peak_sep_bars\\n    out[\\\"trough_sep_bars\\\"] = trough_sep_bars\\n    out[\\\"dist_neck_atr\\\"] = dist_neck_atr\\n    out[\\\"high_slope_atr\\\"] = high_slope_atr\\n    out[\\\"low_slope_atr\\\"] = low_slope_atr\\n    out[\\\"convergence\\\"] = convergence\\n    out[\\\"width_now_atr\\\"] = width_now_atr\\n    out[CHART_PATTERN_COL] = _chart_pattern_ids(\\n        failed_break=failed_break,\\n        bars_since_breakout=bars_since_breakout,\\n        breakout_size_atr=(\\n            out[\\\"breakout_size_atr\\\"].to_numpy(dtype=np.float64)\\n            if \\\"breakout_size_atr\\\" in out.columns\\n            else np.zeros(n)\\n        ),\\n        peak_diff_atr=peak_diff_atr,\\n        trough_diff_atr=trough_diff_atr,\\n        peak_sep_bars=peak_sep_bars,\\n        trough_sep_bars=trough_sep_bars,\\n        dist_neck_atr=dist_neck_atr,\\n        high_slope_atr=high_slope_atr,\\n        low_slope_atr=low_slope_atr,\\n        convergence=convergence,\\n        width_now_atr=width_now_atr,\\n        pole_disp_atr=(\\n            out[\\\"pole_disp_atr\\\"].to_numpy(dtype=np.float64)\\n            if \\\"pole_disp_atr\\\" in out.columns\\n            else np.zeros(n)\\n        ),\\n        flag_width_atr=(\\n            out[\\\"flag_width_atr\\\"].to_numpy(dtype=np.float64)\\n            if \\\"flag_width_atr\\\" in out.columns\\n            else np.zeros(n)\\n        ),\\n        flag_slope_atr=(\\n            out[\\\"flag_slope_atr\\\"].to_numpy(dtype=np.float64)\\n            if \\\"flag_slope_atr\\\" in out.columns\\n            else np.zeros(n)\\n        ),\\n        structure_bias=structure_bias,\\n    )\\n    return out\\n\\n\\ndef _chart_pattern_ids(\\n    *,\\n    failed_break: np.ndarray,\\n    bars_since_breakout: np.ndarray,\\n    breakout_size_atr: np.ndarray,\\n    peak_diff_atr: np.ndarray,\\n    trough_diff_atr: np.ndarray,\\n    peak_sep_bars: np.ndarray,\\n    trough_sep_bars: np.ndarray,\\n    dist_neck_atr: np.ndarray,\\n    high_slope_atr: np.ndarray,\\n    low_slope_atr: np.ndarray,\\n    convergence: np.ndarray,\\n    width_now_atr: np.ndarray,\\n    pole_disp_atr: np.ndarray,\\n    flag_width_atr: np.ndarray,\\n    flag_slope_atr: np.ndarray,\\n    structure_bias: np.ndarray,\\n) -> np.ndarray:\\n    \\\"\\\"\\\"Causal discrete chart-pattern id at bar t (0=NONE ... 8=FAILED_BREAK).\\n\\n    FAILED_BREAK is a rising-edge event inside ``_FAILED_BREAK_BARS`` of the\\n    last Donchian breakout. It does not overwrite FLAG/TRIANGLE/DOUBLE/CHANNEL\\n    and does not stay on for the whole post-breakout regime. BREAKOUT is the\\n    event bar only and also does not overwrite those named geometries.\\n    \\\"\\\"\\\"\\n    _ = breakout_size_atr\\n    n = int(failed_break.shape[0])\\n    ids = np.zeros(n, dtype=np.int64)\\n    channel = (np.abs(high_slope_atr - low_slope_atr) < 0.15) & (width_now_atr > 0.6)\\n    triangle = (np.abs(convergence) >= 0.08) & (width_now_atr < 2.5)\\n    double_top = (\\n        (np.abs(peak_diff_atr) < 0.45)\\n        & (peak_sep_bars >= 4)\\n        & (dist_neck_atr > 0.15)\\n    )\\n    double_bottom = (\\n        (np.abs(trough_diff_atr) < 0.45)\\n        & (trough_sep_bars >= 4)\\n        & (dist_neck_atr < -0.15)\\n    )\\n    flag_bull = (\\n        (pole_disp_atr > 1.0)\\n        & (flag_width_atr < 2.0)\\n        & (flag_slope_atr < 0.0)\\n        & (structure_bias > 0.15)\\n    )\\n    flag_bear = (\\n        (pole_disp_atr < -1.0)\\n        & (flag_width_atr < 2.0)\\n        & (flag_slope_atr > 0.0)\\n        & (structure_bias < -0.15)\\n    )\\n    # Event bar only. abs(size vs Donchian high) is not a two-sided breakout\\n    # and was tagging most 5m bars as BREAKOUT.\\n    fresh_break = bars_since_breakout <= 0.5\\n    failed_now = (failed_break >= 0.5) & (\\n        bars_since_breakout <= float(_FAILED_BREAK_BARS)\\n    )\\n    failed_prev = np.empty_like(failed_now)\\n    failed_prev[0] = False\\n    if n > 1:\\n        failed_prev[1:] = failed_now[:-1]\\n    failed_pulse = failed_now & ~failed_prev\\n    ids = np.where(channel, 6, ids)\\n    ids = np.where(triangle, 3, ids)\\n    ids = np.where(flag_bull, 1, ids)\\n    ids = np.where(flag_bear, 2, ids)\\n    ids = np.where(double_bottom, 5, ids)\\n    ids = np.where(double_top, 4, ids)\\n    named = np.isin(ids, [1, 2, 3, 4, 5, 6])\\n    ids = np.where(fresh_break & ~named, 7, ids)\\n    ids = np.where(failed_pulse & ~named, 8, ids)\\n    return ids\\n\\n\\ndef add_market_structure_features(df: pd.DataFrame) -> pd.DataFrame:\\n    \\\"\\\"\\\"Add causal native-TF structure, trend, S/R, breakout, and geometry columns.\\n\\n    Args:\\n        df: OHLCV frame with atr (computed if missing).\\n\\n    Returns:\\n        Same frame with ``NATIVE_STRUCTURE_COLS`` assigned and finite-filled.\\n    \\\"\\\"\\\"\\n    out = _ensure_atr(df)\\n    out = _vectorized_trend_and_ma(out)\\n    out = _zigzag_structure_loop(out)\\n    for col in NATIVE_STRUCTURE_COLS:\\n        if col not in out.columns:\\n            out[col] = 0.0\\n        out[col] = out[col].astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)\\n    if CHART_PATTERN_COL not in out.columns:\\n        out[CHART_PATTERN_COL] = 0\\n    out[CHART_PATTERN_COL] = (\\n        pd.to_numeric(out[CHART_PATTERN_COL], errors=\\\"coerce\\\").fillna(0).astype(np.int64)\\n    )\\n    return out\\n\\n\\ndef _resample_closed_ohlcv(\\n    df: pd.DataFrame,\\n    htf_minutes: int,\\n    native_minutes: int = _HTF_NATIVE_MINUTES,\\n) -> pd.DataFrame:\\n    \\\"\\\"\\\"Resample native bars to a higher TF, keeping only complete buckets.\\\"\\\"\\\"\\n    expected = max(1, int(round(htf_minutes / native_minutes)))\\n    src = df[[\\\"time\\\", \\\"open\\\", \\\"high\\\", \\\"low\\\", \\\"close\\\"]].copy()\\n    if \\\"volume\\\" in df.columns:\\n        src[\\\"volume\\\"] = df[\\\"volume\\\"]\\n    else:\\n        src[\\\"volume\\\"] = 0.0\\n    src[\\\"time\\\"] = pd.to_datetime(src[\\\"time\\\"], utc=True)\\n    grouped = src.set_index(\\\"time\\\")\\n    rule = f\\\"{int(htf_minutes)}min\\\"\\n    agg = grouped.resample(rule, label=\\\"right\\\", closed=\\\"right\\\").agg(\\n        {\\n            \\\"open\\\": \\\"first\\\",\\n            \\\"high\\\": \\\"max\\\",\\n            \\\"low\\\": \\\"min\\\",\\n            \\\"close\\\": \\\"last\\\",\\n            \\\"volume\\\": \\\"sum\\\",\\n        }\\n    )\\n    counts = grouped[\\\"close\\\"].resample(rule, label=\\\"right\\\", closed=\\\"right\\\").count()\\n    complete = agg.loc[counts >= expected].dropna(subset=[\\\"open\\\", \\\"high\\\", \\\"low\\\", \\\"close\\\"])\\n    if complete.empty:\\n        return complete\\n    complete = complete.reset_index()\\n    return complete\\n\\n\\ndef add_htf_structure_features(df: pd.DataFrame) -> pd.DataFrame:\\n    \\\"\\\"\\\"Merge closed 15m/30m/1h/2h structure onto a 5m frame (backward asof).\\\"\\\"\\\"\\n    out = df.copy()\\n    out[\\\"time\\\"] = pd.to_datetime(out[\\\"time\\\"], utc=True)\\n    out = out.sort_values(\\\"time\\\").reset_index(drop=True)\\n\\n    for tf_name in HTF_SOURCE_TFS:\\n        minutes = RESOLUTION_MINUTES[tf_name]\\n        htf_raw = _resample_closed_ohlcv(out, minutes, _HTF_NATIVE_MINUTES)\\n        prefixed = {f\\\"htf_{tf_name}_{field}\\\": 0.0 for field in HTF_FEATURE_FIELDS}\\n        if htf_raw.empty:\\n            for col, val in prefixed.items():\\n                out[col] = val\\n            continue\\n        htf_feat = add_market_structure_features(_ensure_atr(htf_raw))\\n        merge_cols = {\\\"time\\\": htf_feat[\\\"time\\\"]}\\n        for field in HTF_FEATURE_FIELDS:\\n            src = _HTF_NATIVE_MAP[field]\\n            merge_cols[f\\\"htf_{tf_name}_{field}\\\"] = htf_feat[src].to_numpy(dtype=np.float64)\\n        htf_df = pd.DataFrame(merge_cols).sort_values(\\\"time\\\")\\n        out = pd.merge_asof(\\n            out.sort_values(\\\"time\\\"),\\n            htf_df,\\n            on=\\\"time\\\",\\n            direction=\\\"backward\\\",\\n        )\\n\\n    for col in HTF_STRUCTURE_COLS:\\n        if col not in out.columns:\\n            out[col] = 0.0\\n        out[col] = out[col].astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)\\n    return out\\n\", \"feature_store/transformer_btcusd/features.py\": \"\\\"\\\"\\\"Causal per-TF feature engineering for BTCUSD transformers.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Optional, Sequence\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom feature_store.transformer_btcusd.contract import (\\n    CANDLE_CLASS_CARDINALITY,\\n    CANDLE_CLASS_COL,\\n    FEATURE_COLS,\\n    scale_period,\\n)\\nfrom feature_store.transformer_btcusd.derivatives import (\\n    compute_funding_derivatives,\\n    compute_oi_derivatives,\\n)\\nfrom feature_store.transformer_btcusd.structure import (\\n    add_htf_structure_features,\\n    add_market_structure_features,\\n)\\n\\nWICK_NEGLIGIBLE = 0.05\\nWICK_BALANCE_MAX = 0.25\\nFLAT_ATR_MULT = 1e-4\\n_RATIO_COLS = (\\\"body_ratio\\\", \\\"upper_wick_ratio\\\", \\\"lower_wick_ratio\\\")\\n_STRUCTURE_EPS = 1e-9\\n_ATR_NORM_CLIP = 8.0\\n\\n\\ndef assemble_raw_frame(\\n    df: pd.DataFrame,\\n    *,\\n    funding_df: Optional[pd.DataFrame] = None,\\n    oi_df: Optional[pd.DataFrame] = None,\\n) -> pd.DataFrame:\\n    \\\"\\\"\\\"Merge OHLCV with funding/OI using the same path as live inference.\\\"\\\"\\\"\\n    out = df.copy()\\n    if \\\"timestamp\\\" in out.columns and \\\"time\\\" not in out.columns:\\n        out[\\\"time\\\"] = pd.to_datetime(out[\\\"timestamp\\\"], utc=True)\\n    elif \\\"time\\\" in out.columns:\\n        out[\\\"time\\\"] = pd.to_datetime(out[\\\"time\\\"], utc=True)\\n    else:\\n        raise ValueError(\\\"OHLCV frame requires timestamp or time column\\\")\\n\\n    required = (\\\"open\\\", \\\"high\\\", \\\"low\\\", \\\"close\\\", \\\"volume\\\")\\n    missing = [c for c in required if c not in out.columns]\\n    if missing:\\n        raise ValueError(f\\\"OHLCV frame missing columns: {missing}\\\")\\n\\n    out = out.sort_values(\\\"time\\\").reset_index(drop=True)\\n\\n    if funding_df is not None and not funding_df.empty:\\n        fund = funding_df.copy()\\n        if \\\"timestamp\\\" in fund.columns:\\n            fund[\\\"time\\\"] = pd.to_datetime(fund[\\\"timestamp\\\"], utc=True)\\n        rate_col = \\\"funding_rate\\\" if \\\"funding_rate\\\" in fund.columns else \\\"close\\\"\\n        fund = fund[[\\\"time\\\", rate_col]].rename(columns={rate_col: \\\"funding_rate\\\"})\\n        out = pd.merge_asof(\\n            out.sort_values(\\\"time\\\"),\\n            fund.sort_values(\\\"time\\\"),\\n            on=\\\"time\\\",\\n            direction=\\\"backward\\\",\\n        )\\n    elif \\\"funding_rate\\\" not in out.columns:\\n        out[\\\"funding_rate\\\"] = np.nan\\n\\n    if oi_df is not None and not oi_df.empty:\\n        oi = oi_df.copy()\\n        if \\\"timestamp\\\" in oi.columns:\\n            oi[\\\"time\\\"] = pd.to_datetime(oi[\\\"timestamp\\\"], utc=True)\\n        oi_col = None\\n        for candidate in (\\\"open_interest\\\", \\\"oi_contracts\\\", \\\"close\\\"):\\n            if candidate in oi.columns:\\n                oi_col = candidate\\n                break\\n        if oi_col is not None:\\n            oi = oi[[\\\"time\\\", oi_col]].rename(columns={oi_col: \\\"open_interest\\\"})\\n            out = pd.merge_asof(\\n                out.sort_values(\\\"time\\\"),\\n                oi.sort_values(\\\"time\\\"),\\n                on=\\\"time\\\",\\n                direction=\\\"backward\\\",\\n            )\\n    elif \\\"open_interest\\\" not in out.columns:\\n        out[\\\"open_interest\\\"] = np.nan\\n\\n    if \\\"funding_rate\\\" in out.columns:\\n        out[\\\"funding_rate\\\"] = out[\\\"funding_rate\\\"].ffill()\\n    if \\\"open_interest\\\" in out.columns:\\n        out[\\\"open_interest\\\"] = out[\\\"open_interest\\\"].ffill()\\n\\n    return out\\n\\n\\ndef prepare_raw_frame(\\n    df: pd.DataFrame,\\n    *,\\n    funding_df: Optional[pd.DataFrame] = None,\\n    oi_df: Optional[pd.DataFrame] = None,\\n) -> pd.DataFrame:\\n    \\\"\\\"\\\"Alias for assemble_raw_frame (agent inference entry point).\\\"\\\"\\\"\\n    return assemble_raw_frame(df, funding_df=funding_df, oi_df=oi_df)\\n\\n\\ndef classify_candle_shape(out: pd.DataFrame, *, sr_window: int) -> pd.Series:\\n    \\\"\\\"\\\"Assign mutually exclusive single-bar candle class ids 0..12.\\n\\n    Uses bar-i OHLC geometry plus rolling body-size quantiles up to bar i.\\n    First matching ``np.select`` condition wins.\\n\\n    Args:\\n        out: Frame with open/high/low/close, atr, and wick/body ratio columns.\\n        sr_window: Rolling lookback for doji/marubozu quantiles.\\n\\n    Returns:\\n        int64 Series of class ids aligned with ``out.index``.\\n    \\\"\\\"\\\"\\n    rng_raw = (out[\\\"high\\\"] - out[\\\"low\\\"]).astype(float)\\n    atr = out[\\\"atr\\\"] if \\\"atr\\\" in out.columns else pd.Series(0.0, index=out.index)\\n    is_flat = (rng_raw.fillna(0) <= 0) | (\\n        rng_raw.fillna(0) < atr.fillna(0) * FLAT_ATR_MULT\\n    )\\n\\n    body_abs = out[\\\"body_ratio\\\"].abs()\\n    upper = out[\\\"upper_wick_ratio\\\"]\\n    lower = out[\\\"lower_wick_ratio\\\"]\\n    doji_thresh = body_abs.rolling(sr_window, min_periods=20).quantile(0.15)\\n    marubozu_thresh = body_abs.rolling(sr_window, min_periods=20).quantile(0.85)\\n    body_median = body_abs.rolling(sr_window, min_periods=20).median()\\n\\n    wick_sum = upper + lower\\n    wick_imbalance = (upper - lower).abs() / (wick_sum + 1e-9)\\n    cond_wicks_balanced = (wick_sum > 2 * WICK_NEGLIGIBLE) & (\\n        wick_imbalance < WICK_BALANCE_MAX\\n    )\\n\\n    cond_doji = ~is_flat & (body_abs < doji_thresh)\\n    cond_dragonfly = (\\n        cond_doji\\n        & (upper < WICK_NEGLIGIBLE)\\n        & (lower > 2 * WICK_NEGLIGIBLE)\\n    )\\n    cond_gravestone = (\\n        cond_doji\\n        & (lower < WICK_NEGLIGIBLE)\\n        & (upper > 2 * WICK_NEGLIGIBLE)\\n    )\\n    cond_doji_standard = cond_doji\\n    cond_marubozu_bull = (\\n        ~is_flat\\n        & ~cond_doji\\n        & (out[\\\"body_ratio\\\"] > marubozu_thresh)\\n        & (upper < WICK_NEGLIGIBLE)\\n        & (lower < WICK_NEGLIGIBLE)\\n    )\\n    cond_marubozu_bear = (\\n        ~is_flat\\n        & ~cond_doji\\n        & (-out[\\\"body_ratio\\\"] > marubozu_thresh)\\n        & (upper < WICK_NEGLIGIBLE)\\n        & (lower < WICK_NEGLIGIBLE)\\n    )\\n    cond_hammer = (\\n        ~is_flat\\n        & ~cond_doji\\n        & (lower > 2 * body_abs)\\n        & (upper < body_abs)\\n    )\\n    cond_inv_hammer = (\\n        ~is_flat\\n        & ~cond_doji\\n        & (upper > 2 * body_abs)\\n        & (lower < body_abs)\\n    )\\n    cond_spinning = (\\n        ~is_flat\\n        & ~cond_doji\\n        & (body_abs < body_median)\\n        & cond_wicks_balanced\\n    )\\n    cond_belt_bull = (\\n        ~is_flat\\n        & ~cond_doji\\n        & (out[\\\"close\\\"] > out[\\\"open\\\"])\\n        & (lower < WICK_NEGLIGIBLE)\\n        & (upper > WICK_NEGLIGIBLE)\\n    )\\n    cond_belt_bear = (\\n        ~is_flat\\n        & ~cond_doji\\n        & (out[\\\"close\\\"] < out[\\\"open\\\"])\\n        & (upper < WICK_NEGLIGIBLE)\\n        & (lower > WICK_NEGLIGIBLE)\\n    )\\n    cond_standard_bull = ~is_flat & (out[\\\"close\\\"] > out[\\\"open\\\"])\\n    cond_standard_bear = ~is_flat & (out[\\\"close\\\"] < out[\\\"open\\\"])\\n\\n    conditions = [\\n        is_flat,\\n        cond_dragonfly,\\n        cond_gravestone,\\n        cond_doji_standard,\\n        cond_marubozu_bull,\\n        cond_marubozu_bear,\\n        cond_hammer,\\n        cond_inv_hammer,\\n        cond_spinning,\\n        cond_belt_bull,\\n        cond_belt_bear,\\n        cond_standard_bull,\\n        cond_standard_bear,\\n    ]\\n    choices = list(range(CANDLE_CLASS_CARDINALITY))\\n    ids = np.select(conditions, choices, default=3).astype(np.int64)\\n    return pd.Series(ids, index=out.index, dtype=\\\"int64\\\")\\n\\n\\ndef summarize_candle_class_distribution(\\n    feat_df: pd.DataFrame,\\n    *,\\n    high_frac: float = 0.40,\\n    low_frac: float = 0.001,\\n) -> pd.Series:\\n    \\\"\\\"\\\"Return class fractions and print warnings for extreme imbalance.\\n\\n    Args:\\n        feat_df: Feature frame containing ``candle_class_id``.\\n        high_frac: Warn if any class exceeds this share of bars.\\n        low_frac: Warn if any present class is below this share.\\n\\n    Returns:\\n        Normalized value counts indexed by class id.\\n    \\\"\\\"\\\"\\n    counts = feat_df[CANDLE_CLASS_COL].value_counts(normalize=True).sort_index()\\n    for cid, frac in counts.items():\\n        if float(frac) > high_frac or float(frac) < low_frac:\\n            print(\\n                f\\\"WARNING: candle class {int(cid)} fraction {float(frac):.4f} \\\"\\n                f\\\"(thresholds {low_frac:.4f} / {high_frac:.2f})\\\"\\n            )\\n    return counts\\n\\n\\ndef add_candle_structure_features(out: pd.DataFrame) -> pd.DataFrame:\\n    \\\"\\\"\\\"Add causal bar-geometry and bar-to-bar relation features.\\n\\n    All columns use OHLCV and ATR at or before bar t. Shift-based values on\\n    the first bar fill to 0.\\n\\n    Args:\\n        out: Frame with open/high/low/close and atr.\\n\\n    Returns:\\n        The same frame with seven structure columns assigned.\\n    \\\"\\\"\\\"\\n    rng_raw = (out[\\\"high\\\"] - out[\\\"low\\\"]).astype(float)\\n    atr = out[\\\"atr\\\"] if \\\"atr\\\" in out.columns else pd.Series(0.0, index=out.index)\\n    is_flat = rng_raw.fillna(0) <= 0\\n\\n    close_loc = (out[\\\"close\\\"] - out[\\\"low\\\"]) / (rng_raw + _STRUCTURE_EPS)\\n    out[\\\"close_loc\\\"] = close_loc.where(~is_flat, 0.5).fillna(0.5)\\n\\n    out[\\\"range_atr\\\"] = (rng_raw / (atr + _STRUCTURE_EPS)).clip(\\n        -_ATR_NORM_CLIP, _ATR_NORM_CLIP\\n    )\\n    out[\\\"body_atr\\\"] = (\\n        (out[\\\"close\\\"] - out[\\\"open\\\"]).abs() / (atr + _STRUCTURE_EPS)\\n    ).clip(-_ATR_NORM_CLIP, _ATR_NORM_CLIP)\\n    out[\\\"gap_atr\\\"] = (\\n        (out[\\\"open\\\"] - out[\\\"close\\\"].shift(1)) / (atr + _STRUCTURE_EPS)\\n    ).clip(-_ATR_NORM_CLIP, _ATR_NORM_CLIP)\\n\\n    prev_high = out[\\\"high\\\"].shift(1)\\n    prev_low = out[\\\"low\\\"].shift(1)\\n    out[\\\"inside_bar\\\"] = (\\n        (out[\\\"high\\\"] <= prev_high) & (out[\\\"low\\\"] >= prev_low)\\n    ).astype(np.float32)\\n    out[\\\"outside_bar\\\"] = (\\n        (out[\\\"high\\\"] >= prev_high) & (out[\\\"low\\\"] <= prev_low)\\n    ).astype(np.float32)\\n\\n    prior_body = out[\\\"close\\\"].shift(1) - out[\\\"open\\\"].shift(1)\\n    cur_body = out[\\\"close\\\"] - out[\\\"open\\\"]\\n    opposite = np.sign(cur_body) != np.sign(prior_body)\\n    prior_nonzero = prior_body.abs() > _STRUCTURE_EPS\\n    penetration = (out[\\\"close\\\"] - out[\\\"open\\\"].shift(1)) / (\\n        prior_body.abs() + _STRUCTURE_EPS\\n    )\\n    out[\\\"engulf_score\\\"] = (\\n        penetration.clip(-2.0, 2.0)\\n        * np.sign(cur_body)\\n        * opposite.astype(np.float64)\\n        * prior_nonzero.astype(np.float64)\\n    )\\n\\n    shift_fill_cols = (\\\"gap_atr\\\", \\\"inside_bar\\\", \\\"outside_bar\\\", \\\"engulf_score\\\")\\n    for col in shift_fill_cols:\\n        out[col] = out[col].fillna(0.0)\\n    return out\\n\\n\\ndef add_features(\\n    df: pd.DataFrame,\\n    *,\\n    resolution_minutes: int = 15,\\n    atr_period: int = 14,\\n    rsi_period: int = 14,\\n    adx_period: int = 14,\\n    ema_period: int = 50,\\n    macd_fast: int = 12,\\n    macd_slow: int = 26,\\n    macd_signal: int = 9,\\n    include_htf: bool = True,\\n) -> pd.DataFrame:\\n    \\\"\\\"\\\"Compute causal features on a native TF grid.\\n\\n    Args:\\n        include_htf: When True (default), 5m frames also merge resampled HTF\\n            structure. The fused multi-TF model passes False and encodes each\\n            timeframe from independently sampled OHLCV instead.\\n    \\\"\\\"\\\"\\n    out = df.copy()\\n    rv_short = scale_period(16, resolution_minutes)\\n    rv_long = scale_period(96, resolution_minutes)\\n    sr_window = scale_period(96, resolution_minutes)\\n    obv_window = scale_period(96, resolution_minutes)\\n    vol_window = scale_period(96, resolution_minutes)\\n    ret2_bars = max(1, scale_period(2, resolution_minutes))\\n\\n    atr_period = scale_period(atr_period, resolution_minutes)\\n    rsi_period = scale_period(rsi_period, resolution_minutes)\\n    adx_period = scale_period(adx_period, resolution_minutes)\\n    ema_period = scale_period(ema_period, resolution_minutes)\\n    macd_fast = scale_period(macd_fast, resolution_minutes)\\n    macd_slow = scale_period(macd_slow, resolution_minutes)\\n    macd_signal = scale_period(macd_signal, resolution_minutes)\\n\\n    out[\\\"ret_1\\\"] = np.log(out[\\\"close\\\"] / out[\\\"close\\\"].shift(1))\\n\\n    prev_close = out[\\\"close\\\"].shift(1)\\n    tr = pd.concat(\\n        [\\n            out[\\\"high\\\"] - out[\\\"low\\\"],\\n            (out[\\\"high\\\"] - prev_close).abs(),\\n            (out[\\\"low\\\"] - prev_close).abs(),\\n        ],\\n        axis=1,\\n    ).max(axis=1)\\n    out[\\\"atr\\\"] = tr.rolling(atr_period).mean()\\n\\n    out[\\\"rv_16\\\"] = out[\\\"ret_1\\\"].rolling(rv_short).std()\\n    out[\\\"rv_96\\\"] = out[\\\"ret_1\\\"].rolling(rv_long).std()\\n\\n    out[\\\"ema50\\\"] = out[\\\"close\\\"].ewm(span=ema_period, adjust=False).mean()\\n    out[\\\"ema50_dist_pct\\\"] = (out[\\\"close\\\"] - out[\\\"ema50\\\"]) / out[\\\"ema50\\\"]\\n\\n    ema_fast_s = out[\\\"close\\\"].ewm(span=macd_fast, adjust=False).mean()\\n    ema_slow_s = out[\\\"close\\\"].ewm(span=macd_slow, adjust=False).mean()\\n    macd_line = ema_fast_s - ema_slow_s\\n    macd_signal_line = macd_line.ewm(span=macd_signal, adjust=False).mean()\\n    out[\\\"macd_hist\\\"] = macd_line - macd_signal_line\\n\\n    delta = out[\\\"close\\\"].diff()\\n    gain = delta.clip(lower=0).rolling(rsi_period).mean()\\n    loss = (-delta.clip(upper=0)).rolling(rsi_period).mean()\\n    rs = gain / (loss + 1e-9)\\n    out[\\\"rsi_14\\\"] = 100 - (100 / (1 + rs))\\n\\n    up_move = out[\\\"high\\\"].diff()\\n    down_move = -out[\\\"low\\\"].diff()\\n    plus_dm = np.where((up_move > down_move) & (up_move > 0), up_move, 0.0)\\n    minus_dm = np.where((down_move > up_move) & (down_move > 0), down_move, 0.0)\\n    atr_for_di = tr.rolling(adx_period).mean()\\n    plus_di = 100 * pd.Series(plus_dm, index=out.index).rolling(adx_period).mean() / (\\n        atr_for_di + 1e-9\\n    )\\n    minus_di = 100 * pd.Series(minus_dm, index=out.index).rolling(adx_period).mean() / (\\n        atr_for_di + 1e-9\\n    )\\n    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di + 1e-9)\\n    out[\\\"adx_14\\\"] = dx.rolling(adx_period).mean()\\n\\n    obv_raw = (np.sign(out[\\\"close\\\"].diff()) * out[\\\"volume\\\"]).fillna(0).cumsum()\\n    out[\\\"obv_z\\\"] = (obv_raw - obv_raw.rolling(obv_window).mean()) / (\\n        obv_raw.rolling(obv_window).std() + 1e-9\\n    )\\n\\n    out[\\\"vol_z\\\"] = (out[\\\"volume\\\"] - out[\\\"volume\\\"].rolling(vol_window).mean()) / (\\n        out[\\\"volume\\\"].rolling(vol_window).std() + 1e-9\\n    )\\n\\n    rng_raw = (out[\\\"high\\\"] - out[\\\"low\\\"]).astype(float)\\n    is_flat = (rng_raw.fillna(0) <= 0) | (\\n        rng_raw.fillna(0) < out[\\\"atr\\\"].fillna(0) * FLAT_ATR_MULT\\n    )\\n    rng = rng_raw.replace(0, np.nan)\\n    out[\\\"body_ratio\\\"] = (out[\\\"close\\\"] - out[\\\"open\\\"]) / rng\\n    out[\\\"upper_wick_ratio\\\"] = (\\n        out[\\\"high\\\"] - out[[\\\"open\\\", \\\"close\\\"]].max(axis=1)\\n    ) / rng\\n    out[\\\"lower_wick_ratio\\\"] = (\\n        out[[\\\"open\\\", \\\"close\\\"]].min(axis=1) - out[\\\"low\\\"]\\n    ) / rng\\n    out.loc[is_flat, list(_RATIO_COLS)] = 0.0\\n    out[list(_RATIO_COLS)] = out[list(_RATIO_COLS)].fillna(0.0)\\n\\n    out[\\\"hour\\\"] = out[\\\"time\\\"].dt.hour\\n    out[\\\"hour_sin\\\"] = np.sin(2 * np.pi * out[\\\"hour\\\"] / 24)\\n    out[\\\"hour_cos\\\"] = np.cos(2 * np.pi * out[\\\"hour\\\"] / 24)\\n    out[\\\"dow\\\"] = out[\\\"time\\\"].dt.dayofweek\\n    out[\\\"dow_sin\\\"] = np.sin(2 * np.pi * out[\\\"dow\\\"] / 7)\\n    out[\\\"dow_cos\\\"] = np.cos(2 * np.pi * out[\\\"dow\\\"] / 7)\\n\\n    rolling_high = out[\\\"high\\\"].rolling(sr_window).max()\\n    rolling_low = out[\\\"low\\\"].rolling(sr_window).min()\\n    out[\\\"dist_to_resistance_pct\\\"] = (rolling_high - out[\\\"close\\\"]) / out[\\\"close\\\"]\\n    out[\\\"dist_to_support_pct\\\"] = (out[\\\"close\\\"] - rolling_low) / out[\\\"close\\\"]\\n\\n    if \\\"funding_rate\\\" not in out.columns:\\n        out[\\\"funding_rate\\\"] = np.nan\\n\\n    if \\\"open_interest\\\" in out.columns:\\n        oi_z_window = scale_period(96, resolution_minutes)\\n        out[\\\"oi_z\\\"] = (out[\\\"open_interest\\\"] - out[\\\"open_interest\\\"].rolling(oi_z_window).mean()) / (\\n            out[\\\"open_interest\\\"].rolling(oi_z_window).std() + 1e-9\\n        )\\n    else:\\n        out[\\\"oi_z\\\"] = np.nan\\n\\n    ret_2 = out[\\\"close\\\"].pct_change(ret2_bars)\\n    fund_rate = out[\\\"funding_rate\\\"].fillna(0.0)\\n    funding_deriv = compute_funding_derivatives(\\n        fund_rate, ret_2, resolution_minutes=resolution_minutes\\n    )\\n    for col in funding_deriv.columns:\\n        out[col] = funding_deriv[col].values\\n\\n    oi_deriv = compute_oi_derivatives(out, resolution_minutes=resolution_minutes)\\n    out[\\\"oi_change_2\\\"] = oi_deriv[\\\"oi_change_2\\\"]\\n    out[\\\"oi_delta_z\\\"] = oi_deriv[\\\"oi_delta_z\\\"]\\n    out[\\\"oi_price_divergence\\\"] = oi_deriv[\\\"oi_price_divergence\\\"]\\n    out[\\\"oi_acceleration\\\"] = oi_deriv[\\\"oi_acceleration\\\"]\\n    out[\\\"funding_x_oi\\\"] = out[\\\"funding_zscore\\\"] * oi_deriv[\\\"oi_zscore\\\"]\\n\\n    out = add_candle_structure_features(out)\\n    out[CANDLE_CLASS_COL] = classify_candle_shape(out, sr_window=sr_window)\\n    out = add_market_structure_features(out)\\n    if include_htf and int(resolution_minutes) == 5:\\n        out = add_htf_structure_features(out)\\n\\n    return out\\n\\n\\ndef build_feature_matrix(\\n    df: pd.DataFrame,\\n    *,\\n    resolution_minutes: int = 15,\\n    atr_period: int = 14,\\n    dropna: bool = True,\\n) -> pd.DataFrame:\\n    \\\"\\\"\\\"Return feature columns ready for windowing.\\\"\\\"\\\"\\n    feat = add_features(df, resolution_minutes=resolution_minutes, atr_period=atr_period)\\n    if dropna:\\n        feat = feat.dropna().reset_index(drop=True)\\n    return feat\\n\\n\\ndef latest_closed_feature_row(feat_df: pd.DataFrame) -> pd.Series:\\n    \\\"\\\"\\\"Closed-bar row used for diagnostics (second-to-last after dropna).\\\"\\\"\\\"\\n    if len(feat_df) < 2:\\n        raise ValueError(\\\"Need at least 2 feature rows for closed-bar semantics\\\")\\n    return feat_df.iloc[-2]\\n\\n\\ndef validate_feature_columns(\\n    feat_df: pd.DataFrame,\\n    *,\\n    require_finite_closed_bar: bool = False,\\n    feature_cols: Optional[Sequence[str]] = None,\\n) -> None:\\n    cols = tuple(feature_cols) if feature_cols is not None else FEATURE_COLS\\n    missing = [c for c in cols if c not in feat_df.columns]\\n    if missing:\\n        raise ValueError(f\\\"Feature matrix missing columns: {missing}\\\")\\n    if CANDLE_CLASS_COL not in feat_df.columns:\\n        raise ValueError(f\\\"Feature matrix missing {CANDLE_CLASS_COL}\\\")\\n    ids = feat_df[CANDLE_CLASS_COL]\\n    if ((ids < 0) | (ids > CANDLE_CLASS_CARDINALITY - 1)).any():\\n        raise ValueError(\\n            f\\\"{CANDLE_CLASS_COL} out of range [0, {CANDLE_CLASS_CARDINALITY - 1}]\\\"\\n        )\\n    if require_finite_closed_bar and len(feat_df) >= 2:\\n        closed = latest_closed_feature_row(feat_df)\\n        for col in cols:\\n            val = closed[col]\\n            if not np.isfinite(float(val)):\\n                raise ValueError(f\\\"Non-finite closed-bar value for {col}: {val}\\\")\\n        cid = int(closed[CANDLE_CLASS_COL])\\n        if cid < 0 or cid > CANDLE_CLASS_CARDINALITY - 1:\\n            raise ValueError(f\\\"Closed-bar {CANDLE_CLASS_COL} out of range: {cid}\\\")\\n\", \"feature_store/transformer_btcusd/inference.py\": \"\\\"\\\"\\\"Window building and label un-standardization for per-TF transformer inference.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\nfrom typing import Any, Dict, Mapping, Sequence, Tuple\\n\\nimport numpy as np\\n\\nfrom feature_store.transformer_btcusd.contract import (\\n    CANDLE_CLASS_CARDINALITY,\\n    CANDLE_CLASS_COL,\\n    CANDLE_CLASS_NAMES,\\n    CONTINUOUS_LABEL_COLS,\\n    FEATURE_COLS,\\n    FEATURE_CONTRACT_VERSION,\\n    FEATURE_CONTRACT_VERSION_V6,\\n    FEATURE_CONTRACT_VERSION_V7,\\n    FEATURE_CONTRACT_VERSION_V8,\\n    FEATURE_CONTRACT_VERSION_V10,\\n    FEATURE_CONTRACT_VERSION_V11,\\n    ONNX_OUTPUT_NAMES,\\n    ONNX_OUTPUT_NAMES_V6,\\n    ONNX_OUTPUT_NAMES_V7,\\n    ONNX_OUTPUT_NAMES_V8,\\n    ONNX_OUTPUT_NAMES_V9,\\n    ONNX_OUTPUT_NAMES_V10,\\n    ONNX_OUTPUT_NAMES_V11,\\n    STRUCTURE_OUTCOME_NAMES,\\n    TRANSFORMER_FEATURE_CONFIG_FILENAME,\\n    V8_CONTINUOUS_LABEL_COLS,\\n    default_training_config,\\n    model_family_for_resolution,\\n    onnx_filename_for_resolution,\\n    onnx_output_names_for_contract,\\n)\\n\\n\\ndef load_feature_config(path: Path) -> Dict[str, Any]:\\n    raw = json.loads(path.read_text(encoding=\\\"utf-8\\\"))\\n    if not isinstance(raw, dict):\\n        raise ValueError(f\\\"{path} must contain a JSON object\\\")\\n    return raw\\n\\n\\ndef resolve_feature_config(bundle_dir: Path) -> Dict[str, Any]:\\n    cfg_path = bundle_dir / TRANSFORMER_FEATURE_CONFIG_FILENAME\\n    if not cfg_path.is_file():\\n        raise FileNotFoundError(\\n            f\\\"Missing {TRANSFORMER_FEATURE_CONFIG_FILENAME} in {bundle_dir}\\\"\\n        )\\n    return load_feature_config(cfg_path)\\n\\n\\ndef zscore_window(window: np.ndarray) -> np.ndarray:\\n    \\\"\\\"\\\"Per-window z-score (matches Colab WindowDataset).\\\"\\\"\\\"\\n    mu = window.mean(axis=0, keepdims=True)\\n    sd = window.std(axis=0, keepdims=True) + 1e-6\\n    return ((window - mu) / sd).astype(np.float32)\\n\\n\\ndef build_continuous_window(\\n    feat_values: np.ndarray,\\n    *,\\n    window_len: int,\\n    feature_cols: Sequence[str] = FEATURE_COLS,\\n) -> np.ndarray:\\n    \\\"\\\"\\\"Build a z-scored (1, window_len, n_features) tensor from continuous cols.\\\"\\\"\\\"\\n    if feat_values.shape[0] < window_len:\\n        raise ValueError(\\n            f\\\"Need at least {window_len} feature rows, got {feat_values.shape[0]}\\\"\\n        )\\n    window = feat_values[-window_len:, :].astype(np.float32)\\n    if not np.isfinite(window).all():\\n        raise ValueError(\\\"Feature window contains non-finite values\\\")\\n    normed = zscore_window(window)\\n    return normed[np.newaxis, :, :]\\n\\n\\ndef build_candle_class_window(\\n    class_ids: np.ndarray,\\n    *,\\n    window_len: int,\\n    max_class_id: int = CANDLE_CLASS_CARDINALITY - 1,\\n) -> np.ndarray:\\n    \\\"\\\"\\\"Build a raw (1, window_len) int64 tensor of candle class ids.\\\"\\\"\\\"\\n    ids = np.asarray(class_ids).reshape(-1)\\n    if ids.shape[0] < window_len:\\n        raise ValueError(\\n            f\\\"Need at least {window_len} candle class rows, got {ids.shape[0]}\\\"\\n        )\\n    window = ids[-window_len:].astype(np.int64)\\n    if np.any((window < 0) | (window > max_class_id)):\\n        raise ValueError(\\n            f\\\"{CANDLE_CLASS_COL} out of range [0, {max_class_id}]\\\"\\n        )\\n    return window[np.newaxis, :]\\n\\n\\ndef build_inference_window(\\n    feat_values: np.ndarray,\\n    *,\\n    window_len: int,\\n    feature_cols: Sequence[str] = FEATURE_COLS,\\n) -> np.ndarray:\\n    \\\"\\\"\\\"Build a single (1, window_len, n_features) tensor from feature matrix values.\\\"\\\"\\\"\\n    return build_continuous_window(\\n        feat_values, window_len=window_len, feature_cols=feature_cols\\n    )\\n\\n\\ndef unstandardize_continuous(\\n    pred_z: np.ndarray,\\n    label_mean: Sequence[float],\\n    label_std: Sequence[float],\\n    *,\\n    label_cols: Sequence[str] = CONTINUOUS_LABEL_COLS,\\n) -> Dict[str, float]:\\n    \\\"\\\"\\\"Map standardized ONNX output back to real units.\\\"\\\"\\\"\\n    mean = np.asarray(label_mean, dtype=np.float64)\\n    std = np.asarray(label_std, dtype=np.float64)\\n    flat = np.asarray(pred_z, dtype=np.float64).reshape(-1)\\n    if flat.shape[0] != len(label_cols):\\n        raise ValueError(\\n            f\\\"Expected {len(label_cols)} continuous outputs, got {flat.shape[0]}\\\"\\n        )\\n    real = flat * std + mean\\n    return {str(col): float(val) for col, val in zip(label_cols, real)}\\n\\n\\ndef softmax(logits: np.ndarray) -> np.ndarray:\\n    x = np.asarray(logits, dtype=np.float64).reshape(-1)\\n    x = x - x.max()\\n    exp = np.exp(x)\\n    return exp / (exp.sum() + 1e-12)\\n\\n\\ndef parse_regime_prediction(\\n    regime_logits: np.ndarray,\\n    regime_names: Mapping[str, str],\\n) -> Tuple[int, str, Dict[str, float]]:\\n    probs = softmax(regime_logits)\\n    idx = int(np.argmax(probs))\\n    name = regime_names.get(str(idx), regime_names.get(idx, f\\\"CLASS_{idx}\\\"))\\n    prob_map = {\\n        regime_names.get(str(i), f\\\"CLASS_{i}\\\"): float(probs[i])\\n        for i in range(len(probs))\\n    }\\n    return idx, str(name), prob_map\\n\\n\\ndef require_onnx_output_names(\\n    output_names: Sequence[str],\\n    *,\\n    contract_version: str | None = None,\\n    resolution: str = \\\"15m\\\",\\n) -> None:\\n    \\\"\\\"\\\"Reject bundles that lack the heads required by the given contract.\\\"\\\"\\\"\\n    have = {str(name) for name in output_names}\\n    ver = str(contract_version or FEATURE_CONTRACT_VERSION_V6)\\n    required = onnx_output_names_for_contract(ver, resolution=resolution)\\n    known = (\\n        FEATURE_CONTRACT_VERSION,\\n        FEATURE_CONTRACT_VERSION_V11,\\n        FEATURE_CONTRACT_VERSION_V10,\\n        FEATURE_CONTRACT_VERSION_V8,\\n        FEATURE_CONTRACT_VERSION_V7,\\n        FEATURE_CONTRACT_VERSION_V6,\\n    )\\n    if ver not in known and not required:\\n        required = ONNX_OUTPUT_NAMES_V6\\n    missing = [name for name in required if name not in have]\\n    if missing:\\n        raise RuntimeError(\\n            f\\\"ONNX bundle is not {ver}: missing outputs {missing}. Retrain all TFs.\\\"\\n        )\\n\\n\\ndef feature_config_from_training_export(\\n    *,\\n    feature_cols: Sequence[str],\\n    window_len: int,\\n    label_mean: Sequence[float],\\n    label_std: Sequence[float],\\n    q_edges: Sequence[float],\\n    config: Mapping[str, Any],\\n    contract_version: str | None = None,\\n    onnx_output_names: Sequence[str] | None = None,\\n    continuous_label_cols: Sequence[str] | None = None,\\n) -> Dict[str, Any]:\\n    \\\"\\\"\\\"Build feature_config.json payload written by Colab export cell.\\\"\\\"\\\"\\n    version = str(contract_version or FEATURE_CONTRACT_VERSION_V6)\\n    if onnx_output_names is not None:\\n        names = list(onnx_output_names)\\n    elif version == FEATURE_CONTRACT_VERSION_V11:\\n        names = list(ONNX_OUTPUT_NAMES_V11)\\n    elif version == FEATURE_CONTRACT_VERSION_V10:\\n        names = list(ONNX_OUTPUT_NAMES_V10)\\n    elif version == FEATURE_CONTRACT_VERSION:\\n        names = list(ONNX_OUTPUT_NAMES_V9)\\n    elif version == FEATURE_CONTRACT_VERSION_V8:\\n        names = list(ONNX_OUTPUT_NAMES_V8)\\n    elif version == FEATURE_CONTRACT_VERSION_V7:\\n        names = list(ONNX_OUTPUT_NAMES_V7)\\n    else:\\n        names = list(ONNX_OUTPUT_NAMES)\\n    if continuous_label_cols is not None:\\n        label_cols = list(continuous_label_cols)\\n    elif version in (FEATURE_CONTRACT_VERSION, FEATURE_CONTRACT_VERSION_V8):\\n        label_cols = list(V8_CONTINUOUS_LABEL_COLS)\\n    else:\\n        label_cols = list(CONTINUOUS_LABEL_COLS)\\n    return {\\n        \\\"feature_contract_version\\\": version,\\n        \\\"feature_cols\\\": list(feature_cols),\\n        \\\"categorical_cols\\\": [CANDLE_CLASS_COL],\\n        \\\"categorical_cardinality\\\": {CANDLE_CLASS_COL: CANDLE_CLASS_CARDINALITY},\\n        \\\"candle_class_names\\\": {str(k): v for k, v in CANDLE_CLASS_NAMES.items()},\\n        \\\"window_len\\\": int(window_len),\\n        \\\"continuous_label_cols\\\": label_cols,\\n        \\\"label_mean\\\": [float(x) for x in label_mean],\\n        \\\"label_std\\\": [float(x) for x in label_std],\\n        \\\"regime_names\\\": {\\\"0\\\": \\\"LOW\\\", \\\"1\\\": \\\"NORMAL\\\", \\\"2\\\": \\\"HIGH\\\", \\\"3\\\": \\\"EXTREME\\\"},\\n        \\\"structure_outcome_names\\\": {\\n            str(k): v for k, v in STRUCTURE_OUTCOME_NAMES.items()\\n        },\\n        \\\"onnx_output_names\\\": names,\\n        \\\"vol_regime_quantile_edges\\\": [float(x) for x in q_edges],\\n        \\\"config\\\": dict(config),\\n    }\\n\\n\\ndef metadata_from_training_export(\\n    *,\\n    resolution: str,\\n    label_mean: Sequence[float],\\n    label_std: Sequence[float],\\n    config: Mapping[str, Any],\\n    test_metrics: Mapping[str, Any] | None = None,\\n    export_quality: Mapping[str, Any] | None = None,\\n    onnx_output_names: Sequence[str] | None = None,\\n) -> Dict[str, Any]:\\n    \\\"\\\"\\\"Build metadata_transformer.json for a per-TF bundle.\\\"\\\"\\\"\\n    res = resolution.strip().lower()\\n    cfg = dict(config)\\n    names = list(onnx_output_names or ONNX_OUTPUT_NAMES)\\n    meta: Dict[str, Any] = {\\n        \\\"version\\\": \\\"transformer_per_tf_v1\\\",\\n        \\\"model_name\\\": f\\\"jacksparrow_transformer_BTCUSD_{res}\\\",\\n        \\\"model_family\\\": model_family_for_resolution(res),\\n        \\\"symbol\\\": str(cfg.get(\\\"symbol\\\") or \\\"BTCUSD\\\"),\\n        \\\"resolution\\\": res,\\n        \\\"resolution_minutes\\\": int(cfg.get(\\\"resolution_minutes\\\") or 15),\\n        \\\"onnx_filename\\\": onnx_filename_for_resolution(res),\\n        \\\"feature_config_filename\\\": TRANSFORMER_FEATURE_CONFIG_FILENAME,\\n        \\\"path_label_horizon_bars\\\": int(cfg.get(\\\"path_label_horizon_bars\\\") or 8),\\n        \\\"atr_period\\\": int(cfg.get(\\\"atr_period\\\") or 14),\\n        \\\"default_threshold\\\": float(cfg.get(\\\"default_threshold\\\") or 0.005),\\n        \\\"primary_signal_mode\\\": \\\"path_edge\\\",\\n        \\\"onnx_output_names\\\": names,\\n        \\\"label_mean\\\": [float(x) for x in label_mean],\\n        \\\"label_std\\\": [float(x) for x in label_std],\\n        \\\"test_metrics\\\": dict(test_metrics or {}),\\n        \\\"training_config\\\": cfg,\\n    }\\n    if export_quality:\\n        meta[\\\"export_quality\\\"] = dict(export_quality)\\n    return meta\\n\\n\\ndef training_config_for_resolution(resolution: str) -> Dict[str, Any]:\\n    return default_training_config(resolution)\\n\", \"scripts/colab/transformer_data.py\": \"\\\"\\\"\\\"Public Delta India history fetch for Colab transformer training.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport time\\nfrom datetime import datetime, timedelta, timezone\\nfrom typing import Any, Dict, List, Optional\\n\\nimport pandas as pd\\nimport requests\\n\\nfrom feature_store.transformer_btcusd.features import assemble_raw_frame\\n\\n_RESOLUTION_MINUTES = {\\n    \\\"1m\\\": 1,\\n    \\\"3m\\\": 3,\\n    \\\"5m\\\": 5,\\n    \\\"10m\\\": 10,\\n    \\\"15m\\\": 15,\\n    \\\"30m\\\": 30,\\n    \\\"1h\\\": 60,\\n    \\\"2h\\\": 120,\\n    \\\"4h\\\": 240,\\n    \\\"6h\\\": 360,\\n    \\\"1d\\\": 1440,\\n}\\n\\n# Documented candles cap is 2000; live India API has returned 4000 (newest-in-range\\n# when the window is larger). Request windows stay at the documented 2000. The loop\\n# walks backward from the oldest timestamp actually returned so a lower cap cannot\\n# open gaps. Production loaders use 500 for the same reason.\\nDELTA_CANDLE_PAGE_BARS = 2000\\nREQUEST_DELAY_SECONDS = 0.2\\nEMPTY_PAGE_RETRIES = 3\\nMIN_OHLCV_COMPLETENESS = 0.95\\n\\n\\ndef _bar_seconds(resolution: str) -> int:\\n    if resolution not in _RESOLUTION_MINUTES:\\n        raise ValueError(\\n            f\\\"Unsupported resolution {resolution!r}; \\\"\\n            f\\\"use one of {sorted(_RESOLUTION_MINUTES)}\\\"\\n        )\\n    return int(_RESOLUTION_MINUTES[resolution]) * 60\\n\\n\\ndef _parse_candle_rows(payload: Any) -> List[Dict[str, Any]]:\\n    \\\"\\\"\\\"Normalize Delta history/candles payloads to a list of candle dicts.\\\"\\\"\\\"\\n    if not isinstance(payload, dict):\\n        return []\\n    result = payload.get(\\\"result\\\", [])\\n    if isinstance(result, dict):\\n        result = result.get(\\\"candles\\\", []) or []\\n    if not isinstance(result, list):\\n        return []\\n    rows: List[Dict[str, Any]] = []\\n    for row in result:\\n        if isinstance(row, dict) and \\\"time\\\" in row:\\n            rows.append(row)\\n    return rows\\n\\n\\ndef _candle_epoch(row: Dict[str, Any]) -> Optional[int]:\\n    try:\\n        return int(row[\\\"time\\\"])\\n    except (KeyError, TypeError, ValueError):\\n        return None\\n\\n\\ndef _request_candle_page(\\n    url: str,\\n    *,\\n    symbol: str,\\n    resolution: str,\\n    start_ts: int,\\n    end_ts: int,\\n) -> List[Dict[str, Any]]:\\n    last_error: Optional[Exception] = None\\n    for attempt in range(EMPTY_PAGE_RETRIES):\\n        try:\\n            response = requests.get(\\n                url,\\n                params={\\n                    \\\"symbol\\\": symbol,\\n                    \\\"resolution\\\": resolution,\\n                    \\\"start\\\": int(start_ts),\\n                    \\\"end\\\": int(end_ts),\\n                },\\n                timeout=30,\\n            )\\n            response.raise_for_status()\\n            payload = response.json()\\n        except (requests.RequestException, ValueError) as exc:\\n            last_error = exc\\n            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))\\n            continue\\n        if payload.get(\\\"success\\\") is False:\\n            last_error = RuntimeError(f\\\"Delta candles error for {symbol}: {payload}\\\")\\n            time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))\\n            continue\\n        rows = _parse_candle_rows(payload)\\n        if rows:\\n            return rows\\n        last_error = None\\n        time.sleep(REQUEST_DELAY_SECONDS * (attempt + 1))\\n    if last_error is not None:\\n        raise last_error\\n    return []\\n\\n\\ndef _rows_to_frame(all_rows: List[Dict[str, Any]]) -> pd.DataFrame:\\n    df = pd.DataFrame(all_rows)\\n    expected_cols = [\\\"time\\\", \\\"open\\\", \\\"high\\\", \\\"low\\\", \\\"close\\\", \\\"volume\\\"]\\n    df = df[[c for c in expected_cols if c in df.columns]]\\n    df[\\\"time\\\"] = pd.to_datetime(df[\\\"time\\\"], unit=\\\"s\\\", utc=True)\\n    df = df.drop_duplicates(subset=\\\"time\\\").sort_values(\\\"time\\\").reset_index(drop=True)\\n    for col in [\\\"open\\\", \\\"high\\\", \\\"low\\\", \\\"close\\\", \\\"volume\\\"]:\\n        if col in df.columns:\\n            df[col] = df[col].astype(float)\\n    return df\\n\\n\\ndef fetch_candles(\\n    symbol: str,\\n    resolution: str,\\n    start_ts: int,\\n    end_ts: int,\\n    base_url: str,\\n    *,\\n    page_bars: int = DELTA_CANDLE_PAGE_BARS,\\n) -> pd.DataFrame:\\n    \\\"\\\"\\\"Pull OHLC-style candles with response-driven backward pagination.\\n\\n    Each request covers at most ``page_bars`` of wall-clock. The next ``end`` is the\\n    oldest timestamp actually returned minus one second, so a server cap below the\\n    requested window cannot skip bars.\\n\\n    Args:\\n        symbol: Delta symbol (e.g. ``BTCUSD``, ``FUNDING:BTCUSD``).\\n        resolution: Candle resolution key in ``_RESOLUTION_MINUTES``.\\n        start_ts: Inclusive Unix seconds.\\n        end_ts: Exclusive-ish Unix seconds (API treats the range as a window).\\n        base_url: Exchange origin, without a trailing path.\\n        page_bars: Max bars per request window (default documented 2000).\\n\\n    Returns:\\n        Deduplicated OHLCV frame sorted by ``time`` ascending.\\n\\n    Raises:\\n        ValueError: Unknown resolution.\\n        RuntimeError: No candles in the requested range.\\n    \\\"\\\"\\\"\\n    url = f\\\"{base_url.rstrip('/')}/v2/history/candles\\\"\\n    bar_seconds = _bar_seconds(resolution)\\n    page_bars = max(1, int(page_bars))\\n    start_ts, end_ts = int(start_ts), int(end_ts)\\n    if end_ts <= start_ts:\\n        raise ValueError(\\\"end_ts must be greater than start_ts\\\")\\n\\n    all_rows: List[Dict[str, Any]] = []\\n    cursor_end = end_ts\\n    seen_oldest: Optional[int] = None\\n    max_pages = max(2, (end_ts - start_ts) // bar_seconds + 5)\\n\\n    for _ in range(max_pages):\\n        if cursor_end <= start_ts:\\n            break\\n        page_start = max(start_ts, cursor_end - page_bars * bar_seconds)\\n        if page_start >= cursor_end:\\n            break\\n        rows = _request_candle_page(\\n            url,\\n            symbol=symbol,\\n            resolution=resolution,\\n            start_ts=page_start,\\n            end_ts=cursor_end,\\n        )\\n        if not rows:\\n            break\\n        epochs = [e for e in (_candle_epoch(r) for r in rows) if e is not None]\\n        if not epochs:\\n            break\\n        oldest = min(epochs)\\n        if seen_oldest is not None and oldest >= seen_oldest:\\n            break\\n        seen_oldest = oldest\\n        all_rows.extend(rows)\\n        cursor_end = oldest - 1\\n        time.sleep(REQUEST_DELAY_SECONDS)\\n\\n    if not all_rows:\\n        raise RuntimeError(\\n            f\\\"No candle data returned for symbol={symbol}; \\\"\\n            \\\"check symbol/resolution/date range.\\\"\\n        )\\n    return _rows_to_frame(all_rows)\\n\\n\\ndef validate_ohlcv_completeness(\\n    df: pd.DataFrame,\\n    resolution: str,\\n    *,\\n    min_completeness: float = MIN_OHLCV_COMPLETENESS,\\n    symbol: str = \\\"\\\",\\n) -> Dict[str, Any]:\\n    \\\"\\\"\\\"Measure gap rate on a native-TF OHLCV frame.\\n\\n    Completeness is ``1 - gaps / (n - 1)`` where a gap is a bar-to-bar delta greater\\n    than ``1.5 * bar_seconds``. This catches sawtooth holes from truncated pages.\\n\\n    Args:\\n        df: Frame with a ``time`` column (datetime or Unix seconds).\\n        resolution: Candle resolution key.\\n        min_completeness: Raise if the gap-free fraction is below this.\\n        symbol: Optional label for the error message.\\n\\n    Returns:\\n        Report with ``rows``, ``gaps``, ``completeness``, and ``span_bars``.\\n\\n    Raises:\\n        ValueError: Empty frame or completeness below ``min_completeness``.\\n    \\\"\\\"\\\"\\n    if df is None or df.empty or \\\"time\\\" not in df.columns:\\n        raise ValueError(\\\"Cannot validate completeness on an empty OHLCV frame\\\")\\n    bar_seconds = _bar_seconds(resolution)\\n    time_col = df[\\\"time\\\"]\\n    if pd.api.types.is_datetime64_any_dtype(time_col):\\n        epochs = (pd.to_datetime(time_col, utc=True).astype(\\\"int64\\\") // 10**9).tolist()\\n    else:\\n        epochs = pd.to_numeric(time_col, errors=\\\"coerce\\\").dropna().astype(int).tolist()\\n    if len(epochs) < 2:\\n        raise ValueError(\\\"Need at least 2 OHLCV bars to validate completeness\\\")\\n\\n    gaps = 0\\n    for prev, cur in zip(epochs, epochs[1:]):\\n        if int(cur) - int(prev) > bar_seconds * 1.5:\\n            gaps += 1\\n    completeness = 1.0 - (gaps / max(1, len(epochs) - 1))\\n    span_bars = int((epochs[-1] - epochs[0]) // bar_seconds) + 1\\n    report: Dict[str, Any] = {\\n        \\\"rows\\\": len(epochs),\\n        \\\"gaps\\\": int(gaps),\\n        \\\"completeness\\\": round(float(completeness), 4),\\n        \\\"span_bars\\\": span_bars,\\n        \\\"expected_bars\\\": span_bars,\\n    }\\n    if completeness < min_completeness:\\n        label = f\\\"{symbol} {resolution}\\\".strip()\\n        raise ValueError(\\n            f\\\"OHLCV completeness {completeness:.1%} below {min_completeness:.0%} \\\"\\n            f\\\"for {label or 'candles'} ({gaps} gaps, {len(epochs)} rows, \\\"\\n            f\\\"span {span_bars} bars). Refetch with response-driven pagination.\\\"\\n        )\\n    return report\\n\\n\\ndef validate_derivatives_coverage(\\n    df: pd.DataFrame,\\n    *,\\n    min_coverage: float = 0.5,\\n    warn_coverage: float = 0.9,\\n) -> Dict[str, Any]:\\n    \\\"\\\"\\\"Check funding/OI non-null coverage over the OHLCV timeline.\\\"\\\"\\\"\\n    n = len(df)\\n    if n == 0:\\n        raise ValueError(\\\"Cannot validate derivatives coverage on empty frame\\\")\\n\\n    report: Dict[str, Any] = {\\\"rows\\\": n, \\\"columns\\\": {}}\\n    for col in (\\\"funding_rate\\\", \\\"open_interest\\\"):\\n        if col not in df.columns:\\n            report[\\\"columns\\\"][col] = {\\\"coverage\\\": 0.0, \\\"status\\\": \\\"missing\\\"}\\n            continue\\n        coverage = float(df[col].notna().mean())\\n        status = \\\"ok\\\"\\n        if coverage < min_coverage:\\n            status = \\\"error\\\"\\n        elif coverage < warn_coverage:\\n            status = \\\"warn\\\"\\n        report[\\\"columns\\\"][col] = {\\n            \\\"coverage\\\": round(coverage, 4),\\n            \\\"status\\\": status,\\n        }\\n\\n    worst = min(v[\\\"coverage\\\"] for v in report[\\\"columns\\\"].values())\\n    report[\\\"worst_coverage\\\"] = worst\\n    if worst < min_coverage:\\n        raise ValueError(\\n            \\\"Derivatives coverage below minimum \\\"\\n            f\\\"({worst:.1%} < {min_coverage:.0%}): {report['columns']}\\\"\\n        )\\n    return report\\n\\n\\ndef fetch_history_bundle(\\n    *,\\n    symbol: str,\\n    resolution: str,\\n    history_days: int,\\n    base_url: str,\\n    min_derivatives_coverage: float = 0.5,\\n    derivatives_coverage_warn: float = 0.9,\\n    min_ohlcv_completeness: float = MIN_OHLCV_COMPLETENESS,\\n) -> pd.DataFrame:\\n    \\\"\\\"\\\"Fetch OHLCV + funding + OI merged frame (notebook-compatible).\\\"\\\"\\\"\\n    end_dt = datetime.now(timezone.utc)\\n    start_dt = end_dt - timedelta(days=history_days)\\n    start_ts, end_ts = int(start_dt.timestamp()), int(end_dt.timestamp())\\n\\n    raw_df = fetch_candles(symbol, resolution, start_ts, end_ts, base_url)\\n    ohlcv_report = validate_ohlcv_completeness(\\n        raw_df,\\n        resolution,\\n        min_completeness=min_ohlcv_completeness,\\n        symbol=symbol,\\n    )\\n    print(\\n        f\\\"OHLCV {symbol} {resolution}: {ohlcv_report['rows']} bars, \\\"\\n        f\\\"completeness {ohlcv_report['completeness']:.1%}, \\\"\\n        f\\\"gaps={ohlcv_report['gaps']}\\\"\\n    )\\n\\n    try:\\n        funding_df = fetch_candles(\\n            f\\\"FUNDING:{symbol}\\\", resolution, start_ts, end_ts, base_url\\n        )\\n        funding_df = funding_df[[\\\"time\\\", \\\"close\\\"]].rename(columns={\\\"close\\\": \\\"funding_rate\\\"})\\n    except Exception:\\n        funding_df = pd.DataFrame({\\\"time\\\": [], \\\"funding_rate\\\": []})\\n\\n    try:\\n        oi_df = fetch_candles(f\\\"OI:{symbol}\\\", resolution, start_ts, end_ts, base_url)\\n        oi_df = oi_df[[\\\"time\\\", \\\"close\\\"]].rename(columns={\\\"close\\\": \\\"open_interest\\\"})\\n    except Exception:\\n        oi_df = pd.DataFrame({\\\"time\\\": [], \\\"open_interest\\\": []})\\n\\n    raw_df = assemble_raw_frame(raw_df, funding_df=funding_df, oi_df=oi_df)\\n\\n    report = validate_derivatives_coverage(\\n        raw_df,\\n        min_coverage=min_derivatives_coverage,\\n        warn_coverage=derivatives_coverage_warn,\\n    )\\n    for col, info in report[\\\"columns\\\"].items():\\n        if info[\\\"status\\\"] == \\\"warn\\\":\\n            print(\\n                f\\\"WARNING: {col} coverage {info['coverage']:.1%} \\\"\\n                f\\\"below recommended {derivatives_coverage_warn:.0%}\\\"\\n            )\\n        elif info[\\\"status\\\"] == \\\"ok\\\":\\n            print(f\\\"{col} coverage: {info['coverage']:.1%}\\\")\\n\\n    return raw_df\\n\", \"research/swinging/__init__.py\": \"\\\"\\\"\\\"Hybrid algorithmic and ML research for BTCUSD.\\n\\nThis package does not place orders and does not import the execution stack.\\n\\\"\\\"\\\"\\n\\nfrom research.swinging.contract import SPEC_VERSION\\n\\n__all__ = [\\\"SPEC_VERSION\\\"]\\n\", \"research/swinging/contract.py\": \"\\\"\\\"\\\"Frozen constants for the swinging research track.\\n\\nNothing here enables an order. Cost figures match the forecast v5 round trip\\nand add an explicit funding term at simulation time.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nSPEC_VERSION = \\\"swinging-v1\\\"\\nSYMBOL = \\\"BTCUSD\\\"\\nRESOLUTIONS = (\\\"5m\\\", \\\"15m\\\", \\\"1h\\\", \\\"4h\\\")\\nBAR_MINUTES = {\\\"5m\\\": 5, \\\"15m\\\": 15, \\\"1h\\\": 60, \\\"4h\\\": 240}\\nBAR_SECONDS = {name: minutes * 60 for name, minutes in BAR_MINUTES.items()}\\n\\nDELTA_BASE_URL = \\\"https://api.india.delta.exchange\\\"\\nDELTA_CANDLES_URL = f\\\"{DELTA_BASE_URL}/v2/history/candles\\\"\\nPAGE_BARS = 2000\\nHISTORY_START = \\\"2018-01-01T00:00:00+00:00\\\"\\nCOLAB_DATA_ROOT = \\\"/content/JackSparrow/data\\\"\\nCOLAB_REPO_ROOT = \\\"/content/JackSparrow\\\"\\n\\nTAKER_FEE_RATE = 0.0005\\nSLIPPAGE_BPS = 5.0\\nROUND_TRIP_COST = 2.0 * (TAKER_FEE_RATE + SLIPPAGE_BPS / 10_000.0)\\nBREAKEVEN_BAND = 0.0005\\n\\nSTOP_ATR_MULT = 1.5\\nTAKE_PROFIT_ATR_MULT = 2.5\\nMAX_HOLD_BARS = {\\\"5m\\\": 48, \\\"15m\\\": 32, \\\"1h\\\": 24, \\\"4h\\\": 12}\\nBREAKOUT_LOOKBACK = 20\\nWARMUP_BARS = 30\\nNEUTRAL_STRIDE = 8\\nACCEPT_SCORE = 0.55\\nREJECT_SCORE = 0.35\\n\\nSETUP_TYPES = (\\n    \\\"long_breakout\\\",\\n    \\\"short_breakdown\\\",\\n    \\\"breakout_retest\\\",\\n    \\\"support_bounce\\\",\\n    \\\"resistance_rejection\\\",\\n    \\\"reversal\\\",\\n    \\\"trend_continuation\\\",\\n    \\\"candlestick\\\",\\n    \\\"chart_pattern\\\",\\n)\\nDIRECTIONS = (\\\"long\\\", \\\"short\\\", \\\"neutral\\\")\\nCANDIDATE_STATUSES = (\\\"accepted\\\", \\\"rejected\\\", \\\"neutral\\\")\\nOUTCOME_LABELS = (\\\"win\\\", \\\"loss\\\", \\\"breakeven\\\", \\\"expiry\\\")\\nREGIME_LABELS = (\\n    \\\"trending\\\",\\n    \\\"ranging\\\",\\n    \\\"high_volatility\\\",\\n    \\\"low_volatility\\\",\\n    \\\"transition\\\",\\n)\\nSR_LABELS = (\\\"bounce\\\", \\\"rejection\\\", \\\"breakout\\\", \\\"failed_breakout\\\")\\nSTRATEGIES = (\\\"algorithm_only\\\", \\\"ml_only\\\", \\\"hybrid\\\")\\n\\nSERIES_KINDS = (\\\"ohlcv\\\", \\\"mark\\\", \\\"funding\\\", \\\"oi\\\")\\nDERIVATIVE_PREFIX = {\\\"mark\\\": \\\"MARK:\\\", \\\"funding\\\": \\\"FUNDING:\\\", \\\"oi\\\": \\\"OI:\\\"}\\n\\nSWING_COLUMNS = (\\n    \\\"swing_high_dist_atr\\\",\\n    \\\"swing_low_dist_atr\\\",\\n    \\\"bars_since_swing_high\\\",\\n    \\\"bars_since_swing_low\\\",\\n)\\n\\n# Labels and trade results. These must never appear in a feature snapshot.\\nFORBIDDEN_FEATURE_COLUMNS = frozenset(\\n    {\\n        \\\"outcome_label\\\",\\n        \\\"sr_label\\\",\\n        \\\"regime_target\\\",\\n        \\\"net_return\\\",\\n        \\\"gross_return\\\",\\n        \\\"mfe\\\",\\n        \\\"mae\\\",\\n        \\\"exit_price\\\",\\n        \\\"exit_timestamp\\\",\\n        \\\"funding_pnl\\\",\\n        \\\"breakout_status\\\",\\n        \\\"orders_enabled\\\",\\n    }\\n)\\n\\nSETUP_META_COLUMNS = (\\n    \\\"setup_id\\\",\\n    \\\"signal_timestamp\\\",\\n    \\\"bar_index\\\",\\n    \\\"direction\\\",\\n    \\\"setup_type\\\",\\n    \\\"timeframe\\\",\\n    \\\"algorithmic_score\\\",\\n    \\\"reasoning\\\",\\n    \\\"invalidation\\\",\\n    \\\"status\\\",\\n)\\n\\nOUTCOME_COLUMNS = (\\n    \\\"entry_timestamp\\\",\\n    \\\"entry_price\\\",\\n    \\\"stop_price\\\",\\n    \\\"take_profit_price\\\",\\n    \\\"exit_timestamp\\\",\\n    \\\"exit_price\\\",\\n    \\\"exit_reason\\\",\\n    \\\"holding_seconds\\\",\\n    \\\"mfe\\\",\\n    \\\"mae\\\",\\n    \\\"gross_return\\\",\\n    \\\"fee_return\\\",\\n    \\\"slippage_return\\\",\\n    \\\"funding_pnl\\\",\\n    \\\"missing_funding\\\",\\n    \\\"net_return\\\",\\n    \\\"outcome_label\\\",\\n    \\\"simulated\\\",\\n)\\n\\nPROMOTION_STATUS_RESEARCH = \\\"research\\\"\\nORDERS_ENABLED = False\\nSETUP_BUNDLE_DIR = \\\"JackSparrow_SetupEffectiveness_BTCUSD\\\"\\nSR_BUNDLE_DIR = \\\"JackSparrow_SREffectiveness_BTCUSD\\\"\\nHIGH_VOL_ATR_PCT = 0.02\\nTREND_ADX_MIN = 22.0\\nTREND_HURST_MIN = 0.52\\nLOW_VOL_ATR_PCT_MAX = 0.003\\nLOW_VOL_REGIME_MAX = 0.85\\nRANGE_ADX_MAX = 22.0\\nRANGE_VOL_REGIME_MAX = 1.05\\n\\n\\ndef series_symbol(kind: str, symbol: str = SYMBOL) -> str:\\n    \\\"\\\"\\\"Return the Delta symbol for an OHLCV or derivative candle series.\\n\\n    Args:\\n        kind: ``ohlcv``, ``mark``, ``funding``, or ``oi``.\\n        symbol: Instrument symbol, default ``BTCUSD``.\\n\\n    Returns:\\n        The public history symbol.\\n\\n    Raises:\\n        ValueError: If ``kind`` is unknown.\\n    \\\"\\\"\\\"\\n    if kind == \\\"ohlcv\\\":\\n        return symbol\\n    prefix = DERIVATIVE_PREFIX.get(kind)\\n    if prefix is None:\\n        raise ValueError(f\\\"unknown series kind: {kind}\\\")\\n    return f\\\"{prefix}{symbol}\\\"\\n\\n\\ndef max_hold_bars(resolution: str) -> int:\\n    \\\"\\\"\\\"Return the maximum holding window for one resolution.\\n\\n    Args:\\n        resolution: One of ``RESOLUTIONS``.\\n\\n    Returns:\\n        Bar count.\\n\\n    Raises:\\n        ValueError: If the resolution is unknown.\\n    \\\"\\\"\\\"\\n    if resolution not in MAX_HOLD_BARS:\\n        raise ValueError(f\\\"unknown resolution: {resolution}\\\")\\n    return int(MAX_HOLD_BARS[resolution])\\n\", \"research/swinging/quality.py\": \"\\\"\\\"\\\"OHLCV checks, resample comparisons, and dataset manifests.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nfrom datetime import datetime, timezone\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport pandas as pd\\n\\nfrom research.cpr_mtf.bars import resample_ohlcv\\nfrom research.swinging.contract import BAR_SECONDS, DELTA_CANDLES_URL, RESOLUTIONS\\n\\n_OHLC = (\\\"open\\\", \\\"high\\\", \\\"low\\\", \\\"close\\\")\\n\\n\\ndef empty_ohlcv() -> pd.DataFrame:\\n    \\\"\\\"\\\"Return an empty OHLCV frame with the canonical columns.\\\"\\\"\\\"\\n    return pd.DataFrame(columns=[\\\"time\\\", \\\"open\\\", \\\"high\\\", \\\"low\\\", \\\"close\\\", \\\"volume\\\"])\\n\\n\\ndef normalize_ohlcv(frame: pd.DataFrame) -> pd.DataFrame:\\n    \\\"\\\"\\\"Sort, coerce, and dedupe candles on UTC time.\\n\\n    Args:\\n        frame: Raw candle rows. ``time`` may be unix seconds or a timestamp.\\n            ``timestamp`` is accepted as an alias.\\n\\n    Returns:\\n        A frame sorted by ``time`` with one row per timestamp.\\n\\n    Raises:\\n        ValueError: If a required column is missing.\\n    \\\"\\\"\\\"\\n    if frame is None or frame.empty:\\n        return empty_ohlcv()\\n    work = frame.copy()\\n    if \\\"time\\\" not in work.columns:\\n        if \\\"timestamp\\\" not in work.columns:\\n            raise ValueError(\\\"candle frame needs a time or timestamp column\\\")\\n        work[\\\"time\\\"] = work[\\\"timestamp\\\"]\\n    missing = [name for name in (\\\"open\\\", \\\"high\\\", \\\"low\\\", \\\"close\\\", \\\"volume\\\") if name not in work.columns]\\n    if missing:\\n        raise ValueError(f\\\"candle frame is missing columns: {missing}\\\")\\n    time = pd.to_datetime(work[\\\"time\\\"], utc=True)\\n    if not pd.api.types.is_datetime64_any_dtype(work[\\\"time\\\"]) and pd.api.types.is_numeric_dtype(\\n        work[\\\"time\\\"]\\n    ):\\n        time = pd.to_datetime(work[\\\"time\\\"], unit=\\\"s\\\", utc=True)\\n    work[\\\"time\\\"] = time\\n    for column in (\\\"open\\\", \\\"high\\\", \\\"low\\\", \\\"close\\\", \\\"volume\\\"):\\n        work[column] = pd.to_numeric(work[column], errors=\\\"coerce\\\")\\n    work = work.dropna(subset=[\\\"time\\\", \\\"open\\\", \\\"high\\\", \\\"low\\\", \\\"close\\\"])\\n    work = work.drop_duplicates(subset=\\\"time\\\", keep=\\\"last\\\")\\n    work = work.sort_values(\\\"time\\\").reset_index(drop=True)\\n    return work.loc[:, [\\\"time\\\", \\\"open\\\", \\\"high\\\", \\\"low\\\", \\\"close\\\", \\\"volume\\\"]]\\n\\n\\ndef ohlc_mask(frame: pd.DataFrame) -> pd.Series:\\n    \\\"\\\"\\\"Return True for rows whose high and low contain the open and close.\\n\\n    Args:\\n        frame: Normalized OHLCV.\\n\\n    Returns:\\n        A boolean series aligned to ``frame``.\\n    \\\"\\\"\\\"\\n    if frame.empty:\\n        return pd.Series(dtype=bool)\\n    high_ok = frame[\\\"high\\\"] >= frame[[\\\"open\\\", \\\"close\\\", \\\"low\\\"]].max(axis=1)\\n    low_ok = frame[\\\"low\\\"] <= frame[[\\\"open\\\", \\\"close\\\", \\\"high\\\"]].min(axis=1)\\n    return high_ok & low_ok & frame[\\\"high\\\"].ge(frame[\\\"low\\\"])\\n\\n\\ndef drop_invalid_ohlc(frame: pd.DataFrame) -> tuple[pd.DataFrame, int]:\\n    \\\"\\\"\\\"Remove rows that break OHLC consistency.\\n\\n    Args:\\n        frame: Normalized OHLCV.\\n\\n    Returns:\\n        The valid rows and the number of rows removed.\\n    \\\"\\\"\\\"\\n    if frame.empty:\\n        return frame.copy(), 0\\n    mask = ohlc_mask(frame)\\n    removed = int((~mask).sum())\\n    return frame.loc[mask].reset_index(drop=True), removed\\n\\n\\ndef gap_report(frame: pd.DataFrame, resolution: str) -> dict[str, Any]:\\n    \\\"\\\"\\\"Count missing bars from timestamp spacing.\\n\\n    Args:\\n        frame: Normalized OHLCV.\\n        resolution: Bar size key.\\n\\n    Returns:\\n        Rows, gap count, completeness, and the observed span.\\n\\n    Raises:\\n        ValueError: If the resolution is unknown.\\n    \\\"\\\"\\\"\\n    if resolution not in BAR_SECONDS:\\n        raise ValueError(f\\\"unknown resolution: {resolution}\\\")\\n    if frame.empty:\\n        return {\\n            \\\"rows\\\": 0,\\n            \\\"gaps\\\": 0,\\n            \\\"completeness\\\": 0.0,\\n            \\\"earliest\\\": None,\\n            \\\"latest\\\": None,\\n        }\\n    bar_seconds = BAR_SECONDS[resolution]\\n    stamps = pd.to_datetime(frame[\\\"time\\\"], utc=True)\\n    unit = str(stamps.dtype)\\n    raw = stamps.astype(\\\"int64\\\")\\n    if unit.startswith(\\\"datetime64[us\\\"):\\n        divisor = 1_000_000\\n    elif unit.startswith(\\\"datetime64[ms\\\"):\\n        divisor = 1_000\\n    elif unit.startswith(\\\"datetime64[s\\\"):\\n        divisor = 1\\n    else:\\n        divisor = 1_000_000_000\\n    epochs = (raw // divisor).astype(\\\"int64\\\").tolist()\\n    gaps = 0\\n    for previous, current in zip(epochs, epochs[1:]):\\n        if int(current) - int(previous) > bar_seconds * 1.5:\\n            gaps += 1\\n    span = max(1, len(epochs) - 1)\\n    completeness = 1.0 - (gaps / span) if len(epochs) > 1 else 1.0\\n    return {\\n        \\\"rows\\\": int(len(epochs)),\\n        \\\"gaps\\\": int(gaps),\\n        \\\"completeness\\\": round(float(completeness), 4),\\n        \\\"earliest\\\": pd.Timestamp(frame[\\\"time\\\"].iloc[0]).isoformat(),\\n        \\\"latest\\\": pd.Timestamp(frame[\\\"time\\\"].iloc[-1]).isoformat(),\\n    }\\n\\n\\ndef compare_native_to_resampled(\\n    native: pd.DataFrame,\\n    five_minute: pd.DataFrame,\\n    resolution: str,\\n    *,\\n    price_tolerance: float = 1e-6,\\n) -> dict[str, Any]:\\n    \\\"\\\"\\\"Compare native candles with a resample of the 5-minute series.\\n\\n    Args:\\n        native: Native timeframe candles.\\n        five_minute: 5-minute candles covering the same instrument.\\n        resolution: ``15m``, ``1h``, or ``4h``.\\n        price_tolerance: Relative close tolerance.\\n\\n    Returns:\\n        Overlap counts and the number of closes that disagree.\\n\\n    Raises:\\n        ValueError: If ``resolution`` is not a resample target.\\n    \\\"\\\"\\\"\\n    if resolution not in (\\\"15m\\\", \\\"1h\\\", \\\"4h\\\"):\\n        raise ValueError(f\\\"{resolution} is not resampled from 5-minute bars\\\")\\n    resampled = resample_ohlcv(five_minute, resolution)\\n    if native.empty or resampled.empty:\\n        return {\\n            \\\"resolution\\\": resolution,\\n            \\\"native_rows\\\": int(len(native)),\\n            \\\"resampled_rows\\\": int(len(resampled)),\\n            \\\"overlap\\\": 0,\\n            \\\"close_mismatches\\\": 0,\\n        }\\n    left = native.copy()\\n    right = resampled.copy()\\n    left[\\\"time\\\"] = pd.to_datetime(left[\\\"time\\\"], utc=True)\\n    right[\\\"time\\\"] = pd.to_datetime(right[\\\"time\\\"], utc=True)\\n    merged = left.merge(right, on=\\\"time\\\", suffixes=(\\\"_native\\\", \\\"_resampled\\\"))\\n    if merged.empty:\\n        mismatches = 0\\n    else:\\n        scale = merged[\\\"close_native\\\"].abs().clip(lower=1.0)\\n        delta = (merged[\\\"close_native\\\"] - merged[\\\"close_resampled\\\"]).abs() / scale\\n        mismatches = int((delta > price_tolerance).sum())\\n    return {\\n        \\\"resolution\\\": resolution,\\n        \\\"native_rows\\\": int(len(native)),\\n        \\\"resampled_rows\\\": int(len(resampled)),\\n        \\\"overlap\\\": int(len(merged)),\\n        \\\"close_mismatches\\\": mismatches,\\n    }\\n\\n\\ndef frame_sha256(frame: pd.DataFrame) -> str:\\n    \\\"\\\"\\\"Hash the canonical CSV bytes of a frame.\\n\\n    Args:\\n        frame: Table to fingerprint.\\n\\n    Returns:\\n        Hex sha256.\\n    \\\"\\\"\\\"\\n    payload = frame.to_csv(index=False).encode(\\\"utf-8\\\")\\n    return hashlib.sha256(payload).hexdigest()\\n\\n\\ndef build_manifest(\\n    frame: pd.DataFrame,\\n    *,\\n    symbol: str,\\n    resolution: str,\\n    kind: str,\\n    source_error: str | None = None,\\n) -> dict[str, Any]:\\n    \\\"\\\"\\\"Describe one stored series, including the observed timestamp span.\\n\\n    Args:\\n        frame: Normalized candles. May be empty when the API returned nothing.\\n        symbol: Delta symbol that was requested.\\n        resolution: Candle resolution.\\n        kind: ``ohlcv``, ``mark``, ``funding``, or ``oi``.\\n        source_error: Fetch error text when the series is missing.\\n\\n    Returns:\\n        JSON-ready manifest. Earliest and latest are null when no rows exist.\\n    \\\"\\\"\\\"\\n    report = gap_report(frame, resolution) if resolution in RESOLUTIONS else {\\n        \\\"rows\\\": int(len(frame)),\\n        \\\"gaps\\\": None,\\n        \\\"completeness\\\": None,\\n        \\\"earliest\\\": None if frame.empty else pd.Timestamp(frame[\\\"time\\\"].iloc[0]).isoformat(),\\n        \\\"latest\\\": None if frame.empty else pd.Timestamp(frame[\\\"time\\\"].iloc[-1]).isoformat(),\\n    }\\n    return {\\n        \\\"symbol\\\": symbol,\\n        \\\"resolution\\\": resolution,\\n        \\\"kind\\\": kind,\\n        \\\"rows\\\": int(len(frame)),\\n        \\\"earliest\\\": report.get(\\\"earliest\\\"),\\n        \\\"latest\\\": report.get(\\\"latest\\\"),\\n        \\\"gaps\\\": report.get(\\\"gaps\\\"),\\n        \\\"completeness\\\": report.get(\\\"completeness\\\"),\\n        \\\"sha256\\\": frame_sha256(frame),\\n        \\\"source_url\\\": DELTA_CANDLES_URL,\\n        \\\"source_error\\\": source_error,\\n        \\\"generated_at\\\": datetime.now(timezone.utc).isoformat(),\\n    }\\n\\n\\ndef write_json(path: Path, payload: dict[str, Any]) -> None:\\n    \\\"\\\"\\\"Write a JSON document, creating parent directories.\\n\\n    Args:\\n        path: Destination file.\\n        payload: JSON-ready mapping.\\n    \\\"\\\"\\\"\\n    path.parent.mkdir(parents=True, exist_ok=True)\\n    path.write_text(json.dumps(payload, indent=2), encoding=\\\"utf-8\\\")\\n\\n\\ndef quality_report(\\n    frame: pd.DataFrame,\\n    resolution: str,\\n    *,\\n    invalid_rows: int,\\n    five_minute: pd.DataFrame | None = None,\\n) -> dict[str, Any]:\\n    \\\"\\\"\\\"Assemble the data-quality document for one OHLCV series.\\n\\n    Args:\\n        frame: Validated OHLCV.\\n        resolution: Bar size.\\n        invalid_rows: Rows removed for OHLC inconsistency.\\n        five_minute: Optional 5-minute frame used to check a native higher timeframe.\\n\\n    Returns:\\n        Quality report dictionary.\\n    \\\"\\\"\\\"\\n    report: dict[str, Any] = {\\n        \\\"resolution\\\": resolution,\\n        \\\"invalid_ohlc_rows\\\": int(invalid_rows),\\n        \\\"gaps\\\": gap_report(frame, resolution),\\n    }\\n    if five_minute is not None and resolution in (\\\"15m\\\", \\\"1h\\\", \\\"4h\\\"):\\n        report[\\\"resample_check\\\"] = compare_native_to_resampled(frame, five_minute, resolution)\\n    return report\\n\", \"research/swinging/data.py\": \"\\\"\\\"\\\"Delta public-history collection onto a local directory.\\n\\nPagination, retries, and the 2,000-bar window live in\\n``scripts.colab.transformer_data.fetch_candles``. This module chooses the\\nsymbol, walks the available span, and writes parquet plus a manifest.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nfrom datetime import datetime, timezone\\nfrom pathlib import Path\\nfrom typing import Any, Callable\\n\\nimport pandas as pd\\n\\nfrom research.swinging.contract import (\\n    BAR_SECONDS,\\n    COLAB_DATA_ROOT,\\n    DELTA_BASE_URL,\\n    HISTORY_START,\\n    PAGE_BARS,\\n    RESOLUTIONS,\\n    SERIES_KINDS,\\n    SYMBOL,\\n    series_symbol,\\n)\\nfrom research.swinging.quality import (\\n    build_manifest,\\n    drop_invalid_ohlc,\\n    normalize_ohlcv,\\n    quality_report,\\n    write_json,\\n)\\n\\nFetchCandles = Callable[[str, str, int, int, str], pd.DataFrame]\\n\\n\\ndef _default_fetch(\\n    symbol: str,\\n    resolution: str,\\n    start_ts: int,\\n    end_ts: int,\\n    base_url: str,\\n) -> pd.DataFrame:\\n    \\\"\\\"\\\"Call the shared Colab candle fetcher.\\n\\n    Args:\\n        symbol: Delta symbol, including ``MARK:`` / ``FUNDING:`` / ``OI:`` prefixes.\\n        resolution: Candle resolution.\\n        start_ts: Inclusive unix seconds.\\n        end_ts: Window end unix seconds.\\n        base_url: Exchange origin.\\n\\n    Returns:\\n        Deduplicated OHLCV from ``fetch_candles``.\\n    \\\"\\\"\\\"\\n    from scripts.colab.transformer_data import fetch_candles\\n\\n    return fetch_candles(\\n        symbol,\\n        resolution,\\n        start_ts,\\n        end_ts,\\n        base_url,\\n        page_bars=PAGE_BARS,\\n    )\\n\\n\\ndef _unix(value: datetime | pd.Timestamp) -> int:\\n    stamp = pd.Timestamp(value)\\n    if stamp.tzinfo is None:\\n        stamp = stamp.tz_localize(\\\"UTC\\\")\\n    else:\\n        stamp = stamp.tz_convert(\\\"UTC\\\")\\n    return int(stamp.timestamp())\\n\\n\\ndef fetch_series(\\n    kind: str,\\n    resolution: str,\\n    start: datetime,\\n    end: datetime,\\n    *,\\n    symbol: str = SYMBOL,\\n    base_url: str = DELTA_BASE_URL,\\n    fetcher: FetchCandles | None = None,\\n) -> tuple[pd.DataFrame, str | None]:\\n    \\\"\\\"\\\"Fetch one series and keep an empty frame when the API has no rows.\\n\\n    Args:\\n        kind: ``ohlcv``, ``mark``, ``funding``, or ``oi``.\\n        resolution: Candle resolution.\\n        start: Requested start. The returned frame may begin later.\\n        end: Requested end.\\n        symbol: Instrument.\\n        base_url: Exchange origin.\\n        fetcher: Replacement for ``fetch_candles``, used in tests.\\n\\n    Returns:\\n        Normalized candles and an error string when the fetch failed.\\n    \\\"\\\"\\\"\\n    fetch = fetcher or _default_fetch\\n    requested = series_symbol(kind, symbol)\\n    try:\\n        raw = fetch(requested, resolution, _unix(start), _unix(end), base_url)\\n    except (RuntimeError, ValueError) as exc:\\n        return normalize_ohlcv(pd.DataFrame()), str(exc)\\n    return normalize_ohlcv(raw), None\\n\\n\\ndef _read_parquet(path: Path) -> pd.DataFrame:\\n    if not path.is_file():\\n        return normalize_ohlcv(pd.DataFrame())\\n    return normalize_ohlcv(pd.read_parquet(path))\\n\\n\\ndef _merge_incremental(existing: pd.DataFrame, fresh: pd.DataFrame) -> pd.DataFrame:\\n    if existing.empty:\\n        return fresh\\n    if fresh.empty:\\n        return existing\\n    return normalize_ohlcv(pd.concat([existing, fresh], ignore_index=True))\\n\\n\\ndef series_path(root: Path | str, resolution: str, kind: str, symbol: str = SYMBOL) -> Path:\\n    \\\"\\\"\\\"Return the parquet path for one stored series.\\n\\n    Args:\\n        root: Dataset root. On Colab this is ``/content/JackSparrow/data``.\\n        resolution: Candle resolution.\\n        kind: Series kind.\\n        symbol: Instrument.\\n\\n    Returns:\\n        Parquet path.\\n    \\\"\\\"\\\"\\n    return Path(root) / symbol / resolution / f\\\"{kind}.parquet\\\"\\n\\n\\ndef collect_series(\\n    root: Path | str,\\n    resolution: str,\\n    kind: str,\\n    *,\\n    symbol: str = SYMBOL,\\n    base_url: str = DELTA_BASE_URL,\\n    fetcher: FetchCandles | None = None,\\n    incremental: bool = True,\\n    start: datetime | None = None,\\n    end: datetime | None = None,\\n) -> dict[str, Any]:\\n    \\\"\\\"\\\"Download one series into the local dataset root.\\n\\n    Args:\\n        root: Directory that receives parquet and JSON reports.\\n        resolution: Candle resolution.\\n        kind: Series kind.\\n        symbol: Instrument.\\n        base_url: Exchange origin.\\n        fetcher: Optional fetch replacement.\\n        incremental: When true, request only bars after the stored maximum.\\n        start: Override the probe start. Defaults to ``HISTORY_START`` or the\\n            last stored bar.\\n        end: Override the probe end. Defaults to the current UTC time.\\n\\n    Returns:\\n        The manifest written beside the parquet file.\\n    \\\"\\\"\\\"\\n    destination = series_path(root, resolution, kind, symbol)\\n    existing = _read_parquet(destination) if incremental else normalize_ohlcv(pd.DataFrame())\\n    end_at = end or datetime.now(timezone.utc)\\n    if start is not None:\\n        start_at = start\\n    elif not existing.empty:\\n        start_at = pd.Timestamp(existing[\\\"time\\\"].iloc[-1]).to_pydatetime()\\n        start_at = start_at + pd.Timedelta(seconds=BAR_SECONDS[resolution])\\n    else:\\n        start_at = pd.Timestamp(HISTORY_START).to_pydatetime()\\n    if start_at.tzinfo is None:\\n        start_at = start_at.replace(tzinfo=timezone.utc)\\n    frame, error = fetch_series(\\n        kind,\\n        resolution,\\n        start_at,\\n        end_at,\\n        symbol=symbol,\\n        base_url=base_url,\\n        fetcher=fetcher,\\n    )\\n    combined = _merge_incremental(existing, frame)\\n    if kind == \\\"ohlcv\\\":\\n        combined, invalid_rows = drop_invalid_ohlc(combined)\\n    else:\\n        invalid_rows = 0\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    combined.to_parquet(destination, index=False)\\n    manifest = build_manifest(\\n        combined,\\n        symbol=series_symbol(kind, symbol),\\n        resolution=resolution,\\n        kind=kind,\\n        source_error=error,\\n    )\\n    manifest[\\\"invalid_ohlc_rows\\\"] = invalid_rows\\n    manifest[\\\"incremental\\\"] = bool(incremental and not existing.empty)\\n    write_json(destination.with_name(f\\\"{kind}_manifest.json\\\"), manifest)\\n    return manifest\\n\\n\\ndef collect_resolution(\\n    root: Path | str,\\n    resolution: str,\\n    *,\\n    symbol: str = SYMBOL,\\n    base_url: str = DELTA_BASE_URL,\\n    fetcher: FetchCandles | None = None,\\n    incremental: bool = True,\\n    five_minute: pd.DataFrame | None = None,\\n) -> dict[str, Any]:\\n    \\\"\\\"\\\"Collect OHLCV, mark, funding, and open interest for one resolution.\\n\\n    Args:\\n        root: Dataset root.\\n        resolution: Candle resolution.\\n        symbol: Instrument.\\n        base_url: Exchange origin.\\n        fetcher: Optional fetch replacement.\\n        incremental: Append to parquet already on disk.\\n        five_minute: Optional 5-minute frame for the resample check.\\n\\n    Returns:\\n        Manifests and the OHLCV quality report.\\n    \\\"\\\"\\\"\\n    manifests = {}\\n    for kind in SERIES_KINDS:\\n        manifests[kind] = collect_series(\\n            root,\\n            resolution,\\n            kind,\\n            symbol=symbol,\\n            base_url=base_url,\\n            fetcher=fetcher,\\n            incremental=incremental,\\n        )\\n    ohlcv = _read_parquet(series_path(root, resolution, \\\"ohlcv\\\", symbol))\\n    invalid = int(manifests[\\\"ohlcv\\\"].get(\\\"invalid_ohlc_rows\\\") or 0)\\n    reference = five_minute\\n    if reference is None and resolution != \\\"5m\\\":\\n        reference = _read_parquet(series_path(root, \\\"5m\\\", \\\"ohlcv\\\", symbol))\\n    report = quality_report(\\n        ohlcv,\\n        resolution,\\n        invalid_rows=invalid,\\n        five_minute=None if reference is None or reference.empty else reference,\\n    )\\n    write_json(series_path(root, resolution, \\\"ohlcv\\\", symbol).with_name(\\\"quality.json\\\"), report)\\n    return {\\\"manifests\\\": manifests, \\\"quality\\\": report}\\n\\n\\ndef collect_all(\\n    root: Path | str = COLAB_DATA_ROOT,\\n    *,\\n    symbol: str = SYMBOL,\\n    base_url: str = DELTA_BASE_URL,\\n    fetcher: FetchCandles | None = None,\\n    incremental: bool = True,\\n    resolutions: tuple[str, ...] = RESOLUTIONS,\\n) -> dict[str, Any]:\\n    \\\"\\\"\\\"Collect every swinging resolution under ``root``.\\n\\n    Args:\\n        root: Local dataset root. Colab uses ``/content/JackSparrow/data``.\\n        symbol: Instrument.\\n        base_url: Exchange origin.\\n        fetcher: Optional fetch replacement.\\n        incremental: Append to existing parquet.\\n        resolutions: Resolutions to request. Five-minute data is collected\\n            before higher timeframes so the resample check has a reference.\\n\\n    Returns:\\n        One report per resolution.\\n    \\\"\\\"\\\"\\n    ordered = tuple(dict.fromkeys((\\\"5m\\\", *resolutions)))\\n    reports: dict[str, Any] = {}\\n    for resolution in ordered:\\n        if resolution not in resolutions and resolution != \\\"5m\\\":\\n            continue\\n        reports[resolution] = collect_resolution(\\n            root,\\n            resolution,\\n            symbol=symbol,\\n            base_url=base_url,\\n            fetcher=fetcher,\\n            incremental=incremental,\\n        )\\n    return reports\\n\", \"research/swinging/regime.py\": \"\\\"\\\"\\\"Regime labels aligned with the agent's structure thresholds.\\n\\nThe label is a training target. It is not a model input.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Mapping\\n\\nimport pandas as pd\\n\\nfrom research.swinging.contract import (\\n    HIGH_VOL_ATR_PCT,\\n    LOW_VOL_ATR_PCT_MAX,\\n    LOW_VOL_REGIME_MAX,\\n    RANGE_ADX_MAX,\\n    RANGE_VOL_REGIME_MAX,\\n    TREND_ADX_MIN,\\n    TREND_HURST_MIN,\\n)\\n\\n\\ndef _number(features: Mapping[str, object], key: str, default: float) -> float:\\n    raw = features.get(key, default)\\n    try:\\n        value = float(raw)  # type: ignore[arg-type]\\n    except (TypeError, ValueError):\\n        return default\\n    if value != value:\\n        return default\\n    return value\\n\\n\\ndef regime_target(features: Mapping[str, object]) -> str:\\n    \\\"\\\"\\\"Classify one closed bar the way ``classify_market_structure`` does.\\n\\n    Args:\\n        features: Causal feature values. ``atr_pct`` is ATR divided by close.\\n            Missing keys use the same defaults as the agent helper.\\n\\n    Returns:\\n        One of ``trending``, ``ranging``, ``high_volatility``,\\n        ``low_volatility``, or ``transition``.\\n    \\\"\\\"\\\"\\n    adx = _number(features, \\\"adx_14\\\", 0.0)\\n    atr_pct = _number(features, \\\"atr_pct\\\", 0.0)\\n    vol_regime = _number(features, \\\"vol_regime\\\", 1.0)\\n    hurst = _number(features, \\\"hurst_60\\\", 0.5)\\n    if atr_pct >= HIGH_VOL_ATR_PCT:\\n        return \\\"high_volatility\\\"\\n    if atr_pct > 0.0 and atr_pct < LOW_VOL_ATR_PCT_MAX and vol_regime < LOW_VOL_REGIME_MAX:\\n        return \\\"low_volatility\\\"\\n    if adx >= TREND_ADX_MIN and hurst >= TREND_HURST_MIN:\\n        return \\\"trending\\\"\\n    if adx < RANGE_ADX_MAX and vol_regime < RANGE_VOL_REGIME_MAX:\\n        return \\\"ranging\\\"\\n    return \\\"transition\\\"\\n\\n\\ndef attach_regime_targets(featured: pd.DataFrame) -> pd.DataFrame:\\n    \\\"\\\"\\\"Add ``regime_target`` from each row's own feature values.\\n\\n    Args:\\n        featured: Feature frame that may include ``atr_14`` and ``close``.\\n\\n    Returns:\\n        A copy with ``regime_target``. ``atr_pct`` is filled from ATR and close\\n        when the caller did not supply it.\\n    \\\"\\\"\\\"\\n    frame = featured.copy()\\n    if \\\"atr_pct\\\" not in frame.columns and {\\\"atr_14\\\", \\\"close\\\"}.issubset(frame.columns):\\n        close = pd.to_numeric(frame[\\\"close\\\"], errors=\\\"coerce\\\").replace(0, pd.NA)\\n        frame[\\\"atr_pct\\\"] = pd.to_numeric(frame[\\\"atr_14\\\"], errors=\\\"coerce\\\") / close\\n    labels = [\\n        regime_target(row.dropna().to_dict())\\n        for _, row in frame.iterrows()\\n    ]\\n    frame[\\\"regime_target\\\"] = labels\\n    return frame\\n\", \"research/swinging/features.py\": \"\\\"\\\"\\\"Causal feature snapshots shared by training and inference.\\n\\n``snapshot_features`` builds the frame on bars at or before the as-of index.\\nLater bars are not passed to the engines.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom feature_store.feature_registry import (\\n    CANDLESTICK_FEATURES,\\n    CHART_PATTERN_FEATURES,\\n    FEATURE_LIST,\\n    MTF_CONTEXT_FEATURES,\\n)\\nfrom feature_store.pattern_features.candlestick_patterns import CandlestickPatternEngine\\nfrom feature_store.pattern_features.chart_patterns import ChartPatternEngine\\nfrom feature_store.transformer_btcusd.contract import NATIVE_STRUCTURE_COLS\\nfrom feature_store.transformer_btcusd.structure import add_market_structure_features\\nfrom feature_store.unified_feature_engine import UnifiedFeatureEngine\\nfrom research.cpr_mtf.features import causal_atr\\nfrom research.cpr_mtf.swings import detect_swings\\nfrom research.swinging.contract import (\\n    BAR_MINUTES,\\n    COLAB_DATA_ROOT,\\n    FORBIDDEN_FEATURE_COLUMNS,\\n    RESOLUTIONS,\\n    SWING_COLUMNS,\\n    SYMBOL,\\n)\\nfrom research.swinging.quality import normalize_ohlcv\\n\\n_ATR_PERIOD = 14\\n\\n\\ndef snapshot_columns() -> tuple[str, ...]:\\n    \\\"\\\"\\\"Return the feature names a snapshot always contains.\\n\\n    Returns:\\n        Stable column order. Outcome columns are not included.\\n    \\\"\\\"\\\"\\n    ordered: list[str] = []\\n    for name in (\\n        *FEATURE_LIST,\\n        *CANDLESTICK_FEATURES,\\n        *CHART_PATTERN_FEATURES,\\n        *MTF_CONTEXT_FEATURES,\\n        *NATIVE_STRUCTURE_COLS,\\n        *SWING_COLUMNS,\\n    ):\\n        if name in FORBIDDEN_FEATURE_COLUMNS or name in ordered:\\n            continue\\n        ordered.append(name)\\n    return tuple(ordered)\\n\\n\\ndef _finite(value: object) -> float:\\n    try:\\n        number = float(value)  # type: ignore[arg-type]\\n    except (TypeError, ValueError):\\n        return 0.0\\n    if number != number or number in (float(\\\"inf\\\"), float(\\\"-inf\\\")):\\n        return 0.0\\n    return number\\n\\n\\ndef _prepare_ohlcv(frame: pd.DataFrame) -> pd.DataFrame:\\n    work = normalize_ohlcv(frame)\\n    if work.empty:\\n        raise ValueError(\\\"feature frame is empty\\\")\\n    work[\\\"timestamp\\\"] = pd.to_datetime(work[\\\"time\\\"], utc=True)\\n    return work\\n\\n\\ndef _swing_frame(work: pd.DataFrame, resolution: str) -> pd.DataFrame:\\n    high = work[\\\"high\\\"].to_numpy(dtype=np.float64)\\n    low = work[\\\"low\\\"].to_numpy(dtype=np.float64)\\n    close = work[\\\"close\\\"].to_numpy(dtype=np.float64)\\n    atr = causal_atr(high, low, close, _ATR_PERIOD)\\n    swings = detect_swings(work, resolution, atr)\\n    count = len(work)\\n    high_dist = np.zeros(count, dtype=np.float64)\\n    low_dist = np.zeros(count, dtype=np.float64)\\n    bars_high = np.full(count, -1.0, dtype=np.float64)\\n    bars_low = np.full(count, -1.0, dtype=np.float64)\\n    pending: dict[int, list[Any]] = {}\\n    for swing in swings:\\n        pending.setdefault(int(swing.confirmation_index), []).append(swing)\\n    last_high = None\\n    last_low = None\\n    for index in range(count):\\n        for swing in pending.get(index, []):\\n            if swing.swing_type == \\\"high\\\":\\n                last_high = swing\\n            else:\\n                last_low = swing\\n        scale = atr[index]\\n        if not np.isfinite(scale) or scale <= 0.0:\\n            scale = max(abs(float(close[index])) * 0.01, 1e-9)\\n        if last_high is not None:\\n            high_dist[index] = (float(close[index]) - float(last_high.pivot_price)) / scale\\n            bars_high[index] = float(index - int(last_high.confirmation_index))\\n        if last_low is not None:\\n            low_dist[index] = (float(close[index]) - float(last_low.pivot_price)) / scale\\n            bars_low[index] = float(index - int(last_low.confirmation_index))\\n    return pd.DataFrame(\\n        {\\n            \\\"swing_high_dist_atr\\\": high_dist,\\n            \\\"swing_low_dist_atr\\\": low_dist,\\n            \\\"bars_since_swing_high\\\": bars_high,\\n            \\\"bars_since_swing_low\\\": bars_low,\\n        }\\n    )\\n\\n\\ndef build_feature_frame(frame: pd.DataFrame, resolution: str) -> pd.DataFrame:\\n    \\\"\\\"\\\"Compute swinging features for every row of ``frame``.\\n\\n    The engines see only this frame. Callers that need a causal row pass the\\n    prefix that ends at the prediction bar.\\n\\n    Args:\\n        frame: OHLCV candles.\\n        resolution: Bar size. Selects the unified-engine minute scale.\\n\\n    Returns:\\n        Feature rows in ``snapshot_columns`` order, plus ``time`` and ``close``.\\n\\n    Raises:\\n        ValueError: If the resolution is unknown or the frame is empty.\\n        RuntimeError: If a forbidden future column is present.\\n    \\\"\\\"\\\"\\n    if resolution not in BAR_MINUTES:\\n        raise ValueError(f\\\"unknown resolution: {resolution}\\\")\\n    work = _prepare_ohlcv(frame)\\n    minutes = BAR_MINUTES[resolution]\\n    base = UnifiedFeatureEngine().compute_batch(\\n        work,\\n        resolution_minutes=minutes,\\n        include_pattern_features=False,\\n        include_mtf_context=True,\\n        fill_invalid=True,\\n    )\\n    candles = CandlestickPatternEngine().compute_all(work)\\n    charts = ChartPatternEngine().compute_all(work)\\n    structured = add_market_structure_features(work.drop(columns=[\\\"timestamp\\\"], errors=\\\"ignore\\\"))\\n    structure_cols = [name for name in NATIVE_STRUCTURE_COLS if name in structured.columns]\\n    pieces = [\\n        base.reset_index(drop=True),\\n        candles.reset_index(drop=True),\\n        charts.reset_index(drop=True),\\n        structured.loc[:, structure_cols].reset_index(drop=True),\\n        _swing_frame(work, resolution),\\n    ]\\n    featured = pd.concat(pieces, axis=1)\\n    featured = featured.loc[:, ~featured.columns.duplicated()]\\n    columns = snapshot_columns()\\n    for name in columns:\\n        if name not in featured.columns:\\n            featured[name] = 0.0\\n    featured = featured.loc[:, list(columns)]\\n    leaked = set(featured.columns) & FORBIDDEN_FEATURE_COLUMNS\\n    if leaked:\\n        raise RuntimeError(f\\\"feature frame contains future columns: {sorted(leaked)}\\\")\\n    featured.insert(0, \\\"time\\\", work[\\\"time\\\"].to_numpy())\\n    featured.insert(1, \\\"close\\\", work[\\\"close\\\"].to_numpy())\\n    return featured.replace([np.inf, -np.inf], 0.0).fillna(0.0)\\n\\n\\ndef snapshot_features(frame: pd.DataFrame, asof_index: int, resolution: str) -> dict[str, float]:\\n    \\\"\\\"\\\"Return the feature row known at ``asof_index``.\\n\\n    Args:\\n        frame: Full OHLCV series. Rows after ``asof_index`` are discarded\\n            before any engine runs.\\n        asof_index: Signal bar. Must be inside ``frame``.\\n        resolution: Bar size.\\n\\n    Returns:\\n        Finite feature values in ``snapshot_columns`` order.\\n\\n    Raises:\\n        ValueError: If the index is outside the frame.\\n    \\\"\\\"\\\"\\n    if asof_index < 0 or asof_index >= len(frame):\\n        raise ValueError(f\\\"as-of index {asof_index} is outside the series\\\")\\n    featured = build_feature_frame(frame.iloc[: asof_index + 1], resolution)\\n    row = featured.iloc[-1]\\n    return {name: _finite(row[name]) for name in snapshot_columns()}\\n\\n\\ndef materialize_features(\\n    root: Path | str = COLAB_DATA_ROOT,\\n    *,\\n    symbol: str = SYMBOL,\\n    resolutions: tuple[str, ...] = RESOLUTIONS,\\n) -> dict[str, str]:\\n    \\\"\\\"\\\"Write a feature parquet next to each stored OHLCV file.\\n\\n    Args:\\n        root: Local dataset root.\\n        symbol: Instrument directory name.\\n        resolutions: Resolutions that already have ``ohlcv.parquet``.\\n\\n    Returns:\\n        Map of resolution to the feature parquet path.\\n    \\\"\\\"\\\"\\n    written: dict[str, str] = {}\\n    for resolution in resolutions:\\n        source = Path(root) / symbol / resolution / \\\"ohlcv.parquet\\\"\\n        if not source.is_file():\\n            continue\\n        featured = build_feature_frame(pd.read_parquet(source), resolution)\\n        destination = source.with_name(\\\"features.parquet\\\")\\n        featured.to_parquet(destination, index=False)\\n        written[resolution] = str(destination)\\n    return written\\n\", \"research/swinging/setups.py\": \"\\\"\\\"\\\"Deterministic setup candidates, including rejects and neutrals.\\n\\nChart-pattern rows reuse CPR proposals. A proposal's future breakout status\\nis not copied onto the feature snapshot.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom research.cpr_mtf.candidates import generate_candidates\\nfrom research.cpr_mtf.features import causal_atr\\nfrom research.cpr_mtf.swings import detect_swings\\nfrom research.swinging.contract import (\\n    ACCEPT_SCORE,\\n    BREAKOUT_LOOKBACK,\\n    COLAB_DATA_ROOT,\\n    NEUTRAL_STRIDE,\\n    REJECT_SCORE,\\n    RESOLUTIONS,\\n    SETUP_META_COLUMNS,\\n    SYMBOL,\\n    WARMUP_BARS,\\n)\\nfrom research.swinging.features import snapshot_columns, snapshot_features\\nfrom research.swinging.quality import normalize_ohlcv\\n\\n\\ndef _stamp(value: object) -> str:\\n    return pd.Timestamp(value).isoformat()\\n\\n\\ndef _row(\\n    *,\\n    resolution: str,\\n    index: int,\\n    timestamp: object,\\n    setup_type: str,\\n    direction: str,\\n    score: float,\\n    reasoning: str,\\n    invalidation: str,\\n    status: str,\\n    features: dict[str, float],\\n) -> dict[str, Any]:\\n    record: dict[str, Any] = {\\n        \\\"setup_id\\\": f\\\"{resolution}:{setup_type}:{index}:{direction}\\\",\\n        \\\"signal_timestamp\\\": _stamp(timestamp),\\n        \\\"bar_index\\\": int(index),\\n        \\\"direction\\\": direction,\\n        \\\"setup_type\\\": setup_type,\\n        \\\"timeframe\\\": resolution,\\n        \\\"algorithmic_score\\\": float(score),\\n        \\\"reasoning\\\": reasoning,\\n        \\\"invalidation\\\": invalidation,\\n        \\\"status\\\": status,\\n    }\\n    record.update(features)\\n    return record\\n\\n\\ndef screen_indices(\\n    frame: pd.DataFrame,\\n    *,\\n    warmup: int = WARMUP_BARS,\\n    neutral_stride: int = NEUTRAL_STRIDE,\\n    lookback: int = BREAKOUT_LOOKBACK,\\n) -> list[int]:\\n    \\\"\\\"\\\"Choose bars worth a causal snapshot.\\n\\n    Args:\\n        frame: OHLCV series.\\n        warmup: First usable bar.\\n        neutral_stride: Keep a neutral sample on this stride.\\n        lookback: Window for a close through the prior high or low.\\n\\n    Returns:\\n        Increasing bar indexes.\\n    \\\"\\\"\\\"\\n    work = normalize_ohlcv(frame)\\n    high = work[\\\"high\\\"].to_numpy(dtype=float)\\n    low = work[\\\"low\\\"].to_numpy(dtype=float)\\n    close = work[\\\"close\\\"].to_numpy(dtype=float)\\n    opened = work[\\\"open\\\"].to_numpy(dtype=float)\\n    chosen: set[int] = set()\\n    start = max(warmup, lookback)\\n    for index in range(start, len(work)):\\n        prior_high = float(np.max(high[index - lookback : index]))\\n        prior_low = float(np.min(low[index - lookback : index]))\\n        span = float(high[index] - low[index])\\n        body = abs(float(close[index] - opened[index]))\\n        if close[index] > prior_high or close[index] < prior_low:\\n            chosen.add(index)\\n        if span > 0.0 and body / span <= 0.35:\\n            chosen.add(index)\\n        if neutral_stride > 0 and (index - start) % neutral_stride == 0:\\n            chosen.add(index)\\n    return sorted(chosen)\\n\\n\\ndef _status(score: float, direction: str) -> str:\\n    if direction == \\\"neutral\\\" or score <= 0.0:\\n        return \\\"neutral\\\"\\n    if score < REJECT_SCORE:\\n        return \\\"rejected\\\"\\n    if score < ACCEPT_SCORE:\\n        return \\\"rejected\\\"\\n    return \\\"accepted\\\"\\n\\n\\ndef _level_setups(\\n    work: pd.DataFrame,\\n    index: int,\\n    resolution: str,\\n    features: dict[str, float],\\n) -> list[dict[str, Any]]:\\n    lookback = BREAKOUT_LOOKBACK\\n    window = work.iloc[index - lookback : index]\\n    close = float(work[\\\"close\\\"].iloc[index])\\n    opened = float(work[\\\"open\\\"].iloc[index])\\n    volume = float(work[\\\"volume\\\"].iloc[index])\\n    average_volume = float(window[\\\"volume\\\"].mean()) if len(window) else volume\\n    prior_high = float(window[\\\"high\\\"].max())\\n    prior_low = float(window[\\\"low\\\"].min())\\n    atr = float(features.get(\\\"atr_14\\\") or 0.0)\\n    if atr <= 0.0:\\n        atr = max(abs(close) * 0.01, 1e-9)\\n    volume_boost = 0.15 if average_volume > 0.0 and volume >= average_volume else 0.0\\n    rows: list[dict[str, Any]] = []\\n    stamp = work[\\\"time\\\"].iloc[index]\\n    if close > prior_high:\\n        distance = (close - prior_high) / atr\\n        score = float(np.clip(0.45 + 0.25 * distance + volume_boost, 0.0, 1.0))\\n        rows.append(\\n            _row(\\n                resolution=resolution,\\n                index=index,\\n                timestamp=stamp,\\n                setup_type=\\\"long_breakout\\\",\\n                direction=\\\"long\\\",\\n                score=score,\\n                reasoning=\\\"close traded through the prior lookback high\\\",\\n                invalidation=\\\"close back below the broken high\\\",\\n                status=_status(score, \\\"long\\\"),\\n                features=features,\\n            )\\n        )\\n    if close < prior_low:\\n        distance = (prior_low - close) / atr\\n        score = float(np.clip(0.45 + 0.25 * distance + volume_boost, 0.0, 1.0))\\n        rows.append(\\n            _row(\\n                resolution=resolution,\\n                index=index,\\n                timestamp=stamp,\\n                setup_type=\\\"short_breakdown\\\",\\n                direction=\\\"short\\\",\\n                score=score,\\n                reasoning=\\\"close traded through the prior lookback low\\\",\\n                invalidation=\\\"close back above the broken low\\\",\\n                status=_status(score, \\\"short\\\"),\\n                features=features,\\n            )\\n        )\\n    near_low = abs(close - prior_low) <= 0.35 * atr and close >= opened\\n    near_high = abs(close - prior_high) <= 0.35 * atr and close <= opened\\n    if near_low and close <= prior_high:\\n        score = 0.6 if close > prior_low else 0.3\\n        rows.append(\\n            _row(\\n                resolution=resolution,\\n                index=index,\\n                timestamp=stamp,\\n                setup_type=\\\"support_bounce\\\",\\n                direction=\\\"long\\\",\\n                score=score,\\n                reasoning=\\\"close held the lookback low and finished above the open\\\",\\n                invalidation=\\\"close through the lookback low\\\",\\n                status=_status(score, \\\"long\\\"),\\n                features=features,\\n            )\\n        )\\n    if near_high and close >= prior_low:\\n        score = 0.6 if close < prior_high else 0.3\\n        rows.append(\\n            _row(\\n                resolution=resolution,\\n                index=index,\\n                timestamp=stamp,\\n                setup_type=\\\"resistance_rejection\\\",\\n                direction=\\\"short\\\",\\n                score=score,\\n                reasoning=\\\"close stalled at the lookback high and finished below the open\\\",\\n                invalidation=\\\"close through the lookback high\\\",\\n                status=_status(score, \\\"short\\\"),\\n                features=features,\\n            )\\n        )\\n    return rows\\n\\n\\ndef _pattern_setups(\\n    work: pd.DataFrame,\\n    index: int,\\n    resolution: str,\\n    features: dict[str, float],\\n) -> list[dict[str, Any]]:\\n    rows: list[dict[str, Any]] = []\\n    stamp = work[\\\"time\\\"].iloc[index]\\n    bull = float(features.get(\\\"cdl_bull_score\\\") or 0.0)\\n    bear = float(features.get(\\\"cdl_bear_score\\\") or 0.0)\\n    if bull > 0.0 or bear > 0.0:\\n        direction = \\\"long\\\" if bull >= bear else \\\"short\\\"\\n        score = float(np.clip(max(bull, bear), 0.0, 1.0))\\n        rows.append(\\n            _row(\\n                resolution=resolution,\\n                index=index,\\n                timestamp=stamp,\\n                setup_type=\\\"candlestick\\\",\\n                direction=direction,\\n                score=score,\\n                reasoning=\\\"candlestick engine score on the closed bar\\\",\\n                invalidation=\\\"pattern high or low broken on the next bar\\\",\\n                status=_status(score, direction),\\n                features=features,\\n            )\\n        )\\n    net = float(features.get(\\\"cdl_net_score\\\") or 0.0)\\n    if abs(net) >= 0.5:\\n        direction = \\\"long\\\" if net > 0.0 else \\\"short\\\"\\n        rows.append(\\n            _row(\\n                resolution=resolution,\\n                index=index,\\n                timestamp=stamp,\\n                setup_type=\\\"reversal\\\",\\n                direction=direction,\\n                score=float(np.clip(abs(net), 0.0, 1.0)),\\n                reasoning=\\\"net candlestick score points against the prior close\\\",\\n                invalidation=\\\"signal bar extreme broken\\\",\\n                status=_status(abs(net), direction),\\n                features=features,\\n            )\\n        )\\n    adx = float(features.get(\\\"adx_14\\\") or 0.0)\\n    ema_distance = float(features.get(\\\"price_vs_ema21_atr\\\") or 0.0)\\n    if adx >= 22.0 and abs(ema_distance) > 0.0:\\n        direction = \\\"long\\\" if ema_distance > 0.0 else \\\"short\\\"\\n        score = float(np.clip(0.4 + adx / 100.0, 0.0, 1.0))\\n        rows.append(\\n            _row(\\n                resolution=resolution,\\n                index=index,\\n                timestamp=stamp,\\n                setup_type=\\\"trend_continuation\\\",\\n                direction=direction,\\n                score=score,\\n                reasoning=\\\"ADX and EMA distance agree on the closed bar\\\",\\n                invalidation=\\\"ADX falls below the trend threshold\\\",\\n                status=_status(score, direction),\\n                features=features,\\n            )\\n        )\\n    chart_score = max(float(features.get(name) or 0.0) for name in (\\n        \\\"chp_bull_flag\\\",\\n        \\\"chp_bear_flag\\\",\\n        \\\"chp_double_top\\\",\\n        \\\"chp_double_bottom\\\",\\n        \\\"chp_hs_detected\\\",\\n        \\\"chp_ihs_detected\\\",\\n    ))\\n    if chart_score > 0.0:\\n        direction = \\\"short\\\" if float(features.get(\\\"chp_bear_flag\\\") or 0.0) >= chart_score else \\\"long\\\"\\n        if float(features.get(\\\"chp_double_top\\\") or 0.0) >= chart_score or float(\\n            features.get(\\\"chp_hs_detected\\\") or 0.0\\n        ) >= chart_score:\\n            direction = \\\"short\\\"\\n        rows.append(\\n            _row(\\n                resolution=resolution,\\n                index=index,\\n                timestamp=stamp,\\n                setup_type=\\\"chart_pattern\\\",\\n                direction=direction,\\n                score=float(np.clip(chart_score, 0.0, 1.0)),\\n                reasoning=\\\"chart-pattern engine flag on the closed prefix\\\",\\n                invalidation=\\\"pattern boundary broken the wrong way\\\",\\n                status=_status(chart_score, direction),\\n                features=features,\\n            )\\n        )\\n    return rows\\n\\n\\ndef _retests(\\n    work: pd.DataFrame,\\n    index: int,\\n    resolution: str,\\n    features: dict[str, float],\\n) -> list[dict[str, Any]]:\\n    if index < BREAKOUT_LOOKBACK + 3:\\n        return []\\n    close = work[\\\"close\\\"].to_numpy(dtype=float)\\n    high = work[\\\"high\\\"].to_numpy(dtype=float)\\n    low = work[\\\"low\\\"].to_numpy(dtype=float)\\n    atr = float(features.get(\\\"atr_14\\\") or 0.0)\\n    if atr <= 0.0:\\n        atr = max(abs(float(close[index])) * 0.01, 1e-9)\\n    rows: list[dict[str, Any]] = []\\n    for previous in range(index - 5, index):\\n        if previous < BREAKOUT_LOOKBACK:\\n            continue\\n        prior_high = float(np.max(high[previous - BREAKOUT_LOOKBACK : previous]))\\n        prior_low = float(np.min(low[previous - BREAKOUT_LOOKBACK : previous]))\\n        if close[previous] > prior_high and abs(close[index] - prior_high) <= 0.35 * atr:\\n            score = 0.62\\n            rows.append(\\n                _row(\\n                    resolution=resolution,\\n                    index=index,\\n                    timestamp=work[\\\"time\\\"].iloc[index],\\n                    setup_type=\\\"breakout_retest\\\",\\n                    direction=\\\"long\\\",\\n                    score=score,\\n                    reasoning=\\\"price returned to a high broken within five bars\\\",\\n                    invalidation=\\\"close back below the retested high\\\",\\n                    status=_status(score, \\\"long\\\"),\\n                    features=features,\\n                )\\n            )\\n            break\\n        if close[previous] < prior_low and abs(close[index] - prior_low) <= 0.35 * atr:\\n            score = 0.62\\n            rows.append(\\n                _row(\\n                    resolution=resolution,\\n                    index=index,\\n                    timestamp=work[\\\"time\\\"].iloc[index],\\n                    setup_type=\\\"breakout_retest\\\",\\n                    direction=\\\"short\\\",\\n                    score=score,\\n                    reasoning=\\\"price returned to a low broken within five bars\\\",\\n                    invalidation=\\\"close back above the retested low\\\",\\n                    status=_status(score, \\\"short\\\"),\\n                    features=features,\\n                )\\n            )\\n            break\\n    return rows\\n\\n\\ndef _cpr_setups(\\n    work: pd.DataFrame,\\n    index: int,\\n    resolution: str,\\n    features: dict[str, float],\\n) -> list[dict[str, Any]]:\\n    if resolution not in (\\\"15m\\\", \\\"1h\\\", \\\"4h\\\"):\\n        return []\\n    prefix = work.iloc[: index + 1].reset_index(drop=True)\\n    atr = causal_atr(\\n        prefix[\\\"high\\\"].to_numpy(dtype=float),\\n        prefix[\\\"low\\\"].to_numpy(dtype=float),\\n        prefix[\\\"close\\\"].to_numpy(dtype=float),\\n        14,\\n    )\\n    prefix = prefix.copy()\\n    prefix[\\\"atr\\\"] = atr\\n    swings = detect_swings(prefix, resolution, atr)\\n    proposals = generate_candidates(prefix, swings, resolution, len(prefix) - 1)\\n    rows: list[dict[str, Any]] = []\\n    for proposal in proposals:\\n        direction = {\\\"bullish\\\": \\\"long\\\", \\\"bearish\\\": \\\"short\\\"}.get(proposal.direction, \\\"neutral\\\")\\n        score = float(np.clip(proposal.rule_score, 0.0, 1.0))\\n        rows.append(\\n            _row(\\n                resolution=resolution,\\n                index=index,\\n                timestamp=work[\\\"time\\\"].iloc[index],\\n                setup_type=\\\"chart_pattern\\\",\\n                direction=direction,\\n                score=score,\\n                reasoning=f\\\"cpr proposal {proposal.pattern}\\\",\\n                invalidation=\\\"proposal boundary broken before confirmation\\\",\\n                status=_status(score, direction),\\n                features=features,\\n            )\\n        )\\n    return rows\\n\\n\\ndef generate_setups(\\n    frame: pd.DataFrame,\\n    resolution: str,\\n    *,\\n    warmup: int = WARMUP_BARS,\\n    neutral_stride: int = NEUTRAL_STRIDE,\\n    indexes: list[int] | None = None,\\n) -> pd.DataFrame:\\n    \\\"\\\"\\\"Generate setup rows for one OHLCV frame.\\n\\n    Args:\\n        frame: OHLCV candles.\\n        resolution: Bar size.\\n        warmup: First bar that may emit a setup.\\n        neutral_stride: Sampling stride for bars that are not price events.\\n        indexes: Optional explicit bar list. Skips the screen when provided.\\n\\n    Returns:\\n        One row per candidate. Feature columns follow ``snapshot_columns``.\\n        Failed screens are stored as ``rejected`` or ``neutral``.\\n    \\\"\\\"\\\"\\n    work = normalize_ohlcv(frame)\\n    chosen = indexes if indexes is not None else screen_indices(\\n        work,\\n        warmup=warmup,\\n        neutral_stride=neutral_stride,\\n    )\\n    records: list[dict[str, Any]] = []\\n    for index in chosen:\\n        if index < 0 or index >= len(work):\\n            continue\\n        features = snapshot_features(work, index, resolution)\\n        found = []\\n        found.extend(_level_setups(work, index, resolution, features))\\n        found.extend(_retests(work, index, resolution, features))\\n        found.extend(_pattern_setups(work, index, resolution, features))\\n        found.extend(_cpr_setups(work, index, resolution, features))\\n        if not found:\\n            records.append(\\n                _row(\\n                    resolution=resolution,\\n                    index=index,\\n                    timestamp=work[\\\"time\\\"].iloc[index],\\n                    setup_type=\\\"candlestick\\\",\\n                    direction=\\\"neutral\\\",\\n                    score=0.0,\\n                    reasoning=\\\"no directional rule fired on this bar\\\",\\n                    invalidation=\\\"not a trade candidate\\\",\\n                    status=\\\"neutral\\\",\\n                    features=features,\\n                )\\n            )\\n        else:\\n            records.extend(found)\\n    columns = list(SETUP_META_COLUMNS) + list(snapshot_columns())\\n    if not records:\\n        return pd.DataFrame(columns=columns)\\n    frame_out = pd.DataFrame.from_records(records)\\n    frame_out = frame_out.loc[:, columns]\\n    frame_out[\\\"setup_id\\\"] = [\\n        f\\\"{setup_id}:{position}\\\"\\n        for position, setup_id in enumerate(frame_out[\\\"setup_id\\\"].tolist())\\n    ]\\n    return frame_out\\n\\n\\ndef materialize_setups(\\n    root: Path | str = COLAB_DATA_ROOT,\\n    *,\\n    symbol: str = SYMBOL,\\n    resolutions: tuple[str, ...] = RESOLUTIONS,\\n    neutral_stride: int = NEUTRAL_STRIDE,\\n) -> dict[str, str]:\\n    \\\"\\\"\\\"Write setup parquet files next to stored OHLCV.\\n\\n    Args:\\n        root: Local dataset root.\\n        symbol: Instrument.\\n        resolutions: Resolutions to scan.\\n        neutral_stride: Neutral sampling stride.\\n\\n    Returns:\\n        Map of resolution to the setup parquet path.\\n    \\\"\\\"\\\"\\n    written: dict[str, str] = {}\\n    for resolution in resolutions:\\n        source = Path(root) / symbol / resolution / \\\"ohlcv.parquet\\\"\\n        if not source.is_file():\\n            continue\\n        setups = generate_setups(\\n            pd.read_parquet(source),\\n            resolution,\\n            neutral_stride=neutral_stride,\\n        )\\n        destination = source.with_name(\\\"setups.parquet\\\")\\n        setups.to_parquet(destination, index=False)\\n        written[resolution] = str(destination)\\n    return written\\n\", \"research/swinging/outcomes.py\": \"\\\"\\\"\\\"Next-bar trade simulation with costs, funding, and one open position.\\n\\nEntry is the next bar's open. The signal close is never an entry. Five-minute\\nbars decide whether a stop or a target printed first inside a higher-timeframe\\nbar. Without a finer bar, a dual touch exits at the stop.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport pandas as pd\\n\\nfrom research.swinging.contract import (\\n    BREAKEVEN_BAND,\\n    COLAB_DATA_ROOT,\\n    OUTCOME_COLUMNS,\\n    RESOLUTIONS,\\n    SLIPPAGE_BPS,\\n    STOP_ATR_MULT,\\n    SYMBOL,\\n    TAKER_FEE_RATE,\\n    TAKE_PROFIT_ATR_MULT,\\n    max_hold_bars,\\n)\\nfrom research.swinging.quality import normalize_ohlcv\\n\\n_SLIP = SLIPPAGE_BPS / 10_000.0\\n\\n\\ndef _lookup_index(frame: pd.DataFrame, timestamp: object, bar_index: int | None) -> int | None:\\n    if bar_index is not None and 0 <= int(bar_index) < len(frame):\\n        return int(bar_index)\\n    if \\\"time\\\" not in frame.columns or frame.empty or timestamp is None:\\n        return None\\n    stamp = pd.Timestamp(timestamp)\\n    if stamp.tzinfo is None:\\n        stamp = stamp.tz_localize(\\\"UTC\\\")\\n    times = pd.to_datetime(frame[\\\"time\\\"], utc=True)\\n    hits = list(times[times == stamp].index)\\n    if not hits:\\n        return None\\n    return int(hits[0])\\n\\n\\ndef _atr(setup: dict[str, Any], frame: pd.DataFrame, index: int) -> float:\\n    raw = setup.get(\\\"atr_14\\\")\\n    try:\\n        value = float(raw)\\n    except (TypeError, ValueError):\\n        value = 0.0\\n    if value == value and value > 0.0:\\n        return value\\n    close = float(frame[\\\"close\\\"].iloc[index])\\n    return max(abs(close) * 0.01, 1e-9)\\n\\n\\ndef _brackets(direction: str, entry: float, atr: float) -> tuple[float, float]:\\n    risk = STOP_ATR_MULT * atr\\n    reward = TAKE_PROFIT_ATR_MULT * atr\\n    if direction == \\\"long\\\":\\n        return entry - risk, entry + reward\\n    return entry + risk, entry - reward\\n\\n\\ndef _touch(direction: str, high: float, low: float, stop: float, take: float) -> tuple[str, float] | None:\\n    if direction == \\\"long\\\":\\n        hit_stop = low <= stop\\n        hit_take = high >= take\\n    else:\\n        hit_stop = high >= stop\\n        hit_take = low <= take\\n    if hit_stop and hit_take:\\n        return \\\"stop_first\\\", stop\\n    if hit_stop:\\n        return \\\"stop\\\", stop\\n    if hit_take:\\n        return \\\"take_profit\\\", take\\n    return None\\n\\n\\ndef _funding_pnl(\\n    direction: str,\\n    entry_time: pd.Timestamp,\\n    exit_time: pd.Timestamp,\\n    funding: pd.DataFrame | None,\\n) -> tuple[float, bool]:\\n    if funding is None or funding.empty:\\n        return 0.0, True\\n    frame = funding.copy()\\n    if \\\"time\\\" not in frame.columns:\\n        return 0.0, True\\n    frame[\\\"time\\\"] = pd.to_datetime(frame[\\\"time\\\"], utc=True)\\n    rate_column = \\\"funding_rate\\\" if \\\"funding_rate\\\" in frame.columns else \\\"close\\\"\\n    window = frame[(frame[\\\"time\\\"] > entry_time) & (frame[\\\"time\\\"] <= exit_time)]\\n    if window.empty:\\n        return 0.0, False\\n    rates = pd.to_numeric(window[rate_column], errors=\\\"coerce\\\").fillna(0.0)\\n    sign = 1.0 if direction == \\\"long\\\" else -1.0\\n    return float(-(sign * rates.sum())), False\\n\\n\\ndef _returns(\\n    direction: str,\\n    entry_open: float,\\n    exit_raw: float,\\n    funding_pnl: float,\\n) -> dict[str, float]:\\n    if direction == \\\"long\\\":\\n        entry_fill = entry_open * (1.0 + _SLIP)\\n        exit_fill = exit_raw * (1.0 - _SLIP)\\n        gross = (exit_raw - entry_open) / entry_open\\n        traded = (exit_fill - entry_fill) / entry_fill\\n    else:\\n        entry_fill = entry_open * (1.0 - _SLIP)\\n        exit_fill = exit_raw * (1.0 + _SLIP)\\n        gross = (entry_open - exit_raw) / entry_open\\n        traded = (entry_fill - exit_fill) / entry_fill\\n    fee = 2.0 * TAKER_FEE_RATE\\n    net = traded - fee - funding_pnl\\n    return {\\n        \\\"entry_price\\\": float(entry_fill),\\n        \\\"exit_price\\\": float(exit_fill),\\n        \\\"gross_return\\\": float(gross),\\n        \\\"fee_return\\\": float(fee),\\n        \\\"slippage_return\\\": float(gross - traded),\\n        \\\"net_return\\\": float(net),\\n    }\\n\\n\\ndef _label(reason: str, net_return: float) -> str:\\n    if reason == \\\"expiry\\\" and abs(net_return) <= BREAKEVEN_BAND:\\n        return \\\"expiry\\\"\\n    if abs(net_return) <= BREAKEVEN_BAND:\\n        return \\\"breakeven\\\"\\n    if net_return > 0.0:\\n        return \\\"win\\\"\\n    return \\\"loss\\\"\\n\\n\\ndef _blank(setup: dict[str, Any], reason: str) -> dict[str, Any]:\\n    payload = {column: None for column in OUTCOME_COLUMNS}\\n    payload[\\\"exit_reason\\\"] = reason\\n    payload[\\\"simulated\\\"] = False\\n    payload[\\\"missing_funding\\\"] = True\\n    payload[\\\"setup_id\\\"] = setup.get(\\\"setup_id\\\")\\n    return payload\\n\\n\\ndef _path_bars(\\n    signal_frame: pd.DataFrame,\\n    entry_index: int,\\n    hold: int,\\n    lower: pd.DataFrame | None,\\n) -> pd.DataFrame:\\n    entry_time = pd.Timestamp(signal_frame[\\\"time\\\"].iloc[entry_index])\\n    last_index = min(len(signal_frame) - 1, entry_index + hold - 1)\\n    if len(signal_frame) > 1:\\n        step = pd.Timestamp(signal_frame[\\\"time\\\"].iloc[1]) - pd.Timestamp(signal_frame[\\\"time\\\"].iloc[0])\\n    else:\\n        step = pd.Timedelta(minutes=5)\\n    end_time = pd.Timestamp(signal_frame[\\\"time\\\"].iloc[last_index]) + step - pd.Timedelta(seconds=1)\\n    if lower is None or lower.empty:\\n        return signal_frame.iloc[entry_index : last_index + 1].reset_index(drop=True)\\n    bars = normalize_ohlcv(lower)\\n    bars = bars[(bars[\\\"time\\\"] >= entry_time) & (bars[\\\"time\\\"] <= end_time)]\\n    if bars.empty:\\n        return signal_frame.iloc[entry_index : last_index + 1].reset_index(drop=True)\\n    return bars.reset_index(drop=True)\\n\\n\\ndef _excursions(direction: str, entry_open: float, path: pd.DataFrame) -> tuple[float, float]:\\n    if direction == \\\"long\\\":\\n        favorable = (path[\\\"high\\\"].astype(float) - entry_open) / entry_open\\n        adverse = (entry_open - path[\\\"low\\\"].astype(float)) / entry_open\\n    else:\\n        favorable = (entry_open - path[\\\"low\\\"].astype(float)) / entry_open\\n        adverse = (path[\\\"high\\\"].astype(float) - entry_open) / entry_open\\n    return float(favorable.max()), float(adverse.max())\\n\\n\\ndef simulate_setup(\\n    setup: dict[str, Any],\\n    bars: pd.DataFrame,\\n    *,\\n    lower_bars: pd.DataFrame | None = None,\\n    funding: pd.DataFrame | None = None,\\n) -> dict[str, Any] | None:\\n    \\\"\\\"\\\"Simulate one setup or drop it when the future is incomplete.\\n\\n    Args:\\n        setup: Candidate row. ``direction`` of ``neutral`` is not simulated.\\n        bars: Signal-timeframe OHLCV.\\n        lower_bars: Optional 5-minute bars used to order stop and target.\\n        funding: Optional funding candles. A missing series sets\\n            ``missing_funding``.\\n\\n    Returns:\\n        Outcome fields, or ``None`` when the hold cannot be resolved and\\n        neither stop nor target has printed.\\n    \\\"\\\"\\\"\\n    direction = str(setup.get(\\\"direction\\\") or \\\"neutral\\\")\\n    if direction not in (\\\"long\\\", \\\"short\\\"):\\n        return _blank(setup, \\\"not_directional\\\")\\n    frame = normalize_ohlcv(bars)\\n    raw_index = setup.get(\\\"bar_index\\\")\\n    bar_index = int(raw_index) if raw_index is not None and str(raw_index) != \\\"\\\" else None\\n    signal_index = _lookup_index(frame, setup.get(\\\"signal_timestamp\\\"), bar_index)\\n    if signal_index is None:\\n        return None\\n    entry_index = signal_index + 1\\n    hold = max_hold_bars(str(setup.get(\\\"timeframe\\\") or setup.get(\\\"resolution\\\") or \\\"5m\\\"))\\n    last_needed = entry_index + hold - 1\\n    if entry_index >= len(frame):\\n        return None\\n    entry_open = float(frame[\\\"open\\\"].iloc[entry_index])\\n    if entry_open <= 0.0:\\n        return None\\n    atr = _atr(setup, frame, signal_index)\\n    stop, take = _brackets(direction, entry_open, atr)\\n    path = _path_bars(frame, entry_index, hold, lower_bars)\\n    if path.empty:\\n        return None\\n    exit_reason = None\\n    exit_raw = None\\n    exit_time = None\\n    for _, bar in path.iterrows():\\n        touch = _touch(direction, float(bar[\\\"high\\\"]), float(bar[\\\"low\\\"]), stop, take)\\n        if touch is not None:\\n            exit_reason, exit_raw = touch\\n            exit_time = pd.Timestamp(bar[\\\"time\\\"])\\n            break\\n    resolved_early = exit_reason is not None\\n    if not resolved_early:\\n        if last_needed >= len(frame):\\n            return None\\n        exit_reason = \\\"expiry\\\"\\n        exit_raw = float(frame[\\\"close\\\"].iloc[last_needed])\\n        exit_time = pd.Timestamp(frame[\\\"time\\\"].iloc[last_needed])\\n    assert exit_raw is not None and exit_time is not None and exit_reason is not None\\n    entry_time = pd.Timestamp(frame[\\\"time\\\"].iloc[entry_index])\\n    funding_pnl, missing_funding = _funding_pnl(direction, entry_time, exit_time, funding)\\n    priced = _returns(direction, entry_open, float(exit_raw), funding_pnl)\\n    visible = path.loc[pd.to_datetime(path[\\\"time\\\"], utc=True) <= exit_time]\\n    if visible.empty:\\n        visible = path.iloc[:1]\\n    favorable, adverse = _excursions(direction, entry_open, visible)\\n    return {\\n        \\\"setup_id\\\": setup.get(\\\"setup_id\\\"),\\n        \\\"entry_timestamp\\\": entry_time.isoformat(),\\n        \\\"entry_price\\\": priced[\\\"entry_price\\\"],\\n        \\\"stop_price\\\": float(stop),\\n        \\\"take_profit_price\\\": float(take),\\n        \\\"exit_timestamp\\\": exit_time.isoformat(),\\n        \\\"exit_price\\\": priced[\\\"exit_price\\\"],\\n        \\\"exit_reason\\\": exit_reason,\\n        \\\"holding_seconds\\\": float((exit_time - entry_time).total_seconds()),\\n        \\\"mfe\\\": favorable,\\n        \\\"mae\\\": adverse,\\n        \\\"gross_return\\\": priced[\\\"gross_return\\\"],\\n        \\\"fee_return\\\": priced[\\\"fee_return\\\"],\\n        \\\"slippage_return\\\": priced[\\\"slippage_return\\\"],\\n        \\\"funding_pnl\\\": funding_pnl,\\n        \\\"missing_funding\\\": bool(missing_funding),\\n        \\\"net_return\\\": priced[\\\"net_return\\\"],\\n        \\\"outcome_label\\\": _label(exit_reason, priced[\\\"net_return\\\"]),\\n        \\\"simulated\\\": True,\\n    }\\n\\n\\ndef simulate_portfolio(\\n    setups: pd.DataFrame,\\n    bars: pd.DataFrame,\\n    *,\\n    lower_bars: pd.DataFrame | None = None,\\n    funding: pd.DataFrame | None = None,\\n) -> pd.DataFrame:\\n    \\\"\\\"\\\"Simulate setups in time order and refuse a second open position.\\n\\n    Args:\\n        setups: Candidate rows.\\n        bars: Signal-timeframe OHLCV.\\n        lower_bars: Optional finer bars.\\n        funding: Optional funding candles.\\n\\n    Returns:\\n        Setup columns plus outcome columns. Overlaps stay in the frame with\\n        ``simulated`` false so they are not labeled as flat trades.\\n    \\\"\\\"\\\"\\n    if setups.empty:\\n        return setups.copy()\\n    ordered = setups.copy()\\n    ordered[\\\"_signal_time\\\"] = pd.to_datetime(ordered[\\\"signal_timestamp\\\"], utc=True)\\n    ordered = ordered.sort_values([\\\"_signal_time\\\", \\\"setup_id\\\"]).reset_index(drop=True)\\n    busy_until: pd.Timestamp | None = None\\n    outcome_rows: list[dict[str, Any]] = []\\n    for record in ordered.to_dict(orient=\\\"records\\\"):\\n        signal_time = pd.Timestamp(record[\\\"signal_timestamp\\\"])\\n        if busy_until is not None and signal_time < busy_until:\\n            outcome_rows.append(_blank(record, \\\"overlap_rejected\\\"))\\n            continue\\n        outcome = simulate_setup(record, bars, lower_bars=lower_bars, funding=funding)\\n        if outcome is None:\\n            continue\\n        outcome_rows.append(outcome)\\n        if outcome.get(\\\"simulated\\\") and outcome.get(\\\"exit_timestamp\\\"):\\n            busy_until = pd.Timestamp(outcome[\\\"exit_timestamp\\\"])\\n    if not outcome_rows:\\n        return ordered.drop(columns=[\\\"_signal_time\\\"]).iloc[0:0]\\n    return _align_outcomes(ordered.drop(columns=[\\\"_signal_time\\\"]), outcome_rows)\\n\\n\\ndef _align_outcomes(setups: pd.DataFrame, outcome_rows: list[dict[str, Any]]) -> pd.DataFrame:\\n    \\\"\\\"\\\"Attach outcomes that were produced in the same order as ``setups``.\\n\\n    Unresolved setups are omitted from ``outcome_rows``, so this walks both\\n    sequences using setup id plus signal time and drops missing results.\\n\\n    Args:\\n        setups: Sorted setup rows.\\n        outcome_rows: One result per setup that was not an unresolved tail.\\n\\n    Returns:\\n        Joined frame.\\n    \\\"\\\"\\\"\\n    if len(outcome_rows) == len(setups):\\n        extra = pd.DataFrame(outcome_rows).drop(columns=[\\\"setup_id\\\"], errors=\\\"ignore\\\")\\n        return pd.concat([setups.reset_index(drop=True), extra], axis=1)\\n    # Fallback when unresolved rows were skipped: match by id and timestamp order.\\n    remaining = outcome_rows.copy()\\n    kept_setups = []\\n    kept_outcomes = []\\n    for record in setups.to_dict(orient=\\\"records\\\"):\\n        match = None\\n        for index, outcome in enumerate(remaining):\\n            if outcome.get(\\\"setup_id\\\") == record.get(\\\"setup_id\\\"):\\n                match = remaining.pop(index)\\n                break\\n        if match is None:\\n            continue\\n        kept_setups.append(record)\\n        kept_outcomes.append(match)\\n    if not kept_setups:\\n        return setups.iloc[0:0]\\n    extra = pd.DataFrame(kept_outcomes).drop(columns=[\\\"setup_id\\\"], errors=\\\"ignore\\\")\\n    return pd.concat([pd.DataFrame(kept_setups).reset_index(drop=True), extra], axis=1)\\n\\n\\ndef materialize_outcomes(\\n    root: Path | str = COLAB_DATA_ROOT,\\n    *,\\n    symbol: str = SYMBOL,\\n    resolutions: tuple[str, ...] = RESOLUTIONS,\\n) -> dict[str, str]:\\n    \\\"\\\"\\\"Simulate stored setups and write ``outcomes.parquet``.\\n\\n    Args:\\n        root: Local dataset root.\\n        symbol: Instrument.\\n        resolutions: Resolutions that already have setups and OHLCV.\\n\\n    Returns:\\n        Map of resolution to the outcome parquet path.\\n    \\\"\\\"\\\"\\n    written: dict[str, str] = {}\\n    five_minute_path = Path(root) / symbol / \\\"5m\\\" / \\\"ohlcv.parquet\\\"\\n    five_minute = pd.read_parquet(five_minute_path) if five_minute_path.is_file() else None\\n    for resolution in resolutions:\\n        folder = Path(root) / symbol / resolution\\n        setup_path = folder / \\\"setups.parquet\\\"\\n        bar_path = folder / \\\"ohlcv.parquet\\\"\\n        if not setup_path.is_file() or not bar_path.is_file():\\n            continue\\n        funding_path = folder / \\\"funding.parquet\\\"\\n        funding = pd.read_parquet(funding_path) if funding_path.is_file() else None\\n        lower = None if resolution == \\\"5m\\\" else five_minute\\n        simulated = simulate_portfolio(\\n            pd.read_parquet(setup_path),\\n            pd.read_parquet(bar_path),\\n            lower_bars=lower,\\n            funding=funding,\\n        )\\n        destination = folder / \\\"outcomes.parquet\\\"\\n        simulated.to_parquet(destination, index=False)\\n        written[resolution] = str(destination)\\n    return written\\n\", \"research/swinging/splits.py\": \"\\\"\\\"\\\"Chronological splits with an embargo around overlapping trade outcomes.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Iterator\\n\\n\\ndef chronological_bounds(\\n    row_count: int,\\n    *,\\n    train_fraction: float = 0.70,\\n    validation_fraction: float = 0.15,\\n) -> tuple[int, int, int]:\\n    \\\"\\\"\\\"Return train end, validation end, and the sealed start.\\n\\n    Args:\\n        row_count: Number of rows in time order.\\n        train_fraction: Share used to fit.\\n        validation_fraction: Share used to tune and calibrate. The remainder\\n            is the sealed test.\\n\\n    Returns:\\n        ``(train_end, validation_end, sealed_start)``. ``sealed_start`` equals\\n        ``validation_end``.\\n\\n    Raises:\\n        ValueError: If the fractions do not leave a sealed tail.\\n    \\\"\\\"\\\"\\n    if row_count < 3:\\n        raise ValueError(\\\"need at least 3 rows to split\\\")\\n    if train_fraction <= 0.0 or validation_fraction <= 0.0:\\n        raise ValueError(\\\"train and validation fractions must be positive\\\")\\n    if train_fraction + validation_fraction >= 1.0:\\n        raise ValueError(\\\"fractions must leave a sealed tail\\\")\\n    train_end = max(1, int(row_count * train_fraction))\\n    validation_end = max(train_end + 1, int(row_count * (train_fraction + validation_fraction)))\\n    validation_end = min(validation_end, row_count - 1)\\n    return train_end, validation_end, validation_end\\n\\n\\ndef apply_embargo(train_indexes: list[int], validation_start: int, embargo: int) -> list[int]:\\n    \\\"\\\"\\\"Drop training rows that sit inside the embargo before validation.\\n\\n    Args:\\n        train_indexes: Candidate training positions.\\n        validation_start: First validation row.\\n        embargo: Bars to drop before ``validation_start``. Use at least the\\n            longest holding period so a trade opened in train cannot close\\n            inside validation.\\n\\n    Returns:\\n        Training positions that end before the embargo.\\n    \\\"\\\"\\\"\\n    cutoff = validation_start - max(0, int(embargo))\\n    return [index for index in train_indexes if index < cutoff]\\n\\n\\ndef purged_folds(\\n    row_count: int,\\n    *,\\n    n_splits: int = 3,\\n    embargo: int = 24,\\n) -> Iterator[tuple[list[int], list[int]]]:\\n    \\\"\\\"\\\"Yield expanding walk-forward folds on a chronological frame.\\n\\n    Args:\\n        row_count: Number of rows.\\n        n_splits: Validation blocks carved from the development span. The last\\n            15 percent of rows is not yielded.\\n        embargo: Bars removed from the end of each training side.\\n\\n    Yields:\\n        ``(train_indexes, validation_indexes)``.\\n\\n    Raises:\\n        ValueError: If the frame is too short for the requested folds.\\n    \\\"\\\"\\\"\\n    _, _, sealed = chronological_bounds(row_count)\\n    development = sealed\\n    if development < n_splits * 2:\\n        raise ValueError(\\\"not enough development rows for the requested folds\\\")\\n    block = max(1, development // (n_splits + 1))\\n    for fold in range(n_splits):\\n        validation_start = block * (fold + 1)\\n        validation_end = min(development, validation_start + block)\\n        if validation_end <= validation_start:\\n            continue\\n        train = apply_embargo(list(range(validation_start)), validation_start, embargo)\\n        validation = list(range(validation_start, validation_end))\\n        if not train or not validation:\\n            continue\\n        yield train, validation\\n\", \"research/swinging/fusion.py\": \"\\\"\\\"\\\"Combine algorithmic evidence with calibrated model output.\\n\\nThis module does not import execution, risk, or order placement.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nfrom typing import Any, Mapping\\n\\nfrom research.swinging.contract import ACCEPT_SCORE, ROUND_TRIP_COST, STRATEGIES\\n\\n\\ndef decide(\\n    *,\\n    strategy: str,\\n    direction: str,\\n    algorithmic_score: float,\\n    win_probability: float,\\n    expected_net_return: float,\\n    probability_floor: float = 0.55,\\n    minimum_edge: float = 0.0,\\n) -> dict[str, Any]:\\n    \\\"\\\"\\\"Choose long, short, or abstain for one strategy.\\n\\n    Args:\\n        strategy: ``algorithm_only``, ``ml_only``, or ``hybrid``.\\n        direction: Setup direction. Neutral never trades.\\n        algorithmic_score: Engine A score in ``[0, 1]``.\\n        win_probability: Calibrated probability of a winning outcome.\\n        expected_net_return: Model expected net return after costs.\\n        probability_floor: Minimum win probability for an ML entry.\\n        minimum_edge: Minimum expected net return. Defaults to zero, which\\n            already sits above the cost embedded in the label.\\n\\n    Returns:\\n        Action, reason, agreement, and the expected net return used.\\n\\n    Raises:\\n        ValueError: If ``strategy`` is unknown.\\n    \\\"\\\"\\\"\\n    if strategy not in STRATEGIES:\\n        raise ValueError(f\\\"unknown strategy: {strategy}\\\")\\n    algo_trade = direction in (\\\"long\\\", \\\"short\\\") and float(algorithmic_score) >= ACCEPT_SCORE\\n    ml_trade = (\\n        direction in (\\\"long\\\", \\\"short\\\")\\n        and float(win_probability) >= probability_floor\\n        and float(expected_net_return) > minimum_edge\\n    )\\n    if strategy == \\\"algorithm_only\\\":\\n        action = direction if algo_trade else \\\"abstain\\\"\\n        reason = \\\"algorithm_threshold\\\" if algo_trade else \\\"algorithm_abstain\\\"\\n        agreement = \\\"algorithm_only\\\"\\n    elif strategy == \\\"ml_only\\\":\\n        action = direction if ml_trade else \\\"abstain\\\"\\n        reason = \\\"ml_edge\\\" if ml_trade else \\\"ml_abstain\\\"\\n        agreement = \\\"ml_only\\\"\\n    else:\\n        if algo_trade and ml_trade:\\n            action = direction\\n            reason = \\\"hybrid_agree\\\"\\n            agreement = \\\"agree\\\"\\n        elif algo_trade != ml_trade:\\n            action = \\\"abstain\\\"\\n            reason = \\\"hybrid_disagree\\\"\\n            agreement = \\\"disagree\\\"\\n        else:\\n            action = \\\"abstain\\\"\\n            reason = \\\"hybrid_abstain\\\"\\n            agreement = \\\"disagree\\\"\\n    return {\\n        \\\"strategy\\\": strategy,\\n        \\\"action\\\": action,\\n        \\\"reason\\\": reason,\\n        \\\"agreement\\\": agreement,\\n        \\\"expected_net_return\\\": float(expected_net_return),\\n        \\\"round_trip_cost\\\": ROUND_TRIP_COST,\\n    }\\n\\n\\ndef decide_frame(row: Mapping[str, Any], strategy: str, **kwargs: float) -> dict[str, Any]:\\n    \\\"\\\"\\\"Run ``decide`` on a mapping of setup and model fields.\\n\\n    Args:\\n        row: Must include direction, score, win probability, and expected net.\\n        strategy: One of ``STRATEGIES``.\\n        **kwargs: Forwarded floors.\\n\\n    Returns:\\n        Decision dictionary.\\n    \\\"\\\"\\\"\\n    return decide(\\n        strategy=strategy,\\n        direction=str(row.get(\\\"direction\\\") or \\\"neutral\\\"),\\n        algorithmic_score=float(row.get(\\\"algorithmic_score\\\") or 0.0),\\n        win_probability=float(row.get(\\\"win_probability\\\") or 0.0),\\n        expected_net_return=float(row.get(\\\"expected_net_return\\\") or 0.0),\\n        **kwargs,\\n    )\\n\", \"research/swinging/models.py\": \"\\\"\\\"\\\"XGBoost and LightGBM fits tuned only on the development span.\\n\\nThe sealed tail from ``chronological_bounds`` is never passed to Optuna or to\\n``fit``.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nfrom pathlib import Path\\nfrom typing import Any, Callable, Sequence\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom research.swinging.contract import COLAB_DATA_ROOT, RESOLUTIONS, SYMBOL\\nfrom research.swinging.features import snapshot_columns\\nfrom research.swinging.regime import attach_regime_targets\\nfrom research.swinging.splits import chronological_bounds\\n\\n_SEED = 7\\n\\n\\nclass LabelledClassifier:\\n    \\\"\\\"\\\"Fit a tree classifier on stable integer ids and expose the class names.\\n\\n    XGBoost's multiclass objective expects integer labels. The names stay on\\n    ``classes_`` so scoring code can read them.\\n    \\\"\\\"\\\"\\n\\n    def __init__(self, model: Any, classes: list[str]) -> None:\\n        self.model = model\\n        self.classes_ = np.asarray(list(classes))\\n\\n    def fit(self, features: np.ndarray, labels: np.ndarray) -> \\\"LabelledClassifier\\\":\\n        \\\"\\\"\\\"Fit on string or integer labels.\\n\\n        Args:\\n            features: Training matrix.\\n            labels: Class names. Every name must be in ``classes_``.\\n\\n        Returns:\\n            This wrapper.\\n        \\\"\\\"\\\"\\n        mapping = {str(name): index for index, name in enumerate(self.classes_)}\\n        encoded = np.asarray([mapping[str(label)] for label in labels], dtype=np.int32)\\n        self.model.fit(features, encoded)\\n        return self\\n\\n    def predict_proba(self, features: np.ndarray) -> np.ndarray:\\n        \\\"\\\"\\\"Return class probabilities in ``classes_`` order.\\n\\n        Args:\\n            features: Rows to score.\\n\\n        Returns:\\n            Probability matrix.\\n        \\\"\\\"\\\"\\n        return self.model.predict_proba(features)\\n\\n\\ndef _class_names(labels: np.ndarray) -> list[str]:\\n    return sorted({str(label) for label in labels})\\n\\n\\ndef _xy(\\n    frame: pd.DataFrame,\\n    feature_columns: Sequence[str],\\n    target: str,\\n) -> tuple[np.ndarray, np.ndarray]:\\n    matrix = frame.loc[:, list(feature_columns)].apply(pd.to_numeric, errors=\\\"coerce\\\")\\n    matrix = matrix.replace([np.inf, -np.inf], np.nan).fillna(0.0)\\n    labels = frame[target].to_numpy()\\n    return matrix.to_numpy(dtype=np.float64), labels\\n\\n\\ndef development_slice(frame: pd.DataFrame) -> tuple[pd.DataFrame, int]:\\n    \\\"\\\"\\\"Return rows before the sealed test and the sealed start index.\\n\\n    Args:\\n        frame: Chronological frame.\\n\\n    Returns:\\n        Development rows and the first sealed index.\\n    \\\"\\\"\\\"\\n    _, _, sealed = chronological_bounds(len(frame))\\n    return frame.iloc[:sealed].reset_index(drop=True), sealed\\n\\n\\ndef _estimator(algorithm: str, task: str, params: dict[str, Any]) -> Any:\\n    depth = int(params.get(\\\"max_depth\\\", 3))\\n    trees = int(params.get(\\\"n_estimators\\\", 40))\\n    rate = float(params.get(\\\"learning_rate\\\", 0.1))\\n    if algorithm == \\\"xgboost\\\":\\n        if task == \\\"classifier\\\":\\n            from xgboost import XGBClassifier\\n\\n            return XGBClassifier(\\n                max_depth=depth,\\n                n_estimators=trees,\\n                learning_rate=rate,\\n                objective=\\\"multi:softprob\\\",\\n                eval_metric=\\\"mlogloss\\\",\\n                verbosity=0,\\n                random_state=_SEED,\\n            )\\n        from xgboost import XGBRegressor\\n\\n        return XGBRegressor(\\n            max_depth=depth,\\n            n_estimators=trees,\\n            learning_rate=rate,\\n            objective=\\\"reg:squarederror\\\",\\n            verbosity=0,\\n            random_state=_SEED,\\n        )\\n    if algorithm != \\\"lightgbm\\\":\\n        raise ValueError(f\\\"unknown algorithm: {algorithm}\\\")\\n    return _NativeLightGBM(task, depth, trees, rate)\\n\\n\\nclass _NativeLightGBM:\\n    \\\"\\\"\\\"LightGBM booster trained without the sklearn wrapper.\\n\\n    The installed LightGBM sklearn classes pass ``force_all_finite`` into\\n    scikit-learn, which this environment no longer accepts. ``lightgbm.train``\\n    is the same path the forecast package uses.\\n    \\\"\\\"\\\"\\n\\n    def __init__(self, task: str, depth: int, trees: int, rate: float) -> None:\\n        self.task = task\\n        self.depth = depth\\n        self.trees = trees\\n        self.rate = rate\\n        self.booster: Any = None\\n        self.num_class = 1\\n\\n    def fit(self, features: np.ndarray, labels: np.ndarray) -> \\\"_NativeLightGBM\\\":\\n        \\\"\\\"\\\"Train one booster.\\n\\n        Args:\\n            features: Training matrix.\\n            labels: Integer class ids or numeric targets.\\n\\n        Returns:\\n            This booster.\\n        \\\"\\\"\\\"\\n        import lightgbm as lgb\\n\\n        settings: dict[str, Any] = {\\n            \\\"learning_rate\\\": self.rate,\\n            \\\"max_depth\\\": self.depth,\\n            \\\"verbosity\\\": -1,\\n            \\\"seed\\\": _SEED,\\n            \\\"num_threads\\\": 1,\\n        }\\n        if self.task == \\\"classifier\\\":\\n            encoded = np.asarray(labels, dtype=np.int32)\\n            self.num_class = int(encoded.max()) + 1\\n            settings[\\\"objective\\\"] = \\\"multiclass\\\"\\n            settings[\\\"num_class\\\"] = self.num_class\\n            settings[\\\"metric\\\"] = \\\"multi_logloss\\\"\\n            dataset = lgb.Dataset(np.asarray(features, dtype=np.float64), label=encoded)\\n        else:\\n            settings[\\\"objective\\\"] = \\\"regression\\\"\\n            dataset = lgb.Dataset(\\n                np.asarray(features, dtype=np.float64),\\n                label=np.asarray(labels, dtype=np.float64),\\n            )\\n        self.booster = lgb.train(settings, dataset, num_boost_round=self.trees)\\n        return self\\n\\n    def predict(self, features: np.ndarray) -> np.ndarray:\\n        \\\"\\\"\\\"Return numeric predictions.\\n\\n        Args:\\n            features: Rows to score.\\n\\n        Returns:\\n            One prediction per row.\\n        \\\"\\\"\\\"\\n        return np.asarray(self.booster.predict(features), dtype=np.float64).reshape(-1)\\n\\n    def predict_proba(self, features: np.ndarray) -> np.ndarray:\\n        \\\"\\\"\\\"Return class probabilities in class-id order.\\n\\n        Args:\\n            features: Rows to score.\\n\\n        Returns:\\n            Probability matrix with one column per class.\\n        \\\"\\\"\\\"\\n        predicted = np.asarray(self.booster.predict(features), dtype=np.float64)\\n        if predicted.ndim == 1:\\n            predicted = predicted.reshape(-1, self.num_class)\\n        return predicted\\n\\n\\ndef _validation_loss(model: Any, features: np.ndarray, target: np.ndarray, task: str) -> float:\\n    if len(features) == 0:\\n        return 1.0\\n    if task == \\\"classifier\\\":\\n        from sklearn.metrics import log_loss\\n\\n        probabilities = model.predict_proba(features)\\n        labels = [str(label) for label in model.classes_]\\n        return float(log_loss(target, probabilities, labels=labels))\\n    predictions = model.predict(features)\\n    return float(np.mean((predictions - target.astype(float)) ** 2))\\n\\n\\ndef tune_hyperparameters(\\n    frame: pd.DataFrame,\\n    feature_columns: Sequence[str],\\n    target: str,\\n    *,\\n    algorithm: str,\\n    task: str,\\n    n_trials: int = 10,\\n    storage: str | None = None,\\n    study_name: str = \\\"swinging\\\",\\n) -> dict[str, Any]:\\n    \\\"\\\"\\\"Search hyperparameters on the development train and validation blocks.\\n\\n    Args:\\n        frame: Chronological rows. The sealed tail is removed before search.\\n        feature_columns: Model inputs.\\n        target: Column to predict.\\n        algorithm: ``xgboost`` or ``lightgbm``.\\n        task: ``classifier`` or ``regressor``.\\n        n_trials: Optuna trials.\\n        storage: Optional Optuna storage URL. ``None`` keeps the study in memory.\\n        study_name: Study name when ``storage`` is set.\\n\\n    Returns:\\n        Best parameter dictionary and the sealed start index that was excluded.\\n\\n    Raises:\\n        ValueError: If the development span cannot be split.\\n    \\\"\\\"\\\"\\n    import optuna\\n\\n    optuna.logging.set_verbosity(optuna.logging.WARNING)\\n    development, sealed = development_slice(frame)\\n    train_end, validation_end, _ = chronological_bounds(len(development))\\n    features, labels = _xy(development, feature_columns, target)\\n    if task == \\\"regressor\\\":\\n        labels = labels.astype(np.float64)\\n\\n    def objective(trial: \\\"optuna.Trial\\\") -> float:\\n        params = {\\n            \\\"max_depth\\\": trial.suggest_int(\\\"max_depth\\\", 2, 4),\\n            \\\"learning_rate\\\": trial.suggest_float(\\\"learning_rate\\\", 0.05, 0.3, log=True),\\n            \\\"n_estimators\\\": trial.suggest_int(\\\"n_estimators\\\", 8, 30),\\n        }\\n        if task == \\\"classifier\\\":\\n            model = LabelledClassifier(_estimator(algorithm, task, params), _class_names(labels))\\n        else:\\n            model = _estimator(algorithm, task, params)\\n        model.fit(features[:train_end], labels[:train_end])\\n        return _validation_loss(\\n            model,\\n            features[train_end:validation_end],\\n            labels[train_end:validation_end],\\n            task,\\n        )\\n\\n    study = optuna.create_study(\\n        direction=\\\"minimize\\\",\\n        study_name=study_name,\\n        storage=storage,\\n        load_if_exists=storage is not None,\\n    )\\n    study.optimize(objective, n_trials=int(n_trials))\\n    best = dict(study.best_params)\\n    best[\\\"sealed_start\\\"] = int(sealed)\\n    best[\\\"development_rows\\\"] = int(len(development))\\n    return best\\n\\n\\ndef fit_estimator(\\n    frame: pd.DataFrame,\\n    feature_columns: Sequence[str],\\n    target: str,\\n    *,\\n    algorithm: str,\\n    task: str,\\n    params: dict[str, Any] | None = None,\\n) -> Any:\\n    \\\"\\\"\\\"Fit on the development span and ignore the sealed tail.\\n\\n    Args:\\n        frame: Chronological rows.\\n        feature_columns: Model inputs.\\n        target: Column to predict.\\n        algorithm: ``xgboost`` or ``lightgbm``.\\n        task: ``classifier`` or ``regressor``.\\n        params: Optional fixed parameters. Optuna keys besides the three\\n            tree parameters are ignored.\\n\\n    Returns:\\n        A fitted estimator.\\n    \\\"\\\"\\\"\\n    development, _ = development_slice(frame)\\n    features, labels = _xy(development, feature_columns, target)\\n    if task == \\\"regressor\\\":\\n        labels = labels.astype(np.float64)\\n    if task == \\\"classifier\\\":\\n        model = LabelledClassifier(_estimator(algorithm, task, params or {}), _class_names(labels))\\n    else:\\n        model = _estimator(algorithm, task, params or {})\\n    model.fit(features, labels)\\n    return model\\n\\n\\ndef _calibrate_win(\\n    classifier: Any,\\n    features: np.ndarray,\\n    labels: np.ndarray,\\n    train_end: int,\\n    validation_end: int,\\n) -> Any | None:\\n    from sklearn.isotonic import IsotonicRegression\\n\\n    if \\\"win\\\" not in [str(label) for label in classifier.classes_]:\\n        return None\\n    validation_features = features[train_end:validation_end]\\n    validation_labels = labels[train_end:validation_end]\\n    if len(validation_features) < 2:\\n        return None\\n    probabilities = classifier.predict_proba(validation_features)\\n    classes = [str(label) for label in classifier.classes_]\\n    win_probability = probabilities[:, classes.index(\\\"win\\\")]\\n    observed = (validation_labels.astype(str) == \\\"win\\\").astype(float)\\n    if len(np.unique(observed)) < 2:\\n        return None\\n    calibrator = IsotonicRegression(out_of_bounds=\\\"clip\\\")\\n    calibrator.fit(win_probability, observed)\\n    return calibrator\\n\\n\\ndef fit_bundle(\\n    frame: pd.DataFrame,\\n    feature_columns: Sequence[str],\\n    *,\\n    target: str,\\n    positive_class: str | None,\\n    n_trials: int = 0,\\n    storage: str | None = None,\\n    study_name: str = \\\"swinging\\\",\\n) -> dict[str, Any]:\\n    \\\"\\\"\\\"Fit the classifier and, when the target is numeric, skip it.\\n\\n    The exported classifier and regressor are the algorithms with the lower\\n    validation loss. Search runs only when ``n_trials`` is positive.\\n\\n    Args:\\n        frame: Chronological supervised rows.\\n        feature_columns: Model inputs.\\n        target: Classification target column.\\n        positive_class: Class name treated as the event, or ``None``.\\n        n_trials: Optuna trials per algorithm. Zero uses fixed small trees.\\n        storage: Optional Optuna storage URL.\\n        study_name: Study prefix.\\n\\n    Returns:\\n        Artifact dictionary with ``classifier``, optional ``regressor``, and\\n        ``calibrator``.\\n    \\\"\\\"\\\"\\n    params: dict[str, dict[str, Any]] = {}\\n    for algorithm in (\\\"xgboost\\\", \\\"lightgbm\\\"):\\n        if n_trials > 0:\\n            params[algorithm] = tune_hyperparameters(\\n                frame,\\n                feature_columns,\\n                target,\\n                algorithm=algorithm,\\n                task=\\\"classifier\\\",\\n                n_trials=n_trials,\\n                storage=storage,\\n                study_name=f\\\"{study_name}-{algorithm}-clf\\\",\\n            )\\n        else:\\n            params[algorithm] = {\\\"max_depth\\\": 3, \\\"n_estimators\\\": 20, \\\"learning_rate\\\": 0.1}\\n    development, sealed = development_slice(frame)\\n    features, labels = _xy(development, feature_columns, target)\\n    train_end, validation_end, _ = chronological_bounds(len(development))\\n    fitted = {\\n        name: LabelledClassifier(_estimator(name, \\\"classifier\\\", params[name]), _class_names(labels))\\n        for name in (\\\"xgboost\\\", \\\"lightgbm\\\")\\n    }\\n    losses = {}\\n    for name, model in fitted.items():\\n        model.fit(features[:train_end], labels[:train_end])\\n        losses[name] = _validation_loss(\\n            model,\\n            features[train_end:validation_end],\\n            labels[train_end:validation_end],\\n            \\\"classifier\\\",\\n        )\\n    best_name = min(losses, key=losses.get)\\n    calibrator = _calibrate_win(fitted[best_name], features, labels, train_end, validation_end)\\n    classifier = LabelledClassifier(\\n        _estimator(best_name, \\\"classifier\\\", params[best_name]),\\n        _class_names(labels),\\n    )\\n    classifier.fit(features, labels)\\n    artifact: dict[str, Any] = {\\n        \\\"classifier\\\": classifier,\\n        \\\"classifier_algorithm\\\": best_name,\\n        \\\"classifier_validation_loss\\\": losses[best_name],\\n        \\\"feature_columns\\\": list(feature_columns),\\n        \\\"target\\\": target,\\n        \\\"positive_class\\\": positive_class,\\n        \\\"sealed_start\\\": int(sealed),\\n        \\\"development_rows\\\": int(len(development)),\\n        \\\"calibrator\\\": calibrator,\\n        \\\"regressor\\\": None,\\n        \\\"regressor_algorithm\\\": None,\\n    }\\n    return artifact\\n\\n\\ndef fit_regressor_bundle(\\n    artifact: dict[str, Any],\\n    frame: pd.DataFrame,\\n    feature_columns: Sequence[str],\\n    target: str,\\n    *,\\n    n_trials: int = 0,\\n    storage: str | None = None,\\n    study_name: str = \\\"swinging-reg\\\",\\n) -> dict[str, Any]:\\n    \\\"\\\"\\\"Attach the better of an XGBoost and a LightGBM regressor.\\n\\n    Args:\\n        artifact: Classifier artifact to update.\\n        frame: Chronological rows containing ``target``.\\n        feature_columns: Model inputs.\\n        target: Numeric column.\\n        n_trials: Optuna trials per algorithm.\\n        storage: Optional Optuna storage URL.\\n        study_name: Study prefix.\\n\\n    Returns:\\n        The same artifact with ``regressor`` set.\\n    \\\"\\\"\\\"\\n    params: dict[str, dict[str, Any]] = {}\\n    for algorithm in (\\\"xgboost\\\", \\\"lightgbm\\\"):\\n        if n_trials > 0:\\n            params[algorithm] = tune_hyperparameters(\\n                frame,\\n                feature_columns,\\n                target,\\n                algorithm=algorithm,\\n                task=\\\"regressor\\\",\\n                n_trials=n_trials,\\n                storage=storage,\\n                study_name=f\\\"{study_name}-{algorithm}\\\",\\n            )\\n        else:\\n            params[algorithm] = {\\\"max_depth\\\": 3, \\\"n_estimators\\\": 20, \\\"learning_rate\\\": 0.1}\\n    development, _ = development_slice(frame)\\n    features, labels = _xy(development, feature_columns, target)\\n    labels = labels.astype(np.float64)\\n    train_end, validation_end, _ = chronological_bounds(len(development))\\n    losses = {}\\n    for name in (\\\"xgboost\\\", \\\"lightgbm\\\"):\\n        model = _estimator(name, \\\"regressor\\\", params[name])\\n        model.fit(features[:train_end], labels[:train_end])\\n        losses[name] = _validation_loss(\\n            model,\\n            features[train_end:validation_end],\\n            labels[train_end:validation_end],\\n            \\\"regressor\\\",\\n        )\\n    best_name = min(losses, key=losses.get)\\n    regressor = _estimator(best_name, \\\"regressor\\\", params[best_name])\\n    regressor.fit(features, labels)\\n    artifact[\\\"regressor\\\"] = regressor\\n    artifact[\\\"regressor_algorithm\\\"] = best_name\\n    artifact[\\\"regressor_validation_loss\\\"] = losses[best_name]\\n    artifact[\\\"regressor_target\\\"] = target\\n    return artifact\\n\\n\\ndef score_features(artifact: dict[str, Any], values: np.ndarray) -> dict[str, Any]:\\n    \\\"\\\"\\\"Score one or more feature rows with a fitted artifact.\\n\\n    Args:\\n        artifact: Dictionary produced by ``fit_bundle``.\\n        values: Array shaped ``(rows, features)``.\\n\\n    Returns:\\n        Win or positive-class probability, class probabilities, and expected\\n        net return when a regressor is present.\\n    \\\"\\\"\\\"\\n    matrix = np.asarray(values, dtype=np.float64)\\n    if matrix.ndim == 1:\\n        matrix = matrix.reshape(1, -1)\\n    classifier = artifact[\\\"classifier\\\"]\\n    probabilities = classifier.predict_proba(matrix)\\n    classes = [str(label) for label in classifier.classes_]\\n    positive = artifact.get(\\\"positive_class\\\")\\n    if positive in classes:\\n        event = probabilities[:, classes.index(positive)]\\n    else:\\n        event = probabilities.max(axis=1)\\n    calibrator = artifact.get(\\\"calibrator\\\")\\n    if calibrator is not None and positive == \\\"win\\\":\\n        event = np.asarray(calibrator.predict(event), dtype=float)\\n    expected = np.zeros(len(matrix), dtype=float)\\n    if artifact.get(\\\"regressor\\\") is not None:\\n        expected = np.asarray(artifact[\\\"regressor\\\"].predict(matrix), dtype=float)\\n    return {\\n        \\\"event_probability\\\": event,\\n        \\\"class_probabilities\\\": [\\n            {name: float(probability) for name, probability in zip(classes, row)}\\n            for row in probabilities\\n        ],\\n        \\\"expected_net_return\\\": expected,\\n    }\\n\\n\\ndef class_from_outcome(label: object) -> str:\\n    \\\"\\\"\\\"Map a trade outcome onto win, loss, or other.\\n\\n    Args:\\n        label: Simulator label.\\n\\n    Returns:\\n        Classifier class name.\\n    \\\"\\\"\\\"\\n    text = str(label)\\n    if text == \\\"win\\\":\\n        return \\\"win\\\"\\n    if text == \\\"loss\\\":\\n        return \\\"loss\\\"\\n    return \\\"other\\\"\\n\\n\\ndef sr_class(setup_type: object, outcome: object) -> str | None:\\n    \\\"\\\"\\\"Map a support or breakout setup onto an effectiveness class.\\n\\n    Args:\\n        setup_type: Engine A setup type.\\n        outcome: Simulator label.\\n\\n    Returns:\\n        An S/R class, or ``None`` when the row is not an S/R example.\\n    \\\"\\\"\\\"\\n    kind = str(setup_type)\\n    result = str(outcome)\\n    if kind == \\\"support_bounce\\\" and result == \\\"win\\\":\\n        return \\\"bounce\\\"\\n    if kind == \\\"resistance_rejection\\\" and result == \\\"win\\\":\\n        return \\\"rejection\\\"\\n    if kind in (\\\"long_breakout\\\", \\\"short_breakdown\\\", \\\"breakout_retest\\\") and result == \\\"win\\\":\\n        return \\\"breakout\\\"\\n    if kind in (\\n        \\\"support_bounce\\\",\\n        \\\"resistance_rejection\\\",\\n        \\\"long_breakout\\\",\\n        \\\"short_breakdown\\\",\\n        \\\"breakout_retest\\\",\\n    ) and result == \\\"loss\\\":\\n        return \\\"failed_breakout\\\"\\n    return None\\n\\n\\ndef _load_outcomes(root: Path, symbol: str, resolutions: Sequence[str]) -> pd.DataFrame:\\n    frames = []\\n    for resolution in resolutions:\\n        path = root / symbol / resolution / \\\"outcomes.parquet\\\"\\n        if path.is_file():\\n            frame = pd.read_parquet(path)\\n            frame[\\\"timeframe\\\"] = resolution\\n            frames.append(frame)\\n    if not frames:\\n        raise FileNotFoundError(f\\\"no outcomes parquet under {root}\\\")\\n    combined = pd.concat(frames, ignore_index=True)\\n    combined = combined.sort_values(\\\"signal_timestamp\\\").reset_index(drop=True)\\n    return combined\\n\\n\\ndef _feature_columns(frame: pd.DataFrame) -> list[str]:\\n    return [name for name in snapshot_columns() if name in frame.columns]\\n\\n\\ndef _supervised(frame: pd.DataFrame) -> pd.DataFrame:\\n    if \\\"simulated\\\" in frame.columns:\\n        frame = frame.loc[frame[\\\"simulated\\\"].fillna(False).astype(bool)]\\n    frame = frame.loc[frame[\\\"outcome_label\\\"].notna()].copy()\\n    frame[\\\"outcome_class\\\"] = frame[\\\"outcome_label\\\"].map(class_from_outcome)\\n    return frame.reset_index(drop=True)\\n\\n\\ndef train_setup_effectiveness(\\n    root: Path | str = COLAB_DATA_ROOT,\\n    *,\\n    symbol: str = SYMBOL,\\n    resolutions: tuple[str, ...] = RESOLUTIONS,\\n    n_trials: int = 8,\\n    storage: str | None = None,\\n) -> dict[str, Any]:\\n    \\\"\\\"\\\"Train setup win/loss/other and net-return models.\\n\\n    Args:\\n        root: Local dataset root.\\n        symbol: Instrument.\\n        resolutions: Outcome files to stack, in time order per file.\\n        n_trials: Optuna trials. Zero skips search.\\n        storage: Optional Optuna storage URL under the Colab disk.\\n\\n    Returns:\\n        Fitted artifact. Also written to ``experiments/setup_effectiveness``.\\n    \\\"\\\"\\\"\\n    frame = _supervised(_load_outcomes(Path(root), symbol, resolutions))\\n    columns = _feature_columns(frame)\\n    artifact = fit_bundle(\\n        frame,\\n        columns,\\n        target=\\\"outcome_class\\\",\\n        positive_class=\\\"win\\\",\\n        n_trials=n_trials,\\n        storage=storage,\\n        study_name=\\\"setup-effectiveness\\\",\\n    )\\n    artifact[\\\"task\\\"] = \\\"setup_effectiveness\\\"\\n    for extra in (\\\"net_return\\\", \\\"mfe\\\", \\\"mae\\\"):\\n        if extra in frame.columns and extra == \\\"net_return\\\":\\n            fit_regressor_bundle(\\n                artifact,\\n                frame,\\n                columns,\\n                extra,\\n                n_trials=n_trials,\\n                storage=storage,\\n                study_name=\\\"setup-return\\\",\\n            )\\n    _write_experiment(Path(root), \\\"setup_effectiveness\\\", artifact)\\n    return artifact\\n\\n\\ndef train_sr_effectiveness(\\n    root: Path | str = COLAB_DATA_ROOT,\\n    *,\\n    symbol: str = SYMBOL,\\n    resolutions: tuple[str, ...] = RESOLUTIONS,\\n    n_trials: int = 8,\\n    storage: str | None = None,\\n) -> dict[str, Any]:\\n    \\\"\\\"\\\"Train support, resistance, and breakout effectiveness models.\\n\\n    Args:\\n        root: Local dataset root.\\n        symbol: Instrument.\\n        resolutions: Outcome files to stack.\\n        n_trials: Optuna trials.\\n        storage: Optional Optuna storage URL.\\n\\n    Returns:\\n        Fitted artifact.\\n    \\\"\\\"\\\"\\n    frame = _supervised(_load_outcomes(Path(root), symbol, resolutions))\\n    frame[\\\"sr_class\\\"] = [\\n        sr_class(kind, outcome)\\n        for kind, outcome in zip(frame[\\\"setup_type\\\"], frame[\\\"outcome_label\\\"])\\n    ]\\n    frame = frame.loc[frame[\\\"sr_class\\\"].notna()].reset_index(drop=True)\\n    if frame.empty:\\n        raise ValueError(\\\"no support or breakout rows to train\\\")\\n    columns = _feature_columns(frame)\\n    artifact = fit_bundle(\\n        frame,\\n        columns,\\n        target=\\\"sr_class\\\",\\n        positive_class=None,\\n        n_trials=n_trials,\\n        storage=storage,\\n        study_name=\\\"sr-effectiveness\\\",\\n    )\\n    artifact[\\\"task\\\"] = \\\"sr_effectiveness\\\"\\n    if \\\"net_return\\\" in frame.columns:\\n        fit_regressor_bundle(\\n            artifact,\\n            frame,\\n            columns,\\n            \\\"net_return\\\",\\n            n_trials=n_trials,\\n            storage=storage,\\n            study_name=\\\"sr-return\\\",\\n        )\\n    _write_experiment(Path(root), \\\"sr_effectiveness\\\", artifact)\\n    return artifact\\n\\n\\ndef train_regime_model(\\n    root: Path | str = COLAB_DATA_ROOT,\\n    *,\\n    symbol: str = SYMBOL,\\n    resolutions: tuple[str, ...] = (\\\"1h\\\",),\\n    n_trials: int = 8,\\n    storage: str | None = None,\\n) -> dict[str, Any]:\\n    \\\"\\\"\\\"Train a regime classifier. The label is not one of the inputs.\\n\\n    Args:\\n        root: Local dataset root.\\n        symbol: Instrument.\\n        resolutions: Feature parquets to stack. Defaults to 1-hour bars.\\n        n_trials: Optuna trials.\\n        storage: Optional Optuna storage URL.\\n\\n    Returns:\\n        Fitted artifact.\\n    \\\"\\\"\\\"\\n    frames = []\\n    for resolution in resolutions:\\n        path = Path(root) / symbol / resolution / \\\"features.parquet\\\"\\n        if path.is_file():\\n            frames.append(pd.read_parquet(path))\\n    if not frames:\\n        raise FileNotFoundError(f\\\"no features parquet under {root}\\\")\\n    featured = attach_regime_targets(pd.concat(frames, ignore_index=True))\\n    columns = [name for name in _feature_columns(featured) if name != \\\"regime_target\\\"]\\n    artifact = fit_bundle(\\n        featured,\\n        columns,\\n        target=\\\"regime_target\\\",\\n        positive_class=None,\\n        n_trials=n_trials,\\n        storage=storage,\\n        study_name=\\\"regime\\\",\\n    )\\n    artifact[\\\"task\\\"] = \\\"regime\\\"\\n    _write_experiment(Path(root), \\\"regime\\\", artifact)\\n    return artifact\\n\\n\\ndef _write_experiment(root: Path, name: str, artifact: dict[str, Any]) -> Path:\\n    import joblib\\n\\n    destination = root / \\\"experiments\\\" / name\\n    destination.mkdir(parents=True, exist_ok=True)\\n    joblib.dump(artifact, destination / \\\"models.joblib\\\")\\n    return destination\\n\", \"research/swinging/evaluate.py\": \"\\\"\\\"\\\"Walk-forward comparison of algorithm-only, ML-only, and hybrid decisions.\\n\\nAll three strategies read the same net returns, costs, and fills. This module\\ndoes not place orders.\\n\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport json\\nfrom pathlib import Path\\nfrom typing import Any, Callable, Sequence\\n\\nimport numpy as np\\nimport pandas as pd\\n\\nfrom research.swinging.contract import COLAB_DATA_ROOT, STRATEGIES, max_hold_bars\\nfrom research.swinging.fusion import decide_frame\\nfrom research.swinging.splits import chronological_bounds, purged_folds\\n\\nPredictFn = Callable[[pd.DataFrame, pd.DataFrame], pd.DataFrame]\\n\\n\\ndef trade_metrics(returns: Sequence[float], regimes: Sequence[str] | None = None) -> dict[str, Any]:\\n    \\\"\\\"\\\"Summarize a list of net trade returns.\\n\\n    Args:\\n        returns: Net returns of trades the strategy actually took.\\n        regimes: Optional regime label per trade.\\n\\n    Returns:\\n        Expectancy, profit factor, drawdown, frequency, and per-trade Sharpe.\\n        An empty list scores as no trades rather than as a profit.\\n    \\\"\\\"\\\"\\n    values = np.asarray(list(returns), dtype=float)\\n    if values.size == 0:\\n        return {\\n            \\\"trades\\\": 0,\\n            \\\"net_expectancy\\\": 0.0,\\n            \\\"profit_factor\\\": 0.0,\\n            \\\"max_drawdown\\\": 0.0,\\n            \\\"per_trade_sharpe\\\": 0.0,\\n            \\\"by_regime\\\": {},\\n        }\\n    wins = values[values > 0.0]\\n    losses = values[values < 0.0]\\n    if losses.size == 0:\\n        profit_factor = None if wins.size else 0.0\\n    else:\\n        profit_factor = float(wins.sum() / abs(losses.sum())) if wins.size else 0.0\\n    equity = np.cumsum(values)\\n    peak = np.maximum.accumulate(equity)\\n    drawdown = float((peak - equity).max()) if equity.size else 0.0\\n    deviation = float(values.std(ddof=0))\\n    sharpe = 0.0 if deviation == 0.0 else float(values.mean() / deviation * np.sqrt(values.size))\\n    by_regime: dict[str, float] = {}\\n    if regimes is not None:\\n        grouped: dict[str, list[float]] = {}\\n        for label, value in zip(regimes, values):\\n            grouped.setdefault(str(label), []).append(float(value))\\n        by_regime = {\\n            label: float(np.mean(samples)) for label, samples in grouped.items()\\n        }\\n    return {\\n        \\\"trades\\\": int(values.size),\\n        \\\"net_expectancy\\\": float(values.mean()),\\n        \\\"profit_factor\\\": profit_factor,\\n        \\\"max_drawdown\\\": drawdown,\\n        \\\"per_trade_sharpe\\\": sharpe,\\n        \\\"by_regime\\\": by_regime,\\n    }\\n\\n\\ndef _taken_returns(frame: pd.DataFrame, strategy: str) -> tuple[list[float], list[str]]:\\n    returns: list[float] = []\\n    regimes: list[str] = []\\n    for record in frame.to_dict(orient=\\\"records\\\"):\\n        decision = decide_frame(record, strategy)\\n        if decision[\\\"action\\\"] not in (\\\"long\\\", \\\"short\\\"):\\n            continue\\n        if decision[\\\"action\\\"] != record.get(\\\"direction\\\"):\\n            continue\\n        returns.append(float(record.get(\\\"net_return\\\") or 0.0))\\n        regimes.append(str(record.get(\\\"regime_target\\\") or \\\"unknown\\\"))\\n    return returns, regimes\\n\\n\\ndef compare_strategies(frame: pd.DataFrame) -> dict[str, Any]:\\n    \\\"\\\"\\\"Score the three strategies on one shared outcome frame.\\n\\n    Args:\\n        frame: Rows with ``net_return``, ``direction``, ``algorithmic_score``,\\n            ``win_probability``, and ``expected_net_return``.\\n\\n    Returns:\\n        Metrics for each strategy. Costs and fills are the columns already on\\n        ``frame``; this function does not resimulate them.\\n    \\\"\\\"\\\"\\n    report = {}\\n    for strategy in STRATEGIES:\\n        returns, regimes = _taken_returns(frame, strategy)\\n        report[strategy] = trade_metrics(returns, regimes)\\n    return report\\n\\n\\ndef walk_forward_compare(\\n    frame: pd.DataFrame,\\n    predict: PredictFn,\\n    *,\\n    n_splits: int = 3,\\n    embargo: int | None = None,\\n) -> dict[str, Any]:\\n    \\\"\\\"\\\"Fit on each purged training side and score the three strategies.\\n\\n    Args:\\n        frame: Chronological supervised rows.\\n        predict: ``(train, validation) -> validation`` with ``win_probability``\\n            and ``expected_net_return`` filled from the training side only.\\n        n_splits: Walk-forward folds inside the development span.\\n        embargo: Bars dropped before each validation block. Defaults to the\\n            longest configured hold.\\n\\n    Returns:\\n        Fold metrics, the pooled development comparison, and one sealed-test\\n        comparison.\\n    \\\"\\\"\\\"\\n    hold = embargo if embargo is not None else max(max_hold_bars(name) for name in (\\\"5m\\\", \\\"15m\\\", \\\"1h\\\", \\\"4h\\\"))\\n    folds = []\\n    pooled: list[pd.DataFrame] = []\\n    for train_indexes, validation_indexes in purged_folds(\\n        len(frame),\\n        n_splits=n_splits,\\n        embargo=hold,\\n    ):\\n        train = frame.iloc[train_indexes]\\n        validation = frame.iloc[validation_indexes]\\n        predicted = predict(train, validation.copy())\\n        folds.append(compare_strategies(predicted))\\n        pooled.append(predicted)\\n    development = compare_strategies(pd.concat(pooled, ignore_index=True)) if pooled else {}\\n    _, _, sealed = chronological_bounds(len(frame))\\n    sealed_frame = predict(frame.iloc[:sealed], frame.iloc[sealed:].copy())\\n    return {\\n        \\\"embargo_bars\\\": int(hold),\\n        \\\"folds\\\": folds,\\n        \\\"development\\\": development,\\n        \\\"sealed_test\\\": compare_strategies(sealed_frame),\\n    }\\n\\n\\ndef materialize_comparison(\\n    root: Path | str = COLAB_DATA_ROOT,\\n    *,\\n    symbol: str = \\\"BTCUSD\\\",\\n) -> dict[str, Any]:\\n    \\\"\\\"\\\"Score stored outcomes with the setup-effectiveness artifact.\\n\\n    Args:\\n        root: Local dataset root.\\n        symbol: Instrument.\\n\\n    Returns:\\n        Strategy comparison written to ``experiments/comparison.json``.\\n    \\\"\\\"\\\"\\n    import joblib\\n\\n    from research.swinging.models import score_features\\n\\n    dataset = Path(root)\\n    artifact = joblib.load(dataset / \\\"experiments\\\" / \\\"setup_effectiveness\\\" / \\\"models.joblib\\\")\\n    frames = []\\n    for resolution in (\\\"5m\\\", \\\"15m\\\", \\\"1h\\\", \\\"4h\\\"):\\n        path = dataset / symbol / resolution / \\\"outcomes.parquet\\\"\\n        if path.is_file():\\n            frames.append(pd.read_parquet(path))\\n    if not frames:\\n        raise FileNotFoundError(\\\"outcomes parquet is missing\\\")\\n    frame = pd.concat(frames, ignore_index=True)\\n    frame = frame.sort_values(\\\"signal_timestamp\\\").reset_index(drop=True)\\n    columns = list(artifact[\\\"feature_columns\\\"])\\n    matrix = frame.reindex(columns=columns).apply(pd.to_numeric, errors=\\\"coerce\\\").fillna(0.0)\\n    scored = score_features(artifact, matrix.to_numpy(dtype=float))\\n    frame[\\\"win_probability\\\"] = scored[\\\"event_probability\\\"]\\n    frame[\\\"expected_net_return\\\"] = scored[\\\"expected_net_return\\\"]\\n    report = compare_strategies(frame)\\n    destination = dataset / \\\"experiments\\\" / \\\"comparison.json\\\"\\n    destination.parent.mkdir(parents=True, exist_ok=True)\\n    destination.write_text(json.dumps(report, indent=2), encoding=\\\"utf-8\\\")\\n    return report\\n\", \"research/swinging/export_bundle.py\": \"\\\"\\\"\\\"Research bundle export. Bundles cannot enable orders or enter model storage.\\\"\\\"\\\"\\n\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nimport zipfile\\nfrom pathlib import Path\\nfrom typing import Any\\n\\nimport joblib\\n\\nfrom research.swinging.contract import (\\n    COLAB_DATA_ROOT,\\n    ORDERS_ENABLED,\\n    PROMOTION_STATUS_RESEARCH,\\n    SPEC_VERSION,\\n)\\n\\n\\ndef assert_research_destination(path: Path) -> None:\\n    \\\"\\\"\\\"Refuse a write into the live model directory.\\n\\n    Args:\\n        path: Proposed bundle or zip path.\\n\\n    Raises:\\n        RuntimeError: If a path part is ``model_storage``.\\n    \\\"\\\"\\\"\\n    if \\\"model_storage\\\" in {part.lower() for part in path.parts}:\\n        raise RuntimeError(f\\\"refusing to write a research bundle under model storage: {path}\\\")\\n\\n\\ndef file_sha256(path: Path) -> str:\\n    \\\"\\\"\\\"Hash a file.\\n\\n    Args:\\n        path: File to read.\\n\\n    Returns:\\n        Hex sha256.\\n    \\\"\\\"\\\"\\n    return hashlib.sha256(path.read_bytes()).hexdigest()\\n\\n\\ndef write_research_bundle(\\n    directory: Path | str,\\n    artifact: dict[str, Any],\\n    *,\\n    task: str,\\n    metrics: dict[str, Any],\\n    dataset_version: str,\\n    training_period: dict[str, str],\\n) -> Path:\\n    \\\"\\\"\\\"Write a research-only model bundle.\\n\\n    Args:\\n        directory: Destination directory on the local disk.\\n        artifact: Fitted models and feature column list.\\n        task: ``setup_effectiveness`` or ``sr_effectiveness``.\\n        metrics: Validation and sealed-test metrics.\\n        dataset_version: Manifest version or checksum of the training rows.\\n        training_period: Earliest and latest training timestamps.\\n\\n    Returns:\\n        The bundle directory.\\n\\n    Raises:\\n        RuntimeError: If the destination is inside model storage.\\n    \\\"\\\"\\\"\\n    destination = Path(directory)\\n    assert_research_destination(destination)\\n    destination.mkdir(parents=True, exist_ok=True)\\n    model_path = destination / \\\"models.joblib\\\"\\n    joblib.dump(artifact, model_path)\\n    columns = list(artifact.get(\\\"feature_columns\\\") or [])\\n    schema = {\\n        \\\"spec_version\\\": SPEC_VERSION,\\n        \\\"task\\\": task,\\n        \\\"features\\\": columns,\\n        \\\"feature_count\\\": len(columns),\\n    }\\n    (destination / \\\"feature_schema.json\\\").write_text(json.dumps(schema, indent=2), encoding=\\\"utf-8\\\")\\n    calibration = {\\n        \\\"method\\\": \\\"isotonic\\\" if artifact.get(\\\"calibrator\\\") is not None else \\\"none\\\",\\n        \\\"positive_class\\\": artifact.get(\\\"positive_class\\\"),\\n    }\\n    (destination / \\\"calibration.json\\\").write_text(\\n        json.dumps(calibration, indent=2),\\n        encoding=\\\"utf-8\\\",\\n    )\\n    manifest = {\\n        \\\"spec_version\\\": SPEC_VERSION,\\n        \\\"task\\\": task,\\n        \\\"model_name\\\": f\\\"jacksparrow_{task}_BTCUSD\\\",\\n        \\\"orders_enabled\\\": ORDERS_ENABLED,\\n        \\\"promotion_status\\\": PROMOTION_STATUS_RESEARCH,\\n        \\\"contributes_to_consensus\\\": False,\\n        \\\"feature_columns\\\": columns,\\n        \\\"dataset_version\\\": dataset_version,\\n        \\\"training_period\\\": training_period,\\n        \\\"validation_metrics\\\": metrics.get(\\\"validation\\\", {}),\\n        \\\"test_metrics\\\": metrics.get(\\\"test\\\", {}),\\n        \\\"model_checksum\\\": file_sha256(model_path),\\n        \\\"classifier_algorithm\\\": artifact.get(\\\"classifier_algorithm\\\"),\\n        \\\"regressor_algorithm\\\": artifact.get(\\\"regressor_algorithm\\\"),\\n        \\\"sealed_start\\\": artifact.get(\\\"sealed_start\\\"),\\n    }\\n    if manifest[\\\"orders_enabled\\\"] is not False or manifest[\\\"promotion_status\\\"] != \\\"research\\\":\\n        raise RuntimeError(\\\"research export tried to enable orders\\\")\\n    (destination / \\\"manifest.json\\\").write_text(json.dumps(manifest, indent=2), encoding=\\\"utf-8\\\")\\n    return destination\\n\\n\\ndef zip_directory(directory: Path, zip_path: Path) -> Path:\\n    \\\"\\\"\\\"Zip a bundle for a manual download from Colab.\\n\\n    Args:\\n        directory: Bundle directory.\\n        zip_path: Destination zip on the local disk.\\n\\n    Returns:\\n        ``zip_path``.\\n    \\\"\\\"\\\"\\n    assert_research_destination(zip_path)\\n    zip_path.parent.mkdir(parents=True, exist_ok=True)\\n    with zipfile.ZipFile(zip_path, \\\"w\\\", compression=zipfile.ZIP_DEFLATED) as archive:\\n        for path in sorted(directory.rglob(\\\"*\\\")):\\n            if path.is_file():\\n                archive.write(path, path.relative_to(directory.parent).as_posix())\\n    return zip_path\\n\\n\\ndef export_download_zip(\\n    root: Path | str = COLAB_DATA_ROOT,\\n    *,\\n    task: str = \\\"setup_effectiveness\\\",\\n) -> str:\\n    \\\"\\\"\\\"Copy an experiment artifact into a downloadable research zip.\\n\\n    Args:\\n        root: Local dataset root containing ``experiments``.\\n        task: Experiment directory name.\\n\\n    Returns:\\n        Zip path.\\n    \\\"\\\"\\\"\\n    dataset = Path(root)\\n    source = dataset / \\\"experiments\\\" / task / \\\"models.joblib\\\"\\n    artifact = joblib.load(source)\\n    comparison_path = dataset / \\\"experiments\\\" / \\\"comparison.json\\\"\\n    metrics: dict[str, Any] = {}\\n    if comparison_path.is_file():\\n        metrics = {\\\"validation\\\": json.loads(comparison_path.read_text(encoding=\\\"utf-8\\\"))}\\n    bundle = write_research_bundle(\\n        dataset / \\\"export\\\" / task,\\n        artifact,\\n        task=task,\\n        metrics=metrics,\\n        dataset_version=task,\\n        training_period={\\\"earliest\\\": \\\"\\\", \\\"latest\\\": \\\"\\\"},\\n    )\\n    zip_path = zip_directory(bundle, dataset / \\\"export\\\" / f\\\"{task}.zip\\\")\\n    return str(zip_path)\\n\"}")
for relative, text in FILES.items():
    destination = ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(text, encoding='utf-8')
sys.path.insert(0, str(ROOT))
print('materialized', len(FILES), 'files under', ROOT)


materialized 38 files under /content/JackSparrow


## Run

In [ ]:
from research.swinging.contract import COLAB_DATA_ROOT
from research.swinging.features import materialize_features

written = materialize_features(COLAB_DATA_ROOT)
print(written)


{'5m': '/content/JackSparrow/data/BTCUSD/5m/features.parquet', '15m': '/content/JackSparrow/data/BTCUSD/15m/features.parquet', '1h': '/content/JackSparrow/data/BTCUSD/1h/features.parquet', '4h': '/content/JackSparrow/data/BTCUSD/4h/features.parquet'}
